# 18 — Streamlit Application Development

## Leadership & Management Book Recommendation System

### Objective

This stage transforms the validated recommendation pipeline into an interactive application where users can discover leadership and management books based on their interests.

The application will use the final production architecture established during model evaluation:

**User Input → NLP Processing → Enriched TF-IDF Representation → Cosine Similarity → Ranked Book Recommendations**

The application will also incorporate topic-cluster information and available book metadata to provide additional context for each recommendation.

### Application Features

The Streamlit application will support:

1. **Natural-Language Book Discovery**
   - Users can describe the leadership or management topics they want to explore.
   - The query will be transformed using the validated TF-IDF vectorizer.
   - Books will be ranked using cosine similarity.

2. **Book-to-Book Recommendations**
   - Users can select a book from the catalog.
   - The system will retrieve similar books using the validated content-based recommendation model.

3. **Leadership Topic Exploration**
   - Users can explore books across the 29 topic clusters identified through K-Means clustering.

4. **Recommendation Results**
   - Results may display available information such as:
     - Book title
     - Author(s)
     - Cover image
     - Leadership topic
     - Similarity score
     - Publication information
     - Ratings and engagement metadata when available

5. **Grounded Recommendation Explanations**
   - Recommendation explanations will use only metadata available in the validated project catalog.
   - Missing metadata will not be inferred or fabricated.

### Production Model

The final recommendation engine uses:

- **Enriched TF-IDF**
- **Cosine Similarity**
- **2,067 books in the catalog**
- **2,040 books with usable enriched NLP vectors**
- **29 K-Means leadership and management topic clusters**
- **97.53% recommendation catalog coverage**

The 200-dimensional SVD/LSA representation and K-Means clusters provide supporting semantic and topic information.

The autoencoder remains an experimental model and is **not used as the production recommendation engine**.

### Methodological Note

Similarity scores represent **textual similarity within the project catalog**. They should not be interpreted as book quality scores, predicted ratings, probabilities, or expert evaluations.

The recommendation system is content-based and does not use individual user behavior or collaborative-filtering data.

### Development Goal

The objective of this stage is to create a functional and deployment-ready Streamlit interface while preserving the validated recommendation methodology and avoiding changes to the trained NLP and recommendation artifacts.

## 18.1 — Streamlit Installation and Environment Validation

Streamlit is required to build the interactive recommendation-system interface.

To maintain reproducibility, Streamlit will be installed in the same project virtual environment used throughout the capstone. The installation will then be validated before application development continues.

In [25]:
%pip install streamlit

  Using cached streamlit-1.64.0-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached pyarrow-25.0.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached uvicorn-0.53.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached httptools-0.8.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (3.5 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached websockets-16.1.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
Using cached streamlit-1.64.0-py3-none-any.whl (10.1 MB)
Using cached altair-6.3.0-py3-none-any.whl (797 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached httptools-0.8.0-cp313-cp313-macosx_11_0_arm64.

## 18.2 — Project Path Configuration

The application requires reliable access to the project's processed datasets and validated model artifacts.

Because development is currently being performed inside Jupyter Notebook, the `__file__` variable used by standalone Python applications is not available. The path configuration below therefore supports both:

- **Jupyter development**, using the current project directory.
- **Streamlit deployment**, using the location of the application script.

This ensures that the application can later be moved from the notebook environment to the Streamlit application without relying on hard-coded absolute file paths.

In [27]:
from pathlib import Path

# ---------------------------------------------------------
# PROJECT PATH CONFIGURATION
# ---------------------------------------------------------

if "__file__" in globals():
    # Running as a Streamlit/Python script
    APP_DIR = Path(__file__).resolve().parent
    PROJECT_ROOT = APP_DIR.parent
else:
    # Running inside Jupyter Notebook
    PROJECT_ROOT = Path.cwd()

# If the notebook is inside /notebooks, move up to project root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", DATA_DIR)
print("Models directory:", MODELS_DIR)

print("\nValidation:")
print("Project root exists:", PROJECT_ROOT.exists())
print("Processed data directory exists:", DATA_DIR.exists())
print("Models directory exists:", MODELS_DIR.exists())

Project root: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Processed data directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed
Models directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models

Validation:
Project root exists: True
Processed data directory exists: True
Models directory exists: True


## 18.3 — Loading the Validated Recommendation Artifacts

The Streamlit application will reuse the final recommendation artifacts produced and validated during the NLP and recommendation-system stages.

The production recommendation engine uses the **Enriched TF-IDF representation with cosine similarity**. No model will be retrained during application development.

The following artifacts are loaded:

- Enriched TF-IDF sparse matrix
- Enriched TF-IDF vectorizer
- NLP book index
- Processed book metadata
- Final topic-cluster assignments

Loading the existing artifacts ensures that the application remains consistent with the recommendation system evaluated earlier in the project.

In [29]:
import re
import joblib
import pandas as pd

from scipy.sparse import load_npz
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


# ---------------------------------------------------------
# RESTORE THE VALIDATED NOTEBOOK 09 ANALYZER
# ---------------------------------------------------------

FIELD_BOUNDARY = "zzfieldboundaryzz"
STOP_WORDS = set(ENGLISH_STOP_WORDS)


def boundary_aware_analyzer(document):
    features = []

    fields = document.split(FIELD_BOUNDARY)

    for field in fields:
        tokens = re.findall(
            r"(?u)\b[^\W_][\w'-]+\b",
            field.lower()
        )

        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        features.extend(tokens)

        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(len(tokens) - 1)
            ]
        )

    return features


# ---------------------------------------------------------
# LOAD PRODUCTION ARTIFACTS
# ---------------------------------------------------------

TFIDF_MATRIX_PATH = MODELS_DIR / "enriched_tfidf_matrix.npz"
VECTORIZER_PATH = MODELS_DIR / "enriched_tfidf_vectorizer.joblib"
NLP_INDEX_PATH = DATA_DIR / "nlp_book_index.csv"
BOOK_FEATURES_PATH = DATA_DIR / "books_nlp_features.csv"
CLUSTERS_PATH = DATA_DIR / "books_with_final_topic_clusters.csv"


enriched_tfidf_matrix = load_npz(TFIDF_MATRIX_PATH)

enriched_vectorizer = joblib.load(
    VECTORIZER_PATH
)

nlp_index = pd.read_csv(NLP_INDEX_PATH)
book_features = pd.read_csv(BOOK_FEATURES_PATH)
topic_clusters = pd.read_csv(CLUSTERS_PATH)


# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------

print("TF-IDF matrix shape:", enriched_tfidf_matrix.shape)
print("Vocabulary size:", len(enriched_vectorizer.vocabulary_))
print("NLP index rows:", len(nlp_index))
print("Book feature rows:", len(book_features))
print("Topic-cluster rows:", len(topic_clusters))

print(
    "\nMatrix/index alignment:",
    enriched_tfidf_matrix.shape[0] == len(nlp_index)
)

TF-IDF matrix shape: (2067, 5130)
Vocabulary size: 5130
NLP index rows: 2067
Book feature rows: 2067
Topic-cluster rows: 1884

Matrix/index alignment: True


## 18.4 — Artifact Alignment and Integrity Validation

Before implementing the application recommendation functions, the loaded artifacts must be checked for structural consistency.

The TF-IDF matrix is position-dependent: each matrix row must correspond to the correct book in the NLP index. Book identifiers must therefore remain unique and consistently aligned across the application datasets.

The validation below checks:

- Uniqueness of book identifiers
- Alignment between the NLP index and processed book features
- Topic-cluster coverage
- Missing and duplicated identifiers
- Number of books with usable and zero-vector TF-IDF representations

These checks protect the application from returning metadata for the wrong recommendation.

In [31]:
# ---------------------------------------------------------
# IDENTIFY BOOK-ID COLUMNS
# ---------------------------------------------------------

print("NLP index columns:")
print(nlp_index.columns.tolist())

print("\nBook features columns:")
print(book_features.columns.tolist())

print("\nTopic cluster columns:")
print(topic_clusters.columns.tolist())

NLP index columns:
['matrix_row', 'book_id', 'canonical_title', 'source_group', 'core_zero_vector', 'enriched_zero_vector']

Book features columns:
['book_id', 'canonical_title', 'source_group', 'core_text_boundary', 'enriched_text_boundary', 'core_active_features_final', 'enriched_active_features_final', 'core_zero_vector_final', 'enriched_zero_vector_final']

Topic cluster columns:
['book_id', 'canonical_title', 'authors', 'description', 'subjects', 'first_publish_year', 'average_rating', 'ratings_count', 'edition_count', 'want_to_read_count', 'currently_reading_count', 'already_read_count', 'cover_url', 'openlibrary_key', 'source_openlibrary', 'source_leadershipnow', 'publication_year_observed', 'title_normalized', 'has_authors', 'has_description', 'has_subjects', 'has_cover', 'has_rating', 'has_engagement', 'has_extended_text', 'has_first_publish_year', 'has_observed_publication_year', 'book_age_from_first_publish', 'years_since_observed_publication', 'log1p_ratings_count', 'log1p_

## 18.5 — Book-ID Alignment and Integrity Checks

The recommendation matrix is indexed by row position, while book metadata is linked through `book_id`. Before implementing retrieval functions, the application must verify that these identifiers remain unique and correctly aligned.

The validation checks:

- The NLP index contains 2,067 unique book identifiers.
- `matrix_row` corresponds exactly to the TF-IDF matrix row order.
- The processed NLP feature table contains the same 2,067 books.
- Book identifiers are aligned between the NLP index and feature table.
- The final topic-cluster table contains the expected 1,884 clustered books.
- Clustered book identifiers form a valid subset of the recommendation catalog.
- The expected 27 enriched zero-vector books are preserved.

These checks ensure that similarity scores can be safely mapped back to the correct books and metadata.

In [33]:
# ---------------------------------------------------------
# ARTIFACT INTEGRITY AND ALIGNMENT VALIDATION
# ---------------------------------------------------------

catalog_ids = nlp_index["book_id"].astype(str)
feature_ids = book_features["book_id"].astype(str)
cluster_ids = topic_clusters["book_id"].astype(str)

expected_matrix_rows = np.arange(len(nlp_index))

checks = {
    "TF-IDF rows = NLP index rows":
        enriched_tfidf_matrix.shape[0] == len(nlp_index),

    "NLP book IDs unique":
        catalog_ids.is_unique,

    "Feature book IDs unique":
        feature_ids.is_unique,

    "Cluster book IDs unique":
        cluster_ids.is_unique,

    "NLP matrix_row unique":
        nlp_index["matrix_row"].is_unique,

    "matrix_row matches matrix position":
        np.array_equal(
            nlp_index["matrix_row"].to_numpy(),
            expected_matrix_rows
        ),

    "NLP and feature row counts match":
        len(nlp_index) == len(book_features),

    "NLP and feature book order identical":
        catalog_ids.tolist() == feature_ids.tolist(),

    "Cluster IDs are subset of catalog":
        set(cluster_ids).issubset(set(catalog_ids)),
}

print("ARTIFACT INTEGRITY CHECKS")
print("-" * 55)

for check, result in checks.items():
    print(f"{check:<42} {result}")


# ---------------------------------------------------------
# COVERAGE COUNTS
# ---------------------------------------------------------

zero_vector_count = int(
    nlp_index["enriched_zero_vector"]
    .fillna(False)
    .astype(bool)
    .sum()
)

usable_vector_count = len(nlp_index) - zero_vector_count

clustered_count = len(topic_clusters)
unclustered_count = len(nlp_index) - clustered_count

print("\nCATALOG COVERAGE")
print("-" * 55)
print("Catalog books:", len(nlp_index))
print("Usable enriched vectors:", usable_vector_count)
print("Enriched zero vectors:", zero_vector_count)
print("Clustered books:", clustered_count)
print("Books without final cluster:", unclustered_count)


# ---------------------------------------------------------
# OVERALL VALIDATION
# ---------------------------------------------------------

all_checks_passed = all(checks.values())

print("\nOverall integrity validation:", all_checks_passed)

ARTIFACT INTEGRITY CHECKS
-------------------------------------------------------
TF-IDF rows = NLP index rows               True
NLP book IDs unique                        True
Feature book IDs unique                    True
Cluster book IDs unique                    True
NLP matrix_row unique                      True
matrix_row matches matrix position         True
NLP and feature row counts match           True
NLP and feature book order identical       True
Cluster IDs are subset of catalog          True

CATALOG COVERAGE
-------------------------------------------------------
Catalog books: 2067
Usable enriched vectors: 2040
Enriched zero vectors: 27
Clustered books: 1884
Books without final cluster: 183

Overall integrity validation: True


## 18.6 — Building the Application Metadata Catalog

The recommendation engine covers the full 2,067-book NLP catalog, while final topic-cluster assignments are available for 1,884 books.

The application catalog therefore uses the NLP index as its base and attaches topic and descriptive metadata using a left join. This preserves books that are eligible for recommendation but do not have a final K-Means topic assignment.

The resulting catalog will provide the metadata required by the application interface while maintaining the exact TF-IDF matrix-row alignment established in the previous validation.

In [34]:
# ---------------------------------------------------------
# BUILD APPLICATION METADATA CATALOG
# ---------------------------------------------------------

metadata_columns = [
    "book_id",
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_url",
    "format",
    "page_count",
    "content_depth",
    "topic_cluster",
    "cluster_label",
]

available_metadata_columns = [
    col for col in metadata_columns
    if col in topic_clusters.columns
]

cluster_metadata = (
    topic_clusters[available_metadata_columns]
    .copy()
)

app_catalog = (
    nlp_index[
        [
            "matrix_row",
            "book_id",
            "canonical_title",
            "source_group",
            "enriched_zero_vector",
        ]
    ]
    .copy()
    .merge(
        cluster_metadata,
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)

# ---------------------------------------------------------
# DISPLAY-SAFE FIELDS
# ---------------------------------------------------------

app_catalog["authors_display"] = (
    app_catalog["authors"]
    .fillna("Author information not available")
)

app_catalog["topic_display"] = (
    app_catalog["cluster_label"]
    .fillna("Topic not assigned")
)

app_catalog["cover_available"] = (
    app_catalog["cover_url"]
    .notna()
    & app_catalog["cover_url"].astype(str).str.strip().ne("")
)

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

print("Application catalog shape:", app_catalog.shape)
print("Unique book IDs:", app_catalog["book_id"].nunique())
print(
    "Matrix rows aligned:",
    np.array_equal(
        app_catalog["matrix_row"].to_numpy(),
        np.arange(len(app_catalog))
    )
)

print(
    "Books with topic assignment:",
    app_catalog["cluster_label"].notna().sum()
)

print(
    "Books without topic assignment:",
    app_catalog["cluster_label"].isna().sum()
)

print(
    "Books with cover:",
    app_catalog["cover_available"].sum()
)

print(
    "Usable recommendation vectors:",
    (~app_catalog["enriched_zero_vector"].astype(bool)).sum()
)

display(
    app_catalog[
        [
            "book_id",
            "canonical_title",
            "authors_display",
            "source_group",
            "topic_display",
            "cover_available",
            "enriched_zero_vector",
        ]
    ].head(10)
)

Application catalog shape: (2067, 25)
Unique book IDs: 2067
Matrix rows aligned: True
Books with topic assignment: 1884
Books without topic assignment: 183
Books with cover: 1705
Usable recommendation vectors: 2040


,book_id,canonical_title,authors_display,source_group,topic_display,cover_available,enriched_zero_vector
0,BOOK00001,Principle-Centered Leadership,['Stephen R. Covey'],Open Library only,Broad Leadership and Success,True,False
1,BOOK00002,Leadership in Organizations,['Gary A. Yukl'],Open Library only,Decision Making,True,False
2,BOOK00003,Kepemimpinan =,['Karjadi M.'],Open Library only,Transformational Leadership,True,False
3,BOOK00004,Spiritual leadership,['J. Oswald Sanders'],Open Library only,Transformational Leadership,True,False
4,BOOK00005,Leadership,['Peter Guy Northouse'],Open Library only,Transformational Leadership,True,False
5,BOOK00006,The 21 Irrefutable Laws of Leadership,['John C. Maxwell'],Open Library only,Transformational Leadership,True,False
6,BOOK00007,Leadership,['Peter G. Northouse'],Open Library only,Leadership Development,True,False
7,BOOK00008,Leadership and performance beyond expectations,['Bernard M. Bass'],Open Library only,Transformational Leadership,True,False
8,BOOK00009,A Higher Loyalty,"['James Comey', 'James B. Comey', 'James Comey']",Open Library only,Future Thinking and Personal Development,True,False
9,BOOK00010,Leadership and Self Deception,"['The Arbinger Institute', 'Dick Ruhe']",Open Library only,General Business and Management,True,False


## 18.7 — Application Catalog Enrichment

The initial application catalog successfully preserves all 2,067 books and maintains exact alignment with the production TF-IDF matrix.

However, metadata attached through the final topic-cluster table is necessarily limited to the 1,884 books that received final cluster assignments. This results in reduced metadata coverage for some application fields.

Notebook 15 previously produced a validated grounding catalog covering the full recommendation catalog. Before finalizing the application's metadata layer, this dataset will be inspected and compared with the current catalog.

The objective is to maximize the use of validated metadata without changing the production recommendation representation or fabricating missing information.

In [36]:
# ---------------------------------------------------------
# INSPECT VALIDATED NOTEBOOK 15 GROUNDING CATALOG
# ---------------------------------------------------------

GROUNDING_PATH = DATA_DIR / "llm_grounding_catalog.csv"

grounding_catalog = pd.read_csv(GROUNDING_PATH)

print("Grounding catalog shape:", grounding_catalog.shape)
print("Unique book IDs:", grounding_catalog["book_id"].nunique())

print("\nColumns:")
print(grounding_catalog.columns.tolist())

print("\nMissing-value coverage:")
for column in grounding_catalog.columns:
    available = grounding_catalog[column].notna().sum()
    pct = available / len(grounding_catalog) * 100

    print(
        f"{column:<30} "
        f"{available:>4} / {len(grounding_catalog)} "
        f"({pct:6.2f}%)"
    )

Grounding catalog shape: (2067, 19)
Unique book IDs: 2067

Columns:
['matrix_row', 'book_id', 'canonical_title', 'authors', 'description', 'subjects', 'first_publish_year', 'publication_year_observed', 'average_rating', 'ratings_count', 'edition_count', 'want_to_read_count', 'currently_reading_count', 'already_read_count', 'cover_url', 'source_group', 'topic_cluster', 'cluster_label', 'enriched_zero_vector']

Missing-value coverage:
matrix_row                     2067 / 2067 (100.00%)
book_id                        2067 / 2067 (100.00%)
canonical_title                2067 / 2067 (100.00%)
authors                        1884 / 2067 ( 91.15%)
description                     192 / 2067 (  9.29%)
subjects                       1884 / 2067 ( 91.15%)
first_publish_year              947 / 2067 ( 45.82%)
publication_year_observed       934 / 2067 ( 45.19%)
average_rating                  282 / 2067 ( 13.64%)
ratings_count                   282 / 2067 ( 13.64%)
edition_count                   9

## 18.8 — Final Application Catalog

The validated Notebook 15 grounding catalog is adopted as the primary metadata layer for the Streamlit application.

It contains all 2,067 books in the recommendation catalog and preserves the matrix-row mapping required by the production TF-IDF model.

No missing book metadata will be inferred or fabricated. Display-oriented fields are created only to ensure that unavailable information is communicated clearly in the user interface.

The application catalog therefore separates:

- **Recommendation computation:** Enriched TF-IDF + cosine similarity
- **Book metadata:** Validated grounding catalog
- **Topic context:** Final K-Means cluster assignments where available
- **Display formatting:** Application-only labels for missing information

In [37]:
# ---------------------------------------------------------
# FINAL APPLICATION CATALOG
# ---------------------------------------------------------

app_catalog = grounding_catalog.copy()

# Preserve exact matrix order
app_catalog = (
    app_catalog
    .sort_values("matrix_row")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# DISPLAY-SAFE FIELDS
# ---------------------------------------------------------

app_catalog["authors_display"] = (
    app_catalog["authors"]
    .fillna("Author information not available")
)

app_catalog["topic_display"] = (
    app_catalog["cluster_label"]
    .fillna("Topic not assigned")
)

app_catalog["description_display"] = (
    app_catalog["description"]
    .fillna("Description not available")
)

app_catalog["subjects_display"] = (
    app_catalog["subjects"]
    .fillna("Subject information not available")
)

app_catalog["cover_available"] = (
    app_catalog["cover_url"].notna()
    & app_catalog["cover_url"].astype(str).str.strip().ne("")
)


# ---------------------------------------------------------
# FINAL VALIDATION
# ---------------------------------------------------------

checks = {
    "Catalog has 2,067 books":
        len(app_catalog) == 2067,

    "Book IDs are unique":
        app_catalog["book_id"].is_unique,

    "Matrix rows are unique":
        app_catalog["matrix_row"].is_unique,

    "Matrix order is aligned":
        np.array_equal(
            app_catalog["matrix_row"].to_numpy(),
            np.arange(len(app_catalog))
        ),

    "TF-IDF row count matches catalog":
        enriched_tfidf_matrix.shape[0] == len(app_catalog),

    "Usable vectors = 2,040":
        (~app_catalog["enriched_zero_vector"].astype(bool)).sum() == 2040,

    "Topic assignments = 1,884":
        app_catalog["cluster_label"].notna().sum() == 1884,
}

print("FINAL APPLICATION CATALOG VALIDATION")
print("-" * 55)

for check, result in checks.items():
    print(f"{check:<42} {result}")

print("\nCatalog shape:", app_catalog.shape)
print("Covers available:", int(app_catalog["cover_available"].sum()))
print(
    "Descriptions available:",
    int(app_catalog["description"].notna().sum())
)

print("\nOverall validation:", all(checks.values()))

FINAL APPLICATION CATALOG VALIDATION
-------------------------------------------------------
Catalog has 2,067 books                    True
Book IDs are unique                        True
Matrix rows are unique                     True
Matrix order is aligned                    True
TF-IDF row count matches catalog           True
Usable vectors = 2,040                     True
Topic assignments = 1,884                  True

Catalog shape: (2067, 24)
Covers available: 1705
Descriptions available: 192

Overall validation: True


## 18.9 — Natural-Language Recommendation Retrieval

The first application feature allows users to describe the leadership or management topics they are interested in using natural language.

The retrieval process follows the validated production methodology:

1. The user's text query is transformed using the fitted **Enriched TF-IDF vectorizer**.
2. The query vector is compared with the 2,067-book catalog using **cosine similarity**.
3. Books with unusable enriched vectors are excluded.
4. Only books with positive similarity are retained.
5. Results are ranked from highest to lowest textual similarity.
6. Validated catalog metadata and topic-cluster information are attached to the results.

The similarity score represents textual similarity between the user's query and the available book metadata. It is not a quality score, predicted rating, probability, or expert evaluation.

If the query contains no vocabulary recognized by the fitted TF-IDF model, the system returns no recommendations rather than generating arbitrary results.

In [38]:
from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------
# NATURAL-LANGUAGE RETRIEVAL
# ---------------------------------------------------------

def retrieve_by_query(query, top_n=10):
    """
    Retrieve books using natural-language input and the validated
    enriched TF-IDF + cosine-similarity recommendation pipeline.
    """

    # Validate input
    if not isinstance(query, str) or not query.strip():
        return pd.DataFrame()

    # Transform query with validated fitted vectorizer
    query_vector = enriched_vectorizer.transform([query.strip()])

    # Reject queries containing no recognized TF-IDF vocabulary
    if query_vector.nnz == 0:
        return pd.DataFrame()

    # Calculate cosine similarity against full catalog
    similarities = cosine_similarity(
        query_vector,
        enriched_tfidf_matrix
    ).flatten()

    # Exclude books with unusable enriched vectors
    zero_mask = (
        app_catalog["enriched_zero_vector"]
        .fillna(True)
        .astype(bool)
        .to_numpy()
    )

    similarities[zero_mask] = -1.0

    # Keep positive similarities only
    candidate_indices = np.where(similarities > 0)[0]

    if len(candidate_indices) == 0:
        return pd.DataFrame()

    # Rank candidates by descending similarity
    ranked_indices = candidate_indices[
        np.argsort(similarities[candidate_indices])[::-1]
    ]

    ranked_indices = ranked_indices[:top_n]

    # Retrieve grounded metadata
    results = app_catalog.iloc[ranked_indices].copy()

    results["similarity_score"] = similarities[ranked_indices]

    results["recommendation_rank"] = np.arange(
        1,
        len(results) + 1
    )

    # Return user-facing fields
    result_columns = [
        "recommendation_rank",
        "book_id",
        "canonical_title",
        "authors_display",
        "topic_display",
        "similarity_score",
        "description_display",
        "subjects_display",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "edition_count",
        "want_to_read_count",
        "cover_url",
        "cover_available",
        "source_group",
    ]

    return results[result_columns].reset_index(drop=True)

In [39]:
test_query = (
    "I want to become a better leader by improving decision making, "
    "communication, team leadership, and strategic thinking."
)

test_results = retrieve_by_query(
    test_query,
    top_n=5
)

display(
    test_results[
        [
            "recommendation_rank",
            "canonical_title",
            "authors_display",
            "topic_display",
            "similarity_score",
        ]
    ]
)

,recommendation_rank,canonical_title,authors_display,topic_display,similarity_score
0,1,Team Leadership,['Drikus Kriek'],Team Leadership,0.344048
1,2,Strategic Team Leadership,['Peter Saul'],Team Leadership,0.334949
2,3,The Six Disciplines of Strategic Thinking,['Michael D. Watkins'],CEO and Executive Transformation,0.329029
3,4,Team leadership,['John Apps'],Team Leadership,0.297660
4,5,If You Want Something Done,['Nikki R. Haley'],Future Thinking and Personal Development,0.295807


## 18.10 — Natural-Language Retrieval Validation

Before integrating the recommendation function into the Streamlit interface, its behavior is validated under normal and edge-case conditions.

The validation examines whether the retrieval function:

- Returns the requested number of recommendations when sufficient matches exist
- Produces only positive cosine-similarity scores
- Orders recommendations from highest to lowest similarity
- Returns unique book identifiers
- Excludes books with unusable enriched TF-IDF vectors
- Handles empty queries safely
- Handles queries containing no recognized TF-IDF vocabulary
- Does not generate arbitrary recommendations when meaningful retrieval is impossible

These tests verify application behavior without changing or retraining the validated recommendation model.

In [40]:
# ---------------------------------------------------------
# NATURAL-LANGUAGE RETRIEVAL VALIDATION
# ---------------------------------------------------------

validation_query = (
    "I want to become a better leader by improving decision making, "
    "communication, team leadership, and strategic thinking."
)

validation_results = retrieve_by_query(
    validation_query,
    top_n=10
)

# Normal-query checks
normal_checks = {
    "Returns 10 recommendations":
        len(validation_results) == 10,

    "Book IDs are unique":
        validation_results["book_id"].is_unique,

    "All similarities are positive":
        (validation_results["similarity_score"] > 0).all(),

    "Similarity is descending":
        validation_results["similarity_score"].is_monotonic_decreasing,

    "Ranks are sequential":
        validation_results["recommendation_rank"].tolist()
        == list(range(1, len(validation_results) + 1)),
}

# Verify returned books are not zero vectors
returned_ids = set(validation_results["book_id"])

returned_zero_vectors = app_catalog[
    app_catalog["book_id"].isin(returned_ids)
]["enriched_zero_vector"].astype(bool).sum()

normal_checks["No zero-vector books returned"] = (
    returned_zero_vectors == 0
)


# ---------------------------------------------------------
# EDGE CASES
# ---------------------------------------------------------

empty_query_results = retrieve_by_query("", top_n=10)

whitespace_query_results = retrieve_by_query(
    "     ",
    top_n=10
)

oov_query_results = retrieve_by_query(
    "qxzvplm jjjkkk zzzqqq",
    top_n=10
)

edge_checks = {
    "Empty query returns no results":
        empty_query_results.empty,

    "Whitespace query returns no results":
        whitespace_query_results.empty,

    "OOV query returns no results":
        oov_query_results.empty,
}


# ---------------------------------------------------------
# REPORT
# ---------------------------------------------------------

all_checks = {
    **normal_checks,
    **edge_checks,
}

print("NATURAL-LANGUAGE RETRIEVAL VALIDATION")
print("-" * 60)

for check, result in all_checks.items():
    print(f"{check:<45} {result}")

print("\nRecommendations returned:", len(validation_results))
print(
    "Highest similarity:",
    round(validation_results["similarity_score"].max(), 6)
)
print(
    "Lowest Top-10 similarity:",
    round(validation_results["similarity_score"].min(), 6)
)

print("\nOverall validation:", all(all_checks.values()))

NATURAL-LANGUAGE RETRIEVAL VALIDATION
------------------------------------------------------------
Returns 10 recommendations                    True
Book IDs are unique                           True
All similarities are positive                 True
Similarity is descending                      True
Ranks are sequential                          True
No zero-vector books returned                 True
Empty query returns no results                True
Whitespace query returns no results           True
OOV query returns no results                  True

Recommendations returned: 10
Highest similarity: 0.344048
Lowest Top-10 similarity: 0.236787

Overall validation: True


## 18.11 — Duplicate-Safe Recommendation Retrieval

The integrated catalog may contain different records with identical or similar titles. Some represent genuinely different books, while others may represent duplicate catalog records.

To improve the user-facing recommendation results, duplicate suppression is applied using a normalized combination of:

- Book title
- Author information

Books sharing a title but written by different authors are therefore preserved as distinct recommendations.

Duplicate suppression is performed only after cosine-similarity ranking. The highest-ranked occurrence is retained, preserving the ranking generated by the production recommendation model.

This is an application-level result-filtering step and does not modify the TF-IDF representation, cosine-similarity model, or underlying catalog.

In [41]:
# ---------------------------------------------------------
# NORMALIZATION FOR DUPLICATE SUPPRESSION
# ---------------------------------------------------------

def normalize_duplicate_text(value):
    """
    Normalize text only for duplicate detection.
    This does not modify the recommendation representation.
    """

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    # Keep Unicode word characters while removing punctuation
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)

    # Normalize repeated whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value


app_catalog["duplicate_key"] = (
    app_catalog["canonical_title"]
    .apply(normalize_duplicate_text)
    + " || "
    + app_catalog["authors"]
    .apply(normalize_duplicate_text)
)


print("Catalog rows:", len(app_catalog))
print(
    "Unique normalized title-author keys:",
    app_catalog["duplicate_key"].nunique()
)

print(
    "Potential duplicate rows:",
    len(app_catalog)
    - app_catalog["duplicate_key"].nunique()
)

Catalog rows: 2067
Unique normalized title-author keys: 2015
Potential duplicate rows: 52


## 18.12 — Duplicate-Safe Natural-Language Recommendations

The application catalog contains 2,067 records representing 2,015 unique normalized title-author combinations, indicating 52 records beyond the unique title-author keys.

To prevent duplicate catalog records from occupying multiple recommendation positions, duplicate suppression is incorporated into the natural-language retrieval process.

The procedure:

1. Calculates cosine similarity for all eligible books.
2. Ranks candidates by descending similarity.
3. Processes candidates in ranked order.
4. Retains only the highest-ranked occurrence of each normalized title-author combination.
5. Continues through the candidate list until the requested number of unique recommendations is obtained.

This preserves genuinely different books with the same title when their authors differ while preventing duplicate catalog records from reducing the diversity of the recommendation list.

Duplicate suppression affects only the presentation of ranked results and does not alter the validated TF-IDF model or similarity calculations.

In [42]:
def retrieve_by_query(query, top_n=10):
    """
    Retrieve duplicate-safe book recommendations using natural-language
    input and the validated enriched TF-IDF + cosine-similarity pipeline.
    """

    # -----------------------------------------------------
    # INPUT VALIDATION
    # -----------------------------------------------------

    if not isinstance(query, str) or not query.strip():
        return pd.DataFrame()

    if not isinstance(top_n, int) or top_n <= 0:
        return pd.DataFrame()


    # -----------------------------------------------------
    # QUERY VECTORIZATION
    # -----------------------------------------------------

    query_vector = enriched_vectorizer.transform(
        [query.strip()]
    )

    # No recognized vocabulary
    if query_vector.nnz == 0:
        return pd.DataFrame()


    # -----------------------------------------------------
    # COSINE SIMILARITY
    # -----------------------------------------------------

    similarities = cosine_similarity(
        query_vector,
        enriched_tfidf_matrix
    ).flatten()


    # -----------------------------------------------------
    # EXCLUDE ZERO-VECTOR BOOKS
    # -----------------------------------------------------

    zero_mask = (
        app_catalog["enriched_zero_vector"]
        .fillna(True)
        .astype(bool)
        .to_numpy()
    )

    similarities[zero_mask] = -1.0


    # -----------------------------------------------------
    # POSITIVE-SIMILARITY CANDIDATES
    # -----------------------------------------------------

    candidate_indices = np.where(
        similarities > 0
    )[0]

    if len(candidate_indices) == 0:
        return pd.DataFrame()


    # -----------------------------------------------------
    # RANK BY SIMILARITY
    # -----------------------------------------------------

    ranked_indices = candidate_indices[
        np.argsort(
            similarities[candidate_indices]
        )[::-1]
    ]


    # -----------------------------------------------------
    # DUPLICATE SUPPRESSION
    # -----------------------------------------------------

    selected_indices = []
    seen_duplicate_keys = set()

    for idx in ranked_indices:

        duplicate_key = app_catalog.iloc[idx][
            "duplicate_key"
        ]

        if duplicate_key in seen_duplicate_keys:
            continue

        seen_duplicate_keys.add(duplicate_key)
        selected_indices.append(idx)

        if len(selected_indices) >= top_n:
            break


    if not selected_indices:
        return pd.DataFrame()


    # -----------------------------------------------------
    # RETRIEVE GROUNDED METADATA
    # -----------------------------------------------------

    results = app_catalog.iloc[
        selected_indices
    ].copy()

    results["similarity_score"] = similarities[
        selected_indices
    ]

    results["recommendation_rank"] = np.arange(
        1,
        len(results) + 1
    )


    # -----------------------------------------------------
    # USER-FACING OUTPUT
    # -----------------------------------------------------

    result_columns = [
        "recommendation_rank",
        "book_id",
        "canonical_title",
        "authors_display",
        "topic_display",
        "similarity_score",
        "description_display",
        "subjects_display",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "edition_count",
        "want_to_read_count",
        "cover_url",
        "cover_available",
        "source_group",
        "duplicate_key",
    ]

    return (
        results[result_columns]
        .reset_index(drop=True)
    )

In [43]:
duplicate_safe_results = retrieve_by_query(
    validation_query,
    top_n=10
)

duplicate_safe_checks = {
    "Returns 10 recommendations":
        len(duplicate_safe_results) == 10,

    "Book IDs unique":
        duplicate_safe_results["book_id"].is_unique,

    "Title-author keys unique":
        duplicate_safe_results["duplicate_key"].is_unique,

    "All similarities positive":
        (
            duplicate_safe_results[
                "similarity_score"
            ] > 0
        ).all(),

    "Similarity descending":
        duplicate_safe_results[
            "similarity_score"
        ].is_monotonic_decreasing,

    "Ranks sequential":
        duplicate_safe_results[
            "recommendation_rank"
        ].tolist()
        == list(
            range(
                1,
                len(duplicate_safe_results) + 1
            )
        ),
}

print("DUPLICATE-SAFE RETRIEVAL VALIDATION")
print("-" * 60)

for check, result in duplicate_safe_checks.items():
    print(f"{check:<45} {result}")

print(
    "\nOverall validation:",
    all(duplicate_safe_checks.values())
)

display(
    duplicate_safe_results[
        [
            "recommendation_rank",
            "canonical_title",
            "authors_display",
            "topic_display",
            "similarity_score",
        ]
    ]
)

DUPLICATE-SAFE RETRIEVAL VALIDATION
------------------------------------------------------------
Returns 10 recommendations                    True
Book IDs unique                               True
Title-author keys unique                      True
All similarities positive                     True
Similarity descending                         True
Ranks sequential                              True

Overall validation: True


,recommendation_rank,canonical_title,authors_display,topic_display,similarity_score
0,1,Team Leadership,['Drikus Kriek'],Team Leadership,0.344048
1,2,Strategic Team Leadership,['Peter Saul'],Team Leadership,0.334949
2,3,The Six Disciplines of Strategic Thinking,['Michael D. Watkins'],CEO and Executive Transformation,0.329029
3,4,Team leadership,['John Apps'],Team Leadership,0.297660
4,5,If You Want Something Done,['Nikki R. Haley'],Future Thinking and Personal Development,0.295807
5,6,Creative decision making,['H. B. Gelatt'],Decision Making,0.290526
6,7,DECISION MAKING,['Irving L. Janis'],Decision Making,0.246255
7,8,Team Leadership Questionnaire,"['D. Kayes', 'Anna Kayes']",Team Leadership,0.245542
8,9,Team Leadership Assessments,['LeaderTreks'],Team Leadership,0.236848
9,10,A primer on decision making,['James G. March'],Decision Making,0.236787


## 18.13 — Book-to-Book Recommendation

In addition to natural-language discovery, the application allows users to select an existing book from the catalog and retrieve similar books.

Unlike natural-language retrieval, where a user query is transformed into the TF-IDF feature space, book-to-book recommendation directly compares the selected book's validated Enriched TF-IDF vector with the vectors of the remaining books.

The process:

1. Identifies the selected book using its unique `book_id`.
2. Retrieves its corresponding TF-IDF matrix row.
3. Calculates cosine similarity against the full book catalog.
4. Excludes the selected book itself.
5. Excludes books with unusable enriched TF-IDF vectors.
6. Retains only positive-similarity candidates.
7. Ranks candidates by descending cosine similarity.
8. Suppresses duplicate normalized title-author combinations.
9. Returns the requested number of unique recommendations with validated metadata.

The selected book is identified by `book_id` rather than title alone because multiple books may share identical or similar titles. This prevents ambiguous title matching and avoids silently selecting the wrong catalog record.

As with natural-language retrieval, cosine similarity represents textual similarity rather than book quality, predicted rating, or expert evaluation.

In [44]:
# ---------------------------------------------------------
# BOOK-TO-BOOK RECOMMENDATION
# ---------------------------------------------------------

def recommend_similar_books(book_id, top_n=10):
    """
    Recommend books similar to an existing catalog book using
    validated enriched TF-IDF vectors and cosine similarity.
    """

    # -----------------------------------------------------
    # INPUT VALIDATION
    # -----------------------------------------------------

    if not isinstance(book_id, str) or not book_id.strip():
        return pd.DataFrame()

    if not isinstance(top_n, int) or top_n <= 0:
        return pd.DataFrame()

    book_id = book_id.strip()

    matches = app_catalog[
        app_catalog["book_id"] == book_id
    ]

    if matches.empty:
        return pd.DataFrame()


    # -----------------------------------------------------
    # LOCATE QUERY BOOK
    # -----------------------------------------------------

    query_row = int(
        matches.iloc[0]["matrix_row"]
    )

    query_is_zero = bool(
        matches.iloc[0]["enriched_zero_vector"]
    )

    # A zero-vector book cannot produce meaningful
    # cosine-similarity recommendations.
    if query_is_zero:
        return pd.DataFrame()


    # -----------------------------------------------------
    # COSINE SIMILARITY
    # -----------------------------------------------------

    similarities = cosine_similarity(
        enriched_tfidf_matrix[query_row],
        enriched_tfidf_matrix
    ).flatten()


    # -----------------------------------------------------
    # EXCLUDE INVALID CANDIDATES
    # -----------------------------------------------------

    zero_mask = (
        app_catalog["enriched_zero_vector"]
        .fillna(True)
        .astype(bool)
        .to_numpy()
    )

    similarities[zero_mask] = -1.0

    # Exclude query book itself
    similarities[query_row] = -1.0


    # -----------------------------------------------------
    # POSITIVE-SIMILARITY CANDIDATES
    # -----------------------------------------------------

    candidate_indices = np.where(
        similarities > 0
    )[0]

    if len(candidate_indices) == 0:
        return pd.DataFrame()


    # -----------------------------------------------------
    # RANK CANDIDATES
    # -----------------------------------------------------

    ranked_indices = candidate_indices[
        np.argsort(
            similarities[candidate_indices]
        )[::-1]
    ]


    # -----------------------------------------------------
    # DUPLICATE SUPPRESSION
    # -----------------------------------------------------

    selected_indices = []
    seen_duplicate_keys = set()

    # Also prevent another catalog record representing the
    # same normalized title-author combination as the query.
    query_duplicate_key = app_catalog.iloc[
        query_row
    ]["duplicate_key"]

    seen_duplicate_keys.add(query_duplicate_key)

    for idx in ranked_indices:

        duplicate_key = app_catalog.iloc[
            idx
        ]["duplicate_key"]

        if duplicate_key in seen_duplicate_keys:
            continue

        seen_duplicate_keys.add(duplicate_key)
        selected_indices.append(idx)

        if len(selected_indices) >= top_n:
            break


    if not selected_indices:
        return pd.DataFrame()


    # -----------------------------------------------------
    # ATTACH GROUNDED METADATA
    # -----------------------------------------------------

    results = app_catalog.iloc[
        selected_indices
    ].copy()

    results["similarity_score"] = similarities[
        selected_indices
    ]

    results["recommendation_rank"] = np.arange(
        1,
        len(results) + 1
    )


    # -----------------------------------------------------
    # USER-FACING OUTPUT
    # -----------------------------------------------------

    result_columns = [
        "recommendation_rank",
        "book_id",
        "canonical_title",
        "authors_display",
        "topic_display",
        "similarity_score",
        "description_display",
        "subjects_display",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "edition_count",
        "want_to_read_count",
        "cover_url",
        "cover_available",
        "source_group",
        "duplicate_key",
    ]

    return (
        results[result_columns]
        .reset_index(drop=True)
    )

In [45]:
test_book_id = "BOOK00001"

selected_book = app_catalog[
    app_catalog["book_id"] == test_book_id
].iloc[0]

print("Selected Book")
print("-" * 50)
print("Book ID:", selected_book["book_id"])
print("Title:", selected_book["canonical_title"])
print("Author:", selected_book["authors_display"])
print("Topic:", selected_book["topic_display"])

similar_book_results = recommend_similar_books(
    test_book_id,
    top_n=10
)

print(
    "\nRecommendations returned:",
    len(similar_book_results)
)

display(
    similar_book_results[
        [
            "recommendation_rank",
            "canonical_title",
            "authors_display",
            "topic_display",
            "similarity_score",
        ]
    ]
)

Selected Book
--------------------------------------------------
Book ID: BOOK00001
Title: Principle-Centered Leadership
Author: ['Stephen R. Covey']
Topic: Broad Leadership and Success

Recommendations returned: 10


,recommendation_rank,canonical_title,authors_display,topic_display,similarity_score
0,1,The 7 Habits of Highly Effective People,['Stephen R. Covey'],Broad Leadership and Success,0.220669
1,2,Live Life in Crescendo,['Stephen R. Covey and Cynthia Covey'],CEO and Executive Transformation,0.208324
2,3,The Tao of leadership,['John Heider'],Transformational Leadership,0.198383
3,4,Primal Leadership,"['Daniel Goleman', 'Richard E. Boyatzis', 'Ann...",Emotional Intelligence,0.177451
4,5,Trust and Inspire,['Stephen M.R. Covey'],Organizational Culture and Trust,0.160279
5,6,Leadership development,['Rosemary Ryan'],Leadership Development,0.159162
6,7,7 principles of transformational leadership,['Hugh Blane'],Broad Leadership and Success,0.156437
7,8,Life on the X,['Stephen Drum'],CEO and Executive Transformation,0.144820
8,9,Emotional intelligence,"['Walton, David (Psychologist)']",Emotional Intelligence,0.139195
9,10,Nonviolent Communication,"['Marshall B. Rosenberg', 'Deepak Chopra', 'Ma...",Communication,0.133814


## 18.14 — Book-to-Book Recommendation Validation

The book-to-book recommendation function is validated before integration into the Streamlit interface.

The validation confirms that:

- The requested number of recommendations is returned when sufficient candidates exist.
- The selected query book is excluded from its own recommendations.
- Recommendation book identifiers are unique.
- Normalized title-author combinations are unique.
- All returned cosine-similarity scores are positive.
- Recommendations are ordered from highest to lowest similarity.
- Recommendation ranks are sequential.
- Books with unusable enriched TF-IDF vectors are excluded.
- Invalid book identifiers are handled safely.
- Books with zero-vector representations do not generate arbitrary recommendations.

These checks ensure that the book-selection interface will use the same validated recommendation logic as the underlying production system.

In [46]:
# ---------------------------------------------------------
# BOOK-TO-BOOK RECOMMENDATION VALIDATION
# ---------------------------------------------------------

validation_book_id = "BOOK00001"

book_validation_results = recommend_similar_books(
    validation_book_id,
    top_n=10
)


# ---------------------------------------------------------
# STANDARD RETRIEVAL CHECKS
# ---------------------------------------------------------

returned_ids = set(
    book_validation_results["book_id"]
)

returned_zero_vectors = app_catalog[
    app_catalog["book_id"].isin(returned_ids)
]["enriched_zero_vector"].astype(bool).sum()


book_checks = {
    "Returns 10 recommendations":
        len(book_validation_results) == 10,

    "Query book excluded":
        validation_book_id not in returned_ids,

    "Book IDs unique":
        book_validation_results["book_id"].is_unique,

    "Title-author keys unique":
        book_validation_results["duplicate_key"].is_unique,

    "All similarities positive":
        (
            book_validation_results[
                "similarity_score"
            ] > 0
        ).all(),

    "Similarity descending":
        book_validation_results[
            "similarity_score"
        ].is_monotonic_decreasing,

    "Ranks sequential":
        book_validation_results[
            "recommendation_rank"
        ].tolist()
        == list(
            range(
                1,
                len(book_validation_results) + 1
            )
        ),

    "No zero-vector books returned":
        returned_zero_vectors == 0,
}


# ---------------------------------------------------------
# INVALID BOOK-ID TEST
# ---------------------------------------------------------

invalid_book_results = recommend_similar_books(
    "BOOK_DOES_NOT_EXIST",
    top_n=10
)

book_checks["Invalid book ID returns no results"] = (
    invalid_book_results.empty
)


# ---------------------------------------------------------
# ZERO-VECTOR QUERY TEST
# ---------------------------------------------------------

zero_vector_books = app_catalog[
    app_catalog["enriched_zero_vector"].astype(bool)
]

zero_vector_test_id = (
    zero_vector_books.iloc[0]["book_id"]
    if not zero_vector_books.empty
    else None
)

if zero_vector_test_id is not None:

    zero_vector_query_results = recommend_similar_books(
        zero_vector_test_id,
        top_n=10
    )

    book_checks[
        "Zero-vector query returns no results"
    ] = zero_vector_query_results.empty


# ---------------------------------------------------------
# REPORT
# ---------------------------------------------------------

print("BOOK-TO-BOOK RECOMMENDATION VALIDATION")
print("-" * 60)

for check, result in book_checks.items():
    print(f"{check:<45} {result}")

print("\nQuery book:", validation_book_id)
print(
    "Recommendations returned:",
    len(book_validation_results)
)

print(
    "Highest similarity:",
    round(
        book_validation_results[
            "similarity_score"
        ].max(),
        6
    )
)

print(
    "Lowest Top-10 similarity:",
    round(
        book_validation_results[
            "similarity_score"
        ].min(),
        6
    )
)

if zero_vector_test_id is not None:
    print(
        "Zero-vector test book:",
        zero_vector_test_id
    )

print(
    "\nOverall validation:",
    all(book_checks.values())
)

BOOK-TO-BOOK RECOMMENDATION VALIDATION
------------------------------------------------------------
Returns 10 recommendations                    True
Query book excluded                           True
Book IDs unique                               True
Title-author keys unique                      True
All similarities positive                     True
Similarity descending                         True
Ranks sequential                              True
No zero-vector books returned                 True
Invalid book ID returns no results            True
Zero-vector query returns no results          True

Query book: BOOK00001
Recommendations returned: 10
Highest similarity: 0.220669
Lowest Top-10 similarity: 0.133814
Zero-vector test book: BOOK00963

Overall validation: True


## 18.15 — Topic-Based Book Exploration

The Streamlit application also provides a topic-exploration feature based on the 29 final K-Means clusters developed during the unsupervised-learning stage.

Unlike the two recommendation modes, topic exploration does not calculate a new similarity ranking. Instead, it allows users to browse books that belong to an existing validated leadership or management topic cluster.

The topic clusters serve as a supplementary discovery and interpretability layer. They do not replace the Enriched TF-IDF + cosine-similarity recommendation engine.

For each selected topic, the application can display:

- Books assigned to the topic
- Authors
- Available publication information
- Ratings and engagement metadata where available
- Book-cover images where available
- Source provenance

Because metadata availability differs across the integrated data sources, missing values are preserved rather than inferred or fabricated.

In [47]:
# ---------------------------------------------------------
# TOPIC EXPLORATION SETUP
# ---------------------------------------------------------

topic_options = (
    app_catalog.loc[
        app_catalog["cluster_label"].notna(),
        "cluster_label"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Number of available topics:", len(topic_options))

for i, topic in enumerate(topic_options, start=1):
    print(f"{i:>2}. {topic}")

Number of available topics: 29
 1. Broad Leadership and Success
 2. CEO and Executive Transformation
 3. Change Management
 4. Communication
 5. Contemporary Leadership Practice
 6. Decision Making
 7. Emotional Intelligence
 8. Entrepreneurial and Growth Mindset
 9. Executive Leadership
10. Future Thinking and Personal Development
11. General Business and Management
12. Human Resource Management
13. Information Technology Management
14. Innovation Management
15. Leader Identity and Practice
16. Leadership Development
17. Leading Self and Others
18. Operations Management
19. Organizational Behavior
20. Organizational Culture and Trust
21. People Management
22. Performance Management
23. Power and Influence
24. Project Management
25. Servant Leadership
26. Strategic Management and Planning
27. Team Leadership
28. Transformational Leadership
29. Workplace and Generational Dynamics


In [48]:
# ---------------------------------------------------------
# TOPIC EXPLORATION FUNCTION
# ---------------------------------------------------------

def explore_topic(topic, max_books=20):
    """
    Return books assigned to a validated topic cluster.

    Books are ordered using available engagement information
    where possible. Missing engagement values are not imputed.
    """

    if not isinstance(topic, str) or not topic.strip():
        return pd.DataFrame()

    if not isinstance(max_books, int) or max_books <= 0:
        return pd.DataFrame()

    topic = topic.strip()

    topic_books = app_catalog[
        app_catalog["cluster_label"] == topic
    ].copy()

    if topic_books.empty:
        return pd.DataFrame()

    # -----------------------------------------------------
    # DISPLAY ORDER
    # -----------------------------------------------------
    # Engagement is used only as a browsing/display signal.
    # It is NOT part of the recommendation model.

    topic_books["_engagement_sort"] = (
        pd.to_numeric(
            topic_books["want_to_read_count"],
            errors="coerce"
        )
        .fillna(-1)
    )

    topic_books = (
        topic_books
        .sort_values(
            by=[
                "_engagement_sort",
                "canonical_title",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .head(max_books)
        .copy()
    )

    topic_books["topic_browse_rank"] = np.arange(
        1,
        len(topic_books) + 1
    )

    result_columns = [
        "topic_browse_rank",
        "book_id",
        "canonical_title",
        "authors_display",
        "topic_display",
        "first_publish_year",
        "publication_year_observed",
        "average_rating",
        "ratings_count",
        "want_to_read_count",
        "edition_count",
        "cover_url",
        "cover_available",
        "source_group",
    ]

    return (
        topic_books[result_columns]
        .reset_index(drop=True)
    )

In [49]:
test_topic = "Decision Making"

topic_test_results = explore_topic(
    test_topic,
    max_books=10
)

topic_total_books = (
    app_catalog["cluster_label"]
    .eq(test_topic)
    .sum()
)

print("Selected topic:", test_topic)
print("Total books in topic:", topic_total_books)
print(
    "Books displayed:",
    len(topic_test_results)
)

display(
    topic_test_results[
        [
            "topic_browse_rank",
            "canonical_title",
            "authors_display",
            "want_to_read_count",
            "average_rating",
            "source_group",
        ]
    ]
)

Selected topic: Decision Making
Total books in topic: 33
Books displayed: 10


,topic_browse_rank,canonical_title,authors_display,want_to_read_count,average_rating,source_group
0,1,Business Statistics,['Ken Black'],157.0,4.2,Open Library only
1,2,Leadership in Organizations,['Gary A. Yukl'],110.0,5.0,Open Library only
2,3,The managerial decision-making process,['E. Frank Harrison'],41.0,NaN,Open Library only
3,4,Administrative behavior,['Herbert Alexander Simon'],40.0,NaN,Open Library only
4,5,The psychology of judgment and decision making,['Scott Plous'],40.0,NaN,Open Library only
5,6,Statistical analysis for decision making,['Morris Hamburg'],33.0,4.0,Open Library only
6,7,Business Analytics,"['S. Christian Albright', 'Wayne L. Winston']",29.0,1.0,Open Library only
7,8,An introduction to management science,['David Ray Anderson'],21.0,5.0,Open Library only
8,9,Judgment in managerial decision making,['Max H. Bazerman'],21.0,NaN,Open Library only
9,10,The ethical decision-making manual for helping...,['Sarah O. Steinman'],20.0,NaN,Open Library only


## 18.16 — Topic Exploration Validation

The topic-exploration function is validated before integration into the Streamlit interface.

The validation confirms that:

- All 29 final topic clusters are available for browsing.
- A valid topic returns the requested number of books when sufficient records exist.
- Every returned book belongs to the selected topic.
- Book identifiers are unique.
- Browsing ranks are sequential.
- Available engagement values are ordered from highest to lowest.
- Missing engagement values remain missing rather than being imputed.
- Invalid and empty topic selections are handled safely.

Topic browsing is treated as a supplementary discovery feature rather than a recommendation model. Its display order uses available engagement metadata only to organize books within an already selected topic.

In [50]:
# ---------------------------------------------------------
# TOPIC EXPLORATION VALIDATION
# ---------------------------------------------------------

validation_topic = "Decision Making"

topic_validation_results = explore_topic(
    validation_topic,
    max_books=10
)


# ---------------------------------------------------------
# CHECK ENGAGEMENT ORDER
# ---------------------------------------------------------

observed_engagement = (
    topic_validation_results[
        "want_to_read_count"
    ]
    .dropna()
)

engagement_descending = (
    observed_engagement
    .is_monotonic_decreasing
)


# ---------------------------------------------------------
# STANDARD CHECKS
# ---------------------------------------------------------

topic_checks = {
    "Exactly 29 topic options":
        len(topic_options) == 29,

    "Returns 10 books":
        len(topic_validation_results) == 10,

    "All books belong to selected topic":
        (
            topic_validation_results[
                "topic_display"
            ] == validation_topic
        ).all(),

    "Book IDs unique":
        topic_validation_results[
            "book_id"
        ].is_unique,

    "Browse ranks sequential":
        topic_validation_results[
            "topic_browse_rank"
        ].tolist()
        == list(
            range(
                1,
                len(topic_validation_results) + 1
            )
        ),

    "Observed engagement descending":
        engagement_descending,
}


# ---------------------------------------------------------
# INVALID INPUT TESTS
# ---------------------------------------------------------

invalid_topic_results = explore_topic(
    "TOPIC_DOES_NOT_EXIST",
    max_books=10
)

empty_topic_results = explore_topic(
    "",
    max_books=10
)

topic_checks[
    "Invalid topic returns no results"
] = invalid_topic_results.empty

topic_checks[
    "Empty topic returns no results"
] = empty_topic_results.empty


# ---------------------------------------------------------
# VERIFY TOTAL CLUSTER COVERAGE
# ---------------------------------------------------------

clustered_books_from_topics = (
    app_catalog[
        "cluster_label"
    ]
    .notna()
    .sum()
)

topic_checks[
    "Topic coverage = 1,884 books"
] = clustered_books_from_topics == 1884


# ---------------------------------------------------------
# REPORT
# ---------------------------------------------------------

print("TOPIC EXPLORATION VALIDATION")
print("-" * 60)

for check, result in topic_checks.items():
    print(f"{check:<45} {result}")

print(
    "\nAvailable topics:",
    len(topic_options)
)

print(
    "Decision Making cluster size:",
    (
        app_catalog[
            "cluster_label"
        ] == validation_topic
    ).sum()
)

print(
    "Books displayed:",
    len(topic_validation_results)
)

print(
    "\nOverall validation:",
    all(topic_checks.values())
)

TOPIC EXPLORATION VALIDATION
------------------------------------------------------------
Exactly 29 topic options                      True
Returns 10 books                              True
All books belong to selected topic            True
Book IDs unique                               True
Browse ranks sequential                       True
Observed engagement descending                True
Invalid topic returns no results              True
Empty topic returns no results                True
Topic coverage = 1,884 books                  True

Available topics: 29
Decision Making cluster size: 33
Books displayed: 10

Overall validation: True


## 18.17 — Streamlit Application Architecture

With the three discovery mechanisms validated, development now moves from notebook-based functional testing to the interactive Streamlit application.

The application will provide three primary discovery modes:

1. **Describe What You Need**  
   Users enter a natural-language description of their leadership or management needs. The system transforms the query using the validated Enriched TF-IDF vectorizer and retrieves books using cosine similarity.

2. **Find Similar Books**  
   Users select an existing catalog book and receive content-based recommendations based on similarity between book TF-IDF representations.

3. **Explore Topics**  
   Users browse books within the 29 validated K-Means leadership and management topic clusters.

### Application Architecture

The deployed application will follow this structure:

**User Interface → Validated Retrieval Functions → Production Artifacts → Grounded Book Metadata → Recommendation Display**

The application will reuse the previously validated model artifacts rather than retraining models during execution.

### Model Roles

- **Enriched TF-IDF + Cosine Similarity:** Production recommendation engine
- **K-Means Topic Clusters:** Topic exploration and interpretability
- **200-D LSA Representation:** Supporting analytical representation
- **Autoencoder:** Experimental model only; not used for production recommendations
- **LLM Layer:** Optional grounded explanation layer; live API inference is not required for the application

The interface will explicitly describe similarity scores as textual similarity measures rather than ratings, probabilities, or measures of book quality.

## 18.18 — Creating the Streamlit Application Shell

The first deployment step creates a standalone Streamlit application script inside the project's `app` directory.

The initial application contains only the page configuration, project title, system description, and navigation structure. Recommendation logic will be added after confirming that the standalone Streamlit environment launches successfully.

Separating interface construction from model integration makes application errors easier to identify and debug.

In [51]:
# ---------------------------------------------------------
# CREATE STREAMLIT APPLICATION DIRECTORY
# ---------------------------------------------------------

APP_DIR = PROJECT_ROOT / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

STREAMLIT_APP_PATH = APP_DIR / "streamlit_app.py"

print("Application directory:", APP_DIR)
print("Application file:", STREAMLIT_APP_PATH)
print("Directory exists:", APP_DIR.exists())

Application directory: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app
Application file: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/streamlit_app.py
Directory exists: True


In [52]:
# ---------------------------------------------------------
# INITIAL STREAMLIT APPLICATION SHELL
# ---------------------------------------------------------

streamlit_shell = '''import streamlit as st
from pathlib import Path


# ---------------------------------------------------------
# PAGE CONFIGURATION
# ---------------------------------------------------------

st.set_page_config(
    page_title="Leadership & Management Book Recommender",
    page_icon="📚",
    layout="wide",
    initial_sidebar_state="expanded",
)


# ---------------------------------------------------------
# PROJECT PATHS
# ---------------------------------------------------------

APP_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = APP_DIR.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"


# ---------------------------------------------------------
# APPLICATION HEADER
# ---------------------------------------------------------

st.title("Leadership & Management Book Recommendation System")

st.write(
    """
    Discover leadership and management books using a content-based
    recommendation system developed from an integrated catalog of
    2,067 books.
    """
)

st.caption(
    "Recommendations are based on textual similarity using "
    "Enriched TF-IDF and cosine similarity."
)


# ---------------------------------------------------------
# SIDEBAR
# ---------------------------------------------------------

st.sidebar.title("Book Discovery")

app_mode = st.sidebar.radio(
    "Choose a discovery mode:",
    [
        "Describe What You Need",
        "Find Similar Books",
        "Explore Topics",
        "About the System",
    ],
)


# ---------------------------------------------------------
# PAGE ROUTING
# ---------------------------------------------------------

if app_mode == "Describe What You Need":

    st.header("Describe What You Need")

    st.write(
        """
        Describe the leadership or management skills, challenges,
        or topics you want to explore.
        """
    )

    st.info(
        "Natural-language recommendation functionality will be "
        "connected to the validated retrieval model in the next step."
    )


elif app_mode == "Find Similar Books":

    st.header("Find Similar Books")

    st.write(
        """
        Select a book from the catalog to discover books with
        similar textual content.
        """
    )

    st.info(
        "Book-to-book recommendation functionality will be "
        "connected in the next development step."
    )


elif app_mode == "Explore Topics":

    st.header("Explore Leadership & Management Topics")

    st.write(
        """
        Browse books across the 29 topic clusters identified through
        the project's unsupervised-learning analysis.
        """
    )

    st.info(
        "Topic exploration functionality will be connected in the "
        "next development step."
    )


else:

    st.header("About the System")

    st.markdown(
        """
        ### Production Recommendation Architecture

        **Catalog:** 2,067 leadership and management books  
        **Usable recommendation vectors:** 2,040  
        **Topic-clustered books:** 1,884  
        **Topic clusters:** 29  
        **Catalog recommendation coverage:** 97.53%

        The production recommendation engine uses **Enriched TF-IDF
        with cosine similarity**.

        Topic clusters provide an additional exploration and
        interpretation layer.

        Similarity scores represent textual similarity and should not
        be interpreted as book quality, predicted ratings, probabilities,
        or expert evaluations.
        """
    )
'''

STREAMLIT_APP_PATH.write_text(
    streamlit_shell,
    encoding="utf-8"
)

print("Streamlit application created.")
print("File exists:", STREAMLIT_APP_PATH.exists())
print(
    "File size:",
    STREAMLIT_APP_PATH.stat().st_size,
    "bytes"
)

Streamlit application created.
File exists: True
File size: 3530 bytes


## 18.19 — Production Artifact Loading and Streamlit Caching

The standalone Streamlit application loads the same validated production artifacts used during notebook development.

Because Streamlit reruns the application script whenever a user interacts with the interface, resource and data caching are used to avoid unnecessary repeated loading.

- `st.cache_resource` is used for the fitted Enriched TF-IDF vectorizer and sparse TF-IDF matrix.
- `st.cache_data` is used for the validated application metadata catalog.

The caching implementation is contained in the standalone `app/streamlit_app.py` file and is not executed directly inside the Jupyter Notebook environment.

This preserves a clear separation between model validation in Notebook 18 and application execution through Streamlit.

## 18.20 — Creating a Portable TF-IDF Vectorizer for Deployment

The validated TF-IDF vectorizer was originally serialized during notebook development. Its custom `boundary_aware_analyzer` was therefore stored with a reference to the Jupyter `__main__` environment.

This works within the notebook environment but is not portable to a standalone Streamlit application.

To preserve the validated production model while enabling deployment, a separate application-safe copy of the vectorizer is created. The original Notebook 09 artifact remains unchanged.

The portable copy uses the exact same:

- field-boundary token
- English stop-word set
- Unicode-aware tokenization
- within-field unigram and bigram construction
- fitted vocabulary
- IDF weights

No model retraining or feature modification is performed.

In [2]:
# ---------------------------------------------------------
# CREATE PORTABLE ANALYZER MODULE
# ---------------------------------------------------------

from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

APP_DIR = PROJECT_ROOT / "app"
APP_DIR.mkdir(exist_ok=True)

NLP_UTILS_PATH = APP_DIR / "nlp_utils.py"

nlp_utils_code = r'''import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


FIELD_BOUNDARY = "zzfieldboundaryzz"
STOP_WORDS = set(ENGLISH_STOP_WORDS)


def boundary_aware_analyzer(document):
    features = []

    fields = document.split(FIELD_BOUNDARY)

    for field in fields:
        tokens = re.findall(
            r"(?u)\b[^\W_][\w'-]+\b",
            field.lower()
        )

        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        features.extend(tokens)

        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(len(tokens) - 1)
            ]
        )

    return features
'''

NLP_UTILS_PATH.write_text(
    nlp_utils_code,
    encoding="utf-8"
)

print("Created:", NLP_UTILS_PATH)
print("Exists:", NLP_UTILS_PATH.exists())

Created: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/nlp_utils.py
Exists: True


In [4]:
# ---------------------------------------------------------
# CREATE PORTABLE APPLICATION VECTORIZER
# ---------------------------------------------------------

import re
import sys
import joblib

from pathlib import Path
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


# ---------------------------------------------------------
# PROJECT PATHS
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ---------------------------------------------------------
# RESTORE ORIGINAL NOTEBOOK ANALYZER IN __main__
# ---------------------------------------------------------

FIELD_BOUNDARY = "zzfieldboundaryzz"
STOP_WORDS = set(ENGLISH_STOP_WORDS)


def boundary_aware_analyzer(document):
    features = []

    fields = document.split(FIELD_BOUNDARY)

    for field in fields:
        tokens = re.findall(
            r"(?u)\b[^\W_][\w'-]+\b",
            field.lower()
        )

        tokens = [
            token
            for token in tokens
            if token not in STOP_WORDS
        ]

        features.extend(tokens)

        features.extend(
            [
                f"{tokens[i]} {tokens[i + 1]}"
                for i in range(len(tokens) - 1)
            ]
        )

    return features


print(
    "Original analyzer restored as:",
    boundary_aware_analyzer.__module__
)


# ---------------------------------------------------------
# LOAD ORIGINAL VALIDATED VECTORIZER
# ---------------------------------------------------------

ORIGINAL_VECTORIZER_PATH = (
    PROJECT_ROOT
    / "models"
    / "enriched_tfidf_vectorizer.joblib"
)

original_vectorizer = joblib.load(
    ORIGINAL_VECTORIZER_PATH
)

print("Original vectorizer loaded successfully.")
print(
    "Vocabulary size:",
    len(original_vectorizer.vocabulary_)
)


# ---------------------------------------------------------
# IMPORT PORTABLE ANALYZER
# ---------------------------------------------------------

from app.nlp_utils import (
    boundary_aware_analyzer as portable_analyzer
)

print(
    "Portable analyzer module:",
    portable_analyzer.__module__
)


# ---------------------------------------------------------
# CHANGE ONLY THE ANALYZER REFERENCE
# ---------------------------------------------------------

original_vectorizer.set_params(
    analyzer=portable_analyzer
)


# ---------------------------------------------------------
# SAVE AS NEW APPLICATION ARTIFACT
# ---------------------------------------------------------

APP_VECTORIZER_PATH = (
    PROJECT_ROOT
    / "models"
    / "enriched_tfidf_vectorizer_app.joblib"
)

joblib.dump(
    original_vectorizer,
    APP_VECTORIZER_PATH
)


# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

print("\nPortable artifact:")
print(APP_VECTORIZER_PATH)

print(
    "\nPortable artifact exists:",
    APP_VECTORIZER_PATH.exists()
)

print(
    "Vocabulary size:",
    len(original_vectorizer.vocabulary_)
)

print(
    "Analyzer module:",
    original_vectorizer.analyzer.__module__
)

print(
    "Analyzer name:",
    original_vectorizer.analyzer.__name__
)

Original analyzer restored as: __main__
Original vectorizer loaded successfully.
Vocabulary size: 5130
Portable analyzer module: app.nlp_utils

Portable artifact:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_tfidf_vectorizer_app.joblib

Portable artifact exists: True
Vocabulary size: 5130
Analyzer module: app.nlp_utils
Analyzer name: boundary_aware_analyzer


### 18.21 — Finalizing the Portable Vectorizer for Streamlit

The first portable vectorizer referenced the analyzer through the package path `app.nlp_utils`. During standalone Streamlit execution, the application directory itself is placed on Python's module search path, causing the `app` package reference to be resolved inconsistently.

For deployment, the vectorizer is therefore serialized with the analyzer referenced directly from the `nlp_utils` module located beside `streamlit_app.py`.

This change affects only Python module portability. The fitted vocabulary, IDF weights, feature space, and validated TF-IDF methodology remain unchanged.

In [5]:
# ---------------------------------------------------------
# CREATE STREAMLIT-SAFE VECTORIZER
# ---------------------------------------------------------

import importlib
import joblib
import sys

# Make app/ directly importable
APP_DIR = PROJECT_ROOT / "app"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

# Import nlp_utils directly rather than as app.nlp_utils
nlp_utils = importlib.import_module("nlp_utils")

streamlit_analyzer = nlp_utils.boundary_aware_analyzer

print("Streamlit analyzer module:")
print(streamlit_analyzer.__module__)


# Use the vectorizer already loaded successfully in this notebook.
# Only change the callable reference.
original_vectorizer.set_params(
    analyzer=streamlit_analyzer
)


STREAMLIT_VECTORIZER_PATH = (
    PROJECT_ROOT
    / "models"
    / "enriched_tfidf_vectorizer_streamlit.joblib"
)

joblib.dump(
    original_vectorizer,
    STREAMLIT_VECTORIZER_PATH
)


print("\nSaved:")
print(STREAMLIT_VECTORIZER_PATH)

print("\nExists:")
print(STREAMLIT_VECTORIZER_PATH.exists())

print("\nVocabulary size:")
print(len(original_vectorizer.vocabulary_))

print("\nAnalyzer module:")
print(original_vectorizer.analyzer.__module__)

print("\nAnalyzer name:")
print(original_vectorizer.analyzer.__name__)

Streamlit analyzer module:
nlp_utils

Saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_tfidf_vectorizer_streamlit.joblib

Exists:
True

Vocabulary size:
5130

Analyzer module:
nlp_utils

Analyzer name:
boundary_aware_analyzer


### 18.23 — LeadWise Product and UI/UX Architecture

With the validated production artifacts successfully deployed, the Streamlit application is expanded from a standalone recommendation interface into **LeadWise — Leadership & Management Book Intelligence**.

LeadWise is designed as an interactive leadership and management reading-intelligence platform that integrates the project's validated machine-learning outputs with book discovery, comparison, topic exploration, personal reading management, user feedback, and inquiry functionality.

#### Core Application Modules

1. **Home**
   - Visual introduction to LeadWise
   - Catalog and model statistics
   - Natural-language book discovery
   - Featured topics and books
   - Overview of the machine-learning pipeline

2. **Discover for Me**
   - Natural-language user intent
   - Enriched TF-IDF representation
   - Cosine-similarity retrieval
   - Grounded recommendation explanations

3. **Book Explorer**
   - Browse and search the full catalog
   - Book covers and available metadata
   - Topic and metadata filtering

4. **Book Details**
   - Book cover, title, author, and topic
   - Available description/synopsis
   - Subjects and publication metadata
   - Ratings and engagement metadata when available
   - Similar-book discovery

5. **Compare Books**
   - Side-by-side book comparison
   - Shared and distinguishing metadata/themes
   - Content similarity
   - Related books

6. **Topic Explorer**
   - Exploration of the 29 validated topic clusters
   - Cluster membership and representative books
   - Topic-level catalog browsing

7. **My Library**
   - Want to Read
   - Currently Reading
   - Completed
   - Saved books

8. **Reviews & Feedback**
   - Community book reviews
   - LeadWise application feedback
   - User-generated ratings stored separately from source ratings

9. **Insights**
   - Catalog analytics
   - Topic landscape
   - Recommendation performance
   - Model architecture and evaluation

10. **Ask LeadWise**
    - Book and topic inquiries
    - Book discovery and comparison assistance
    - Application help
    - Client inquiry collection
    - Reviews and feedback

#### Visual Design

LeadWise will use a premium navy, gold, and warm-cream visual identity with:

- Library and leadership imagery
- Book-cover cards
- Background photography and selected video
- Custom Streamlit CSS
- Topic cards
- Interactive navigation
- Responsive content panels
- Consistent icons and visual hierarchy

All displayed book information, recommendations, explanations, and analytical results must remain grounded in the validated project data. Missing descriptions, ratings, covers, or other metadata will not be fabricated.

### 18.24 — LeadWise Visual Asset Architecture

To support the visual identity of LeadWise, a dedicated asset structure is created for application branding, backgrounds, imagery, video, icons, and fallback graphics.

Keeping visual assets separate from application logic improves maintainability and prepares the Streamlit application for later cloud deployment.

The asset structure includes:

- **branding** — LeadWise logo and brand graphics
- **backgrounds** — hero and section background images
- **images** — general application imagery
- **videos** — optional homepage and explanatory video assets
- **icons** — custom interface icons
- **placeholders** — fallback graphics for books without available cover images

Book covers already available through the validated catalog remain linked to their existing cover URLs and are not duplicated into the application assets.

In [6]:
from pathlib import Path

# ---------------------------------------------------------
# LeadWise application asset directories
# ---------------------------------------------------------

APP_DIR = PROJECT_ROOT / "app"
ASSETS_DIR = APP_DIR / "assets"

asset_directories = {
    "branding": ASSETS_DIR / "branding",
    "backgrounds": ASSETS_DIR / "backgrounds",
    "images": ASSETS_DIR / "images",
    "videos": ASSETS_DIR / "videos",
    "icons": ASSETS_DIR / "icons",
    "placeholders": ASSETS_DIR / "placeholders",
}

for directory in asset_directories.values():
    directory.mkdir(parents=True, exist_ok=True)

print("LeadWise asset structure")
print("-" * 50)

for name, directory in asset_directories.items():
    print(f"{name:<15} {directory.exists()}  {directory}")

print("-" * 50)
print("Assets root:", ASSETS_DIR)
print("Assets root exists:", ASSETS_DIR.exists())

LeadWise asset structure
--------------------------------------------------
branding        True  /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/branding
backgrounds     True  /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/backgrounds
images          True  /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/images
videos          True  /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/videos
icons           True  /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/icons
placeholders    True  /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/placeholders
--------------------------------------------------
Assets root: /Users/jannoelvero/Documents/Ironhack/

### 18.25 — LeadWise Visual Identity and Design System

A consistent visual identity is defined for LeadWise before individual interface components are developed.

The design follows the approved premium digital-library concept, combining a dark navy interface with warm gold accents and cream content areas. Photography, book covers, icons, and selected video assets will provide visual richness while maintaining readability and professional presentation.

#### Brand Direction

**Product Name:** LeadWise  
**Descriptor:** Leadership & Management Book Intelligence  
**Tagline:** Discover the right ideas for the leader you want to become.

#### Core Visual Palette

- Deep Navy — primary navigation and dark backgrounds
- Midnight Blue — secondary panels and overlays
- Warm Gold — primary accent and interactive emphasis
- Soft Gold — highlights and hover states
- Warm Cream — primary content background
- White — high-contrast text and cards
- Slate — secondary text
- Success Green — system and validation indicators

#### Interface Principles

- Premium digital-library appearance
- Strong visual hierarchy
- Book covers as primary content imagery
- Large photographic hero area
- Gold accents used selectively
- Rounded cards and subtle shadows
- Consistent spacing and typography
- Responsive layouts suitable for desktop and smaller screens
- Visual assets must not replace analytical clarity
- Missing book metadata or imagery must never be fabricated

In [7]:
# ---------------------------------------------------------
# LeadWise visual design tokens
# ---------------------------------------------------------

LEADWISE_THEME = {
    "navy": "#0B1F33",
    "midnight": "#102A43",
    "gold": "#D8A84E",
    "soft_gold": "#F0C96B",
    "cream": "#F8F4EC",
    "white": "#FFFFFF",
    "slate": "#66788A",
    "dark_text": "#14213D",
    "success": "#19B66A",
}

LEADWISE_BRAND = {
    "name": "LeadWise",
    "descriptor": "Leadership & Management Book Intelligence",
    "tagline": "Discover the right ideas for the leader you want to become.",
}

print("LeadWise Visual Identity")
print("-" * 50)

for key, value in LEADWISE_BRAND.items():
    print(f"{key:<12}: {value}")

print("\nColor Palette")
print("-" * 50)

for name, hex_code in LEADWISE_THEME.items():
    print(f"{name:<12}: {hex_code}")

LeadWise Visual Identity
--------------------------------------------------
name        : LeadWise
descriptor  : Leadership & Management Book Intelligence
tagline     : Discover the right ideas for the leader you want to become.

Color Palette
--------------------------------------------------
navy        : #0B1F33
midnight    : #102A43
gold        : #D8A84E
soft_gold   : #F0C96B
cream       : #F8F4EC
white       : #FFFFFF
slate       : #66788A
dark_text   : #14213D
success     : #19B66A


### 18.26 — LeadWise Brand Asset Development

The first production visual assets are created for the LeadWise interface.

The LeadWise brand mark combines an open-book symbol with the product name to represent leadership development, learning, knowledge discovery, and book intelligence.

Brand assets are maintained separately from the application code so that they can be reused consistently across the sidebar, homepage, chatbot, loading screens, and future deployment environments.

The initial branding assets include:

- LeadWise primary logo
- Compact book icon
- Missing book-cover placeholder
- Hero/background imagery developed separately for the homepage

These assets follow the established navy, gold, and cream visual system.

In [9]:
# ---------------------------------------------------------
# Validate LeadWise primary logo
# ---------------------------------------------------------

LEADWISE_LOGO = (
    PROJECT_ROOT
    / "app"
    / "assets"
    / "logo"
    / "leadwise_logo.png"
)

print("LeadWise Primary Logo")
print("-" * 50)
print("Path:  ", LEADWISE_LOGO)
print("Exists:", LEADWISE_LOGO.exists())

if LEADWISE_LOGO.exists():
    print("Size:  ", f"{LEADWISE_LOGO.stat().st_size / 1024:.2f} KB")
else:
    print("\nSave the approved logo as:")
    print(LEADWISE_LOGO)

LeadWise Primary Logo
--------------------------------------------------
Path:   /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/logo/leadwise_logo.png
Exists: True
Size:   1063.03 KB


### 18.27 — LeadWise Homepage Hero Asset

A dedicated photographic hero background is used to establish the visual identity of the LeadWise homepage.

The hero image follows the approved premium digital-library aesthetic with warm architectural lighting, dark navy tones, bookshelves, and an elegant contemporary reading environment.

The background itself contains no application text, buttons, statistics, or interface elements. These components are rendered separately by Streamlit so that they remain responsive, accessible, and interactive.

The composition intentionally preserves darker visual space for the LeadWise title, tagline, catalog statistics, discovery field, and primary navigation actions.

### 18.28 — LeadWise Hero Asset Validation

The custom LeadWise hero image is stored as a local application asset rather than loaded from an external website.

This improves visual consistency and reduces dependency on external image availability. The hero will be used as the photographic background of the homepage, while all titles, statistics, search controls, and interactive elements remain native Streamlit components layered over the visual design.

In [10]:
# ---------------------------------------------------------
# Validate LeadWise homepage hero image
# ---------------------------------------------------------

LEADWISE_HERO = (
    PROJECT_ROOT
    / "app"
    / "assets"
    / "backgrounds"
    / "leadwise_hero.png"
)

print("LeadWise Homepage Hero")
print("-" * 50)
print("Path:  ", LEADWISE_HERO)
print("Exists:", LEADWISE_HERO.exists())

if LEADWISE_HERO.exists():
    print("Size:  ", f"{LEADWISE_HERO.stat().st_size / 1024:.2f} KB")
else:
    print("\nExpected hero image:")
    print(LEADWISE_HERO)

LeadWise Homepage Hero
--------------------------------------------------
Path:   /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/app/assets/backgrounds/leadwise_hero.png
Exists: True
Size:   1791.09 KB


### 18.29 — LeadWise Streamlit Theme Integration

The validated Streamlit application is now styled using the LeadWise visual design system.

Custom CSS is introduced as a presentation layer only. It does not modify the validated TF-IDF model, recommendation logic, clustering outputs, or application data.

The initial interface transformation establishes:

- Deep navy application navigation
- Warm cream content background
- Gold interactive accents
- Consistent typography
- Styled buttons and form controls
- Rounded interface components
- Reduced default Streamlit visual clutter

The visual layer is developed incrementally to preserve the validated production artifact-loading architecture.

### 18.30 — Responsive Layout and Typography Refinement

Before integrating the final photographic homepage hero, the LeadWise interface is refined to improve readability and responsive behavior.

This stage addresses visual issues identified during local application testing, including:

- clipped navigation labels and subtitles
- insufficient vertical spacing
- inconsistent line heights
- sidebar content density
- card height consistency
- multi-line text rendering
- responsive content width
- navigation spacing
- application readability across different browser and screen heights

The objective is to establish a stable and responsive visual foundation before introducing additional photographic assets and interactive interface components.

### 18.30 — LeadWise Photographic Homepage Hero

The LeadWise homepage is enhanced with a custom photographic hero section based on the approved visual concept.

The locally stored `leadwise_hero.png` image provides the visual background, while the application interface remains dynamically rendered by Streamlit.

The hero introduces the platform through:

- LeadWise branding
- Leadership & Management Book Intelligence identity
- Product tagline
- validated catalog statistics
- natural-language discovery entry point

A dark gradient overlay maintains sufficient contrast between the photographic background and interface content.

The hero image is treated exclusively as a presentation asset. All catalog statistics and machine-learning outputs displayed by the application continue to originate from the validated production artifacts.

### 18.31 — Natural-Language Book Discovery Interface

The validated natural-language retrieval engine is integrated into the LeadWise application to support personalized book discovery.

Users can describe a leadership challenge, management topic, professional objective, or learning interest in natural language. The application transforms the query using the validated enriched TF-IDF vectorizer and compares it with the production book matrix using cosine similarity.

The discovery workflow applies the same methodological safeguards established during recommendation-system development:

- zero-vector books are excluded from retrieval
- only positive cosine similarities are returned
- results are ranked by textual content similarity
- duplicate normalized title-author combinations are suppressed
- available catalog metadata is displayed without fabrication
- missing descriptions or other metadata remain explicitly unavailable
- similarity scores represent textual/content similarity rather than book quality, expert evaluation, or probability of relevance

This interface provides the primary personalized discovery capability of LeadWise while preserving the validated production recommendation architecture.

### 18.32 — LeadWise Book Metadata & Enrichment Audit

Before expanding the LeadWise recommendation cards and Book Details interface, the available metadata is audited to determine which book attributes can be reliably displayed in the application.

The audit focuses on information relevant to book discovery and book intelligence, including:

- book title and author information
- cover images
- descriptions and synopsis information
- subjects and categories
- publication year
- publisher
- ISBN-10 and ISBN-13
- page count
- average rating and ratings count
- reader-interest metrics
- observed price and currency
- original publication country
- original language
- available languages
- translation information
- topic-cluster information
- source provenance

The purpose of this audit is to separate metadata into three categories:

1. **Available for application use** — fields already present and sufficiently populated in the processed project data.
2. **Available elsewhere in the project** — fields that may require integration from another processed or source dataset.
3. **Requires external enrichment** — information that is not currently available and would require a reliable external data source.

LeadWise does not infer publication origin, original language, translation availability, price, or other factual book metadata from author names, ISBN patterns, titles, or similar indirect indicators.

Missing metadata remains explicitly unavailable until it can be supported by a reliable source.

#### 18.32.1 — Inventory of Candidate Metadata Files

The project data directories are inspected to identify processed, final, SQL-ready, and other structured datasets that may contain metadata required by the LeadWise application.

This step only inventories existing project files. No datasets are modified or merged.

In [11]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# Project data directory
# ---------------------------------------------------------

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_root = project_root / "data"

print("Project root:", project_root)
print("Data root:", data_root)
print()

# ---------------------------------------------------------
# Find structured data files
# ---------------------------------------------------------

candidate_files = []

extensions = {
    ".csv",
    ".parquet",
    ".xlsx",
    ".json",
}

for file_path in data_root.rglob("*"):
    
    if (
        file_path.is_file()
        and file_path.suffix.lower() in extensions
    ):
        
        candidate_files.append(file_path)


candidate_files = sorted(candidate_files)

print("Structured data files found:", len(candidate_files))
print()

for i, file_path in enumerate(candidate_files, start=1):
    
    relative_path = file_path.relative_to(project_root)
    
    print(
        f"{i:>3}. {relative_path}"
    )

Project root: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Data root: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data

Structured data files found: 78

  1. data/final/book_metadata_coverage.csv
  2. data/final/books_master.csv
  3. data/final/integration_quality_summary.csv
  4. data/final/leadershipnow_editions.csv
  5. data/final/openlibrary_integrated.csv
  6. data/processed/autoencoder_data_split.csv
  7. data/processed/autoencoder_evaluation_summary.csv
  8. data/processed/autoencoder_latent_32.csv
  9. data/processed/autoencoder_training_history.csv
 10. data/processed/books_features.csv
 11. data/processed/books_nlp_features.csv
 12. data/processed/books_with_final_topic_clusters.csv
 13. data/processed/core_lsa_200.csv
 14. data/processed/core_lsa_topic_terms.csv
 15. data/processed/core_vs_enriched_coverage.csv
 16. data/processed/core_vs_enriched_recommendations.csv
 17. dat

#### 18.32.2 — Metadata Column Audit

Selected integrated, cleaned, and relational datasets are inspected to identify the exact fields available for the expanded LeadWise book-information interface.

The audit focuses on metadata required for recommendation cards and detailed book profiles, including cover images, ratings, prices, publication information, identifiers, language, geographic origin, translations, descriptions, subjects, and reader-interest metrics.

At this stage, the analysis identifies existing fields only. No metadata is inferred, merged, or externally enriched.

In [12]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# Candidate files most likely to contain book metadata
# ---------------------------------------------------------

metadata_files = {
    "books_master":
        data_root / "final" / "books_master.csv",

    "openlibrary_integrated":
        data_root / "final" / "openlibrary_integrated.csv",

    "leadershipnow_editions":
        data_root / "final" / "leadershipnow_editions.csv",

    "open_library_search_clean":
        data_root / "processed" / "open_library_search_clean.csv",

    "open_library_work_enrichment_clean":
        data_root / "processed" / "open_library_work_enrichment_clean.csv",

    "leadershipnow_books_clean":
        data_root / "processed" / "leadershipnow_books_clean.csv",

    "books_features":
        data_root / "processed" / "books_features.csv",

    "llm_grounding_catalog":
        data_root / "processed" / "llm_grounding_catalog.csv",

    "sql_books":
        data_root / "sql_ready" / "books.csv",

    "sql_book_editions":
        data_root / "sql_ready" / "book_editions.csv",

    "sql_book_metrics":
        data_root / "sql_ready" / "book_metrics.csv",
}


# ---------------------------------------------------------
# Load and inspect columns
# ---------------------------------------------------------

loaded_metadata = {}

for name, path in metadata_files.items():

    if not path.exists():
        print(f"\n❌ MISSING: {name}")
        print(path)
        continue

    df = pd.read_csv(path)

    loaded_metadata[name] = df

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    print(f"Shape: {df.shape}")
    print()

    for i, column in enumerate(df.columns, start=1):
        print(f"{i:>3}. {column}")


BOOKS_MASTER
Shape: (2067, 18)

  1. book_id
  2. canonical_title
  3. authors
  4. description
  5. subjects
  6. first_publish_year
  7. average_rating
  8. ratings_count
  9. edition_count
 10. want_to_read_count
 11. currently_reading_count
 12. already_read_count
 13. cover_url
 14. openlibrary_key
 15. source_openlibrary
 16. source_leadershipnow
 17. publication_year_observed
 18. title_normalized

OPENLIBRARY_INTEGRATED
Shape: (950, 50)

  1. openlibrary_key
  2. title
  3. authors
  4. author_keys
  5. first_publish_year
  6. publish_dates
  7. publishers
  8. isbn_10
  9. isbn_13
 10. all_isbns
 11. languages
 12. subjects
 13. edition_count
 14. ratings_average
 15. ratings_count
 16. ratings_count_1
 17. ratings_count_2
 18. ratings_count_3
 19. ratings_count_4
 20. ratings_count_5
 21. want_to_read_count
 22. currently_reading_count
 23. already_read_count
 24. cover_id
 25. cover_url
 26. ebook_access
 27. has_fulltext
 28. public_scan
 29. source
 30. source_url
 31. co

#### 18.32.3 — Metadata Coverage Audit

The identified project datasets are evaluated to measure the availability of metadata required for the expanded LeadWise book profiles.

Coverage is measured using the actual populated values in the existing datasets. Particular attention is given to book identity, publication information, ratings, reader-interest metrics, cover images, language information, edition metadata, and content descriptions.

Fields that are absent from the current project datasets—such as observed price, currency, verified original publication country, and explicit translation metadata—are not inferred. These attributes are classified as candidates for later external enrichment using reliable sources.

In [13]:
# ---------------------------------------------------------
# 18.32.3 Metadata Coverage Audit
# ---------------------------------------------------------

def populated_count(series):
    """
    Count genuinely populated values.
    Handles NaN, blank strings, and common empty-text values.
    """

    if series is None:
        return 0

    cleaned = (
        series
        .astype("string")
        .str.strip()
    )

    missing_values = {
        "",
        "nan",
        "none",
        "null",
        "[]",
        "{}",
    }

    valid = (
        series.notna()
        & ~cleaned.str.lower().isin(missing_values)
    )

    return int(valid.sum())


# ---------------------------------------------------------
# Fields already available in the 2,067-book master catalog
# ---------------------------------------------------------

master = loaded_metadata["books_master"]

master_fields = [
    "canonical_title",
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_url",
    "openlibrary_key",
]


master_coverage = []

for field in master_fields:

    count = populated_count(
        master[field]
    )

    master_coverage.append(
        {
            "dataset": "books_master",
            "field": field,
            "records_with_values": count,
            "total_records": len(master),
            "coverage_pct": round(
                count / len(master) * 100,
                2
            ),
        }
    )


# ---------------------------------------------------------
# Open Library metadata
# ---------------------------------------------------------

openlibrary = loaded_metadata[
    "openlibrary_integrated"
]

openlibrary_fields = [
    "publish_dates",
    "publishers",
    "isbn_10",
    "isbn_13",
    "all_isbns",
    "languages",
    "subjects",
    "ratings_average",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_url",
    "ebook_access",
    "description",
    "first_sentence",
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
]


openlibrary_coverage = []

for field in openlibrary_fields:

    count = populated_count(
        openlibrary[field]
    )

    openlibrary_coverage.append(
        {
            "dataset": "openlibrary_integrated",
            "field": field,
            "records_with_values": count,
            "total_records": len(openlibrary),
            "coverage_pct": round(
                count / len(openlibrary) * 100,
                2
            ),
        }
    )


# ---------------------------------------------------------
# LeadershipNow edition metadata
# ---------------------------------------------------------

editions = loaded_metadata[
    "leadershipnow_editions"
]

edition_fields = [
    "subtitle",
    "isbn_source",
    "isbn_normalized",
    "isbn_13",
    "publisher",
    "publication_date",
    "publication_year",
    "format",
    "page_count",
    "edition_note",
    "cover_url",
    "external_book_url",
]


edition_coverage = []

for field in edition_fields:

    count = populated_count(
        editions[field]
    )

    edition_coverage.append(
        {
            "dataset": "leadershipnow_editions",
            "field": field,
            "records_with_values": count,
            "total_records": len(editions),
            "coverage_pct": round(
                count / len(editions) * 100,
                2
            ),
        }
    )


# ---------------------------------------------------------
# Combine results
# ---------------------------------------------------------

metadata_coverage_audit = pd.DataFrame(
    master_coverage
    + openlibrary_coverage
    + edition_coverage
)


metadata_coverage_audit = (
    metadata_coverage_audit
    .sort_values(
        [
            "dataset",
            "coverage_pct",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    metadata_coverage_audit
)


# ---------------------------------------------------------
# Explicitly identify requested metadata not found
# ---------------------------------------------------------

all_columns = set()

for df in loaded_metadata.values():
    all_columns.update(
        column.lower()
        for column in df.columns
    )


requested_missing_fields = [
    "price",
    "currency",
    "original_publication_country",
    "original_country",
    "country_of_origin",
    "original_language",
    "translation_count",
    "translations",
    "translated_languages",
]


print("\nREQUESTED METADATA FIELD CHECK")
print("=" * 60)

for field in requested_missing_fields:

    if field.lower() in all_columns:
        print(
            f"✓ Existing field found: {field}"
        )

    else:
        print(
            f"✗ No existing field found: {field}"
        )

,dataset,field,records_with_values,total_records,coverage_pct
0,books_master,canonical_title,2067,2067,100.00
1,books_master,authors,2059,2067,99.61
2,books_master,cover_url,1888,2067,91.34
3,books_master,publication_year_observed,1117,2067,54.04
4,books_master,edition_count,950,2067,45.96
5,books_master,openlibrary_key,950,2067,45.96
6,books_master,first_publish_year,947,2067,45.82
7,books_master,subjects,826,2067,39.96
8,books_master,want_to_read_count,807,2067,39.04
9,books_master,currently_reading_count,807,2067,39.04



REQUESTED METADATA FIELD CHECK
✗ No existing field found: price
✗ No existing field found: currency
✗ No existing field found: original_publication_country
✗ No existing field found: original_country
✗ No existing field found: country_of_origin
✗ No existing field found: original_language
✗ No existing field found: translation_count
✗ No existing field found: translations
✗ No existing field found: translated_languages


In [14]:
# ---------------------------------------------------------
# 18.32.5 Existing Language Metadata Inspection
# ---------------------------------------------------------

language_data = (
    loaded_metadata["openlibrary_integrated"][
        [
            "book_id",
            "title",
            "languages",
            "isbn_13",
        ]
    ]
    .copy()
)

language_data = (
    language_data[
        language_data["languages"].notna()
    ]
    .copy()
)

print(
    "Books with language metadata:",
    len(language_data)
)

print(
    "Coverage within Open Library:",
    f"{len(language_data) / len(loaded_metadata['openlibrary_integrated']) * 100:.2f}%"
)

print("\nSample language values:")
print("=" * 70)

display(
    language_data[
        [
            "title",
            "languages",
        ]
    ]
    .head(30)
)

print("\nMost common raw language values:")
print("=" * 70)

display(
    language_data["languages"]
    .value_counts()
    .head(30)
    .rename_axis("languages")
    .reset_index(name="book_count")
)

Books with language metadata: 950
Coverage within Open Library: 100.00%

Sample language values:


,title,languages
0,Principle-Centered Leadership,['eng']
1,Leadership in Organizations,"['tur', 'eng']"
2,Kepemimpinan =,['ind']
3,Spiritual leadership,['eng']
4,Leadership,['eng']
5,The 21 Irrefutable Laws of Leadership,"['eng', 'spa', 'cmn']"
6,Leadership,['eng']
7,Leadership and performance beyond expectations,['eng']
8,A Higher Loyalty,['eng']
9,Leadership and Self Deception,"['chi', 'eng', 'spa']"



Most common raw language values:


,languages,book_count
0,['eng'],844
1,[],48
2,"['eng', 'und']",6
3,"['eng', 'spa']",5
4,"['und', 'eng']",5
5,"['spa', 'eng']",4
6,"['eng', 'chi']",3
7,['ger'],3
8,"['eng', 'ger']",2
9,"['chi', 'eng']",2


#### 18.32.6 — Known Language Normalization

The Open Library language metadata is normalized into human-readable language names for use in the LeadWise application.

Raw three-letter language codes are converted into standardized display names. Empty language lists and undefined (`und`) language values are treated as unavailable.

Where multiple language codes are associated with a book, LeadWise records these as **Known Languages**. Multiple language values indicate that Open Library contains records or editions associated with more than one language; they are not treated as definitive evidence of the work's original language or a complete list of published translations.

Original language and verified translation information remain separate enrichment attributes.

In [15]:
import ast
import pandas as pd


# ---------------------------------------------------------
# Open Library language-code normalization
# ---------------------------------------------------------

LANGUAGE_MAP = {
    "eng": "English",
    "spa": "Spanish",
    "fre": "French",
    "fra": "French",
    "ger": "German",
    "deu": "German",
    "chi": "Chinese",
    "zho": "Chinese",
    "cmn": "Mandarin Chinese",
    "jpn": "Japanese",
    "kor": "Korean",
    "rus": "Russian",
    "por": "Portuguese",
    "ita": "Italian",
    "dut": "Dutch",
    "nld": "Dutch",
    "tur": "Turkish",
    "ind": "Indonesian",
    "heb": "Hebrew",
    "ara": "Arabic",
    "nor": "Norwegian",
    "fin": "Finnish",
    "rum": "Romanian",
    "ron": "Romanian",
    "swe": "Swedish",
    "dan": "Danish",
    "pol": "Polish",
    "cze": "Czech",
    "ces": "Czech",
    "hun": "Hungarian",
    "gre": "Greek",
    "ell": "Greek",
}


def parse_language_codes(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        codes = value

    else:
        try:
            codes = ast.literal_eval(str(value))
        except (ValueError, SyntaxError):
            return []

    if not isinstance(codes, list):
        return []

    cleaned = []

    for code in codes:

        code = str(code).strip().lower()

        if not code or code == "und":
            continue

        if code not in cleaned:
            cleaned.append(code)

    return cleaned


def convert_language_names(codes):

    return [
        LANGUAGE_MAP.get(
            code,
            code.upper()
        )
        for code in codes
    ]


language_normalized = (
    loaded_metadata["openlibrary_integrated"][
        [
            "book_id",
            "openlibrary_key",
            "title",
            "languages",
        ]
    ]
    .copy()
)


language_normalized["language_codes"] = (
    language_normalized["languages"]
    .apply(parse_language_codes)
)


language_normalized["known_languages"] = (
    language_normalized["language_codes"]
    .apply(convert_language_names)
)


language_normalized["known_languages_display"] = (
    language_normalized["known_languages"]
    .apply(
        lambda x:
        ", ".join(x)
        if x
        else pd.NA
    )
)


language_normalized["known_language_count"] = (
    language_normalized["known_languages"]
    .apply(len)
)


language_normalized["has_known_language"] = (
    language_normalized["known_language_count"] > 0
)


language_normalized["has_multiple_known_languages"] = (
    language_normalized["known_language_count"] > 1
)


print("LANGUAGE NORMALIZATION SUMMARY")
print("=" * 60)

print(
    "Open Library books:",
    len(language_normalized)
)

print(
    "Books with known language:",
    language_normalized["has_known_language"].sum()
)

print(
    "Books without usable language:",
    (~language_normalized["has_known_language"]).sum()
)

print(
    "Books with multiple known languages:",
    language_normalized[
        "has_multiple_known_languages"
    ].sum()
)

print(
    "Maximum known languages for one book:",
    language_normalized[
        "known_language_count"
    ].max()
)


print("\nNORMALIZED SAMPLE")
print("=" * 60)

display(
    language_normalized[
        [
            "title",
            "languages",
            "known_languages_display",
            "known_language_count",
        ]
    ]
    .head(30)
)


print("\nBOOKS WITH MULTIPLE KNOWN LANGUAGES")
print("=" * 60)

display(
    language_normalized.loc[
        language_normalized[
            "has_multiple_known_languages"
        ],
        [
            "title",
            "known_languages_display",
            "known_language_count",
        ]
    ]
    .sort_values(
        "known_language_count",
        ascending=False
    )
    .head(30)
)

LANGUAGE NORMALIZATION SUMMARY
Open Library books: 950
Books with known language: 902
Books without usable language: 48
Books with multiple known languages: 40
Maximum known languages for one book: 9

NORMALIZED SAMPLE


,title,languages,known_languages_display,known_language_count
0,Principle-Centered Leadership,['eng'],English,1
1,Leadership in Organizations,"['tur', 'eng']","Turkish, English",2
2,Kepemimpinan =,['ind'],Indonesian,1
3,Spiritual leadership,['eng'],English,1
4,Leadership,['eng'],English,1
5,The 21 Irrefutable Laws of Leadership,"['eng', 'spa', 'cmn']","English, Spanish, Mandarin Chinese",3
6,Leadership,['eng'],English,1
7,Leadership and performance beyond expectations,['eng'],English,1
8,A Higher Loyalty,['eng'],English,1
9,Leadership and Self Deception,"['chi', 'eng', 'spa']","Chinese, English, Spanish",3



BOOKS WITH MULTIPLE KNOWN LANGUAGES


,title,known_languages_display,known_language_count
332,The 7 Habits of Highly Effective People,"Japanese, Portuguese, Spanish, Korean, Russian...",9
808,Nonviolent Communication,"Spanish, GEM, English, French, Czech, German",6
858,Working with Emotional Intelligence,"Chinese, Polish, English, Spanish, Italian",5
280,Marketing management,"English, German, Norwegian, Chinese, French",5
350,The People of the Abyss,"French, Spanish, Finnish, English",4
5,The 21 Irrefutable Laws of Leadership,"English, Spanish, Mandarin Chinese",3
470,Rework,"French, Romanian, English",3
288,The Night Manager,"Chinese, German, English",3
149,Team of Rivals,"Korean, Chinese, English",3
931,Innovation and Entrepreneurship,"Spanish, English, Chinese",3


### 18.33 — Post-Collection Metadata Enrichment

The original LeadWise data-collection pipeline successfully collected the core bibliographic, textual, rating, engagement, and recommendation metadata required for the machine-learning workflow. However, several book-intelligence attributes planned for the final application were not extracted during the initial collection stage.

A post-collection enrichment stage is therefore introduced before construction of the final application catalog.

The enrichment targets are:

- verified original language
- original publication country or associated publication origin
- author country or geographic association
- known languages and multilingual availability
- translation information where verifiable
- observed book price
- currency
- price source and observation date

Existing project identifiers—including `book_id`, Open Library work identifiers, and ISBNs—will be used whenever possible to connect enrichment results with the existing catalog.

The enrichment process follows several data-quality safeguards:

1. Geographic origin is not inferred from author names, publishers, ISBN prefixes, or titles.
2. Original language is not inferred from the first language appearing in the existing language list.
3. Multiple known languages are not automatically interpreted as a complete translation history.
4. Prices represent observed source prices rather than universal or permanent retail prices.
5. Missing enrichment values remain unavailable when reliable evidence cannot be obtained.
6. Source provenance is retained for externally enriched factual metadata.

The enrichment layer supplements the existing recommendation architecture and does not alter the validated TF-IDF and cosine-similarity ranking model.

#### 18.33.1 — Enrichment Identifier Base

A consolidated identifier table is constructed from the existing LeadWise datasets before external enrichment.

The table preserves the project's canonical `book_id` while incorporating available Open Library identifiers, ISBNs, titles, authors, publisher information, and existing language metadata.

This identifier layer provides the matching foundation for subsequent enrichment and prevents external metadata from being attached to books solely through ambiguous title matching.

In [16]:
# ---------------------------------------------------------
# 18.33.1 Build enrichment identifier base
# ---------------------------------------------------------

master = loaded_metadata["books_master"].copy()

ol = loaded_metadata["openlibrary_integrated"][
    [
        "book_id",
        "openlibrary_key",
        "isbn_10",
        "isbn_13",
        "publishers",
        "languages",
        "source_url",
    ]
].copy()

ln = loaded_metadata["leadershipnow_editions"][
    [
        "book_id",
        "isbn_13",
        "publisher",
        "publication_date",
        "publication_year",
        "format",
        "page_count",
        "external_book_url",
    ]
].copy()


# ---------------------------------------------------------
# Rename source-specific fields
# ---------------------------------------------------------

ol = ol.rename(
    columns={
        "isbn_10": "ol_isbn_10",
        "isbn_13": "ol_isbn_13",
        "publishers": "ol_publishers",
        "languages": "ol_languages_raw",
        "source_url": "ol_source_url",
    }
)

ln = ln.rename(
    columns={
        "isbn_13": "ln_isbn_13",
        "publisher": "ln_publisher",
        "publication_date": "ln_publication_date",
        "publication_year": "ln_publication_year",
        "format": "ln_format",
        "page_count": "ln_page_count",
        "external_book_url": "ln_external_book_url",
    }
)


# ---------------------------------------------------------
# Verify source uniqueness before merging
# ---------------------------------------------------------

print("SOURCE UNIQUENESS CHECK")
print("=" * 60)

print(
    "Master unique book_id:",
    master["book_id"].nunique(),
    "/",
    len(master)
)

print(
    "Open Library unique book_id:",
    ol["book_id"].nunique(),
    "/",
    len(ol)
)

print(
    "LeadershipNow unique book_id:",
    ln["book_id"].nunique(),
    "/",
    len(ln)
)


# ---------------------------------------------------------
# Merge onto canonical master
# ---------------------------------------------------------

enrichment_base = (
    master
    .merge(
        ol,
        on="book_id",
        how="left",
        suffixes=("", "_ol"),
        validate="one_to_one",
    )
    .merge(
        ln,
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# Add normalized known-language fields from 18.32.6
# ---------------------------------------------------------

language_merge = language_normalized[
    [
        "book_id",
        "known_languages_display",
        "known_language_count",
        "has_known_language",
        "has_multiple_known_languages",
    ]
].copy()


enrichment_base = enrichment_base.merge(
    language_merge,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print("\nENRICHMENT BASE VALIDATION")
print("=" * 60)

print(
    "Rows:",
    len(enrichment_base)
)

print(
    "Unique book_id:",
    enrichment_base["book_id"].nunique()
)

print(
    "Open Library IDs:",
    enrichment_base["openlibrary_key"].notna().sum()
)

print(
    "OL ISBN-13:",
    enrichment_base["ol_isbn_13"].notna().sum()
)

print(
    "LeadershipNow ISBN-13:",
    enrichment_base["ln_isbn_13"].notna().sum()
)

print(
    "Known languages:",
    enrichment_base[
        "known_languages_display"
    ].notna().sum()
)

print(
    "Publishers from LeadershipNow:",
    enrichment_base[
        "ln_publisher"
    ].notna().sum()
)

print(
    "Page counts from LeadershipNow:",
    enrichment_base[
        "ln_page_count"
    ].notna().sum()
)


assert len(enrichment_base) == 2067
assert enrichment_base["book_id"].nunique() == 2067

print(
    "\n✓ Canonical 2,067-book catalog preserved."
)

SOURCE UNIQUENESS CHECK
Master unique book_id: 2067 / 2067
Open Library unique book_id: 950 / 950
LeadershipNow unique book_id: 1120 / 1120

ENRICHMENT BASE VALIDATION
Rows: 2067
Unique book_id: 2067
Open Library IDs: 950
OL ISBN-13: 950
LeadershipNow ISBN-13: 1120
Known languages: 902
Publishers from LeadershipNow: 1120
Page counts from LeadershipNow: 1119

✓ Canonical 2,067-book catalog preserved.


#### 18.33.2 — Open Library External Enrichment Preparation

The existing Open Library work identifiers and ISBNs are used to retrieve additional bibliographic metadata that was not captured during the original collection stage.

This enrichment focuses on factual metadata that may support the LeadWise Book Details interface, including:

- work-level and edition-level language information
- author identifiers and author metadata
- edition publication information
- publishers
- publication places where explicitly provided
- edition identifiers that may indicate multilingual publication availability

The enrichment process preserves raw API responses and source identifiers to maintain reproducibility and provenance.

No geographic origin, original language, or translation relationship is inferred from indirect indicators. A field is populated only when the retrieved source explicitly supports the corresponding metadata interpretation.

Commercial price information is excluded from this stage because Open Library is a bibliographic source rather than a reliable source of current retail prices.

In [17]:
import requests
import json
import pandas as pd


# ---------------------------------------------------------
# Select one Open Library book for enrichment testing
# ---------------------------------------------------------

test_book = (
    enrichment_base[
        enrichment_base["openlibrary_key"].notna()
    ]
    .iloc[0]
)


book_id = test_book["book_id"]
title = test_book["canonical_title"]
openlibrary_key = str(
    test_book["openlibrary_key"]
).strip()


print("TEST BOOK")
print("=" * 60)
print("book_id:", book_id)
print("title:", title)
print("openlibrary_key:", openlibrary_key)


# ---------------------------------------------------------
# Normalize Open Library work key
# ---------------------------------------------------------

if openlibrary_key.startswith("/works/"):
    work_path = openlibrary_key

elif openlibrary_key.startswith("OL"):
    work_path = f"/works/{openlibrary_key}"

else:
    raise ValueError(
        f"Unexpected Open Library key format: {openlibrary_key}"
    )


work_url = (
    f"https://openlibrary.org"
    f"{work_path}.json"
)

editions_url = (
    f"https://openlibrary.org"
    f"{work_path}/editions.json"
)


# ---------------------------------------------------------
# Retrieve work metadata
# ---------------------------------------------------------

headers = {
    "User-Agent":
        "LeadWise Academic Research Project"
}

work_response = requests.get(
    work_url,
    headers=headers,
    timeout=30,
)

print("\nWORK REQUEST")
print("=" * 60)
print("Status:", work_response.status_code)
print("URL:", work_url)

work_response.raise_for_status()

work_json = work_response.json()


# ---------------------------------------------------------
# Retrieve edition metadata
# ---------------------------------------------------------

edition_response = requests.get(
    editions_url,
    headers=headers,
    params={"limit": 50},
    timeout=30,
)

print("\nEDITION REQUEST")
print("=" * 60)
print("Status:", edition_response.status_code)
print("URL:", edition_response.url)

edition_response.raise_for_status()

edition_json = edition_response.json()


# ---------------------------------------------------------
# Inspect work-level keys
# ---------------------------------------------------------

print("\nWORK-LEVEL FIELDS")
print("=" * 60)

for key in sorted(work_json.keys()):
    print(key)


# ---------------------------------------------------------
# Inspect edition response
# ---------------------------------------------------------

entries = edition_json.get(
    "entries",
    []
)

print("\nEDITION SUMMARY")
print("=" * 60)

print(
    "Edition records returned:",
    len(entries)
)

print(
    "Total editions reported:",
    edition_json.get("size")
)


# ---------------------------------------------------------
# Union of available edition fields
# ---------------------------------------------------------

edition_fields = set()

for entry in entries:
    edition_fields.update(
        entry.keys()
    )


print("\nAVAILABLE EDITION FIELDS")
print("=" * 60)

for key in sorted(edition_fields):
    print(key)


# ---------------------------------------------------------
# Inspect fields relevant to enrichment
# ---------------------------------------------------------

edition_preview = []

for entry in entries[:20]:

    languages = []

    for language in entry.get(
        "languages",
        []
    ):
        if isinstance(language, dict):
            languages.append(
                language.get("key")
            )

    edition_preview.append(
        {
            "title":
                entry.get("title"),

            "publish_date":
                entry.get("publish_date"),

            "publishers":
                entry.get("publishers"),

            "publish_places":
                entry.get("publish_places"),

            "languages":
                languages,

            "isbn_10":
                entry.get("isbn_10"),

            "isbn_13":
                entry.get("isbn_13"),

            "number_of_pages":
                entry.get("number_of_pages"),

            "edition_key":
                entry.get("key"),
        }
    )


edition_preview_df = pd.DataFrame(
    edition_preview
)


print("\nEDITION METADATA PREVIEW")
print("=" * 60)

display(
    edition_preview_df
)

TEST BOOK
book_id: BOOK00001
title: Principle-Centered Leadership
openlibrary_key: /works/OL2630041W

WORK REQUEST
Status: 200
URL: https://openlibrary.org/works/OL2630041W.json

EDITION REQUEST
Status: 200
URL: https://openlibrary.org/works/OL2630041W/editions.json?limit=50

WORK-LEVEL FIELDS
authors
covers
created
description
dewey_number
excerpts
first_publish_date
first_sentence
key
last_modified
latest_revision
lc_classifications
revision
subjects
title
type

EDITION SUMMARY
Edition records returned: 21
Total editions reported: 21

AVAILABLE EDITION FIELDS
authors
by_statement
classifications
contributions
covers
created
description
dewey_decimal_class
edition_name
first_sentence
full_title
ia_box_id
ia_loaded_id
identifiers
isbn_10
isbn_13
key
languages
last_modified
latest_revision
lc_classifications
lccn
local_id
notes
number_of_pages
ocaid
oclc_number
oclc_numbers
other_titles
pagination
physical_dimensions
physical_format
publish_country
publish_date
publish_places
publishers

,title,publish_date,publishers,publish_places,languages,isbn_10,isbn_13,number_of_pages,edition_key
0,Principle-Centered Leadership,"Apr 01, 2012",[Franklin Covey on Brilliance Audio],None,[],[145589348X],[9781455893485],NaN,/books/OL32050650M
1,Principle Centred Leadership,"March 7, 2005",[Simon & Schuster Audio],None,[],[0743501551],[9780743501552],1.0,/books/OL7951825M
2,Principle-centred leadership,1991,[Simon & Schuster],[London],[/languages/eng],[068485841X],None,332.0,/books/OL22138988M
3,Principle-centred Leadership,"October 26, 1992",[Simon & Schuster (Trade Division)],None,[],[0671711350],[9780671711351],336.0,/books/OL7662929M
4,Principle-centred Leadership,"March 1, 1992",[Simon & Schuster (Trade Division)],None,[],[0671711164],[9780671711160],336.0,/books/OL9779667M
5,Principle Centered Leadership,September 1991,[Summit Books],None,[],[0916095312],[9780916095314],NaN,/books/OL8336551M
6,Principle-Centered Leadership,"October 1, 2001",[Covey],None,[/languages/eng],[188321906X],[9781883219062],NaN,/books/OL8685301M
7,Principle Centered Leadership,"January 4, 1999",[Simon Schuster Trade],None,[],[068485841X],[9780684858418],332.0,/books/OL7722396M
8,Principle-Centered Leadership,"October 1, 2001",[Covey],None,[/languages/eng],[1883219248],[9781883219246],15.0,/books/OL8685305M
9,Principle-centered Leadership,"January 4, 1999",[Simon & Schuster Audio],None,[],[0671011138],[9780671011130],NaN,/books/OL9699055M


#### 18.33.3 — Open Library Edition-Level Enrichment

Edition-level metadata is retrieved for the 950 books in the LeadWise catalog that have validated Open Library work identifiers.

The enrichment captures bibliographic evidence relevant to the final Book Intelligence interface, including:

- edition identifiers
- edition titles
- publication dates
- publication countries
- publication places
- languages
- publishers
- ISBN-10 and ISBN-13 identifiers
- page counts
- physical formats

All available editions are requested rather than assuming that the first edition returned by the API represents the original publication.

Raw edition-level observations are retained before book-level attributes are derived. This distinction is important because publication country, publication place, language, publisher, and format are edition-specific characteristics.

The resulting edition history may subsequently support derived attributes such as known publication languages, multilingual availability, earliest observed edition, geographic publication evidence, and international publication reach.

These derived attributes remain separate from stronger claims such as verified original language or definitive country of origin unless the available evidence explicitly supports those interpretations.

In [18]:
import requests
import pandas as pd
import time
from pathlib import Path


# ---------------------------------------------------------
# 18.33.3 Open Library edition-level enrichment
# ---------------------------------------------------------

raw_dir = data_root / "raw"
raw_dir.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_path = (
    raw_dir /
    "openlibrary_edition_enrichment_checkpoint.csv"
)

log_path = (
    raw_dir /
    "openlibrary_edition_enrichment_log.csv"
)


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def normalize_work_key(value):

    value = str(value).strip()

    if value.startswith("/works/"):
        return value

    if value.startswith("OL"):
        return f"/works/{value}"

    raise ValueError(
        f"Unexpected Open Library key: {value}"
    )


def extract_language_keys(value):

    results = []

    if not isinstance(value, list):
        return results

    for item in value:

        if isinstance(item, dict):

            key = item.get("key")

            if key:
                results.append(
                    key.replace(
                        "/languages/",
                        ""
                    )
                )

    return results


# ---------------------------------------------------------
# Books requiring enrichment
# ---------------------------------------------------------

ol_books = (
    enrichment_base.loc[
        enrichment_base[
            "openlibrary_key"
        ].notna(),
        [
            "book_id",
            "canonical_title",
            "openlibrary_key",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Open Library works scheduled:",
    len(ol_books)
)


# ---------------------------------------------------------
# Resume from checkpoint when available
# ---------------------------------------------------------

if checkpoint_path.exists():

    existing_editions = pd.read_csv(
        checkpoint_path
    )

    completed_book_ids = set(
        existing_editions[
            "book_id"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    print(
        "Existing checkpoint edition rows:",
        len(existing_editions)
    )

    print(
        "Books already completed:",
        len(completed_book_ids)
    )

else:

    existing_editions = pd.DataFrame()

    completed_book_ids = set()

    print(
        "No existing checkpoint found."
    )


# ---------------------------------------------------------
# Collection
# ---------------------------------------------------------

headers = {
    "User-Agent":
        "LeadWise Academic Research Project"
}

session = requests.Session()
session.headers.update(headers)

new_rows = []
collection_log = []

SAVE_EVERY = 25


for index, row in ol_books.iterrows():

    book_id = str(
        row["book_id"]
    )

    if book_id in completed_book_ids:
        continue

    title = row[
        "canonical_title"
    ]

    work_path = normalize_work_key(
        row["openlibrary_key"]
    )

    editions_url = (
        f"https://openlibrary.org"
        f"{work_path}/editions.json"
    )

    try:

        offset = 0
        limit = 50

        book_rows = []

        while True:

            response = session.get(
                editions_url,
                params={
                    "limit": limit,
                    "offset": offset,
                },
                timeout=30,
            )

            response.raise_for_status()

            payload = response.json()

            entries = payload.get(
                "entries",
                []
            )

            total_size = payload.get(
                "size"
            )

            for edition in entries:

                book_rows.append(
                    {
                        "book_id":
                            book_id,

                        "openlibrary_work_key":
                            work_path,

                        "canonical_title":
                            title,

                        "edition_key":
                            edition.get(
                                "key"
                            ),

                        "edition_title":
                            edition.get(
                                "title"
                            ),

                        "full_title":
                            edition.get(
                                "full_title"
                            ),

                        "edition_name":
                            edition.get(
                                "edition_name"
                            ),

                        "publish_date":
                            edition.get(
                                "publish_date"
                            ),

                        "publish_country":
                            edition.get(
                                "publish_country"
                            ),

                        "publish_places":
                            edition.get(
                                "publish_places"
                            ),

                        "publishers":
                            edition.get(
                                "publishers"
                            ),

                        "languages":
                            extract_language_keys(
                                edition.get(
                                    "languages",
                                    []
                                )
                            ),

                        "isbn_10":
                            edition.get(
                                "isbn_10"
                            ),

                        "isbn_13":
                            edition.get(
                                "isbn_13"
                            ),

                        "number_of_pages":
                            edition.get(
                                "number_of_pages"
                            ),

                        "physical_format":
                            edition.get(
                                "physical_format"
                            ),
                    }
                )

            if not entries:
                break

            offset += len(entries)

            if (
                total_size is not None
                and offset >= total_size
            ):
                break

            if len(entries) < limit:
                break

            # Small delay between pagination requests
            time.sleep(0.15)


        new_rows.extend(
            book_rows
        )

        collection_log.append(
            {
                "book_id":
                    book_id,

                "openlibrary_work_key":
                    work_path,

                "status":
                    "success",

                "edition_rows":
                    len(book_rows),

                "reported_size":
                    total_size,

                "error":
                    None,
            }
        )


    except Exception as exc:

        collection_log.append(
            {
                "book_id":
                    book_id,

                "openlibrary_work_key":
                    work_path,

                "status":
                    "error",

                "edition_rows":
                    0,

                "reported_size":
                    None,

                "error":
                    str(exc),
            }
        )


    # -----------------------------------------------------
    # Periodic checkpoint
    # -----------------------------------------------------

    processed_now = len(
        collection_log
    )

    if (
        processed_now % SAVE_EVERY == 0
    ):

        current_new = pd.DataFrame(
            new_rows
        )

        if existing_editions.empty:
            checkpoint_df = (
                current_new.copy()
            )

        else:
            checkpoint_df = pd.concat(
                [
                    existing_editions,
                    current_new,
                ],
                ignore_index=True,
            )

        checkpoint_df.to_csv(
            checkpoint_path,
            index=False,
        )

        pd.DataFrame(
            collection_log
        ).to_csv(
            log_path,
            index=False,
        )

        print(
            f"Processed this run: "
            f"{processed_now} | "
            f"Edition rows collected: "
            f"{len(new_rows)}"
        )


    # Polite delay between works
    time.sleep(0.20)


# ---------------------------------------------------------
# Final save
# ---------------------------------------------------------

current_new = pd.DataFrame(
    new_rows
)


if existing_editions.empty:

    final_editions = (
        current_new.copy()
    )

else:

    final_editions = pd.concat(
        [
            existing_editions,
            current_new,
        ],
        ignore_index=True,
    )


final_editions = (
    final_editions
    .drop_duplicates(
        subset=[
            "book_id",
            "edition_key",
        ]
    )
    .reset_index(drop=True)
)


final_editions.to_csv(
    checkpoint_path,
    index=False,
)


collection_log_df = pd.DataFrame(
    collection_log
)

collection_log_df.to_csv(
    log_path,
    index=False,
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("\nCOLLECTION COMPLETE")
print("=" * 60)

print(
    "Total edition rows:",
    len(final_editions)
)

print(
    "Books represented:",
    final_editions[
        "book_id"
    ].nunique()
)

print(
    "Successful works this run:",
    (
        collection_log_df["status"]
        == "success"
    ).sum()
    if not collection_log_df.empty
    else 0
)

print(
    "Errors this run:",
    (
        collection_log_df["status"]
        == "error"
    ).sum()
    if not collection_log_df.empty
    else 0
)

print(
    "Checkpoint:",
    checkpoint_path
)

print(
    "Log:",
    log_path
)

Open Library works scheduled: 950
No existing checkpoint found.
Processed this run: 25 | Edition rows collected: 262
Processed this run: 50 | Edition rows collected: 517
Processed this run: 75 | Edition rows collected: 619
Processed this run: 100 | Edition rows collected: 706
Processed this run: 125 | Edition rows collected: 807
Processed this run: 150 | Edition rows collected: 867
Processed this run: 175 | Edition rows collected: 947
Processed this run: 200 | Edition rows collected: 1020
Processed this run: 225 | Edition rows collected: 1079
Processed this run: 250 | Edition rows collected: 1140
Processed this run: 275 | Edition rows collected: 1197
Processed this run: 300 | Edition rows collected: 1780
Processed this run: 325 | Edition rows collected: 2207
Processed this run: 350 | Edition rows collected: 2475
Processed this run: 375 | Edition rows collected: 2620
Processed this run: 400 | Edition rows collected: 2906
Processed this run: 425 | Edition rows collected: 3049
Processed t

#### 18.33.4 — Open Library Enrichment Error Review

The edition-level enrichment retrieved 6,913 edition records representing 948 of the 950 Open Library works in the LeadWise catalog.

Two work-level requests did not complete successfully. Before edition metadata is aggregated into book-level attributes, these failed requests are inspected individually.

Only failed records are considered for retry. Successfully collected records are preserved in the existing checkpoint to avoid unnecessary duplicate API requests and maintain the integrity of the completed enrichment data.

In [19]:
# ---------------------------------------------------------
# 18.33.4 Inspect failed enrichment requests
# ---------------------------------------------------------

enrichment_log = pd.read_csv(
    data_root
    / "raw"
    / "openlibrary_edition_enrichment_log.csv"
)

failed_requests = (
    enrichment_log[
        enrichment_log["status"] == "error"
    ]
    .copy()
    .reset_index(drop=True)
)

print("FAILED REQUEST SUMMARY")
print("=" * 70)

print(
    "Failed works:",
    len(failed_requests)
)

display(
    failed_requests[
        [
            "book_id",
            "openlibrary_work_key",
            "status",
            "error",
        ]
    ]
)


# ---------------------------------------------------------
# Match failures back to canonical catalog
# ---------------------------------------------------------

failed_books = (
    failed_requests[
        [
            "book_id",
            "openlibrary_work_key",
            "error",
        ]
    ]
    .merge(
        enrichment_base[
            [
                "book_id",
                "canonical_title",
                "authors",
                "openlibrary_key",
            ]
        ],
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)

print("\nFAILED BOOK DETAILS")
print("=" * 70)

display(
    failed_books[
        [
            "book_id",
            "canonical_title",
            "authors",
            "openlibrary_key",
            "error",
        ]
    ]
)

FAILED REQUEST SUMMARY
Failed works: 2


,book_id,openlibrary_work_key,status,error
0,BOOK00123,/works/OL17671901M,error,404 Client Error: Not Found for url: https://o...
1,BOOK00852,/works/OL20690760M,error,404 Client Error: Not Found for url: https://o...



FAILED BOOK DETAILS


,book_id,canonical_title,authors,openlibrary_key,error
0,BOOK00123,Executive leadership.,[],/works/OL17671901M,404 Client Error: Not Found for url: https://o...
1,BOOK00852,The changing face of western communism,[],/works/OL20690760M,404 Client Error: Not Found for url: https://o...


##### 18.33.4A — Resolution of Misclassified Open Library Identifiers

Inspection of the two failed enrichment requests identified an identifier-type inconsistency in the existing integrated dataset.

Both failed Open Library identifiers end in `M` and were stored under the `openlibrary_key` field as work paths. The edition-enrichment workflow expects Open Library work identifiers suitable for the `/works/{work_id}/editions.json` endpoint.

The affected records are therefore treated as identifier-quality exceptions rather than API collection failures.

The corresponding Open Library edition records are inspected to determine whether they contain a valid parent work reference. If a parent work identifier is available, that work identifier is used for edition-level enrichment while preserving the originally collected identifier for provenance.

No identifier is inferred from the title alone.

In [20]:
# ---------------------------------------------------------
# 18.33.4A Resolve edition IDs stored as work identifiers
# ---------------------------------------------------------

problem_records = failed_books.copy()

resolved_records = []

headers = {
    "User-Agent":
        "LeadWise Academic Research Project"
}


for _, row in problem_records.iterrows():

    book_id = row["book_id"]
    stored_key = str(
        row["openlibrary_key"]
    ).strip()

    # Extract identifier from incorrectly stored path
    identifier = stored_key.split("/")[-1]

    edition_url = (
        f"https://openlibrary.org/books/"
        f"{identifier}.json"
    )

    print("\n" + "=" * 70)
    print(book_id)
    print(row["canonical_title"])
    print("Stored key:", stored_key)
    print("Edition URL:", edition_url)

    try:

        response = requests.get(
            edition_url,
            headers=headers,
            timeout=30,
        )

        print(
            "Status:",
            response.status_code
        )

        response.raise_for_status()

        payload = response.json()

        parent_works = payload.get(
            "works",
            []
        )

        work_keys = []

        for work in parent_works:

            if isinstance(work, dict):

                key = work.get("key")

                if key:
                    work_keys.append(key)

        print(
            "Edition title:",
            payload.get("title")
        )

        print(
            "Parent work keys:",
            work_keys
        )

        print(
            "ISBN-10:",
            payload.get("isbn_10")
        )

        print(
            "ISBN-13:",
            payload.get("isbn_13")
        )

        print(
            "Publish date:",
            payload.get("publish_date")
        )

        resolved_records.append(
            {
                "book_id":
                    book_id,

                "canonical_title":
                    row["canonical_title"],

                "stored_openlibrary_key":
                    stored_key,

                "edition_identifier":
                    identifier,

                "edition_status":
                    response.status_code,

                "edition_title":
                    payload.get("title"),

                "parent_work_keys":
                    work_keys,

                "resolved_work_key":
                    (
                        work_keys[0]
                        if len(work_keys) == 1
                        else None
                    ),
            }
        )

    except Exception as exc:

        print(
            "Error:",
            str(exc)
        )

        resolved_records.append(
            {
                "book_id":
                    book_id,

                "canonical_title":
                    row["canonical_title"],

                "stored_openlibrary_key":
                    stored_key,

                "edition_identifier":
                    identifier,

                "edition_status":
                    None,

                "edition_title":
                    None,

                "parent_work_keys":
                    [],

                "resolved_work_key":
                    None,

                "error":
                    str(exc),
            }
        )


resolved_problem_ids = pd.DataFrame(
    resolved_records
)


print("\nRESOLUTION SUMMARY")
print("=" * 70)

display(
    resolved_problem_ids
)


BOOK00123
Executive leadership.
Stored key: /works/OL17671901M
Edition URL: https://openlibrary.org/books/OL17671901M.json
Status: 200
Edition title: Executive leadership.
Parent work keys: []
ISBN-10: None
ISBN-13: None
Publish date: 1987

BOOK00852
The changing face of western communism
Stored key: /works/OL20690760M
Edition URL: https://openlibrary.org/books/OL20690760M.json
Status: 200
Edition title: The changing face of western communism
Parent work keys: []
ISBN-10: None
ISBN-13: None
Publish date: 1980

RESOLUTION SUMMARY


,book_id,canonical_title,stored_openlibrary_key,edition_identifier,edition_status,edition_title,parent_work_keys,resolved_work_key
0,BOOK00123,Executive leadership.,/works/OL17671901M,OL17671901M,200,Executive leadership.,[],None
1,BOOK00852,The changing face of western communism,/works/OL20690760M,OL20690760M,200,The changing face of western communism,[],None


#### 18.33.5 — Edition-Level Metadata Aggregation

The Open Library edition-enrichment dataset is aggregated from individual edition observations into book-level metadata suitable for the LeadWise application.

The aggregation preserves the distinction between directly observed bibliographic evidence and stronger claims about a work's original publication.

For each Open Library work, the enrichment derives:

- number of observed editions
- known publication languages
- number of known publication languages
- multilingual publication evidence
- observed publication countries
- observed publication places
- number of geographic publication locations
- observed publishers
- ISBN coverage
- earliest observed publication date
- earliest observed publication year
- publication metadata associated with the earliest observed edition

The terms **earliest observed**, **known publication languages**, and **observed publication locations** are used deliberately. Open Library edition histories may be incomplete, and the earliest edition represented in the API is not automatically interpreted as the definitive original edition.

The two Open Library records that contain valid edition identifiers but no parent work relationship remain documented as edition-only exceptions and are not assigned an inferred work identifier.

In [21]:
import ast
import re
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# Load collected edition enrichment
# ---------------------------------------------------------

edition_history = pd.read_csv(
    data_root
    / "raw"
    / "openlibrary_edition_enrichment_checkpoint.csv"
)

print("EDITION HISTORY")
print("=" * 70)

print(
    "Rows:",
    len(edition_history)
)

print(
    "Books represented:",
    edition_history["book_id"].nunique()
)


# ---------------------------------------------------------
# Safely parse serialized list columns
# ---------------------------------------------------------

def safe_list(value):

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()

    if (
        not text
        or text.lower()
        in {"nan", "none", "null", "[]"}
    ):
        return []

    try:

        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):

        return [text]


list_columns = [
    "publish_places",
    "publishers",
    "languages",
    "isbn_10",
    "isbn_13",
]


for column in list_columns:

    edition_history[
        f"{column}_parsed"
    ] = (
        edition_history[column]
        .apply(safe_list)
    )


# ---------------------------------------------------------
# Normalize individual list values
# ---------------------------------------------------------

def clean_list(values):

    cleaned = []

    for value in values:

        if pd.isna(value):
            continue

        value = str(value).strip()

        if (
            not value
            or value.lower()
            in {"nan", "none", "null"}
        ):
            continue

        if value not in cleaned:
            cleaned.append(value)

    return cleaned


for column in list_columns:

    parsed_column = (
        f"{column}_parsed"
    )

    edition_history[
        parsed_column
    ] = (
        edition_history[
            parsed_column
        ]
        .apply(clean_list)
    )


# ---------------------------------------------------------
# Normalize publication country
# ---------------------------------------------------------

edition_history[
    "publish_country_clean"
] = (
    edition_history[
        "publish_country"
    ]
    .astype("string")
    .str.strip()
)


edition_history.loc[
    edition_history[
        "publish_country_clean"
    ].str.lower().isin(
        [
            "",
            "nan",
            "none",
            "null",
        ]
    ),
    "publish_country_clean",
] = pd.NA


# ---------------------------------------------------------
# Extract publication year conservatively
# ---------------------------------------------------------

def extract_year(value):

    if pd.isna(value):
        return np.nan

    text = str(value)

    match = re.search(
        r"\b(1[5-9]\d{2}|20\d{2}|21\d{2})\b",
        text
    )

    if match:
        return int(
            match.group(1)
        )

    return np.nan


edition_history[
    "publish_year_extracted"
] = (
    edition_history[
        "publish_date"
    ]
    .apply(extract_year)
)


# ---------------------------------------------------------
# Coverage diagnostics
# ---------------------------------------------------------

print("\nEDITION-LEVEL METADATA COVERAGE")
print("=" * 70)

coverage_fields = {
    "Publication date":
        edition_history[
            "publish_date"
        ].notna().sum(),

    "Extractable publication year":
        edition_history[
            "publish_year_extracted"
        ].notna().sum(),

    "Publication country":
        edition_history[
            "publish_country_clean"
        ].notna().sum(),

    "Publication places":
        edition_history[
            "publish_places_parsed"
        ].apply(bool).sum(),

    "Languages":
        edition_history[
            "languages_parsed"
        ].apply(bool).sum(),

    "Publishers":
        edition_history[
            "publishers_parsed"
        ].apply(bool).sum(),

    "ISBN-10":
        edition_history[
            "isbn_10_parsed"
        ].apply(bool).sum(),

    "ISBN-13":
        edition_history[
            "isbn_13_parsed"
        ].apply(bool).sum(),

    "Page count":
        edition_history[
            "number_of_pages"
        ].notna().sum(),

    "Physical format":
        edition_history[
            "physical_format"
        ].notna().sum(),
}


coverage_table = pd.DataFrame(
    [
        {
            "field": field,
            "edition_records_with_value": count,
            "total_edition_records":
                len(edition_history),
            "coverage_pct":
                round(
                    count
                    / len(edition_history)
                    * 100,
                    2
                ),
        }
        for field, count
        in coverage_fields.items()
    ]
)


display(
    coverage_table
)


print("\nPUBLICATION YEAR RANGE")
print("=" * 70)

print(
    "Minimum extracted year:",
    edition_history[
        "publish_year_extracted"
    ].min()
)

print(
    "Maximum extracted year:",
    edition_history[
        "publish_year_extracted"
    ].max()
)

EDITION HISTORY
Rows: 6913
Books represented: 948

EDITION-LEVEL METADATA COVERAGE


,field,edition_records_with_value,total_edition_records,coverage_pct
0,Publication date,6837,6913,98.90
1,Extractable publication year,6836,6913,98.89
2,Publication country,2238,6913,32.37
3,Publication places,2256,6913,32.63
4,Languages,6117,6913,88.49
5,Publishers,6853,6913,99.13
6,ISBN-10,3823,6913,55.30
7,ISBN-13,5053,6913,73.09
8,Page count,4443,6913,64.27
9,Physical format,1633,6913,23.62



PUBLICATION YEAR RANGE
Minimum extracted year: 1900.0
Maximum extracted year: 2026.0


#### 18.33.6 — Book-Level Edition Intelligence

The 6,913 Open Library edition observations are aggregated into book-level bibliographic intelligence for the 948 works with retrievable edition histories.

The aggregation derives descriptive attributes from the observed edition records, including:

- observed edition count
- earliest and latest observed publication year
- known publication languages
- multilingual publication evidence
- observed publication countries
- observed publication places
- observed publishers
- observed ISBN-10 and ISBN-13 identifiers
- observed page-count range
- observed physical formats

These variables summarize the available Open Library edition history rather than asserting a complete publication history.

In particular:

- `earliest_observed_year` represents the earliest year found among retrieved editions and is not automatically interpreted as the definitive original publication year;
- `observed_publication_countries` and `observed_publication_places` describe edition-level publication geography rather than country of origin;
- `known_publication_languages` represents languages explicitly attached to retrieved editions and does not establish the original language;
- `multilingual_publication_evidence` indicates that editions in more than one known language were observed.

This aggregation provides a reproducible international-publication intelligence layer while retaining conservative interpretations of incomplete bibliographic evidence.

In [22]:
# ---------------------------------------------------------
# 18.33.6 Aggregate edition history to book level
# ---------------------------------------------------------

def unique_flatten(series):
    """
    Flatten lists across rows while preserving unique values.
    """

    values = []

    for item in series:

        if not isinstance(item, list):
            continue

        for value in item:

            value = str(value).strip()

            if (
                value
                and value.lower()
                not in {"nan", "none", "null", "und"}
                and value not in values
            ):
                values.append(value)

    return values


def unique_scalar(series):
    """
    Return unique non-missing scalar values.
    """

    values = []

    for value in series.dropna():

        value = str(value).strip()

        if (
            value
            and value.lower()
            not in {"nan", "none", "null"}
            and value not in values
        ):
            values.append(value)

    return values


# ---------------------------------------------------------
# Aggregate each book
# ---------------------------------------------------------

book_level_rows = []


for book_id, group in edition_history.groupby(
    "book_id",
    sort=False
):

    languages = unique_flatten(
        group["languages_parsed"]
    )

    countries = unique_scalar(
        group["publish_country_clean"]
    )

    places = unique_flatten(
        group["publish_places_parsed"]
    )

    publishers = unique_flatten(
        group["publishers_parsed"]
    )

    isbn_10 = unique_flatten(
        group["isbn_10_parsed"]
    )

    isbn_13 = unique_flatten(
        group["isbn_13_parsed"]
    )

    formats = unique_scalar(
        group["physical_format"]
    )

    years = (
        group["publish_year_extracted"]
        .dropna()
        .astype(int)
    )

    pages = pd.to_numeric(
        group["number_of_pages"],
        errors="coerce"
    ).dropna()


    book_level_rows.append(
        {
            "book_id":
                book_id,

            "observed_edition_count":
                group["edition_key"]
                .nunique(),

            "earliest_observed_year":
                (
                    int(years.min())
                    if len(years)
                    else pd.NA
                ),

            "latest_observed_year":
                (
                    int(years.max())
                    if len(years)
                    else pd.NA
                ),

            "known_publication_language_codes":
                languages,

            "known_publication_language_count":
                len(languages),

            "multilingual_publication_evidence":
                len(languages) > 1,

            "observed_publication_countries":
                countries,

            "observed_publication_country_count":
                len(countries),

            "observed_publication_places":
                places,

            "observed_publication_place_count":
                len(places),

            "observed_publishers":
                publishers,

            "observed_publisher_count":
                len(publishers),

            "observed_isbn_10":
                isbn_10,

            "observed_isbn_10_count":
                len(isbn_10),

            "observed_isbn_13":
                isbn_13,

            "observed_isbn_13_count":
                len(isbn_13),

            "observed_physical_formats":
                formats,

            "observed_format_count":
                len(formats),

            "minimum_observed_pages":
                (
                    int(pages.min())
                    if len(pages)
                    else pd.NA
                ),

            "maximum_observed_pages":
                (
                    int(pages.max())
                    if len(pages)
                    else pd.NA
                ),
        }
    )


book_edition_intelligence = pd.DataFrame(
    book_level_rows
)


# ---------------------------------------------------------
# Convert language codes to readable names
# ---------------------------------------------------------

def language_names(codes):

    names = []

    for code in codes:

        code = str(code).lower()

        name = LANGUAGE_MAP.get(
            code,
            code.upper()
        )

        if name not in names:
            names.append(name)

    return names


book_edition_intelligence[
    "known_publication_languages"
] = (
    book_edition_intelligence[
        "known_publication_language_codes"
    ]
    .apply(language_names)
)


book_edition_intelligence[
    "known_publication_languages_display"
] = (
    book_edition_intelligence[
        "known_publication_languages"
    ]
    .apply(
        lambda values:
        ", ".join(values)
        if values
        else pd.NA
    )
)


# ---------------------------------------------------------
# Merge titles for inspection
# ---------------------------------------------------------

book_edition_intelligence = (
    book_edition_intelligence
    .merge(
        enrichment_base[
            [
                "book_id",
                "canonical_title",
                "authors",
            ]
        ],
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# Reorder useful columns
# ---------------------------------------------------------

front_columns = [
    "book_id",
    "canonical_title",
    "authors",
    "observed_edition_count",
    "earliest_observed_year",
    "latest_observed_year",
    "known_publication_languages_display",
    "known_publication_language_count",
    "multilingual_publication_evidence",
    "observed_publication_countries",
    "observed_publication_country_count",
    "observed_publication_places",
    "observed_publication_place_count",
    "observed_publisher_count",
    "observed_isbn_13_count",
    "observed_format_count",
    "minimum_observed_pages",
    "maximum_observed_pages",
]


remaining_columns = [
    column
    for column in book_edition_intelligence.columns
    if column not in front_columns
]


book_edition_intelligence = (
    book_edition_intelligence[
        front_columns
        + remaining_columns
    ]
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print("BOOK-LEVEL EDITION INTELLIGENCE")
print("=" * 70)

print(
    "Rows:",
    len(book_edition_intelligence)
)

print(
    "Unique book_id:",
    book_edition_intelligence[
        "book_id"
    ].nunique()
)

print(
    "Books with known publication language:",
    (
        book_edition_intelligence[
            "known_publication_language_count"
        ] > 0
    ).sum()
)

print(
    "Books with multilingual evidence:",
    book_edition_intelligence[
        "multilingual_publication_evidence"
    ].sum()
)

print(
    "Books with observed publication country:",
    (
        book_edition_intelligence[
            "observed_publication_country_count"
        ] > 0
    ).sum()
)

print(
    "Books with observed publication place:",
    (
        book_edition_intelligence[
            "observed_publication_place_count"
        ] > 0
    ).sum()
)

print(
    "Books with publisher evidence:",
    (
        book_edition_intelligence[
            "observed_publisher_count"
        ] > 0
    ).sum()
)


assert (
    len(book_edition_intelligence)
    == 948
)

assert (
    book_edition_intelligence[
        "book_id"
    ].nunique()
    == 948
)

print(
    "\n✓ One row per enriched Open Library work."
)


# ---------------------------------------------------------
# Preview richest multilingual records
# ---------------------------------------------------------

print(
    "\nMULTILINGUAL PUBLICATION EXAMPLES"
)
print("=" * 70)

display(
    book_edition_intelligence[
        [
            "canonical_title",
            "observed_edition_count",
            "earliest_observed_year",
            "latest_observed_year",
            "known_publication_languages_display",
            "known_publication_language_count",
            "observed_publication_countries",
            "observed_publication_places",
        ]
    ]
    .sort_values(
        [
            "known_publication_language_count",
            "observed_edition_count",
        ],
        ascending=False,
    )
    .head(20)
)

BOOK-LEVEL EDITION INTELLIGENCE
Rows: 948
Unique book_id: 948
Books with known publication language: 900
Books with multilingual evidence: 40
Books with observed publication country: 583
Books with observed publication place: 575
Books with publisher evidence: 942

✓ One row per enriched Open Library work.

MULTILINGUAL PUBLICATION EXAMPLES


,canonical_title,observed_edition_count,earliest_observed_year,latest_observed_year,known_publication_languages_display,known_publication_language_count,observed_publication_countries,observed_publication_places
331,The 7 Habits of Highly Effective People,78,1989,2020,"English, Portuguese, Chinese, VIE, Spanish, Ru...",9,"[cc, vm, mx, ru, ch, nyu, ja, xxu, pau, meu, s...","[Beijing, TP. Hồ Chí Minh, Moskva, Tai bei shi..."
807,Nonviolent Communication,38,1999,2022,"French, Spanish, GEM, German, English, Czech",6,"[cau, nyu]","[Reggio Emilia, Paris, France, Warszawa, Polan..."
279,Marketing management,66,1973,2016,"English, French, German, Chinese, Norwegian",5,"[si, fr, xxk, gw, nju, onc, enk, xxc, xxu, ii,...","[Singapore, New York, Paris, Englewood Cliffs,..."
856,Working with Emotional Intelligence,20,1998,2016,"English, Italian, Spanish, Polish, Chinese",5,"[it, ag, pl, nyu, enk, ch]","[Milano, Poznań, New York, London, Taibei Shi]"
349,The People of the Abyss,72,1900,2020,"French, Finnish, English, Spanish",4,"[ag, tnu, enk, nyu, xxk, ilu, xx, nju, onc]","[Santa Fe, Memphis, TN, London, New York, [Chi..."
287,The Night Manager,39,1993,2020,"English, Chinese, German",3,"[enk, nyu, ch, onc, gw]","[New York, USA, New York, Taibei Xian Xindian ..."
853,Emotional Intelligence,39,1995,2020,"English, Spanish, Chinese",3,"[enk, xxk, ch, nyu]","[Great Britain, London, New York, Taibei Shi, ..."
929,Innovation and Entrepreneurship,33,1985,2015,"English, Chinese, Spanish",3,"[cc, enk, xxu, xxk, mx, nyu]","[Beijing, London, New York, Mexico, D.F]"
9,Leadership and Self Deception,31,2000,2025,"English, Spanish, Chinese",3,"[cc, cau, nyu]","[Beijing, San Francisco, San Francisco, Calif]"
46,Leadership Wisdom from the Monk Who Sold His F...,22,1998,2018,"Spanish, Hebrew, English",3,"[is, sp]","[Yerushalayim, Barcelona]"


#### 18.33.7 — Enrichment Quality Validation

Before the aggregated edition intelligence is incorporated into the LeadWise application catalog, two data-quality dimensions are validated.

First, all observed Open Library language codes are compared with the existing language mapping to identify codes that still require normalization.

Second, the earliest publication year observed in the edition-history enrichment is compared with the existing Open Library `first_publish_year`. Large discrepancies are flagged rather than automatically replacing previously collected publication-year metadata.

This validation prevents potentially noisy edition-history records from being interpreted as definitive publication origins and ensures that language values displayed in LeadWise use consistent human-readable labels.

In [23]:
# ---------------------------------------------------------
# 18.33.7A Identify all unmapped language codes
# ---------------------------------------------------------

all_language_codes = sorted(
    {
        str(code).lower()
        for codes in book_edition_intelligence[
            "known_publication_language_codes"
        ]
        for code in codes
        if str(code).strip()
    }
)


unmapped_language_codes = [
    code
    for code in all_language_codes
    if code not in LANGUAGE_MAP
]


print("LANGUAGE CODE VALIDATION")
print("=" * 70)

print(
    "Unique observed language codes:",
    len(all_language_codes)
)

print(
    "Mapped codes:",
    len(all_language_codes)
    - len(unmapped_language_codes)
)

print(
    "Unmapped codes:",
    len(unmapped_language_codes)
)

print(
    "\nUnmapped language codes:"
)

print(
    unmapped_language_codes
)


# ---------------------------------------------------------
# 18.33.7B Publication-year consistency
# ---------------------------------------------------------

year_validation = (
    book_edition_intelligence[
        [
            "book_id",
            "canonical_title",
            "earliest_observed_year",
            "latest_observed_year",
            "observed_edition_count",
        ]
    ]
    .merge(
        enrichment_base[
            [
                "book_id",
                "first_publish_year",
                "publication_year_observed",
            ]
        ],
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)


year_validation[
    "earliest_vs_first_difference"
] = (
    pd.to_numeric(
        year_validation[
            "earliest_observed_year"
        ],
        errors="coerce",
    )
    -
    pd.to_numeric(
        year_validation[
            "first_publish_year"
        ],
        errors="coerce",
    )
)


year_validation[
    "absolute_year_difference"
] = (
    year_validation[
        "earliest_vs_first_difference"
    ]
    .abs()
)


# ---------------------------------------------------------
# Flag discrepancies
# ---------------------------------------------------------

year_validation[
    "year_discrepancy_flag"
] = (
    year_validation[
        "absolute_year_difference"
    ] > 5
)


comparable_years = (
    year_validation[
        "earliest_observed_year"
    ].notna()
    &
    year_validation[
        "first_publish_year"
    ].notna()
)


print(
    "\nPUBLICATION YEAR VALIDATION"
)
print("=" * 70)

print(
    "Books with both year measures:",
    comparable_years.sum()
)

print(
    "Exact year agreement:",
    (
        comparable_years
        &
        (
            year_validation[
                "absolute_year_difference"
            ] == 0
        )
    ).sum()
)

print(
    "Within 1 year:",
    (
        comparable_years
        &
        (
            year_validation[
                "absolute_year_difference"
            ] <= 1
        )
    ).sum()
)

print(
    "Within 5 years:",
    (
        comparable_years
        &
        (
            year_validation[
                "absolute_year_difference"
            ] <= 5
        )
    ).sum()
)

print(
    "Difference greater than 5 years:",
    (
        comparable_years
        &
        year_validation[
            "year_discrepancy_flag"
        ]
    ).sum()
)


print(
    "\nLARGEST PUBLICATION-YEAR DISCREPANCIES"
)
print("=" * 70)

display(
    year_validation.loc[
        comparable_years
    ]
    .sort_values(
        "absolute_year_difference",
        ascending=False,
    )
    [
        [
            "book_id",
            "canonical_title",
            "first_publish_year",
            "earliest_observed_year",
            "latest_observed_year",
            "earliest_vs_first_difference",
            "observed_edition_count",
        ]
    ]
    .head(25)
)

LANGUAGE CODE VALIDATION
Unique observed language codes: 22
Mapped codes: 20
Unmapped codes: 2

Unmapped language codes:
['gem', 'vie']

PUBLICATION YEAR VALIDATION
Books with both year measures: 945
Exact year agreement: 945
Within 1 year: 945
Within 5 years: 945
Difference greater than 5 years: 0

LARGEST PUBLICATION-YEAR DISCREPANCIES


,book_id,canonical_title,first_publish_year,earliest_observed_year,latest_observed_year,earliest_vs_first_difference,observed_edition_count
0,BOOK00001,Principle-Centered Leadership,1989.0,1989,2012,0.0,21
637,BOOK00639,Human resources management,1974.0,1974,1992,0.0,8
625,BOOK00627,Understanding Human Resources Management,2019.0,2019,2019,0.0,1
626,BOOK00628,Human Resource Management,2018.0,2018,2018,0.0,1
627,BOOK00629,Human Resource Management,1995.0,1995,2020,0.0,11
628,BOOK00630,Human resource management,1998.0,1998,2002,0.0,5
629,BOOK00631,Human resource management,2007.0,2007,2007,0.0,1
630,BOOK00632,Human resource management,1994.0,1994,2013,0.0,7
631,BOOK00633,Fundamentals of Human Resource Management,2008.0,2008,2018,0.0,6
632,BOOK00634,Human resource management,1990.0,1990,2003,0.0,7


#### 18.33.8 — Finalization of Edition Intelligence

Quality validation demonstrates strong internal consistency between the existing Open Library first-publication metadata and the newly collected edition histories.

Among the 945 books for which both measures are available, the earliest publication year observed in the edition history exactly matches the existing `first_publish_year` for every record. No discrepancies greater than one or five years were identified.

The language-code validation identified only two previously unmapped values:

- `vie` — Vietnamese
- `gem` — Germanic languages, a collective language-family classification rather than the German language specifically

These codes are incorporated into the language normalization mapping without changing their semantic meaning.

The validated edition-intelligence dataset is then saved as a separate processed artifact. This preserves the raw API enrichment while providing a cleaned book-level layer for subsequent integration into the LeadWise application catalog.

In [24]:
# ---------------------------------------------------------
# 18.33.8 Finalize language mapping
# ---------------------------------------------------------

LANGUAGE_MAP.update(
    {
        "vie": "Vietnamese",
        "gem": "Germanic languages",
    }
)


# Rebuild readable language names
book_edition_intelligence[
    "known_publication_languages"
] = (
    book_edition_intelligence[
        "known_publication_language_codes"
    ]
    .apply(language_names)
)


book_edition_intelligence[
    "known_publication_languages_display"
] = (
    book_edition_intelligence[
        "known_publication_languages"
    ]
    .apply(
        lambda values:
        ", ".join(values)
        if values
        else pd.NA
    )
)


# ---------------------------------------------------------
# Recheck unmapped codes
# ---------------------------------------------------------

remaining_unmapped = [
    code
    for code in all_language_codes
    if code not in LANGUAGE_MAP
]


print("LANGUAGE NORMALIZATION")
print("=" * 70)

print(
    "Observed language codes:",
    len(all_language_codes)
)

print(
    "Remaining unmapped codes:",
    len(remaining_unmapped)
)

print(
    "Remaining:",
    remaining_unmapped
)


# ---------------------------------------------------------
# Add publication-year validation information
# ---------------------------------------------------------

publication_validation_fields = (
    year_validation[
        [
            "book_id",
            "first_publish_year",
            "earliest_vs_first_difference",
            "year_discrepancy_flag",
        ]
    ]
    .copy()
)


book_edition_intelligence = (
    book_edition_intelligence
    .merge(
        publication_validation_fields,
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# Save processed edition intelligence
# ---------------------------------------------------------

edition_intelligence_path = (
    data_root
    / "processed"
    / "openlibrary_book_edition_intelligence.csv"
)


book_edition_intelligence.to_csv(
    edition_intelligence_path,
    index=False,
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print("\nFINAL EDITION INTELLIGENCE")
print("=" * 70)

print(
    "Rows:",
    len(book_edition_intelligence)
)

print(
    "Unique book_id:",
    book_edition_intelligence[
        "book_id"
    ].nunique()
)

print(
    "Known-language books:",
    (
        book_edition_intelligence[
            "known_publication_language_count"
        ] > 0
    ).sum()
)

print(
    "Multilingual books:",
    book_edition_intelligence[
        "multilingual_publication_evidence"
    ].sum()
)

print(
    "Books with publication-country evidence:",
    (
        book_edition_intelligence[
            "observed_publication_country_count"
        ] > 0
    ).sum()
)

print(
    "Publication-year discrepancies > 5 years:",
    book_edition_intelligence[
        "year_discrepancy_flag"
    ].fillna(False).sum()
)

print(
    "\nSaved:",
    edition_intelligence_path
)

LANGUAGE NORMALIZATION
Observed language codes: 22
Remaining unmapped codes: 0
Remaining: []

FINAL EDITION INTELLIGENCE
Rows: 948
Unique book_id: 948
Known-language books: 900
Multilingual books: 40
Books with publication-country evidence: 583
Publication-year discrepancies > 5 years: 0

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_book_edition_intelligence.csv


### 18.34 — Author and Geographic Metadata Enrichment

LeadWise originally planned to include geographic and international-context information alongside bibliographic and recommendation metadata.

The initial data-collection pipeline did not retain verified author geography or publication-origin attributes. A separate enrichment stage is therefore introduced using existing source identifiers wherever possible.

The enrichment investigates:

- author identifiers
- author names
- birth and death dates where available
- author biography metadata
- explicitly recorded geographic information
- geographic associations supported by source metadata

Geographic attributes are not inferred from author names, language, publisher, ISBN prefixes, or book titles.

The objective is to distinguish clearly between:

- **author geographic association**, when explicitly supported;
- **observed publication geography**, derived from edition metadata;
- **original publication country**, only when supported by appropriate evidence.

These concepts remain separate within the LeadWise metadata architecture.

#### 18.34.1 — Open Library Author Metadata Inspection

Existing Open Library author identifiers are inspected to determine which author-level attributes are available for geographic and biographical enrichment.

A single author record is tested before batch collection so that the enrichment schema is based on fields actually returned by the source rather than assumed metadata availability.

In [25]:
# ---------------------------------------------------------
# 18.34.1 Inspect one Open Library author record
# ---------------------------------------------------------

ol_integrated = loaded_metadata[
    "openlibrary_integrated"
].copy()


# ---------------------------------------------------------
# Parse author_keys safely
# ---------------------------------------------------------

def parse_list_field(value):

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()

    if (
        not text
        or text.lower()
        in {"nan", "none", "null", "[]"}
    ):
        return []

    try:

        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):

        return [text]


ol_integrated[
    "author_keys_parsed"
] = (
    ol_integrated[
        "author_keys"
    ]
    .apply(parse_list_field)
)


# ---------------------------------------------------------
# Author-key coverage
# ---------------------------------------------------------

books_with_author_keys = (
    ol_integrated[
        "author_keys_parsed"
    ]
    .apply(bool)
)


all_author_keys = sorted(
    {
        str(author_key).strip()
        for keys in ol_integrated[
            "author_keys_parsed"
        ]
        for author_key in keys
        if str(author_key).strip()
    }
)


print("AUTHOR IDENTIFIER COVERAGE")
print("=" * 70)

print(
    "Open Library books:",
    len(ol_integrated)
)

print(
    "Books with author keys:",
    books_with_author_keys.sum()
)

print(
    "Unique author keys:",
    len(all_author_keys)
)


# ---------------------------------------------------------
# Test one author
# ---------------------------------------------------------

test_author_key = all_author_keys[0]

if test_author_key.startswith(
    "/authors/"
):

    author_path = test_author_key

else:

    author_path = (
        f"/authors/"
        f"{test_author_key}"
    )


author_url = (
    f"https://openlibrary.org"
    f"{author_path}.json"
)


response = requests.get(
    author_url,
    headers={
        "User-Agent":
            "LeadWise Academic Research Project"
    },
    timeout=30,
)


print("\nAUTHOR TEST REQUEST")
print("=" * 70)

print(
    "Author key:",
    test_author_key
)

print(
    "Status:",
    response.status_code
)

print(
    "URL:",
    author_url
)


response.raise_for_status()

author_json = response.json()


print("\nAVAILABLE AUTHOR FIELDS")
print("=" * 70)

for key in sorted(
    author_json.keys()
):
    print(key)


print("\nAUTHOR RECORD")
print("=" * 70)

for key, value in author_json.items():

    if key not in {
        "photos",
        "links",
        "remote_ids",
    }:

        print(
            f"{key}: {value}"
        )

AUTHOR IDENTIFIER COVERAGE
Open Library books: 950
Books with author keys: 942
Unique author keys: 1237

AUTHOR TEST REQUEST
Author key: OL10000962A
Status: 200
URL: https://openlibrary.org/authors/OL10000962A.json

AVAILABLE AUTHOR FIELDS
created
key
last_modified
latest_revision
name
revision
source_records
type

AUTHOR RECORD
type: {'key': '/type/author'}
name: King, John, III
key: /authors/OL10000962A
source_records: ['bwb:9781476822709']
latest_revision: 1
revision: 1
created: {'type': '/type/datetime', 'value': '2021-12-26T23:47:18.489219'}
last_modified: {'type': '/type/datetime', 'value': '2021-12-26T23:47:18.489219'}


#### 18.34.2 — Author Identity Bridge for External Enrichment

The Open Library author metadata test demonstrates that author identifiers are widely available in the LeadWise dataset, but geographic attributes are not consistently represented in the tested author record.

Open Library is therefore retained as the author-identity source rather than being treated as the definitive source of author geography.

A structured author identity bridge is created before external enrichment. The bridge preserves:

- LeadWise book identifier
- canonical book title
- Open Library author identifier
- author name recorded in the book dataset
- author position within multi-author works

This bridge enables author-level enrichment to remain separate from book-level metadata and reduces the risk of incorrectly assigning geographic attributes across authors with similar names.

Subsequent geographic enrichment will prioritize structured identifiers and explicitly supported source relationships. Name-only matches will not automatically be accepted as verified identities.

In [26]:
# ---------------------------------------------------------
# 18.34.2 Build Open Library author identity bridge
# ---------------------------------------------------------

author_bridge_rows = []


for _, row in ol_integrated.iterrows():

    book_id = row["book_id"]
    title = row["title"]

    author_keys = row[
        "author_keys_parsed"
    ]

    author_names = parse_list_field(
        row["authors"]
    )

    max_authors = max(
        len(author_keys),
        len(author_names),
    )


    for position in range(
        max_authors
    ):

        author_key = (
            author_keys[position]
            if position < len(author_keys)
            else None
        )

        author_name = (
            author_names[position]
            if position < len(author_names)
            else None
        )


        author_bridge_rows.append(
            {
                "book_id":
                    book_id,

                "canonical_title":
                    title,

                "author_position":
                    position + 1,

                "openlibrary_author_key":
                    author_key,

                "author_name":
                    author_name,
            }
        )


author_identity_bridge = pd.DataFrame(
    author_bridge_rows
)


# ---------------------------------------------------------
# Clean identifiers and names
# ---------------------------------------------------------

author_identity_bridge[
    "openlibrary_author_key"
] = (
    author_identity_bridge[
        "openlibrary_author_key"
    ]
    .astype("string")
    .str.replace(
        "/authors/",
        "",
        regex=False,
    )
    .str.strip()
)


author_identity_bridge[
    "author_name"
] = (
    author_identity_bridge[
        "author_name"
    ]
    .astype("string")
    .str.strip()
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print("AUTHOR IDENTITY BRIDGE")
print("=" * 70)

print(
    "Book-author relationships:",
    len(author_identity_bridge)
)

print(
    "Books represented:",
    author_identity_bridge[
        "book_id"
    ].nunique()
)

print(
    "Relationships with author ID:",
    author_identity_bridge[
        "openlibrary_author_key"
    ].notna().sum()
)

print(
    "Relationships with author name:",
    author_identity_bridge[
        "author_name"
    ].notna().sum()
)

print(
    "Unique Open Library author IDs:",
    author_identity_bridge[
        "openlibrary_author_key"
    ].dropna().nunique()
)


# ---------------------------------------------------------
# Check alignment of author names and IDs
# ---------------------------------------------------------

alignment_check = (
    author_identity_bridge
    .groupby("book_id")
    .agg(
        relationship_count=(
            "author_position",
            "count",
        ),
        author_id_count=(
            "openlibrary_author_key",
            "count",
        ),
        author_name_count=(
            "author_name",
            "count",
        ),
    )
    .reset_index()
)


alignment_check[
    "author_alignment_complete"
] = (
    alignment_check[
        "author_id_count"
    ]
    ==
    alignment_check[
        "author_name_count"
    ]
)


print(
    "\nBooks with complete name-ID alignment:",
    alignment_check[
        "author_alignment_complete"
    ].sum()
)

print(
    "Books with incomplete alignment:",
    (
        ~alignment_check[
            "author_alignment_complete"
        ]
    ).sum()
)


# ---------------------------------------------------------
# Preview
# ---------------------------------------------------------

print(
    "\nAUTHOR BRIDGE PREVIEW"
)
print("=" * 70)

display(
    author_identity_bridge.head(20)
)


# ---------------------------------------------------------
# Save bridge
# ---------------------------------------------------------

author_bridge_path = (
    data_root
    / "processed"
    / "openlibrary_author_identity_bridge.csv"
)


author_identity_bridge.to_csv(
    author_bridge_path,
    index=False,
)


print(
    "\nSaved:",
    author_bridge_path
)

AUTHOR IDENTITY BRIDGE
Book-author relationships: 1468
Books represented: 942
Relationships with author ID: 1468
Relationships with author name: 1468
Unique Open Library author IDs: 1237

Books with complete name-ID alignment: 942
Books with incomplete alignment: 0

AUTHOR BRIDGE PREVIEW


,book_id,canonical_title,author_position,openlibrary_author_key,author_name
0,BOOK00001,Principle-Centered Leadership,1,OL383159A,Stephen R. Covey
1,BOOK00002,Leadership in Organizations,1,OL400156A,Gary A. Yukl
2,BOOK00003,Kepemimpinan =,1,OL1268A,Karjadi M.
3,BOOK00004,Spiritual leadership,1,OL25389A,J. Oswald Sanders
4,BOOK00005,Leadership,1,OL32040A,Peter Guy Northouse
5,BOOK00006,The 21 Irrefutable Laws of Leadership,1,OL26423A,John C. Maxwell
6,BOOK00007,Leadership,1,OL2655012A,Peter G. Northouse
7,BOOK00008,Leadership and performance beyond expectations,1,OL333669A,Bernard M. Bass
8,BOOK00009,A Higher Loyalty,1,OL7442788A,James Comey
9,BOOK00009,A Higher Loyalty,2,OL8118723A,James B. Comey



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_author_identity_bridge.csv


#### 18.34.3 — Wikidata Author Identity Resolution Pilot

The Open Library author bridge contains complete alignment between author names and author identifiers for the 942 represented books. However, inspection also reveals that Open Library may assign multiple identifiers to different name variants of the same real-world author.

Consequently, Open Library author identifiers are treated as source-record identifiers rather than automatically interpreted as unique people.

Wikidata is evaluated as a complementary structured source for author identity and geographic enrichment. The pilot focuses on a small set of recognizable author records before any large-scale collection is attempted.

Potential Wikidata attributes include:

- Wikidata entity identifier
- canonical person or organization name
- country of citizenship
- place of birth
- place of death
- date of birth
- occupation
- external identifiers, where available

A Wikidata result is not automatically accepted merely because the name matches. Candidate matches are retained with supporting metadata so that identity resolution can be evaluated before geographic information is incorporated into LeadWise.

Organizations and institutional authors are also kept distinct from individual authors.

In [28]:
# ---------------------------------------------------------
# 18.34.3A Wikidata author candidate pilot
# ---------------------------------------------------------

import requests
import pandas as pd
import time


pilot_authors = [
    {
        "openlibrary_author_key": "OL383159A",
        "author_name": "Stephen R. Covey",
    },
    {
        "openlibrary_author_key": "OL32040A",
        "author_name": "Peter Guy Northouse",
    },
    {
        "openlibrary_author_key": "OL26423A",
        "author_name": "John C. Maxwell",
    },
    {
        "openlibrary_author_key": "OL7442788A",
        "author_name": "James Comey",
    },
    {
        "openlibrary_author_key": "OL3080130A",
        "author_name": "The Arbinger Institute",
    },
]


wikidata_api = (
    "https://www.wikidata.org/w/api.php"
)

headers = {
    "User-Agent":
        "LeadWise Academic Research Project"
}


candidate_rows = []


for author in pilot_authors:

    params = {
        "action": "wbsearchentities",
        "search": author["author_name"],
        "language": "en",
        "format": "json",
        "limit": 5,
        "type": "item",
    }

    response = requests.get(
        wikidata_api,
        params=params,
        headers=headers,
        timeout=30,
    )

    response.raise_for_status()

    results = (
        response.json()
        .get("search", [])
    )


    for rank, result in enumerate(
        results,
        start=1,
    ):

        candidate_rows.append(
            {
                "openlibrary_author_key":
                    author[
                        "openlibrary_author_key"
                    ],

                "searched_author_name":
                    author["author_name"],

                "candidate_rank":
                    rank,

                "wikidata_id":
                    result.get("id"),

                "wikidata_label":
                    result.get("label"),

                "wikidata_description":
                    result.get(
                        "description"
                    ),

                "wikidata_url":
                    result.get(
                        "concepturi"
                    ),
            }
        )

    time.sleep(0.20)


wikidata_candidates = pd.DataFrame(
    candidate_rows
)


print("WIKIDATA AUTHOR PILOT")
print("=" * 70)

print(
    "Authors searched:",
    len(pilot_authors)
)

print(
    "Candidates returned:",
    len(wikidata_candidates)
)

print(
    "Authors with at least one candidate:",
    wikidata_candidates[
        "searched_author_name"
    ].nunique()
)


display(
    wikidata_candidates[
        [
            "searched_author_name",
            "openlibrary_author_key",
            "candidate_rank",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
        ]
    ]
)

WIKIDATA AUTHOR PILOT
Authors searched: 5
Candidates returned: 5
Authors with at least one candidate: 3


,searched_author_name,openlibrary_author_key,candidate_rank,wikidata_id,wikidata_label,wikidata_description
0,Stephen R. Covey,OL383159A,1,Q313482,Stephen Covey,"American educator, author, businessman and mot..."
1,Stephen R. Covey,OL383159A,2,Q123654696,"Stephen R. Covey, Herald of Good Habits, Dies ...","The New York Times article (July 17, 2012)"
2,John C. Maxwell,OL26423A,1,Q3304739,John C. Maxwell,"American author, speaker and neopentecostal pa..."
3,John C. Maxwell,OL26423A,2,Q60220754,John C. Maxwell,geologist
4,James Comey,OL7442788A,1,Q167607,James Comey,American lawyer and 7th director of the Federa...


#### 18.34.4 — Structured Wikidata Author Metadata Pilot

The initial Wikidata candidate-search pilot demonstrates that name-based retrieval can identify relevant author entities, but candidate ranking alone is insufficient for reliable automated identity resolution.

Search results may include:

- the intended author;
- people with identical or similar names;
- publications or other entities associated with the author's name;
- no candidate for some authors or institutional authors.

Therefore, candidate rank is not treated as verification.

A second pilot retrieves structured Wikidata properties for credible candidate entities identified during the initial test. The objective is to evaluate the availability of attributes relevant to LeadWise, including:

- entity type
- country of citizenship
- place of birth
- date of birth
- occupation
- country or geographic associations
- external identifiers

Structured properties are retained separately from search descriptions. Geographic metadata will only be incorporated after author identity has been sufficiently supported.

In [29]:
# ---------------------------------------------------------
# 18.34.4 Structured Wikidata metadata pilot
# ---------------------------------------------------------

pilot_qids = {
    "Stephen R. Covey": "Q313482",
    "John C. Maxwell": "Q3304739",
    "James Comey": "Q167607",
}


wikidata_entity_api = (
    "https://www.wikidata.org/w/api.php"
)


params = {
    "action": "wbgetentities",
    "ids": "|".join(
        pilot_qids.values()
    ),
    "languages": "en",
    "props": "labels|descriptions|claims",
    "format": "json",
}


response = requests.get(
    wikidata_entity_api,
    params=params,
    headers={
        "User-Agent":
            "LeadWise Academic Research Project"
    },
    timeout=30,
)

response.raise_for_status()

entities = (
    response.json()
    .get("entities", {})
)


# ---------------------------------------------------------
# Helper: extract entity IDs from Wikidata claims
# ---------------------------------------------------------

def get_claim_entity_ids(
    entity,
    property_id,
):

    values = []

    for claim in (
        entity
        .get("claims", {})
        .get(property_id, [])
    ):

        try:

            value = (
                claim["mainsnak"]
                ["datavalue"]
                ["value"]
            )

            if isinstance(value, dict):

                entity_id = value.get("id")

                if entity_id:
                    values.append(
                        entity_id
                    )

        except (
            KeyError,
            TypeError,
        ):
            continue

    return list(
        dict.fromkeys(values)
    )


# ---------------------------------------------------------
# Helper: extract time values
# ---------------------------------------------------------

def get_claim_times(
    entity,
    property_id,
):

    values = []

    for claim in (
        entity
        .get("claims", {})
        .get(property_id, [])
    ):

        try:

            value = (
                claim["mainsnak"]
                ["datavalue"]
                ["value"]
                ["time"]
            )

            values.append(value)

        except (
            KeyError,
            TypeError,
        ):
            continue

    return list(
        dict.fromkeys(values)
    )


# ---------------------------------------------------------
# Extract selected properties
# ---------------------------------------------------------
#
# P31  = instance of
# P27  = country of citizenship
# P19  = place of birth
# P569 = date of birth
# P106 = occupation
#
# ---------------------------------------------------------

structured_rows = []


for searched_name, qid in (
    pilot_qids.items()
):

    entity = entities.get(
        qid,
        {}
    )

    structured_rows.append(
        {
            "searched_author_name":
                searched_name,

            "wikidata_id":
                qid,

            "wikidata_label":
                (
                    entity
                    .get("labels", {})
                    .get("en", {})
                    .get("value")
                ),

            "wikidata_description":
                (
                    entity
                    .get(
                        "descriptions",
                        {},
                    )
                    .get("en", {})
                    .get("value")
                ),

            "instance_of_qids":
                get_claim_entity_ids(
                    entity,
                    "P31",
                ),

            "citizenship_qids":
                get_claim_entity_ids(
                    entity,
                    "P27",
                ),

            "birthplace_qids":
                get_claim_entity_ids(
                    entity,
                    "P19",
                ),

            "birth_dates":
                get_claim_times(
                    entity,
                    "P569",
                ),

            "occupation_qids":
                get_claim_entity_ids(
                    entity,
                    "P106",
                ),
        }
    )


structured_author_pilot = (
    pd.DataFrame(
        structured_rows
    )
)


print(
    "STRUCTURED WIKIDATA AUTHOR PILOT"
)
print("=" * 70)

display(
    structured_author_pilot
)

STRUCTURED WIKIDATA AUTHOR PILOT


,searched_author_name,wikidata_id,wikidata_label,wikidata_description,instance_of_qids,citizenship_qids,birthplace_qids,birth_dates,occupation_qids
0,Stephen R. Covey,Q313482,Stephen Covey,"American educator, author, businessman and mot...",[Q5],[Q30],[Q23337],[+1932-10-24T00:00:00Z],"[Q121594, Q43845, Q36180, Q1622272]"
1,John C. Maxwell,Q3304739,John C. Maxwell,"American author, speaker and neopentecostal pa...",[Q5],[Q30],[Q1013879],[+1947-02-20T00:00:00Z],"[Q36180, Q15982858, Q152002, Q6051619, Q128592..."
2,James Comey,Q167607,James Comey,American lawyer and 7th director of the Federa...,[Q5],[Q30],[Q128114],[+1960-12-14T00:00:00Z],"[Q40348, Q1622272, Q82955, Q185351]"


#### 18.34.5 — Context-Aware Author Identity Resolution

The structured Wikidata pilot confirms that verified author entities can provide useful geographic and professional metadata, including country of citizenship, birthplace, birth date, and occupation.

The principal remaining challenge is reliable entity resolution.

Name-only matching is insufficient because:

- different people may share the same name;
- Wikidata search may return non-person entities;
- Open Library may contain multiple identifiers for the same real-world author;
- institutional authors require different treatment from individual authors;
- some authors may not have a Wikidata entity.

A context-aware pilot is therefore conducted using both the author name and the associated book title.

Candidate records are retained rather than automatically accepted. This allows identity resolution to consider:

1. author-name similarity;
2. book-title context;
3. Wikidata entity type;
4. candidate description;
5. structured biographical metadata.

Unresolved or ambiguous authors remain explicitly unverified rather than being assigned inferred geographic attributes.

In [30]:
# ---------------------------------------------------------
# 18.34.5 Context-aware Wikidata resolution pilot
# ---------------------------------------------------------

context_pilot = pd.DataFrame(
    [
        {
            "author_name": "Stephen R. Covey",
            "book_title": "Principle-Centered Leadership",
        },
        {
            "author_name": "Peter Guy Northouse",
            "book_title": "Leadership",
        },
        {
            "author_name": "John C. Maxwell",
            "book_title":
                "The 21 Irrefutable Laws of Leadership",
        },
        {
            "author_name": "James Comey",
            "book_title": "A Higher Loyalty",
        },
        {
            "author_name": "The Arbinger Institute",
            "book_title":
                "Leadership and Self Deception",
        },
    ]
)


search_rows = []


for _, row in context_pilot.iterrows():

    author_name = row["author_name"]
    book_title = row["book_title"]

    # ---------------------------------------------
    # Strategy A: exact author-name search
    # ---------------------------------------------

    search_queries = [
        (
            "author_name",
            author_name,
        ),
        (
            "author_plus_book",
            f"{author_name} {book_title}",
        ),
    ]


    for strategy, query in search_queries:

        params = {
            "action": "wbsearchentities",
            "search": query,
            "language": "en",
            "format": "json",
            "limit": 5,
            "type": "item",
        }

        response = requests.get(
            wikidata_api,
            params=params,
            headers=headers,
            timeout=30,
        )

        response.raise_for_status()

        results = (
            response.json()
            .get("search", [])
        )


        for rank, result in enumerate(
            results,
            start=1,
        ):

            search_rows.append(
                {
                    "author_name":
                        author_name,

                    "book_title":
                        book_title,

                    "search_strategy":
                        strategy,

                    "search_query":
                        query,

                    "candidate_rank":
                        rank,

                    "wikidata_id":
                        result.get("id"),

                    "wikidata_label":
                        result.get("label"),

                    "wikidata_description":
                        result.get(
                            "description"
                        ),
                }
            )

        time.sleep(0.20)


context_candidates = pd.DataFrame(
    search_rows
)


print(
    "CONTEXT-AWARE WIKIDATA PILOT"
)
print("=" * 70)

print(
    "Authors tested:",
    len(context_pilot)
)

print(
    "Total candidates:",
    len(context_candidates)
)


# ---------------------------------------------------------
# Summary by strategy
# ---------------------------------------------------------

strategy_summary = (
    context_candidates
    .groupby(
        [
            "author_name",
            "search_strategy",
        ]
    )
    .agg(
        candidates=(
            "wikidata_id",
            "count",
        )
    )
    .reset_index()
)


print(
    "\nSEARCH STRATEGY SUMMARY"
)
print("=" * 70)

display(
    strategy_summary
)


# ---------------------------------------------------------
# Candidate details
# ---------------------------------------------------------

print(
    "\nCANDIDATE DETAILS"
)
print("=" * 70)

display(
    context_candidates[
        [
            "author_name",
            "book_title",
            "search_strategy",
            "candidate_rank",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
        ]
    ]
)

CONTEXT-AWARE WIKIDATA PILOT
Authors tested: 5
Total candidates: 5

SEARCH STRATEGY SUMMARY


,author_name,search_strategy,candidates
0,James Comey,author_name,1
1,John C. Maxwell,author_name,2
2,Stephen R. Covey,author_name,2



CANDIDATE DETAILS


,author_name,book_title,search_strategy,candidate_rank,wikidata_id,wikidata_label,wikidata_description
0,Stephen R. Covey,Principle-Centered Leadership,author_name,1,Q313482,Stephen Covey,"American educator, author, businessman and mot..."
1,Stephen R. Covey,Principle-Centered Leadership,author_name,2,Q123654696,"Stephen R. Covey, Herald of Good Habits, Dies ...","The New York Times article (July 17, 2012)"
2,John C. Maxwell,The 21 Irrefutable Laws of Leadership,author_name,1,Q3304739,John C. Maxwell,"American author, speaker and neopentecostal pa..."
3,John C. Maxwell,The 21 Irrefutable Laws of Leadership,author_name,2,Q60220754,John C. Maxwell,geologist
4,James Comey,A Higher Loyalty,author_name,1,Q167607,James Comey,American lawyer and 7th director of the Federa...


#### 18.34.6 — Author Name Normalization for Entity Resolution

Contextual search using the combination of author name and book title did not improve Wikidata candidate retrieval. All candidates returned during the pilot originated from the author-name-only search strategy.

The enrichment pipeline therefore avoids increasingly broad or fuzzy contextual searches.

Before full-scale author resolution, conservative author-name normalization is evaluated. The objective is not to infer author identity, but to improve candidate discovery for differences in punctuation, initials, and name formatting.

Multiple search variants may be generated from the source author name, while the original Open Library name remains preserved.

Candidate discovery and candidate verification remain separate processes. A normalized-name search result is not automatically treated as a verified author identity.

In [31]:
# ---------------------------------------------------------
# 18.34.6 Author-name normalization pilot
# ---------------------------------------------------------

import re


def generate_author_search_variants(name):
    """
    Generate conservative search variants.
    Does not infer identity.
    """

    if pd.isna(name):
        return []

    original = str(name).strip()

    variants = [original]


    # Remove repeated whitespace
    whitespace_clean = re.sub(
        r"\s+",
        " ",
        original,
    ).strip()

    variants.append(
        whitespace_clean
    )


    # Remove periods from initials
    no_periods = (
        whitespace_clean
        .replace(".", "")
    )

    variants.append(
        no_periods
    )


    # Remove common trailing generational suffixes
    no_suffix = re.sub(
        r",?\s+"
        r"(Jr|Sr|II|III|IV)"
        r"\.?$",
        "",
        whitespace_clean,
        flags=re.IGNORECASE,
    ).strip()

    variants.append(
        no_suffix
    )


    # Preserve unique variants only
    unique_variants = []

    for variant in variants:

        if (
            variant
            and variant
            not in unique_variants
        ):
            unique_variants.append(
                variant
            )

    return unique_variants


normalization_pilot = [
    "Stephen R. Covey",
    "Peter Guy Northouse",
    "John C. Maxwell",
    "James Comey",
    "The Arbinger Institute",
]


variant_rows = []


for author_name in normalization_pilot:

    variants = (
        generate_author_search_variants(
            author_name
        )
    )

    for variant in variants:

        params = {
            "action":
                "wbsearchentities",

            "search":
                variant,

            "language":
                "en",

            "format":
                "json",

            "limit":
                5,

            "type":
                "item",
        }


        response = requests.get(
            wikidata_api,
            params=params,
            headers=headers,
            timeout=30,
        )

        response.raise_for_status()

        results = (
            response.json()
            .get("search", [])
        )


        for rank, result in enumerate(
            results,
            start=1,
        ):

            variant_rows.append(
                {
                    "original_author_name":
                        author_name,

                    "search_variant":
                        variant,

                    "candidate_rank":
                        rank,

                    "wikidata_id":
                        result.get("id"),

                    "wikidata_label":
                        result.get("label"),

                    "wikidata_description":
                        result.get(
                            "description"
                        ),
                }
            )

        time.sleep(0.20)


variant_candidates = pd.DataFrame(
    variant_rows
)


print(
    "AUTHOR NORMALIZATION PILOT"
)
print("=" * 70)


for author_name in normalization_pilot:

    variants = (
        generate_author_search_variants(
            author_name
        )
    )

    candidates = (
        variant_candidates[
            variant_candidates[
                "original_author_name"
            ] == author_name
        ]
        if len(variant_candidates)
        else pd.DataFrame()
    )

    print(
        f"\n{author_name}"
    )

    print(
        "Search variants:",
        variants
    )

    print(
        "Candidates:",
        len(candidates)
    )


print(
    "\nCANDIDATE DETAILS"
)
print("=" * 70)


if len(variant_candidates):

    display(
        variant_candidates[
            [
                "original_author_name",
                "search_variant",
                "candidate_rank",
                "wikidata_id",
                "wikidata_label",
                "wikidata_description",
            ]
        ]
        .drop_duplicates()
    )

else:

    print(
        "No candidates returned."
    )

AUTHOR NORMALIZATION PILOT

Stephen R. Covey
Search variants: ['Stephen R. Covey', 'Stephen R Covey']
Candidates: 2

Peter Guy Northouse
Search variants: ['Peter Guy Northouse']
Candidates: 0

John C. Maxwell
Search variants: ['John C. Maxwell', 'John C Maxwell']
Candidates: 3

James Comey
Search variants: ['James Comey']
Candidates: 1

The Arbinger Institute
Search variants: ['The Arbinger Institute']
Candidates: 0

CANDIDATE DETAILS


,original_author_name,search_variant,candidate_rank,wikidata_id,wikidata_label,wikidata_description
0,Stephen R. Covey,Stephen R. Covey,1,Q313482,Stephen Covey,"American educator, author, businessman and mot..."
1,Stephen R. Covey,Stephen R. Covey,2,Q123654696,"Stephen R. Covey, Herald of Good Habits, Dies ...","The New York Times article (July 17, 2012)"
2,John C. Maxwell,John C. Maxwell,1,Q3304739,John C. Maxwell,"American author, speaker and neopentecostal pa..."
3,John C. Maxwell,John C. Maxwell,2,Q60220754,John C. Maxwell,geologist
4,John C. Maxwell,John C Maxwell,1,Q60220754,John C. Maxwell,geologist
5,James Comey,James Comey,1,Q167607,James Comey,American lawyer and 7th director of the Federa...


#### 18.34.7 — Wikidata Author Candidate Collection

Pilot testing demonstrates that conservative author-name normalization does not materially improve Wikidata candidate discovery and may alter candidate ranking in undesirable ways.

The original Open Library author name is therefore retained as the primary search query.

A batch candidate-discovery stage is performed for the unique Open Library author identifiers represented in LeadWise. Candidate discovery does not constitute identity verification.

For each source author, the process records:

- Open Library author identifier
- original author name
- Wikidata candidate identifier
- candidate rank
- Wikidata label
- Wikidata description
- candidate URL

Up to five candidate entities are retained per author.

Authors for whom Wikidata returns no candidate remain explicitly unresolved. Authors with multiple candidates remain unresolved until structured metadata provides sufficient evidence for identity validation.

This design separates three distinct stages:

1. source-author identification;
2. external candidate discovery;
3. identity verification and geographic enrichment.

No geographic information is assigned during candidate discovery.

In [32]:
# ---------------------------------------------------------
# 18.34.7 Batch Wikidata author candidate collection
# ---------------------------------------------------------

from pathlib import Path
import time
import requests
import pandas as pd


# ---------------------------------------------------------
# Build unique Open Library author table
# ---------------------------------------------------------

unique_authors = (
    author_identity_bridge[
        [
            "openlibrary_author_key",
            "author_name",
        ]
    ]
    .dropna(
        subset=[
            "openlibrary_author_key",
            "author_name",
        ]
    )
    .drop_duplicates(
        subset=[
            "openlibrary_author_key"
        ]
    )
    .reset_index(drop=True)
)


print("UNIQUE AUTHORS")
print("=" * 70)

print(
    "Unique Open Library author IDs:",
    len(unique_authors)
)


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

raw_dir = (
    data_root
    / "raw"
)

candidate_path = (
    raw_dir
    / "wikidata_author_candidates.csv"
)

search_log_path = (
    raw_dir
    / "wikidata_author_search_log.csv"
)


# ---------------------------------------------------------
# Session
# ---------------------------------------------------------

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "LeadWise Academic Research Project"
    }
)


wikidata_api = (
    "https://www.wikidata.org/w/api.php"
)


# ---------------------------------------------------------
# Resume existing collection if available
# ---------------------------------------------------------

if candidate_path.exists():

    existing_candidates = (
        pd.read_csv(
            candidate_path
        )
    )

else:

    existing_candidates = (
        pd.DataFrame()
    )


if search_log_path.exists():

    existing_log = (
        pd.read_csv(
            search_log_path
        )
    )

else:

    existing_log = (
        pd.DataFrame()
    )


if (
    len(existing_log)
    and
    "openlibrary_author_key"
    in existing_log.columns
):

    completed_author_ids = set(
        existing_log.loc[
            existing_log[
                "status"
            ].isin(
                [
                    "success",
                    "no_candidate",
                ]
            ),
            "openlibrary_author_key",
        ]
        .astype(str)
    )

else:

    completed_author_ids = set()


authors_to_process = (
    unique_authors[
        ~unique_authors[
            "openlibrary_author_key"
        ]
        .astype(str)
        .isin(
            completed_author_ids
        )
    ]
    .copy()
)


print(
    "Already completed:",
    len(completed_author_ids)
)

print(
    "Authors scheduled:",
    len(authors_to_process)
)


# ---------------------------------------------------------
# Collection
# ---------------------------------------------------------

new_candidate_rows = []
new_log_rows = []

SAVE_EVERY = 50


for counter, (_, row) in enumerate(
    authors_to_process.iterrows(),
    start=1,
):

    author_id = str(
        row[
            "openlibrary_author_key"
        ]
    ).strip()

    author_name = str(
        row[
            "author_name"
        ]
    ).strip()


    try:

        params = {
            "action":
                "wbsearchentities",

            "search":
                author_name,

            "language":
                "en",

            "format":
                "json",

            "limit":
                5,

            "type":
                "item",
        }


        response = session.get(
            wikidata_api,
            params=params,
            timeout=30,
        )

        response.raise_for_status()

        results = (
            response.json()
            .get(
                "search",
                []
            )
        )


        if results:

            for rank, result in enumerate(
                results,
                start=1,
            ):

                new_candidate_rows.append(
                    {
                        "openlibrary_author_key":
                            author_id,

                        "author_name":
                            author_name,

                        "candidate_rank":
                            rank,

                        "wikidata_id":
                            result.get(
                                "id"
                            ),

                        "wikidata_label":
                            result.get(
                                "label"
                            ),

                        "wikidata_description":
                            result.get(
                                "description"
                            ),

                        "wikidata_url":
                            result.get(
                                "concepturi"
                            ),
                    }
                )


            status = "success"

        else:

            status = "no_candidate"


        new_log_rows.append(
            {
                "openlibrary_author_key":
                    author_id,

                "author_name":
                    author_name,

                "status":
                    status,

                "candidate_count":
                    len(results),

                "error":
                    None,
            }
        )


    except Exception as exc:

        new_log_rows.append(
            {
                "openlibrary_author_key":
                    author_id,

                "author_name":
                    author_name,

                "status":
                    "error",

                "candidate_count":
                    0,

                "error":
                    str(exc),
            }
        )


    # ---------------------------------------------
    # Periodic checkpoint
    # ---------------------------------------------

    if (
        counter % SAVE_EVERY == 0
        or counter == len(
            authors_to_process
        )
    ):

        current_candidates = (
            pd.concat(
                [
                    existing_candidates,
                    pd.DataFrame(
                        new_candidate_rows
                    ),
                ],
                ignore_index=True,
            )
        )


        current_log = (
            pd.concat(
                [
                    existing_log,
                    pd.DataFrame(
                        new_log_rows
                    ),
                ],
                ignore_index=True,
            )
        )


        current_candidates = (
            current_candidates
            .drop_duplicates(
                subset=[
                    "openlibrary_author_key",
                    "wikidata_id",
                ]
            )
        )


        current_log = (
            current_log
            .drop_duplicates(
                subset=[
                    "openlibrary_author_key"
                ],
                keep="last",
            )
        )


        current_candidates.to_csv(
            candidate_path,
            index=False,
        )


        current_log.to_csv(
            search_log_path,
            index=False,
        )


        print(
            f"Processed this run: "
            f"{counter} | "
            f"Candidate rows: "
            f"{len(current_candidates)}"
        )


    time.sleep(0.20)


# ---------------------------------------------------------
# Final load
# ---------------------------------------------------------

final_candidates = pd.read_csv(
    candidate_path
)

final_search_log = pd.read_csv(
    search_log_path
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "\nWIKIDATA CANDIDATE COLLECTION COMPLETE"
)
print("=" * 70)

print(
    "Authors searched:",
    len(final_search_log)
)

print(
    "Authors with candidates:",
    (
        final_search_log[
            "status"
        ] == "success"
    ).sum()
)

print(
    "Authors with no candidate:",
    (
        final_search_log[
            "status"
        ] == "no_candidate"
    ).sum()
)

print(
    "Errors:",
    (
        final_search_log[
            "status"
        ] == "error"
    ).sum()
)

print(
    "Total candidate entities:",
    len(final_candidates)
)

print(
    "Unique Wikidata candidate IDs:",
    final_candidates[
        "wikidata_id"
    ].nunique()
)

print(
    "\nCandidates saved:",
    candidate_path
)

print(
    "Search log saved:",
    search_log_path
)

UNIQUE AUTHORS
Unique Open Library author IDs: 1237
Already completed: 0
Authors scheduled: 1237
Processed this run: 50 | Candidate rows: 8
Processed this run: 100 | Candidate rows: 28
Processed this run: 150 | Candidate rows: 28
Processed this run: 200 | Candidate rows: 28
Processed this run: 250 | Candidate rows: 28
Processed this run: 300 | Candidate rows: 40
Processed this run: 350 | Candidate rows: 40
Processed this run: 400 | Candidate rows: 40
Processed this run: 450 | Candidate rows: 55
Processed this run: 500 | Candidate rows: 55
Processed this run: 550 | Candidate rows: 55
Processed this run: 600 | Candidate rows: 55
Processed this run: 650 | Candidate rows: 67
Processed this run: 700 | Candidate rows: 67
Processed this run: 750 | Candidate rows: 67
Processed this run: 800 | Candidate rows: 67
Processed this run: 850 | Candidate rows: 78
Processed this run: 900 | Candidate rows: 78
Processed this run: 950 | Candidate rows: 78
Processed this run: 1000 | Candidate rows: 78
Proc

#### 18.34.8 — Wikidata Batch Error Diagnosis

The initial batch candidate-discovery process searched 1,237 unique Open Library author identifiers.

Only a small portion of the requests completed successfully, while 1,160 requests returned errors. This failure rate is too high to interpret as missing Wikidata coverage and instead indicates a request-level collection issue such as API throttling, rate limiting, timeout behavior, or another HTTP response condition.

Before candidate validation or geographic enrichment proceeds, the failed requests are analyzed by error type.

Successfully completed searches and confirmed zero-result searches remain preserved in the checkpoint log. Only failed requests will be considered for subsequent retry.

This prevents unnecessary repetition of successful API calls and separates source-coverage limitations from technical collection failures.

In [33]:
# ---------------------------------------------------------
# 18.34.8 Diagnose Wikidata batch errors
# ---------------------------------------------------------

wikidata_log = pd.read_csv(
    data_root
    / "raw"
    / "wikidata_author_search_log.csv"
)


# ---------------------------------------------------------
# Overall status
# ---------------------------------------------------------

status_summary = (
    wikidata_log[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("status")
    .reset_index(
        name="records"
    )
)


print("WIKIDATA SEARCH STATUS")
print("=" * 70)

display(status_summary)


# ---------------------------------------------------------
# Failed requests
# ---------------------------------------------------------

error_log = (
    wikidata_log[
        wikidata_log[
            "status"
        ] == "error"
    ]
    .copy()
)


print(
    "\nTotal errors:",
    len(error_log)
)


# ---------------------------------------------------------
# Inspect raw error messages
# ---------------------------------------------------------

print(
    "\nMOST COMMON RAW ERRORS"
)
print("=" * 70)

raw_errors = (
    error_log[
        "error"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
    .rename_axis("error")
    .reset_index(
        name="count"
    )
)

display(raw_errors)


# ---------------------------------------------------------
# Categorize error types
# ---------------------------------------------------------

def classify_wikidata_error(value):

    text = str(value).lower()

    if (
        "429" in text
        or "too many requests" in text
    ):
        return "HTTP 429 / Rate limit"

    if (
        "403" in text
        or "forbidden" in text
    ):
        return "HTTP 403 / Forbidden"

    if (
        "502" in text
        or "bad gateway" in text
    ):
        return "HTTP 502 / Bad Gateway"

    if (
        "503" in text
        or "service unavailable" in text
    ):
        return "HTTP 503 / Service unavailable"

    if (
        "504" in text
        or "gateway timeout" in text
    ):
        return "HTTP 504 / Gateway timeout"

    if (
        "timeout" in text
        or "timed out" in text
    ):
        return "Timeout"

    if (
        "connection" in text
        or "connectionerror" in text
    ):
        return "Connection error"

    if "json" in text:
        return "JSON / response parsing"

    return "Other"


error_log[
    "error_category"
] = (
    error_log[
        "error"
    ]
    .apply(
        classify_wikidata_error
    )
)


error_category_summary = (
    error_log[
        "error_category"
    ]
    .value_counts()
    .rename_axis(
        "error_category"
    )
    .reset_index(
        name="count"
    )
)


error_category_summary[
    "pct_of_errors"
] = (
    error_category_summary[
        "count"
    ]
    / len(error_log)
    * 100
).round(2)


print(
    "\nERROR CATEGORY SUMMARY"
)
print("=" * 70)

display(
    error_category_summary
)


# ---------------------------------------------------------
# First and last failed requests
# ---------------------------------------------------------

print(
    "\nSAMPLE FAILED REQUESTS"
)
print("=" * 70)

display(
    error_log[
        [
            "openlibrary_author_key",
            "author_name",
            "error_category",
            "error",
        ]
    ]
    .head(15)
)

WIKIDATA SEARCH STATUS


,status,records
0,error,1160
1,success,49
2,no_candidate,28



Total errors: 1160

MOST COMMON RAW ERRORS


,error,count
0,429 Client Error: Too Many Requests for url: h...,2
1,429 Client Error: Too Many Requests for url: h...,2
2,429 Client Error: Too Many Requests for url: h...,2
3,429 Client Error: Too many requests (f061ab2) ...,2
4,429 Client Error: Too many requests (f061ab2) ...,2
5,429 Client Error: Too Many Requests for url: h...,2
6,429 Client Error: Too many requests (f061ab2) ...,2
7,429 Client Error: Too many requests (f061ab2) ...,2
8,429 Client Error: Too Many Requests for url: h...,2
9,429 Client Error: Too many requests (f061ab2) ...,2



ERROR CATEGORY SUMMARY


,error_category,count,pct_of_errors
0,HTTP 429 / Rate limit,1160,100.0



SAMPLE FAILED REQUESTS


,openlibrary_author_key,author_name,error_category,error
10,OL7964852A,James Comey,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
11,OL3080130A,The Arbinger Institute,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
12,OL3010393A,Dick Ruhe,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
13,OL7596247A,Elesa Zehndorfer,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
14,OL233688A,Richard L. Hughes,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
15,OL2635692A,Robert C. Ginnett,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
16,OL2639183A,Gordon J Curphy,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
17,OL957796A,Richard Hughes,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
18,OL2639184A,Robert Ginnett,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...
19,OL2639185A,Gordon Curphy,HTTP 429 / Rate limit,429 Client Error: Too Many Requests for url: h...


#### 18.34.9 — Rate-Limit-Safe Wikidata Candidate Retry

Error diagnosis confirms that all 1,160 failed author searches resulted from HTTP 429 rate-limit responses rather than missing Wikidata records or invalid author identifiers.

The initial batch therefore does not provide a valid estimate of Wikidata author coverage.

A rate-limit-aware retry strategy is introduced with the following safeguards:

- previously successful searches are not repeated;
- confirmed zero-result searches are not repeated;
- only failed author searches are retried;
- requests are sent at a substantially slower rate;
- HTTP 429 responses trigger exponential backoff;
- the `Retry-After` response header is respected when available;
- repeated rate limiting causes the collection to stop rather than continue sending unsuccessful requests;
- progress is checkpointed continuously.

This approach preserves completed work while reducing unnecessary load on the external service.

In [34]:
# ---------------------------------------------------------
# 18.34.9 Rate-limit-safe Wikidata retry
# ---------------------------------------------------------

import time
import random
import requests
import pandas as pd


candidate_path = (
    data_root
    / "raw"
    / "wikidata_author_candidates.csv"
)

search_log_path = (
    data_root
    / "raw"
    / "wikidata_author_search_log.csv"
)


# ---------------------------------------------------------
# Reload current checkpoint
# ---------------------------------------------------------

existing_candidates = pd.read_csv(
    candidate_path
)

existing_log = pd.read_csv(
    search_log_path
)


# ---------------------------------------------------------
# Only retry failed records
# ---------------------------------------------------------

retry_authors = (
    existing_log[
        existing_log["status"]
        == "error"
    ]
    [
        [
            "openlibrary_author_key",
            "author_name",
        ]
    ]
    .drop_duplicates(
        subset=[
            "openlibrary_author_key"
        ]
    )
    .reset_index(drop=True)
)


print("RATE-LIMIT-SAFE RETRY")
print("=" * 70)

print(
    "Authors requiring retry:",
    len(retry_authors)
)


# ---------------------------------------------------------
# Session
# ---------------------------------------------------------

retry_session = requests.Session()

retry_session.headers.update(
    {
        "User-Agent":
            "LeadWise Academic Research Project"
    }
)


wikidata_api = (
    "https://www.wikidata.org/w/api.php"
)


# ---------------------------------------------------------
# Conservative settings
# ---------------------------------------------------------

NORMAL_DELAY = 2.0

MAX_RETRIES = 5

BACKOFF_BASE = 10

SAVE_EVERY = 10

MAX_CONSECUTIVE_429 = 5


new_candidate_rows = []

updated_log_rows = []

consecutive_429 = 0

stop_collection = False


# ---------------------------------------------------------
# Process failed authors
# ---------------------------------------------------------

for counter, (_, row) in enumerate(
    retry_authors.iterrows(),
    start=1,
):

    author_id = str(
        row["openlibrary_author_key"]
    ).strip()

    author_name = str(
        row["author_name"]
    ).strip()


    final_status = None

    final_results = []

    final_error = None


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        params = {
            "action":
                "wbsearchentities",

            "search":
                author_name,

            "language":
                "en",

            "format":
                "json",

            "limit":
                5,

            "type":
                "item",
        }


        try:

            response = (
                retry_session.get(
                    wikidata_api,
                    params=params,
                    timeout=30,
                )
            )


            # -----------------------------------------
            # Explicit rate-limit handling
            # -----------------------------------------

            if response.status_code == 429:

                consecutive_429 += 1

                retry_after = (
                    response.headers.get(
                        "Retry-After"
                    )
                )


                if retry_after:

                    try:
                        wait_seconds = float(
                            retry_after
                        )

                    except ValueError:
                        wait_seconds = (
                            BACKOFF_BASE
                            * (2 ** (attempt - 1))
                        )

                else:

                    wait_seconds = (
                        BACKOFF_BASE
                        * (2 ** (attempt - 1))
                    )


                # Small jitter so retries are not
                # perfectly periodic.
                wait_seconds += random.uniform(
                    0.5,
                    2.0,
                )


                print(
                    f"429 | {author_id} | "
                    f"attempt {attempt}/"
                    f"{MAX_RETRIES} | "
                    f"waiting "
                    f"{wait_seconds:.1f}s"
                )


                # Stop rather than repeatedly
                # hammering the service.
                if (
                    consecutive_429
                    >= MAX_CONSECUTIVE_429
                ):

                    final_status = (
                        "rate_limited"
                    )

                    final_error = (
                        "Collection stopped after "
                        "persistent HTTP 429 "
                        "responses."
                    )

                    stop_collection = True

                    break


                time.sleep(
                    wait_seconds
                )

                continue


            # -----------------------------------------
            # Other HTTP errors
            # -----------------------------------------

            response.raise_for_status()


            payload = response.json()

            results = payload.get(
                "search",
                []
            )


            consecutive_429 = 0

            final_results = results


            if results:

                final_status = "success"

            else:

                final_status = (
                    "no_candidate"
                )


            final_error = None

            break


        except requests.RequestException as exc:

            final_error = str(exc)

            if attempt < MAX_RETRIES:

                wait_seconds = (
                    BACKOFF_BASE
                    * (2 ** (attempt - 1))
                    + random.uniform(
                        0.5,
                        2.0,
                    )
                )

                time.sleep(
                    wait_seconds
                )

            else:

                final_status = "error"


        except Exception as exc:

            final_status = "error"

            final_error = str(exc)

            break


    # ---------------------------------------------
    # Store candidate results
    # ---------------------------------------------

    if final_status == "success":

        for rank, result in enumerate(
            final_results,
            start=1,
        ):

            new_candidate_rows.append(
                {
                    "openlibrary_author_key":
                        author_id,

                    "author_name":
                        author_name,

                    "candidate_rank":
                        rank,

                    "wikidata_id":
                        result.get("id"),

                    "wikidata_label":
                        result.get("label"),

                    "wikidata_description":
                        result.get(
                            "description"
                        ),

                    "wikidata_url":
                        result.get(
                            "concepturi"
                        ),
                }
            )


    # ---------------------------------------------
    # Update status
    # ---------------------------------------------

    updated_log_rows.append(
        {
            "openlibrary_author_key":
                author_id,

            "author_name":
                author_name,

            "status":
                final_status,

            "candidate_count":
                len(final_results),

            "error":
                final_error,
        }
    )


    # ---------------------------------------------
    # Save checkpoint
    # ---------------------------------------------

    if (
        counter % SAVE_EVERY == 0
        or stop_collection
        or counter == len(
            retry_authors
        )
    ):

        candidate_update = (
            pd.DataFrame(
                new_candidate_rows
            )
        )


        if len(candidate_update):

            current_candidates = (
                pd.concat(
                    [
                        existing_candidates,
                        candidate_update,
                    ],
                    ignore_index=True,
                )
                .drop_duplicates(
                    subset=[
                        "openlibrary_author_key",
                        "wikidata_id",
                    ]
                )
            )

        else:

            current_candidates = (
                existing_candidates.copy()
            )


        log_update = pd.DataFrame(
            updated_log_rows
        )


        # Remove old records for authors
        # already retried in this run.
        updated_ids = set(
            log_update[
                "openlibrary_author_key"
            ].astype(str)
        )


        retained_log = (
            existing_log[
                ~existing_log[
                    "openlibrary_author_key"
                ]
                .astype(str)
                .isin(updated_ids)
            ]
        )


        current_log = (
            pd.concat(
                [
                    retained_log,
                    log_update,
                ],
                ignore_index=True,
            )
            .drop_duplicates(
                subset=[
                    "openlibrary_author_key"
                ],
                keep="last",
            )
        )


        current_candidates.to_csv(
            candidate_path,
            index=False,
        )


        current_log.to_csv(
            search_log_path,
            index=False,
        )


        print(
            f"Processed retries: "
            f"{counter} | "
            f"Success: "
            f"{(log_update['status'] == 'success').sum()} | "
            f"No candidate: "
            f"{(log_update['status'] == 'no_candidate').sum()} | "
            f"Errors: "
            f"{(log_update['status'] == 'error').sum()} | "
            f"Rate limited: "
            f"{(log_update['status'] == 'rate_limited').sum()}"
        )


    if stop_collection:

        print(
            "\nPersistent rate limiting "
            "detected. Collection stopped "
            "safely."
        )

        break


    # ---------------------------------------------
    # Normal inter-request delay
    # ---------------------------------------------

    time.sleep(
        NORMAL_DELAY
        + random.uniform(
            0.0,
            0.5,
        )
    )


# ---------------------------------------------------------
# Reload checkpoint and summarize
# ---------------------------------------------------------

final_candidates = pd.read_csv(
    candidate_path
)

final_search_log = pd.read_csv(
    search_log_path
)


print(
    "\nCURRENT WIKIDATA COLLECTION STATUS"
)
print("=" * 70)


status_counts = (
    final_search_log[
        "status"
    ]
    .value_counts(
        dropna=False
    )
)


for status, count in (
    status_counts.items()
):

    print(
        f"{status}: {count}"
    )


print(
    "\nCandidate rows:",
    len(final_candidates)
)

print(
    "Unique candidate QIDs:",
    final_candidates[
        "wikidata_id"
    ].nunique()
)

RATE-LIMIT-SAFE RETRY
Authors requiring retry: 1160
Processed retries: 10 | Success: 5 | No candidate: 5 | Errors: 0 | Rate limited: 0
429 | OL2635693A | attempt 1/5 | waiting 12.6s
Processed retries: 20 | Success: 13 | No candidate: 7 | Errors: 0 | Rate limited: 0
429 | OL222155A | attempt 1/5 | waiting 34.6s
Processed retries: 30 | Success: 20 | No candidate: 10 | Errors: 0 | Rate limited: 0
429 | OL302271A | attempt 1/5 | waiting 35.4s
Processed retries: 40 | Success: 27 | No candidate: 13 | Errors: 0 | Rate limited: 0
429 | OL220279A | attempt 1/5 | waiting 32.7s
Processed retries: 50 | Success: 34 | No candidate: 16 | Errors: 0 | Rate limited: 0
429 | OL3925630A | attempt 1/5 | waiting 35.1s
Processed retries: 60 | Success: 39 | No candidate: 21 | Errors: 0 | Rate limited: 0
429 | OL72319A | attempt 1/5 | waiting 34.6s
Processed retries: 70 | Success: 44 | No candidate: 26 | Errors: 0 | Rate limited: 0
429 | OL948501A | attempt 1/5 | waiting 34.8s
Processed retries: 80 | Success: 

#### 18.34.10 — Structured Wikidata Candidate Metadata Collection

Wikidata candidate discovery is now complete for all Open Library author identifiers represented in the enrichment dataset.

Of the 1,237 unique Open Library author identifiers:

- 753 returned at least one Wikidata candidate;
- 484 returned no candidate;
- no unresolved technical API errors remain.

Candidate discovery does not establish author identity. A returned entity may represent another person with the same name, an article about the author, an organization, or another unrelated entity.

Structured Wikidata claims are therefore retrieved for the candidate entities before identity decisions are made.

The validation dataset captures, where available:

- entity type;
- country of citizenship;
- place of birth;
- date of birth;
- occupation;
- selected external identifiers;
- candidate label and description.

Wikidata QIDs are requested in batches to reduce the number of external API calls.

No candidate is automatically accepted based solely on search rank, label similarity, or geographic information.

In [35]:
# ---------------------------------------------------------
# 18.34.10 Structured Wikidata candidate metadata
# ---------------------------------------------------------

import time
import requests
import pandas as pd
from pathlib import Path


candidate_path = (
    data_root
    / "raw"
    / "wikidata_author_candidates.csv"
)

output_path = (
    data_root
    / "processed"
    / "wikidata_author_candidate_metadata.csv"
)


candidates = pd.read_csv(
    candidate_path
)


candidate_qids = (
    candidates[
        "wikidata_id"
    ]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)


print(
    "STRUCTURED CANDIDATE METADATA COLLECTION"
)
print("=" * 70)

print(
    "Candidate rows:",
    len(candidates)
)

print(
    "Unique Wikidata QIDs:",
    len(candidate_qids)
)


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def extract_entity_ids(
    claims,
    property_id,
):

    values = []

    for claim in claims.get(
        property_id,
        [],
    ):

        try:

            value = (
                claim[
                    "mainsnak"
                ][
                    "datavalue"
                ][
                    "value"
                ]
            )

            if isinstance(
                value,
                dict,
            ):

                qid = value.get(
                    "id"
                )

                if qid:
                    values.append(
                        qid
                    )

        except (
            KeyError,
            TypeError,
        ):
            continue

    return list(
        dict.fromkeys(values)
    )


def extract_times(
    claims,
    property_id,
):

    values = []

    for claim in claims.get(
        property_id,
        [],
    ):

        try:

            value = (
                claim[
                    "mainsnak"
                ][
                    "datavalue"
                ][
                    "value"
                ]
            )

            if isinstance(
                value,
                dict,
            ):

                time_value = (
                    value.get(
                        "time"
                    )
                )

                if time_value:
                    values.append(
                        time_value
                    )

        except (
            KeyError,
            TypeError,
        ):
            continue

    return list(
        dict.fromkeys(values)
    )


def extract_external_ids(
    claims,
):

    """
    Preserve external-ID claims without
    assuming which identifier will ultimately
    be useful for identity resolution.
    """

    external_ids = {}

    for property_id, claim_list in (
        claims.items()
    ):

        values = []

        for claim in claim_list:

            try:

                datavalue = (
                    claim[
                        "mainsnak"
                    ].get(
                        "datavalue"
                    )
                )

                if not datavalue:
                    continue

                if (
                    datavalue.get(
                        "type"
                    )
                    == "string"
                ):

                    value = (
                        datavalue.get(
                            "value"
                        )
                    )

                    if value:
                        values.append(
                            str(value)
                        )

            except (
                KeyError,
                TypeError,
            ):
                continue

        if values:

            external_ids[
                property_id
            ] = list(
                dict.fromkeys(
                    values
                )
            )

    return external_ids


# ---------------------------------------------------------
# API session
# ---------------------------------------------------------

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "LeadWise Academic Research Project"
    }
)


wikidata_api = (
    "https://www.wikidata.org/w/api.php"
)


# ---------------------------------------------------------
# Batch collection
# ---------------------------------------------------------

BATCH_SIZE = 25

NORMAL_DELAY = 2.0

MAX_RETRIES = 5


metadata_rows = []


for start in range(
    0,
    len(candidate_qids),
    BATCH_SIZE,
):

    batch = candidate_qids[
        start:
        start + BATCH_SIZE
    ]

    success = False


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        params = {
            "action":
                "wbgetentities",

            "ids":
                "|".join(batch),

            "props":
                "labels|descriptions|claims",

            "languages":
                "en",

            "format":
                "json",
        }


        response = session.get(
            wikidata_api,
            params=params,
            timeout=60,
        )


        if response.status_code == 429:

            retry_after = (
                response.headers.get(
                    "Retry-After"
                )
            )


            try:

                wait_seconds = (
                    float(
                        retry_after
                    )
                    if retry_after
                    else 15 * attempt
                )

            except ValueError:

                wait_seconds = (
                    15 * attempt
                )


            print(
                f"429 | batch "
                f"{start // BATCH_SIZE + 1} | "
                f"attempt {attempt} | "
                f"waiting {wait_seconds:.1f}s"
            )

            time.sleep(
                wait_seconds
            )

            continue


        response.raise_for_status()

        payload = response.json()

        entities = payload.get(
            "entities",
            {}
        )

        success = True

        break


    if not success:

        print(
            "Batch failed after retries:",
            batch
        )

        continue


    # -----------------------------------------------------
    # Parse entities
    # -----------------------------------------------------

    for qid in batch:

        entity = entities.get(
            qid,
            {}
        )

        claims = entity.get(
            "claims",
            {}
        )


        label = (
            entity
            .get(
                "labels",
                {}
            )
            .get(
                "en",
                {}
            )
            .get(
                "value"
            )
        )


        description = (
            entity
            .get(
                "descriptions",
                {}
            )
            .get(
                "en",
                {}
            )
            .get(
                "value"
            )
        )


        metadata_rows.append(
            {
                "wikidata_id":
                    qid,

                "wikidata_label":
                    label,

                "wikidata_description":
                    description,

                # P31 = instance of
                "instance_of_qids":
                    extract_entity_ids(
                        claims,
                        "P31",
                    ),

                # P27 = country of citizenship
                "citizenship_qids":
                    extract_entity_ids(
                        claims,
                        "P27",
                    ),

                # P19 = place of birth
                "birthplace_qids":
                    extract_entity_ids(
                        claims,
                        "P19",
                    ),

                # P569 = date of birth
                "birth_dates":
                    extract_times(
                        claims,
                        "P569",
                    ),

                # P106 = occupation
                "occupation_qids":
                    extract_entity_ids(
                        claims,
                        "P106",
                    ),

                # Preserve string-valued identifiers
                # for later identity verification.
                "external_id_claims":
                    extract_external_ids(
                        claims
                    ),
            }
        )


    print(
        f"Processed QIDs: "
        f"{min(start + BATCH_SIZE, len(candidate_qids))}"
        f"/{len(candidate_qids)}"
    )


    time.sleep(
        NORMAL_DELAY
    )


# ---------------------------------------------------------
# Save structured candidate metadata
# ---------------------------------------------------------

candidate_metadata = (
    pd.DataFrame(
        metadata_rows
    )
    .drop_duplicates(
        subset=[
            "wikidata_id"
        ]
    )
)


candidate_metadata.to_csv(
    output_path,
    index=False,
)


# ---------------------------------------------------------
# Coverage summary
# ---------------------------------------------------------

print(
    "\nSTRUCTURED CANDIDATE METADATA COMPLETE"
)
print("=" * 70)

print(
    "Metadata rows:",
    len(candidate_metadata)
)

print(
    "Unique QIDs:",
    candidate_metadata[
        "wikidata_id"
    ].nunique()
)


def has_values(series):

    return series.apply(
        lambda x:
            isinstance(x, list)
            and len(x) > 0
    ).sum()


print(
    "With instance-of:",
    has_values(
        candidate_metadata[
            "instance_of_qids"
        ]
    )
)

print(
    "With citizenship:",
    has_values(
        candidate_metadata[
            "citizenship_qids"
        ]
    )
)

print(
    "With birthplace:",
    has_values(
        candidate_metadata[
            "birthplace_qids"
        ]
    )
)

print(
    "With birth date:",
    has_values(
        candidate_metadata[
            "birth_dates"
        ]
    )
)

print(
    "With occupation:",
    has_values(
        candidate_metadata[
            "occupation_qids"
        ]
    )
)

print(
    "\nSaved:",
    output_path
)

STRUCTURED CANDIDATE METADATA COLLECTION
Candidate rows: 1633
Unique Wikidata QIDs: 1554
Processed QIDs: 25/1554
Processed QIDs: 50/1554
Processed QIDs: 75/1554
Processed QIDs: 100/1554
Processed QIDs: 125/1554
Processed QIDs: 150/1554
Processed QIDs: 175/1554
Processed QIDs: 200/1554
Processed QIDs: 225/1554
Processed QIDs: 250/1554
429 | batch 11 | attempt 1 | waiting 10.0s
Processed QIDs: 275/1554
Processed QIDs: 300/1554
Processed QIDs: 325/1554
Processed QIDs: 350/1554
Processed QIDs: 375/1554
Processed QIDs: 400/1554
Processed QIDs: 425/1554
Processed QIDs: 450/1554
Processed QIDs: 475/1554
Processed QIDs: 500/1554
429 | batch 21 | attempt 1 | waiting 29.0s
Processed QIDs: 525/1554
Processed QIDs: 550/1554
Processed QIDs: 575/1554
Processed QIDs: 600/1554
Processed QIDs: 625/1554
Processed QIDs: 650/1554
Processed QIDs: 675/1554
Processed QIDs: 700/1554
Processed QIDs: 725/1554
Processed QIDs: 750/1554
429 | batch 31 | attempt 1 | waiting 28.0s
Processed QIDs: 775/1554
Processed 

#### 18.34.11 — External Identifier Bridge Audit

Structured metadata has been collected for all unique Wikidata candidate entities identified during author candidate discovery.

Before applying name-based or biographical identity matching, the candidate metadata is examined for external identifiers that may provide a direct bridge between Wikidata and Open Library.

A direct identifier correspondence is stronger evidence of identity than:

- search-result rank;
- name similarity;
- candidate description;
- citizenship;
- birthplace;
- occupation.

The external-ID claims retained during structured metadata collection are therefore audited to identify:

1. which Wikidata properties occur most frequently;
2. whether an Open Library author identifier is present;
3. whether any candidate identifier directly matches the Open Library author key used in LeadWise.

No author identity is assigned during this audit.

In [37]:
# ---------------------------------------------------------
# 18.34.11 External identifier bridge audit
# ---------------------------------------------------------

import ast
import pandas as pd
from collections import Counter


metadata_path = (
    data_root
    / "processed"
    / "wikidata_author_candidate_metadata.csv"
)

candidate_path = (
    data_root
    / "raw"
    / "wikidata_author_candidates.csv"
)


candidate_metadata = pd.read_csv(
    metadata_path
)

author_candidates = pd.read_csv(
    candidate_path
)


# ---------------------------------------------------------
# Parse external-ID dictionaries
# ---------------------------------------------------------

def parse_external_claims(value):

    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, dict):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return {}


candidate_metadata[
    "external_claims_parsed"
] = (
    candidate_metadata[
        "external_id_claims"
    ]
    .apply(
        parse_external_claims
    )
)


# ---------------------------------------------------------
# Count properties
# ---------------------------------------------------------

property_counter = Counter()


for claims in candidate_metadata[
    "external_claims_parsed"
]:

    property_counter.update(
        claims.keys()
    )


property_frequency = (
    pd.DataFrame(
        property_counter.items(),
        columns=[
            "property_id",
            "candidate_entities",
        ],
    )
    .sort_values(
        "candidate_entities",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "EXTERNAL IDENTIFIER PROPERTY AUDIT"
)
print("=" * 70)

print(
    "Candidate entities:",
    len(candidate_metadata)
)

print(
    "Entities with at least one "
    "string-valued claim:",
    candidate_metadata[
        "external_claims_parsed"
    ]
    .apply(
        lambda x:
            isinstance(x, dict)
            and len(x) > 0
    )
    .sum()
)

print(
    "Unique string-valued "
    "Wikidata properties:",
    len(property_frequency)
)


print(
    "\nMOST COMMON PROPERTY IDs"
)
print("=" * 70)

display(
    property_frequency.head(40)
)


# ---------------------------------------------------------
# Search the property IDs themselves through Wikidata
# so we do not assume their meaning.
# ---------------------------------------------------------

property_ids = (
    property_frequency[
        "property_id"
    ]
    .tolist()
)


property_label_rows = []


PROPERTY_BATCH_SIZE = 50


for start in range(
    0,
    len(property_ids),
    PROPERTY_BATCH_SIZE,
):

    batch = property_ids[
        start:
        start + PROPERTY_BATCH_SIZE
    ]

    params = {
        "action":
            "wbgetentities",

        "ids":
            "|".join(batch),

        "props":
            "labels|descriptions",

        "languages":
            "en",

        "format":
            "json",
    }


    response = session.get(
        wikidata_api,
        params=params,
        timeout=60,
    )

    response.raise_for_status()

    entities = (
        response.json()
        .get(
            "entities",
            {}
        )
    )


    for property_id in batch:

        entity = entities.get(
            property_id,
            {}
        )

        label = (
            entity
            .get(
                "labels",
                {}
            )
            .get(
                "en",
                {}
            )
            .get(
                "value"
            )
        )

        description = (
            entity
            .get(
                "descriptions",
                {}
            )
            .get(
                "en",
                {}
            )
            .get(
                "value"
            )
        )


        property_label_rows.append(
            {
                "property_id":
                    property_id,

                "property_label":
                    label,

                "property_description":
                    description,
            }
        )


    time.sleep(2)


property_labels = pd.DataFrame(
    property_label_rows
)


property_audit = (
    property_frequency
    .merge(
        property_labels,
        on="property_id",
        how="left",
        validate="one_to_one",
    )
)


print(
    "\nIDENTIFIED EXTERNAL-ID PROPERTIES"
)
print("=" * 70)

display(
    property_audit.head(50)
)


# ---------------------------------------------------------
# Find properties related to Open Library
# ---------------------------------------------------------

openlibrary_properties = (
    property_audit[
        property_audit[
            [
                "property_label",
                "property_description",
            ]
        ]
        .fillna("")
        .agg(
            " ".join,
            axis=1,
        )
        .str.contains(
            "open library",
            case=False,
            regex=False,
        )
    ]
    .copy()
)


print(
    "\nOPEN LIBRARY RELATED PROPERTIES"
)
print("=" * 70)

if len(openlibrary_properties):

    display(
        openlibrary_properties
    )

else:

    print(
        "No Open Library-related property "
        "was found among the collected "
        "string-valued claims."
    )

EXTERNAL IDENTIFIER PROPERTY AUDIT
Candidate entities: 1554
Entities with at least one string-valued claim: 1461
Unique string-valued Wikidata properties: 1362

MOST COMMON PROPERTY IDs


,property_id,candidate_entities
0,P214,711
1,P213,573
2,P227,518
3,P244,506
4,P691,472
5,P13591,472
6,P8189,448
7,P269,427
8,P646,396
9,P1207,394


HTTPError: 429 Client Error: Too Many Requests for url: https://www.wikidata.org/w/api.php?action=wbgetentities&ids=P11012%7CP8483%7CP6778%7CP11835%7CP3544%7CP8854%7CP7085%7CP10381%7CP836%7CP9426%7CP13766%7CP1281%7CP613%7CP3120%7CP3615%7CP7350%7CP9284%7CP2041%7CP1937%7CP7314%7CP6705%7CP13648%7CP6891%7CP4928%7CP5430%7CP8534%7CP4576%7CP6777%7CP9466%7CP6735%7CP7430%7CP7286%7CP8210%7CP8705%7CP10031%7CP13818%7CP14674%7CP9543%7CP10404%7CP4903%7CP6723%7CP5502%7CP3786%7CP4112%7CP14774%7CP4589%7CP4601%7CP3144%7CP3107%7CP3933&props=labels%7Cdescriptions&languages=en&format=json

#### 18.34.11A — Targeted Verification of the Open Library Property

The broad Wikidata property-label lookup encountered an HTTP 429 rate-limit response after the external identifier frequency audit had already completed.

Rather than repeating requests for all observed Wikidata properties, a targeted lookup is performed for property P648, which occurs in 110 candidate entities.

This targeted verification determines whether P648 represents an Open Library identifier before it is used for author identity resolution.

In [38]:
# ---------------------------------------------------------
# 18.34.11A Targeted P648 verification
# ---------------------------------------------------------

import time
import requests


property_id = "P648"

params = {
    "action": "wbgetentities",
    "ids": property_id,
    "props": "labels|descriptions",
    "languages": "en",
    "format": "json",
}


for attempt in range(1, 6):

    response = session.get(
        wikidata_api,
        params=params,
        timeout=30,
    )


    if response.status_code == 429:

        retry_after = response.headers.get(
            "Retry-After"
        )

        try:
            wait_seconds = (
                float(retry_after)
                if retry_after
                else 15 * attempt
            )

        except ValueError:
            wait_seconds = 15 * attempt


        print(
            f"429 | attempt {attempt}/5 | "
            f"waiting {wait_seconds:.1f}s"
        )

        time.sleep(wait_seconds)

        continue


    response.raise_for_status()

    entity = (
        response.json()
        ["entities"]
        [property_id]
    )


    property_label = (
        entity
        .get("labels", {})
        .get("en", {})
        .get("value")
    )


    property_description = (
        entity
        .get("descriptions", {})
        .get("en", {})
        .get("value")
    )


    print(
        "TARGET PROPERTY VERIFICATION"
    )
    print("=" * 70)

    print(
        "Property:",
        property_id
    )

    print(
        "Label:",
        property_label
    )

    print(
        "Description:",
        property_description
    )

    break

else:

    print(
        "P648 could not be verified "
        "because Wikidata remained rate limited."
    )

TARGET PROPERTY VERIFICATION
Property: P648
Label: Open Library ID
Description: identifier for a work ("W"), edition ("M") or author ("A") for book data of the Internet Archive


#### 18.34.12 — Direct Open Library–Wikidata Author Identity Matching

Wikidata property P648 has been verified as the Open Library identifier used for works, editions, and authors.

This enables a direct identifier-based author identity test.

For each Wikidata candidate containing P648, the stored Open Library identifiers are compared with the Open Library author identifier associated with the LeadWise source record.

An exact match between the two identifiers provides substantially stronger identity evidence than:

- search-result rank;
- author-name similarity;
- candidate descriptions;
- occupation;
- citizenship;
- birthplace.

Only identifiers representing Open Library author records (`A`) are considered for direct author matching.

Candidates with an exact Open Library author-ID correspondence are classified as identifier-verified matches. Candidates without such a correspondence are not rejected; they remain available for subsequent validation using other structured evidence.

No geographic metadata is assigned to unresolved candidates at this stage.

In [39]:
# ---------------------------------------------------------
# 18.34.12 Direct Open Library ↔ Wikidata ID matching
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Reload source datasets
# ---------------------------------------------------------

candidate_metadata = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_candidate_metadata.csv"
)

author_candidates = pd.read_csv(
    data_root
    / "raw"
    / "wikidata_author_candidates.csv"
)


# ---------------------------------------------------------
# Parse external-ID dictionaries
# ---------------------------------------------------------

def parse_external_claims(value):

    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, dict):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return {}


candidate_metadata[
    "external_claims_parsed"
] = (
    candidate_metadata[
        "external_id_claims"
    ]
    .apply(
        parse_external_claims
    )
)


# ---------------------------------------------------------
# Extract P648 values
# ---------------------------------------------------------

def extract_openlibrary_ids(
    claims
):

    if not isinstance(
        claims,
        dict,
    ):
        return []

    values = claims.get(
        "P648",
        []
    )

    if not isinstance(
        values,
        list,
    ):
        values = [values]


    cleaned = []

    for value in values:

        value = str(
            value
        ).strip()

        if value:
            cleaned.append(
                value
            )


    return list(
        dict.fromkeys(cleaned)
    )


candidate_metadata[
    "wikidata_openlibrary_ids"
] = (
    candidate_metadata[
        "external_claims_parsed"
    ]
    .apply(
        extract_openlibrary_ids
    )
)


# ---------------------------------------------------------
# Keep candidate entities containing P648
# ---------------------------------------------------------

metadata_with_p648 = (
    candidate_metadata[
        candidate_metadata[
            "wikidata_openlibrary_ids"
        ]
        .apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        )
    ]
    .copy()
)


print(
    "OPEN LIBRARY IDENTIFIER COVERAGE"
)
print("=" * 70)

print(
    "Candidate QIDs with P648:",
    len(metadata_with_p648)
)


# ---------------------------------------------------------
# Merge structured metadata onto search candidates
# ---------------------------------------------------------

candidate_validation = (
    author_candidates
    .merge(
        candidate_metadata[
            [
                "wikidata_id",
                "instance_of_qids",
                "citizenship_qids",
                "birthplace_qids",
                "birth_dates",
                "occupation_qids",
                "wikidata_openlibrary_ids",
            ]
        ],
        on="wikidata_id",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Normalize Open Library identifiers
# ---------------------------------------------------------

def normalize_ol_author_id(
    value
):

    if pd.isna(value):
        return None

    value = str(
        value
    ).strip()


    # Handle values that may include
    # /authors/ prefix.
    value = (
        value
        .replace(
            "https://openlibrary.org/authors/",
            ""
        )
        .replace(
            "http://openlibrary.org/authors/",
            ""
        )
        .replace(
            "/authors/",
            ""
        )
        .strip("/")
    )


    return value


candidate_validation[
    "source_ol_author_id"
] = (
    candidate_validation[
        "openlibrary_author_key"
    ]
    .apply(
        normalize_ol_author_id
    )
)


candidate_validation[
    "wikidata_ol_author_ids_normalized"
] = (
    candidate_validation[
        "wikidata_openlibrary_ids"
    ]
    .apply(
        lambda values: [
            normalize_ol_author_id(v)
            for v in values
            if normalize_ol_author_id(v)
        ]
        if isinstance(
            values,
            list,
        )
        else []
    )
)


# ---------------------------------------------------------
# Author-ID exact match
# ---------------------------------------------------------

candidate_validation[
    "exact_openlibrary_id_match"
] = (
    candidate_validation
    .apply(
        lambda row:
            row[
                "source_ol_author_id"
            ]
            in row[
                "wikidata_ol_author_ids_normalized"
            ]
            if row[
                "source_ol_author_id"
            ]
            else False,
        axis=1,
    )
)


# ---------------------------------------------------------
# Ensure matched P648 IDs are author identifiers
# ---------------------------------------------------------

candidate_validation[
    "matched_author_identifier"
] = (
    candidate_validation
    .apply(
        lambda row:
            bool(
                row[
                    "exact_openlibrary_id_match"
                ]
                and
                str(
                    row[
                        "source_ol_author_id"
                    ]
                ).endswith(
                    "A"
                )
            ),
        axis=1,
    )
)


# ---------------------------------------------------------
# Identifier-verified candidates
# ---------------------------------------------------------

identifier_verified = (
    candidate_validation[
        candidate_validation[
            "matched_author_identifier"
        ]
    ]
    .copy()
)


print(
    "\nDIRECT IDENTITY MATCHING"
)
print("=" * 70)

print(
    "Candidate rows:",
    len(candidate_validation)
)

print(
    "Candidates with exact "
    "Open Library author-ID match:",
    len(identifier_verified)
)

print(
    "Unique Open Library authors "
    "identifier-verified:",
    identifier_verified[
        "openlibrary_author_key"
    ].nunique()
)

print(
    "Unique Wikidata entities "
    "identifier-verified:",
    identifier_verified[
        "wikidata_id"
    ].nunique()
)


# ---------------------------------------------------------
# Check for conflicting direct matches
# ---------------------------------------------------------

verified_per_author = (
    identifier_verified
    .groupby(
        "openlibrary_author_key"
    )[
        "wikidata_id"
    ]
    .nunique()
)


conflicting_direct_matches = (
    verified_per_author[
        verified_per_author > 1
    ]
)


print(
    "Authors with >1 exact "
    "Wikidata ID match:",
    len(
        conflicting_direct_matches
    )
)


# ---------------------------------------------------------
# Preview verified identities
# ---------------------------------------------------------

print(
    "\nIDENTIFIER-VERIFIED AUTHOR SAMPLE"
)
print("=" * 70)

display(
    identifier_verified[
        [
            "openlibrary_author_key",
            "author_name",
            "candidate_rank",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "wikidata_ol_author_ids_normalized",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save validation table
# ---------------------------------------------------------

validation_path = (
    data_root
    / "processed"
    / "wikidata_author_candidate_validation.csv"
)


candidate_validation.to_csv(
    validation_path,
    index=False,
)


verified_path = (
    data_root
    / "processed"
    / "wikidata_author_identifier_verified.csv"
)


identifier_verified.to_csv(
    verified_path,
    index=False,
)


print(
    "\nSaved candidate validation:",
    validation_path
)

print(
    "Saved identifier-verified authors:",
    verified_path
)

OPEN LIBRARY IDENTIFIER COVERAGE
Candidate QIDs with P648: 110

DIRECT IDENTITY MATCHING
Candidate rows: 1633
Candidates with exact Open Library author-ID match: 68
Unique Open Library authors identifier-verified: 68
Unique Wikidata entities identifier-verified: 68
Authors with >1 exact Wikidata ID match: 0

IDENTIFIER-VERIFIED AUTHOR SAMPLE


,openlibrary_author_key,author_name,candidate_rank,wikidata_id,wikidata_label,wikidata_description,wikidata_ol_author_ids_normalized
0,OL383159A,Stephen R. Covey,1,Q313482,Stephen Covey,"American educator, author, businessman and mot...",[OL383159A]
3,OL26423A,John C. Maxwell,1,Q3304739,John C. Maxwell,"American author, speaker and neopentecostal pa...",[OL26423A]
18,OL236243A,Henry Kissinger,1,Q66107,Henry Kissinger,"American diplomat and geopolitical advisor, se...",[OL236243A]
55,OL2804440A,Jason Fried,1,Q23795888,Jason Fried,software entrepreneur,[OL2804440A]
85,OL39038A,Deepak Chopra,1,Q318506,Deepak Chopra,Indian-born American author and alternative-me...,[OL39038A]
110,OL957796A,Richard Hughes,1,Q302433,Richard Hughes,British writer (1900–1976),[OL957796A]
118,OL27560A,James MacGregor Burns,1,Q1529607,James MacGregor Burns,American historian and political scientist (19...,[OL27560A]
121,OL237290A,James M. Kouzes,1,Q55865983,James M. Kouzes,1945-,[OL237290A]
132,OL222155A,Doris Kearns Goodwin,1,Q442008,Doris Kearns Goodwin,American biographer and historian (born 1943),[OL222155A]
147,OL2632098A,John King,1,Q671520,John King,English writer (born 1960),[OL2632098A]



Saved candidate validation: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_author_candidate_validation.csv
Saved identifier-verified authors: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_author_identifier_verified.csv


#### 18.34.13 — Calibration of Secondary Author Identity Evidence

Direct Open Library–Wikidata identifier matching produced a high-confidence reference set of verified author identities.

A total of 68 Open Library author identifiers were matched directly to Wikidata entities through property P648, with no Open Library author producing more than one exact Wikidata identity match.

These identifier-verified records provide a reference set for evaluating weaker identity evidence among candidates that do not contain a matching Open Library identifier.

Secondary identity evidence is calibrated rather than assigned arbitrarily.

The analysis compares:

- normalized source and candidate names;
- character-based name similarity;
- Wikidata search rank;
- whether the candidate is explicitly represented as a human entity;
- candidate descriptions.

The purpose of this stage is diagnostic. No additional authors are classified as verified solely from name similarity or candidate rank.

In [40]:
# ---------------------------------------------------------
# 18.34.13 Calibrate secondary identity evidence
# ---------------------------------------------------------

import ast
import re
import unicodedata
from difflib import SequenceMatcher
import pandas as pd


validation_path = (
    data_root
    / "processed"
    / "wikidata_author_candidate_validation.csv"
)


candidate_validation = pd.read_csv(
    validation_path
)


# ---------------------------------------------------------
# Parse list-like fields saved to CSV
# ---------------------------------------------------------

def parse_list_value(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return []


candidate_validation[
    "instance_of_parsed"
] = (
    candidate_validation[
        "instance_of_qids"
    ]
    .apply(
        parse_list_value
    )
)


# ---------------------------------------------------------
# Conservative name normalization
# ---------------------------------------------------------

def normalize_author_name(
    value
):

    if pd.isna(value):
        return ""

    value = str(value)

    value = unicodedata.normalize(
        "NFKD",
        value,
    )

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(
            char
        )
    )

    value = value.lower()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


candidate_validation[
    "source_name_normalized"
] = (
    candidate_validation[
        "author_name"
    ]
    .apply(
        normalize_author_name
    )
)


candidate_validation[
    "candidate_name_normalized"
] = (
    candidate_validation[
        "wikidata_label"
    ]
    .apply(
        normalize_author_name
    )
)


# ---------------------------------------------------------
# Name similarity
# ---------------------------------------------------------

def calculate_name_similarity(
    source,
    candidate,
):

    if not source or not candidate:
        return 0.0

    return SequenceMatcher(
        None,
        source,
        candidate,
    ).ratio()


candidate_validation[
    "name_similarity"
] = (
    candidate_validation
    .apply(
        lambda row:
            calculate_name_similarity(
                row[
                    "source_name_normalized"
                ],
                row[
                    "candidate_name_normalized"
                ],
            ),
        axis=1,
    )
)


# ---------------------------------------------------------
# Exact normalized-name agreement
# ---------------------------------------------------------

candidate_validation[
    "exact_normalized_name"
] = (
    candidate_validation[
        "source_name_normalized"
    ]
    ==
    candidate_validation[
        "candidate_name_normalized"
    ]
)


# ---------------------------------------------------------
# Human entity
# Q5 = human
# ---------------------------------------------------------

candidate_validation[
    "is_human"
] = (
    candidate_validation[
        "instance_of_parsed"
    ]
    .apply(
        lambda values:
            "Q5" in values
    )
)


# ---------------------------------------------------------
# Identifier-verified flag
# ---------------------------------------------------------

if (
    "matched_author_identifier"
    in candidate_validation.columns
):

    candidate_validation[
        "identifier_verified"
    ] = (
        candidate_validation[
            "matched_author_identifier"
        ]
        .astype(bool)
    )

else:

    candidate_validation[
        "identifier_verified"
    ] = False


verified_reference = (
    candidate_validation[
        candidate_validation[
            "identifier_verified"
        ]
    ]
    .copy()
)


unverified_candidates = (
    candidate_validation[
        ~candidate_validation[
            "identifier_verified"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Calibration summary
# ---------------------------------------------------------

print(
    "IDENTIFIER-VERIFIED REFERENCE SET"
)
print("=" * 70)

print(
    "Verified candidate rows:",
    len(verified_reference)
)

print(
    "Exact normalized names:",
    verified_reference[
        "exact_normalized_name"
    ].sum()
)

print(
    "Human entities:",
    verified_reference[
        "is_human"
    ].sum()
)

print(
    "Rank-1 candidates:",
    (
        verified_reference[
            "candidate_rank"
        ] == 1
    ).sum()
)

print(
    "Mean name similarity:",
    round(
        verified_reference[
            "name_similarity"
        ].mean(),
        4,
    )
)

print(
    "Minimum name similarity:",
    round(
        verified_reference[
            "name_similarity"
        ].min(),
        4,
    )
)

print(
    "Median name similarity:",
    round(
        verified_reference[
            "name_similarity"
        ].median(),
        4,
    )
)


# ---------------------------------------------------------
# Distribution
# ---------------------------------------------------------

similarity_bins = [
    0.0,
    0.50,
    0.70,
    0.80,
    0.90,
    0.95,
    0.99,
    1.001,
]


verified_reference[
    "similarity_band"
] = pd.cut(
    verified_reference[
        "name_similarity"
    ],
    bins=similarity_bins,
    right=False,
)


verified_distribution = (
    verified_reference[
        "similarity_band"
    ]
    .value_counts(
        sort=False
    )
    .rename_axis(
        "similarity_band"
    )
    .reset_index(
        name="verified_records"
    )
)


print(
    "\nVERIFIED NAME-SIMILARITY DISTRIBUTION"
)
print("=" * 70)

display(
    verified_distribution
)


# ---------------------------------------------------------
# Strong-looking candidates without direct ID match
# ---------------------------------------------------------

strong_unverified = (
    unverified_candidates[
        (
            unverified_candidates[
                "is_human"
            ]
        )
        &
        (
            unverified_candidates[
                "candidate_rank"
            ] == 1
        )
        &
        (
            unverified_candidates[
                "name_similarity"
            ] >= 0.90
        )
    ]
    .copy()
    .sort_values(
        [
            "name_similarity",
            "author_name",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


print(
    "\nSTRONG-LOOKING BUT NOT "
    "IDENTIFIER-VERIFIED"
)
print("=" * 70)

print(
    "Candidate rows:",
    len(
        strong_unverified
    )
)

print(
    "Unique source authors:",
    strong_unverified[
        "openlibrary_author_key"
    ].nunique()
)


display(
    strong_unverified[
        [
            "openlibrary_author_key",
            "author_name",
            "candidate_rank",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "name_similarity",
            "exact_normalized_name",
            "is_human",
        ]
    ]
    .head(40)
)

IDENTIFIER-VERIFIED REFERENCE SET
Verified candidate rows: 68
Exact normalized names: 54
Human entities: 67
Rank-1 candidates: 64
Mean name similarity: 0.9731
Minimum name similarity: 0.5294
Median name similarity: 1.0

VERIFIED NAME-SIMILARITY DISTRIBUTION


,similarity_band,verified_records
0,"[0.0, 0.5)",0
1,"[0.5, 0.7)",1
2,"[0.7, 0.8)",2
3,"[0.8, 0.9)",3
4,"[0.9, 0.95)",7
5,"[0.95, 0.99)",1
6,"[0.99, 1.001)",54



STRONG-LOOKING BUT NOT IDENTIFIER-VERIFIED
Candidate rows: 595
Unique source authors: 595


,openlibrary_author_key,author_name,candidate_rank,wikidata_id,wikidata_label,wikidata_description,name_similarity,exact_normalized_name,is_human
593,OL1193951A,Aaron Antonovsky,1,Q301653,Aaron Antonovsky,Israeli American sociologist,1.0,True,True
306,OL1346979A,Abraham Holtzman,1,Q23722834,Abraham Holtzman,American political scientist,1.0,True,True
603,OL2694332A,Adrian Wilkinson,1,Q112427096,Adrian Wilkinson,NaN,1.0,True,True
1158,OL25174A,Afsaneh Nahavandi,1,Q100154628,Afsaneh Nahavandi,NaN,1.0,True,True
1171,OL492603A,Alain Verbeke,1,Q59609905,Alain Verbeke,"born:1960|; Verbeke, Alain.; Verbeke, A. (Alai...",1.0,True,True
500,OL2640639A,Alan B. Eisner,1,Q139478718,Alan B. Eisner,NaN,1.0,True,True
717,OL24085A,Alan M. Rugman,1,Q16150461,Alan M. Rugman,NaN,1.0,True,True
1565,OL7084372A,Alexander Brem,1,Q43196955,Alexander Brem,German researcher and professor,1.0,True,True
188,OL383301A,Allan A. Glatthorn,1,Q52008212,Allan A. Glatthorn,1924-,1.0,True,True
1539,OL385663A,Allan Afuah,1,Q139435257,Allan Afuah,NaN,1.0,True,True


#### 18.34.14 — Book-Context Evidence for Secondary Author Resolution

Calibration demonstrates that author-name agreement alone is insufficient for reliable identity resolution.

Among the unresolved candidates, several rank-first human entities have exact normalized names but clearly incompatible descriptions, including individuals from unrelated professional domains.

Secondary author verification therefore incorporates the bibliographic context already present in LeadWise.

Each Open Library author identifier is linked back to the books with which it appears in the project dataset. This creates an author-context table containing:

- Open Library author identifier;
- source author name;
- LeadWise book identifier;
- book title;
- publication year where available;
- subjects where available;
- number of LeadWise books associated with the source author identifier.

This context is used as supporting evidence for subsequent candidate validation. It does not itself establish identity, and no geographic metadata is assigned at this stage.

In [43]:
# ---------------------------------------------------------
# 18.34.14 Build author → LeadWise book context
# Self-contained version
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Load required datasets
# ---------------------------------------------------------

books_master_path = (
    data_root
    / "final"
    / "books_master.csv"
)

author_bridge_path = (
    data_root
    / "processed"
    / "openlibrary_author_identity_bridge.csv"
)


books_master = pd.read_csv(
    books_master_path
)

author_identity_bridge = pd.read_csv(
    author_bridge_path
)


print(
    "SOURCE DATASETS"
)
print("=" * 70)

print(
    "Books master:",
    books_master.shape
)

print(
    "Author identity bridge:",
    author_identity_bridge.shape
)

print(
    "\nAUTHOR IDENTITY BRIDGE COLUMNS"
)
print("=" * 70)

print(
    author_identity_bridge.columns.tolist()
)


# ---------------------------------------------------------
# Required-column validation
# ---------------------------------------------------------

required_master_columns = {
    "book_id",
    "canonical_title",
    "first_publish_year",
    "publication_year_observed",
    "subjects",
}

required_bridge_columns = {
    "book_id",
    "openlibrary_author_key",
    "author_name",
}


missing_master = (
    required_master_columns
    - set(books_master.columns)
)

missing_bridge = (
    required_bridge_columns
    - set(author_identity_bridge.columns)
)


assert not missing_master, (
    f"Missing books_master columns: "
    f"{missing_master}"
)

assert not missing_bridge, (
    f"Missing author bridge columns: "
    f"{missing_bridge}"
)


# ---------------------------------------------------------
# Prepare master book context
# ---------------------------------------------------------

book_context = (
    books_master[
        [
            "book_id",
            "canonical_title",
            "first_publish_year",
            "publication_year_observed",
            "subjects",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Merge author relationships with book metadata
# ---------------------------------------------------------

author_book_context = (
    author_identity_bridge
    .merge(
        book_context,
        on="book_id",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Count LeadWise books associated with each OL author ID
# ---------------------------------------------------------

author_book_counts = (
    author_book_context
    .groupby(
        "openlibrary_author_key"
    )["book_id"]
    .nunique()
    .rename(
        "leadwise_book_count"
    )
    .reset_index()
)


author_book_context = (
    author_book_context
    .merge(
        author_book_counts,
        on="openlibrary_author_key",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Basic validation
# ---------------------------------------------------------

print(
    "\nAUTHOR-BOOK CONTEXT"
)
print("=" * 70)

print(
    "Relationships:",
    len(author_book_context)
)

print(
    "Unique Open Library authors:",
    author_book_context[
        "openlibrary_author_key"
    ].nunique()
)

print(
    "Unique LeadWise books:",
    author_book_context[
        "book_id"
    ].nunique()
)

print(
    "Authors associated with >1 "
    "LeadWise book:",
    (
        author_book_counts[
            "leadwise_book_count"
        ] > 1
    ).sum()
)

print(
    "Maximum books for one "
    "Open Library author ID:",
    author_book_counts[
        "leadwise_book_count"
    ].max()
)


# ---------------------------------------------------------
# Inspect suspicious perfect-name matches
# ---------------------------------------------------------

suspicious_names = [
    "Andrew Greasley",
    "Angela Baron",
    "Ann Davis",
    "Arthur Jensen",
    "Barry Berman",
    "Bill Houston",
    "Brian O'Neil",
]


suspicious_context = (
    author_book_context[
        author_book_context[
            "author_name"
        ].isin(
            suspicious_names
        )
    ]
    [
        [
            "openlibrary_author_key",
            "author_name",
            "book_id",
            "canonical_title",
            "first_publish_year",
            "publication_year_observed",
            "subjects",
            "leadwise_book_count",
        ]
    ]
    .sort_values(
        [
            "author_name",
            "canonical_title",
        ]
    )
)


print(
    "\nBOOK CONTEXT FOR "
    "SUSPICIOUS NAME MATCHES"
)
print("=" * 70)

display(
    suspicious_context
)


# ---------------------------------------------------------
# Authors represented by multiple books
# ---------------------------------------------------------

multi_book_authors = (
    author_book_context[
        author_book_context[
            "leadwise_book_count"
        ] > 1
    ]
    .sort_values(
        [
            "leadwise_book_count",
            "author_name",
            "canonical_title",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
)


print(
    "\nMULTI-BOOK AUTHOR SAMPLE"
)
print("=" * 70)

display(
    multi_book_authors[
        [
            "openlibrary_author_key",
            "author_name",
            "canonical_title",
            "leadwise_book_count",
        ]
    ]
    .head(40)
)


# ---------------------------------------------------------
# Save reusable author-book context
# ---------------------------------------------------------

author_context_path = (
    data_root
    / "processed"
    / "openlibrary_author_book_context.csv"
)


author_book_context.to_csv(
    author_context_path,
    index=False,
)


print(
    "\nSaved:",
    author_context_path
)

SOURCE DATASETS
Books master: (2067, 18)
Author identity bridge: (1468, 5)

AUTHOR IDENTITY BRIDGE COLUMNS
['book_id', 'canonical_title', 'author_position', 'openlibrary_author_key', 'author_name']

AUTHOR-BOOK CONTEXT
Relationships: 1468
Unique Open Library authors: 1237
Unique LeadWise books: 942
Authors associated with >1 LeadWise book: 143
Maximum books for one Open Library author ID: 8


KeyError: "['canonical_title'] not in index"

#### 18.34.14A — Author-Book Context Merge Correction

The Open Library author identity bridge already contains the canonical LeadWise book title.

During the initial author-context merge, the canonical title was also selected from the master book table, causing pandas to create duplicate suffixed title columns.

The context table is rebuilt using the canonical title already present in the author identity bridge while adding only the supplementary publication and subject metadata from the master catalog.

This preserves one canonical title field per author-book relationship.

In [44]:
# ---------------------------------------------------------
# 18.34.14A Correct author → LeadWise book context
# ---------------------------------------------------------

# The bridge already contains canonical_title,
# so do not merge that field again from books_master.

book_context = (
    books_master[
        [
            "book_id",
            "first_publish_year",
            "publication_year_observed",
            "subjects",
        ]
    ]
    .copy()
)


author_book_context = (
    author_identity_bridge
    .merge(
        book_context,
        on="book_id",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Count books associated with each OL author ID
# ---------------------------------------------------------

author_book_counts = (
    author_book_context
    .groupby(
        "openlibrary_author_key"
    )["book_id"]
    .nunique()
    .rename(
        "leadwise_book_count"
    )
    .reset_index()
)


author_book_context = (
    author_book_context
    .merge(
        author_book_counts,
        on="openlibrary_author_key",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Confirm corrected schema
# ---------------------------------------------------------

print(
    "CORRECTED AUTHOR-BOOK CONTEXT"
)
print("=" * 70)

print(
    "Relationships:",
    len(author_book_context)
)

print(
    "Unique Open Library authors:",
    author_book_context[
        "openlibrary_author_key"
    ].nunique()
)

print(
    "Unique LeadWise books:",
    author_book_context[
        "book_id"
    ].nunique()
)

print(
    "Authors associated with >1 "
    "LeadWise book:",
    (
        author_book_counts[
            "leadwise_book_count"
        ] > 1
    ).sum()
)

print(
    "Maximum books for one "
    "Open Library author ID:",
    author_book_counts[
        "leadwise_book_count"
    ].max()
)

print(
    "\nColumns:"
)

print(
    author_book_context.columns.tolist()
)


# ---------------------------------------------------------
# Suspicious perfect-name candidates
# ---------------------------------------------------------

suspicious_names = [
    "Andrew Greasley",
    "Angela Baron",
    "Ann Davis",
    "Arthur Jensen",
    "Barry Berman",
    "Bill Houston",
    "Brian O'Neil",
]


suspicious_context = (
    author_book_context[
        author_book_context[
            "author_name"
        ].isin(
            suspicious_names
        )
    ]
    [
        [
            "openlibrary_author_key",
            "author_name",
            "book_id",
            "canonical_title",
            "first_publish_year",
            "publication_year_observed",
            "subjects",
            "leadwise_book_count",
        ]
    ]
    .sort_values(
        [
            "author_name",
            "canonical_title",
        ]
    )
)


print(
    "\nBOOK CONTEXT FOR SUSPICIOUS NAME MATCHES"
)
print("=" * 70)

display(
    suspicious_context
)


# ---------------------------------------------------------
# Multi-book authors
# ---------------------------------------------------------

multi_book_authors = (
    author_book_context[
        author_book_context[
            "leadwise_book_count"
        ] > 1
    ]
    .sort_values(
        [
            "leadwise_book_count",
            "author_name",
            "canonical_title",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
)


print(
    "\nMULTI-BOOK AUTHOR SAMPLE"
)
print("=" * 70)

display(
    multi_book_authors[
        [
            "openlibrary_author_key",
            "author_name",
            "canonical_title",
            "leadwise_book_count",
        ]
    ]
    .head(40)
)


# ---------------------------------------------------------
# Save corrected context
# ---------------------------------------------------------

author_context_path = (
    data_root
    / "processed"
    / "openlibrary_author_book_context.csv"
)


author_book_context.to_csv(
    author_context_path,
    index=False,
)


print(
    "\nSaved:",
    author_context_path
)

CORRECTED AUTHOR-BOOK CONTEXT
Relationships: 1468
Unique Open Library authors: 1237
Unique LeadWise books: 942
Authors associated with >1 LeadWise book: 143
Maximum books for one Open Library author ID: 8

Columns:
['book_id', 'canonical_title', 'author_position', 'openlibrary_author_key', 'author_name', 'first_publish_year', 'publication_year_observed', 'subjects', 'leadwise_book_count']

BOOK CONTEXT FOR SUSPICIOUS NAME MATCHES


,openlibrary_author_key,author_name,book_id,canonical_title,first_publish_year,publication_year_observed,subjects,leadwise_book_count
715,OL392324A,Andrew Greasley,BOOK00482,Operations management,2005.0,NaN,"['Examinations', 'Production management', 'Stu...",1
861,OL2824602A,Angela Baron,BOOK00573,Managing performance,2005.0,NaN,"['Management', 'Performance', 'Management & ma...",1
973,OL739572A,Ann Davis,BOOK00653,Human Resource Management,2014.0,NaN,"['Personnel management', 'Business and Managem...",1
1285,OL1879979A,Arthur Jensen,BOOK00829,Interpersonal communication,1988.0,NaN,"['Interpersonal communication', 'Psychologie',...",1
448,OL398560A,Barry Berman,BOOK00320,Retail management,1979.0,NaN,"['Management', 'Retail trade', 'Detailhandel',...",1
1135,OL7791069A,Bill Houston,BOOK00747,Business Strategy,2003.0,NaN,[],1
1085,OL237651A,Brian O'Neil,BOOK00721,Acting as a business,1993.0,NaN,"['Acting', 'Vocational guidance', 'Acting, voc...",1



MULTI-BOOK AUTHOR SAMPLE


,openlibrary_author_key,author_name,canonical_title,leadwise_book_count
943,OL238534A,Gary Dessler,Fundamentals of Human Resource Management,8
407,OL238534A,Gary Dessler,Human Resource Management,8
917,OL238534A,Gary Dessler,Human Resource Management,8
960,OL238534A,Gary Dessler,"Human Resource Management, 16th edition",8
379,OL238534A,Gary Dessler,Human Resource management,8
923,OL238534A,Gary Dessler,Human resource management,8
927,OL238534A,Gary Dessler,Human resource management,8
970,OL238534A,Gary Dessler,Human resource management,8
716,OL2648619A,Jay Heizer,Operations Management,7
758,OL2648619A,Jay Heizer,Operations Management,7



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_author_book_context.csv


#### 18.34.15 — Wikidata Candidate Occupation Evidence Audit

Book-context analysis demonstrates that exact author-name agreement can still produce incorrect Wikidata identities.

To strengthen secondary identity resolution, Wikidata occupation identifiers are converted into human-readable occupation labels.

Occupation evidence provides an additional contextual signal that can be compared with the subject matter and bibliographic context of each LeadWise book.

Examples of potentially compatible evidence may include:

- author or writer;
- academic or professor;
- economist;
- psychologist;
- management consultant;
- business theorist;
- entrepreneur;
- executive or businessperson.

Conversely, an occupation that is clearly unrelated to the associated book context may provide evidence against a candidate identity.

Occupation evidence is treated as supporting evidence only. It is not used independently to establish or reject an author identity.

In [45]:
# ---------------------------------------------------------
# 18.34.15 Wikidata occupation evidence audit
# ---------------------------------------------------------

import ast
import time
import pandas as pd


metadata_path = (
    data_root
    / "processed"
    / "wikidata_author_candidate_metadata.csv"
)


candidate_metadata = pd.read_csv(
    metadata_path
)


# ---------------------------------------------------------
# Parse occupation QID lists
# ---------------------------------------------------------

def parse_qid_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return []


candidate_metadata[
    "occupation_qids_parsed"
] = (
    candidate_metadata[
        "occupation_qids"
    ]
    .apply(
        parse_qid_list
    )
)


# ---------------------------------------------------------
# Collect unique occupation QIDs
# ---------------------------------------------------------

occupation_qids = sorted(
    {
        qid
        for values
        in candidate_metadata[
            "occupation_qids_parsed"
        ]
        for qid in values
        if isinstance(qid, str)
        and qid.startswith("Q")
    }
)


print(
    "OCCUPATION IDENTIFIER AUDIT"
)
print("=" * 70)

print(
    "Candidate entities:",
    len(candidate_metadata)
)

print(
    "Candidates with occupation:",
    (
        candidate_metadata[
            "occupation_qids_parsed"
        ]
        .apply(len)
        > 0
    ).sum()
)

print(
    "Unique occupation QIDs:",
    len(occupation_qids)
)


# ---------------------------------------------------------
# Resolve occupation QIDs
# Rate-limit-safe
# ---------------------------------------------------------

occupation_rows = []

BATCH_SIZE = 25
MAX_RETRIES = 5


for batch_number, start in enumerate(
    range(
        0,
        len(occupation_qids),
        BATCH_SIZE,
    ),
    start=1,
):

    batch = occupation_qids[
        start:
        start + BATCH_SIZE
    ]


    params = {
        "action": "wbgetentities",
        "ids": "|".join(batch),
        "props": "labels|descriptions",
        "languages": "en",
        "format": "json",
    }


    success = False


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        response = session.get(
            wikidata_api,
            params=params,
            timeout=60,
        )


        if response.status_code == 429:

            retry_after = (
                response.headers.get(
                    "Retry-After"
                )
            )

            try:

                wait_seconds = (
                    float(retry_after)
                    if retry_after
                    else 15 * attempt
                )

            except ValueError:

                wait_seconds = (
                    15 * attempt
                )


            print(
                f"429 | batch "
                f"{batch_number} | "
                f"attempt {attempt} | "
                f"waiting "
                f"{wait_seconds:.1f}s"
            )

            time.sleep(
                wait_seconds
            )

            continue


        response.raise_for_status()

        entities = (
            response.json()
            .get(
                "entities",
                {}
            )
        )


        for qid in batch:

            entity = entities.get(
                qid,
                {}
            )

            label = (
                entity
                .get(
                    "labels",
                    {}
                )
                .get(
                    "en",
                    {}
                )
                .get(
                    "value"
                )
            )

            description = (
                entity
                .get(
                    "descriptions",
                    {}
                )
                .get(
                    "en",
                    {}
                )
                .get(
                    "value"
                )
            )


            occupation_rows.append(
                {
                    "occupation_qid":
                        qid,

                    "occupation_label":
                        label,

                    "occupation_description":
                        description,
                }
            )


        success = True
        break


    if not success:

        print(
            "FAILED BATCH:",
            batch
        )


    time.sleep(2)


occupation_lookup = pd.DataFrame(
    occupation_rows
)


print(
    "\nOCCUPATION LOOKUP"
)
print("=" * 70)

print(
    "Resolved occupation QIDs:",
    occupation_lookup[
        "occupation_qid"
    ].nunique()
)

print(
    "Missing occupation QIDs:",
    len(
        set(occupation_qids)
        -
        set(
            occupation_lookup[
                "occupation_qid"
            ]
        )
    )
)


display(
    occupation_lookup.head(40)
)


# ---------------------------------------------------------
# Save lookup
# ---------------------------------------------------------

occupation_path = (
    data_root
    / "processed"
    / "wikidata_occupation_lookup.csv"
)


occupation_lookup.to_csv(
    occupation_path,
    index=False,
)


print(
    "\nSaved:",
    occupation_path
)

OCCUPATION IDENTIFIER AUDIT
Candidate entities: 1554
Candidates with occupation: 1134
Unique occupation QIDs: 416
429 | batch 11 | attempt 1 | waiting 30.0s

OCCUPATION LOOKUP
Resolved occupation QIDs: 416
Missing occupation QIDs: 0


,occupation_qid,occupation_label,occupation_description
0,Q100983844,garden writer,writers about gardening
1,Q101444631,English teacher,person who teaches the English language and En...
2,Q1028181,painter,artist who practices painting
3,Q10297252,crime fiction writer,writer writing fiction about crime
4,Q10349745,racing automobile driver,occupation driving an automobile in competition
5,Q10355417,advertising person,person concerned with the creation of advertis...
6,Q104828277,biology teacher,teacher of biology at any level
7,Q10497074,military flight engineer,military aircraft crew member
8,Q1051798,statutory auditor,profession
9,Q1056391,head teacher,most senior teacher at a school



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_occupation_lookup.csv


#### 18.34.16 — Occupation Compatibility Audit of Secondary Author Candidates

All Wikidata occupation identifiers observed among the candidate entities have been successfully resolved to human-readable occupation labels.

These occupation labels are now joined to the strong secondary author candidates identified during identity calibration.

The purpose is to examine whether candidate professional context is consistent with the bibliographic context of the corresponding LeadWise author record.

This stage distinguishes three forms of evidence:

- **potentially compatible evidence** — occupations commonly associated with authorship, academia, management, business, leadership, psychology, economics, consulting, education, or related professional fields;
- **potentially incompatible evidence** — occupations from clearly unrelated domains that may indicate a same-name false match;
- **insufficient or ambiguous evidence** — candidates without sufficiently informative occupation metadata.

These classifications are diagnostic screening signals rather than final identity decisions. No candidate is automatically verified or rejected solely because of occupation.

In [46]:
# ---------------------------------------------------------
# 18.34.16 Occupation compatibility audit
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

validation = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_candidate_validation.csv"
)

occupation_lookup = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_occupation_lookup.csv"
)

author_context = pd.read_csv(
    data_root
    / "processed"
    / "openlibrary_author_book_context.csv"
)


# ---------------------------------------------------------
# Parse occupation QIDs
# ---------------------------------------------------------

def parse_list_value(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):
            return parsed

    except (ValueError, SyntaxError):
        pass

    return []


validation["occupation_qids_parsed"] = (
    validation["occupation_qids"]
    .apply(parse_list_value)
)


# ---------------------------------------------------------
# Build QID → occupation label mapping
# ---------------------------------------------------------

occupation_map = (
    occupation_lookup
    .dropna(
        subset=[
            "occupation_qid",
            "occupation_label",
        ]
    )
    .set_index(
        "occupation_qid"
    )["occupation_label"]
    .to_dict()
)


def resolve_occupations(qids):

    labels = [
        occupation_map[qid]
        for qid in qids
        if qid in occupation_map
    ]

    return list(
        dict.fromkeys(labels)
    )


validation["occupation_labels"] = (
    validation[
        "occupation_qids_parsed"
    ]
    .apply(resolve_occupations)
)


# ---------------------------------------------------------
# Reconstruct strong secondary candidate group
# ---------------------------------------------------------

strong_candidates = (
    validation[
        (~validation["matched_author_identifier"])
        &
        (validation["is_human"])
        &
        (validation["candidate_rank"] == 1)
        &
        (validation["name_similarity"] >= 0.90)
    ]
    .copy()
)


# ---------------------------------------------------------
# Add compact LeadWise book context
# ---------------------------------------------------------

author_context_summary = (
    author_context
    .groupby(
        "openlibrary_author_key"
    )
    .agg(
        leadwise_titles=(
            "canonical_title",
            lambda x:
                list(
                    dict.fromkeys(
                        x.dropna()
                        .astype(str)
                    )
                )[:8]
        ),

        leadwise_book_count=(
            "book_id",
            "nunique",
        ),
    )
    .reset_index()
)


strong_candidates = (
    strong_candidates
    .merge(
        author_context_summary,
        on="openlibrary_author_key",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Diagnostic keyword groups
#
# These are screening categories only.
# They are NOT identity-verification rules.
# ---------------------------------------------------------

compatible_keywords = {
    "author",
    "writer",
    "academic",
    "professor",
    "researcher",
    "scientist",
    "psychologist",
    "sociologist",
    "economist",
    "consultant",
    "entrepreneur",
    "business",
    "management",
    "manager",
    "executive",
    "theorist",
    "teacher",
    "educator",
    "coach",
    "accounting",
    "marketing",
    "leadership",
    "organizational",
    "organisation",
    "human resources",
}


potentially_unrelated_keywords = {
    "footballer",
    "football player",
    "cricketer",
    "baseball player",
    "tennis player",
    "basketball player",
    "athletics competitor",
    "boxer",
    "racing automobile driver",
    "actor",
    "actress",
    "model",
}


def occupation_screen(labels):

    if not labels:
        return "insufficient"

    text = " | ".join(
        labels
    ).lower()

    compatible = any(
        keyword in text
        for keyword
        in compatible_keywords
    )

    unrelated = any(
        keyword in text
        for keyword
        in potentially_unrelated_keywords
    )

    # Mixed evidence remains ambiguous.
    if compatible and unrelated:
        return "ambiguous"

    if compatible:
        return "potentially_compatible"

    if unrelated:
        return "potentially_incompatible"

    return "ambiguous"


strong_candidates[
    "occupation_screen"
] = (
    strong_candidates[
        "occupation_labels"
    ]
    .apply(
        occupation_screen
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "SECONDARY CANDIDATE OCCUPATION AUDIT"
)
print("=" * 70)

print(
    "Strong secondary candidates:",
    len(strong_candidates)
)

print(
    "Unique source authors:",
    strong_candidates[
        "openlibrary_author_key"
    ].nunique()
)


screen_summary = (
    strong_candidates[
        "occupation_screen"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "occupation_screen"
    )
    .reset_index(
        name="candidate_count"
    )
)


print(
    "\nOCCUPATION SCREEN SUMMARY"
)
print("=" * 70)

display(
    screen_summary
)


# ---------------------------------------------------------
# Potentially incompatible candidates
# ---------------------------------------------------------

potentially_incompatible = (
    strong_candidates[
        strong_candidates[
            "occupation_screen"
        ]
        == "potentially_incompatible"
    ]
    .copy()
)


print(
    "\nPOTENTIALLY INCOMPATIBLE CANDIDATES"
)
print("=" * 70)

print(
    "Candidate rows:",
    len(
        potentially_incompatible
    )
)


display(
    potentially_incompatible[
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "occupation_labels",
            "leadwise_titles",
            "name_similarity",
        ]
    ]
    .head(40)
)


# ---------------------------------------------------------
# Potentially compatible candidates
# ---------------------------------------------------------

potentially_compatible = (
    strong_candidates[
        strong_candidates[
            "occupation_screen"
        ]
        == "potentially_compatible"
    ]
    .copy()
)


print(
    "\nPOTENTIALLY COMPATIBLE SAMPLE"
)
print("=" * 70)

display(
    potentially_compatible[
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "occupation_labels",
            "leadwise_titles",
            "name_similarity",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save diagnostic audit
# ---------------------------------------------------------

audit_path = (
    data_root
    / "processed"
    / "wikidata_author_secondary_occupation_audit.csv"
)


strong_candidates.to_csv(
    audit_path,
    index=False,
)


print(
    "\nSaved:",
    audit_path
)

KeyError: 'is_human'

In [47]:
# ---------------------------------------------------------
# 18.34.16 Occupation compatibility audit
# Self-contained corrected version
# ---------------------------------------------------------

import ast
import re
import unicodedata
from difflib import SequenceMatcher

import pandas as pd


# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

validation = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_candidate_validation.csv"
)

occupation_lookup = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_occupation_lookup.csv"
)

author_context = pd.read_csv(
    data_root
    / "processed"
    / "openlibrary_author_book_context.csv"
)


print(
    "SOURCE DATASETS"
)
print("=" * 70)

print(
    "Candidate validation:",
    validation.shape
)

print(
    "Occupation lookup:",
    occupation_lookup.shape
)

print(
    "Author-book context:",
    author_context.shape
)


# ---------------------------------------------------------
# Generic parser for list-like CSV fields
# ---------------------------------------------------------

def parse_list_value(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return []


# ---------------------------------------------------------
# Reconstruct is_human
#
# Wikidata Q5 = human
# ---------------------------------------------------------

validation[
    "instance_of_parsed"
] = (
    validation[
        "instance_of_qids"
    ]
    .apply(
        parse_list_value
    )
)


validation[
    "is_human"
] = (
    validation[
        "instance_of_parsed"
    ]
    .apply(
        lambda values:
            "Q5" in values
    )
)


# ---------------------------------------------------------
# Reconstruct normalized names and similarity
# ---------------------------------------------------------

def normalize_author_name(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = unicodedata.normalize(
        "NFKD",
        value,
    )

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(
            char
        )
    )

    value = value.lower()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


validation[
    "source_name_normalized"
] = (
    validation[
        "author_name"
    ]
    .apply(
        normalize_author_name
    )
)


validation[
    "candidate_name_normalized"
] = (
    validation[
        "wikidata_label"
    ]
    .apply(
        normalize_author_name
    )
)


def calculate_name_similarity(
    source,
    candidate,
):

    if not source or not candidate:
        return 0.0

    return SequenceMatcher(
        None,
        source,
        candidate,
    ).ratio()


validation[
    "name_similarity"
] = (
    validation
    .apply(
        lambda row:
            calculate_name_similarity(
                row[
                    "source_name_normalized"
                ],
                row[
                    "candidate_name_normalized"
                ],
            ),
        axis=1,
    )
)


validation[
    "exact_normalized_name"
] = (
    validation[
        "source_name_normalized"
    ]
    ==
    validation[
        "candidate_name_normalized"
    ]
)


# ---------------------------------------------------------
# Ensure identifier-match flag is Boolean
# ---------------------------------------------------------

def to_boolean(value):

    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return (
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
        }
    )


validation[
    "matched_author_identifier"
] = (
    validation[
        "matched_author_identifier"
    ]
    .apply(
        to_boolean
    )
)


# ---------------------------------------------------------
# Parse occupation QIDs
# ---------------------------------------------------------

validation[
    "occupation_qids_parsed"
] = (
    validation[
        "occupation_qids"
    ]
    .apply(
        parse_list_value
    )
)


# ---------------------------------------------------------
# QID → occupation label dictionary
# ---------------------------------------------------------

occupation_map = (
    occupation_lookup
    .dropna(
        subset=[
            "occupation_qid",
            "occupation_label",
        ]
    )
    .set_index(
        "occupation_qid"
    )[
        "occupation_label"
    ]
    .to_dict()
)


def resolve_occupations(qids):

    labels = [
        occupation_map[qid]
        for qid in qids
        if qid in occupation_map
    ]

    return list(
        dict.fromkeys(
            labels
        )
    )


validation[
    "occupation_labels"
] = (
    validation[
        "occupation_qids_parsed"
    ]
    .apply(
        resolve_occupations
    )
)


# ---------------------------------------------------------
# Reconstruct strong secondary candidate group
# ---------------------------------------------------------

strong_candidates = (
    validation[
        (
            ~validation[
                "matched_author_identifier"
            ]
        )
        &
        (
            validation[
                "is_human"
            ]
        )
        &
        (
            validation[
                "candidate_rank"
            ] == 1
        )
        &
        (
            validation[
                "name_similarity"
            ] >= 0.90
        )
    ]
    .copy()
)


# ---------------------------------------------------------
# Add compact LeadWise book context
# ---------------------------------------------------------

author_context_summary = (
    author_context
    .groupby(
        "openlibrary_author_key"
    )
    .agg(
        leadwise_titles=(
            "canonical_title",
            lambda x:
                list(
                    dict.fromkeys(
                        x
                        .dropna()
                        .astype(str)
                    )
                )[:8]
        ),

        leadwise_book_count=(
            "book_id",
            "nunique",
        ),
    )
    .reset_index()
)


strong_candidates = (
    strong_candidates
    .merge(
        author_context_summary,
        on="openlibrary_author_key",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Diagnostic occupation categories
#
# These are screening signals,
# NOT final identity rules.
# ---------------------------------------------------------

compatible_keywords = {
    "author",
    "writer",
    "academic",
    "professor",
    "researcher",
    "scientist",
    "psychologist",
    "sociologist",
    "economist",
    "consultant",
    "entrepreneur",
    "business",
    "management",
    "manager",
    "executive",
    "theorist",
    "teacher",
    "educator",
    "coach",
    "accounting",
    "marketing",
    "leadership",
    "organizational",
    "organisation",
    "human resources",
}


potentially_unrelated_keywords = {
    "footballer",
    "football player",
    "cricketer",
    "baseball player",
    "tennis player",
    "basketball player",
    "athletics competitor",
    "boxer",
    "racing automobile driver",
    "actor",
    "actress",
    "model",
}


def occupation_screen(labels):

    if not labels:
        return "insufficient"

    text = " | ".join(
        labels
    ).lower()


    compatible = any(
        keyword in text
        for keyword
        in compatible_keywords
    )


    unrelated = any(
        keyword in text
        for keyword
        in potentially_unrelated_keywords
    )


    if compatible and unrelated:
        return "ambiguous"

    if compatible:
        return "potentially_compatible"

    if unrelated:
        return "potentially_incompatible"

    return "ambiguous"


strong_candidates[
    "occupation_screen"
] = (
    strong_candidates[
        "occupation_labels"
    ]
    .apply(
        occupation_screen
    )
)


# ---------------------------------------------------------
# Validation — important
# ---------------------------------------------------------

print(
    "\nSECONDARY CANDIDATE RECONSTRUCTION"
)
print("=" * 70)

print(
    "Strong secondary candidates:",
    len(
        strong_candidates
    )
)

print(
    "Unique source authors:",
    strong_candidates[
        "openlibrary_author_key"
    ].nunique()
)

print(
    "Human candidates:",
    strong_candidates[
        "is_human"
    ].sum()
)

print(
    "Exact normalized names:",
    strong_candidates[
        "exact_normalized_name"
    ].sum()
)


# ---------------------------------------------------------
# Occupation screen summary
# ---------------------------------------------------------

screen_summary = (
    strong_candidates[
        "occupation_screen"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "occupation_screen"
    )
    .reset_index(
        name="candidate_count"
    )
)


print(
    "\nOCCUPATION SCREEN SUMMARY"
)
print("=" * 70)

display(
    screen_summary
)


# ---------------------------------------------------------
# Potentially incompatible
# ---------------------------------------------------------

potentially_incompatible = (
    strong_candidates[
        strong_candidates[
            "occupation_screen"
        ]
        == "potentially_incompatible"
    ]
    .copy()
)


print(
    "\nPOTENTIALLY INCOMPATIBLE CANDIDATES"
)
print("=" * 70)

print(
    "Candidate rows:",
    len(
        potentially_incompatible
    )
)


display(
    potentially_incompatible[
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "occupation_labels",
            "leadwise_titles",
            "name_similarity",
        ]
    ]
    .head(40)
)


# ---------------------------------------------------------
# Potentially compatible
# ---------------------------------------------------------

potentially_compatible = (
    strong_candidates[
        strong_candidates[
            "occupation_screen"
        ]
        == "potentially_compatible"
    ]
    .copy()
)


print(
    "\nPOTENTIALLY COMPATIBLE SAMPLE"
)
print("=" * 70)

display(
    potentially_compatible[
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "occupation_labels",
            "leadwise_titles",
            "name_similarity",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save enriched audit
# ---------------------------------------------------------

audit_path = (
    data_root
    / "processed"
    / "wikidata_author_secondary_occupation_audit.csv"
)


strong_candidates.to_csv(
    audit_path,
    index=False,
)


print(
    "\nSaved:",
    audit_path
)

SOURCE DATASETS
Candidate validation: (1633, 17)
Occupation lookup: (416, 3)
Author-book context: (1468, 9)

SECONDARY CANDIDATE RECONSTRUCTION
Strong secondary candidates: 595
Unique source authors: 595
Human candidates: 595
Exact normalized names: 537

OCCUPATION SCREEN SUMMARY


,occupation_screen,candidate_count
0,potentially_compatible,401
1,ambiguous,91
2,insufficient,71
3,potentially_incompatible,32



POTENTIALLY INCOMPATIBLE CANDIDATES
Candidate rows: 32


,openlibrary_author_key,author_name,wikidata_id,wikidata_label,wikidata_description,occupation_labels,leadwise_titles,name_similarity
14,OL1478161A,Terry Anderson,Q7704060,Terry Anderson,English association footballer (1944-1980),[association football player],[Transforming Leadership],1.0
27,OL392324A,Andrew Greasley,Q4757168,Andrew Greasley,English cricketer (born 1960),[cricketer],[Operations management],1.0
51,OL4929480A,David Logan,Q509794,David Logan,American basketball player,[basketball player],[Tribal leadership],1.0
87,OL2877334A,Louis Carter,Q6686867,Louis Carter,American football player,[American football player],[Best practices in leadership development and ...,1.0
91,OL2434686A,Jack Gordon,Q117100642,Jack Gordon,English footballer (1911–?),[association football player],[The Pfeiffer book of successful leadership de...,1.0
113,OL410958A,Peter Hawkins,Q7174587,Peter Hawkins,British actor (1924–2006),"[voice actor, television actor, art collector,...",[Leadership Team Coaching],1.0
118,OL577733A,Pat Wellington,Q7144106,Pat Wellington,Australian rules footballer,[Australian rules football player],[Effective Team Leadership for Engineers],1.0
123,OL8937874A,David Silverman,Q919608,David Silverman,American animator and director,"[film director, animator, actor, film producer]",[Team of Teams],1.0
133,OL5480270A,Terry Anderson,Q7704060,Terry Anderson,English association footballer (1944-1980),[association football player],[Transforming Leadership],1.0
145,OL5310633A,Michael Rucker,Q107736835,Michael Rucker,American baseball player (born 1994),[baseball player],[Servant Leadership],1.0



POTENTIALLY COMPATIBLE SAMPLE


,openlibrary_author_key,author_name,wikidata_id,wikidata_label,wikidata_description,occupation_labels,leadwise_titles,name_similarity
0,OL400156A,Gary A. Yukl,Q130733922,Gary A. Yukl,"psychologist, business manager, university tea...","[psychologist, business manager, university te...","[Leadership in Organizations, Charismatic and ...",1.000000
1,OL333669A,Bernard M. Bass,Q52162437,Bernard M. Bass,American psychologist and leadership pioneer,"[psychologist, university teacher]",[Leadership and performance beyond expectation...,1.000000
2,OL7442788A,James Comey,Q167607,James Comey,American lawyer and 7th director of the Federa...,"[lawyer, university teacher, politician, jurist]",[A Higher Loyalty],1.000000
3,OL8118723A,James B. Comey,Q167607,James Comey,American lawyer and 7th director of the Federa...,"[lawyer, university teacher, politician, jurist]",[A Higher Loyalty],0.916667
4,OL2762041A,Erich Joachimsthaler,Q5388045,Erich Joachimsthaler,American businessman,"[economist, business consultant]",[Brand leadership],1.000000
5,OL772556A,Stephen P. Gordon,Q114569791,Stephen P. Gordon,American academic in educational leadership,"[university teacher, academic, researcher]",[SuperVision and Instructional Leadership],1.000000
6,OL2672226A,Jovita M. Ross-Gordon,Q112517260,Jovita M. Ross-Gordon,academic in adult education,"[university teacher, academic, pedagogue]",[SuperVision and Instructional Leadership],1.000000
7,OL3714825A,Simon Western,Q120188564,Simon Western,British academic,[academic],[Leadership],1.000000
8,OL7524018A,Jocko Willink,Q22935809,Jocko Willink,American author,"[podcaster, martial artist, writer, military c...",[Leadership Strategy and Tactics],1.000000
9,OL241136A,Bruce J. Avolio,Q52152510,Bruce Avolio,American leadership studies scholar,[non-fiction writer],[Improving organizational effectiveness throug...,0.923077



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_author_secondary_occupation_audit.csv


#### 18.34.17 — Conservative Author Identity Evidence Classification

Author identity enrichment combines direct identifiers, name agreement, entity type, candidate rank, occupation evidence, and bibliographic context.

The evidence is retained using explicit confidence tiers rather than treating all Wikidata search results as equivalent.

The classification hierarchy is:

1. **Identifier verified**  
   The Wikidata entity contains an Open Library P648 author identifier that exactly matches the Open Library author record used by LeadWise.

2. **Strong secondary evidence**  
   No direct identifier match is available, but the candidate is the first-ranked Wikidata result, represents a human, has an exact normalized author-name match, and has occupation evidence compatible with the LeadWise bibliographic context.

3. **Probable secondary evidence**  
   The candidate is first-ranked and human, has high but non-exact name similarity, and has compatible occupation evidence.

4. **Conflicting evidence**  
   The candidate has a strong name match but occupation evidence appears inconsistent with the associated LeadWise book context.

5. **Ambiguous evidence**  
   Available evidence is insufficiently discriminating to establish a reliable identity.

6. **Unresolved**  
   No candidate satisfies the minimum secondary matching conditions.

Only direct Open Library–Wikidata identifier matches are described as **verified**. Secondary classifications represent evidence strength and remain distinguishable from identifier verification.

Geographic attributes derived from secondary matches must retain the corresponding identity-confidence classification so that provenance and uncertainty remain visible.

In [48]:
# ---------------------------------------------------------
# 18.34.17 Conservative author identity evidence tiers
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Load enriched secondary audit
# ---------------------------------------------------------

secondary_audit = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_secondary_occupation_audit.csv"
)

candidate_validation = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_candidate_validation.csv"
)


# ---------------------------------------------------------
# Boolean parser
# ---------------------------------------------------------

def parse_bool(value):

    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return (
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
        }
    )


candidate_validation[
    "matched_author_identifier"
] = (
    candidate_validation[
        "matched_author_identifier"
    ]
    .apply(
        parse_bool
    )
)


secondary_audit[
    "exact_normalized_name"
] = (
    secondary_audit[
        "exact_normalized_name"
    ]
    .apply(
        parse_bool
    )
)


secondary_audit[
    "is_human"
] = (
    secondary_audit[
        "is_human"
    ]
    .apply(
        parse_bool
    )
)


# ---------------------------------------------------------
# Direct identifier-verified identities
# ---------------------------------------------------------

identifier_verified = (
    candidate_validation[
        candidate_validation[
            "matched_author_identifier"
        ]
    ]
    .copy()
)


identifier_verified[
    "identity_evidence_tier"
] = (
    "identifier_verified"
)


# ---------------------------------------------------------
# Classify secondary evidence
# ---------------------------------------------------------

def classify_secondary(row):

    if (
        row["candidate_rank"] == 1
        and row["is_human"]
        and row["exact_normalized_name"]
        and row["occupation_screen"]
        == "potentially_compatible"
    ):
        return "strong_secondary"

    if (
        row["candidate_rank"] == 1
        and row["is_human"]
        and row["name_similarity"] >= 0.90
        and row["occupation_screen"]
        == "potentially_compatible"
    ):
        return "probable_secondary"

    if (
        row["occupation_screen"]
        == "potentially_incompatible"
    ):
        return "conflicting"

    return "ambiguous"


secondary_audit[
    "identity_evidence_tier"
] = (
    secondary_audit.apply(
        classify_secondary,
        axis=1,
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "SECONDARY IDENTITY EVIDENCE"
)
print("=" * 70)


secondary_summary = (
    secondary_audit[
        "identity_evidence_tier"
    ]
    .value_counts()
    .rename_axis(
        "identity_evidence_tier"
    )
    .reset_index(
        name="author_count"
    )
)


display(
    secondary_summary
)


print(
    "Identifier-verified authors:",
    identifier_verified[
        "openlibrary_author_key"
    ].nunique()
)


# ---------------------------------------------------------
# Combine accepted evidence candidates
#
# We retain identifier-verified,
# strong secondary and probable secondary
# separately.
# ---------------------------------------------------------

secondary_selected = (
    secondary_audit[
        secondary_audit[
            "identity_evidence_tier"
        ].isin(
            [
                "strong_secondary",
                "probable_secondary",
            ]
        )
    ]
    .copy()
)


print(
    "\nSELECTED SECONDARY IDENTITIES"
)
print("=" * 70)

print(
    "Strong secondary:",
    (
        secondary_selected[
            "identity_evidence_tier"
        ]
        == "strong_secondary"
    ).sum()
)

print(
    "Probable secondary:",
    (
        secondary_selected[
            "identity_evidence_tier"
        ]
        == "probable_secondary"
    ).sum()
)

print(
    "Unique secondary source authors:",
    secondary_selected[
        "openlibrary_author_key"
    ].nunique()
)


# ---------------------------------------------------------
# Inspect non-exact probable cases
# ---------------------------------------------------------

probable_preview = (
    secondary_selected[
        secondary_selected[
            "identity_evidence_tier"
        ]
        == "probable_secondary"
    ]
    [
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "name_similarity",
            "occupation_labels",
            "leadwise_titles",
        ]
    ]
    .sort_values(
        "name_similarity",
        ascending=False,
    )
)


print(
    "\nPROBABLE SECONDARY SAMPLE"
)
print("=" * 70)

display(
    probable_preview.head(30)
)


# ---------------------------------------------------------
# Inspect conflicting cases
# ---------------------------------------------------------

conflicting_preview = (
    secondary_audit[
        secondary_audit[
            "identity_evidence_tier"
        ]
        == "conflicting"
    ]
)


print(
    "\nCONFLICTING EVIDENCE:"
)

print(
    len(
        conflicting_preview
    )
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

tier_path = (
    data_root
    / "processed"
    / "wikidata_author_identity_evidence_tiers.csv"
)


secondary_audit.to_csv(
    tier_path,
    index=False,
)


print(
    "\nSaved:",
    tier_path
)

SECONDARY IDENTITY EVIDENCE


,identity_evidence_tier,author_count
0,strong_secondary,353
1,ambiguous,162
2,probable_secondary,48
3,conflicting,32


Identifier-verified authors: 68

SELECTED SECONDARY IDENTITIES
Strong secondary: 353
Probable secondary: 48
Unique secondary source authors: 401

PROBABLE SECONDARY SAMPLE


,openlibrary_author_key,author_name,wikidata_id,wikidata_label,wikidata_description,name_similarity,occupation_labels,leadwise_titles
64,OL3925630A,Max DePree,Q3302067,Max De Pree,American businessman and writer (1924–2017),0.952381,"['writer', 'business executive', 'entrepreneur']",['Leadership Is an Art']
323,OL2626801A,Richard J. Schonberger,Q112378637,Richard Schonberger,economist,0.950000,['economist'],['Operations management']
36,OL342495A,Theodore S. Rappaport,Q18619463,Theodore Rappaport,American electrical engineering professor,0.947368,"['scientist', 'electrotechnician', 'university...",['Wireless communications']
242,OL5649743A,Melissa A. Schilling,Q55622741,Melissa Schilling,American innovation scholar and professor,0.944444,"['scientist', 'economist']",['Strategic management of technological innova...
102,OL9482201A,Richard D. Heimovics,Q112385927,Richard Heimovics,NaN,0.944444,['economist'],['Executive Leadership in Nonprofit Organizati...
527,OL219702A,William B. Gudykunst,Q20723884,William Gudykunst,sociologist (1947-2005),0.944444,['sociologist'],['Communicating with strangers']
545,OL401772A,Richard E. Boyatzis,Q7324289,Richard Boyatzis,American business theorist,0.941176,"['psychologist', 'sociologist']",['Primal leadership : realizing the power of e...
294,OL218618A,Marcus Gonçalves,Q56507044,Marcus D Goncalves,researcher,0.941176,['researcher'],['Change management']
141,OL399219A,Alfons Trompenaars,Q712041,Fons Trompenaars,Dutch-French anthropologist,0.941176,"['anthropologist', 'sociologist', 'business co...",['Servant-leadership across cultures']
576,OL7356617A,Carolina Machado,Q57018851,Carolina B Machado,researcher,0.941176,['researcher'],['Innovation Management']



CONFLICTING EVIDENCE:
32

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_author_identity_evidence_tiers.csv


#### 18.34.18 — Author Geographic Evidence Preparation

Following conservative author identity resolution, structured geographic evidence can be attached to author records while preserving the confidence and provenance of the underlying identity match.

Two Wikidata geographic properties collected during candidate enrichment are used:

- **country of citizenship (P27)** — documented citizenship or nationality association recorded by Wikidata;
- **place of birth (P19)** — documented birthplace entity recorded by Wikidata.

These fields are kept separate because they represent different geographic concepts.

Neither field is interpreted as the "country of the book," "book origin," or "original publication country."

Geographic information is retained only for author identities classified as:

- identifier verified;
- strong secondary evidence;
- probable secondary evidence.

Ambiguous and conflicting identities are excluded from geographic enrichment.

The identity-evidence tier is retained alongside every geographic attribute so that LeadWise can distinguish directly verified author identities from secondary identity resolutions.

In [49]:
# ---------------------------------------------------------
# 18.34.18 Prepare author geographic evidence
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

candidate_metadata = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_candidate_metadata.csv"
)

candidate_validation = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_candidate_validation.csv"
)

secondary_tiers = pd.read_csv(
    data_root
    / "processed"
    / "wikidata_author_identity_evidence_tiers.csv"
)


# ---------------------------------------------------------
# Boolean parser
# ---------------------------------------------------------

def parse_bool(value):

    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return (
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
        }
    )


candidate_validation[
    "matched_author_identifier"
] = (
    candidate_validation[
        "matched_author_identifier"
    ]
    .apply(
        parse_bool
    )
)


# ---------------------------------------------------------
# Identifier-verified author identities
# ---------------------------------------------------------

identifier_selected = (
    candidate_validation[
        candidate_validation[
            "matched_author_identifier"
        ]
    ]
    [
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
        ]
    ]
    .copy()
)


identifier_selected[
    "identity_evidence_tier"
] = (
    "identifier_verified"
)


# ---------------------------------------------------------
# Selected secondary identities
# ---------------------------------------------------------

secondary_selected = (
    secondary_tiers[
        secondary_tiers[
            "identity_evidence_tier"
        ].isin(
            [
                "strong_secondary",
                "probable_secondary",
            ]
        )
    ]
    [
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "wikidata_label",
            "wikidata_description",
            "identity_evidence_tier",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Combine selected identities
# ---------------------------------------------------------

selected_identities = pd.concat(
    [
        identifier_selected,
        secondary_selected,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# Protect against duplicate source-author selection
# ---------------------------------------------------------

tier_priority = {
    "identifier_verified": 1,
    "strong_secondary": 2,
    "probable_secondary": 3,
}


selected_identities[
    "tier_priority"
] = (
    selected_identities[
        "identity_evidence_tier"
    ]
    .map(
        tier_priority
    )
)


selected_identities = (
    selected_identities
    .sort_values(
        [
            "openlibrary_author_key",
            "tier_priority",
        ]
    )
    .drop_duplicates(
        subset=[
            "openlibrary_author_key"
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Attach structured Wikidata geography
# ---------------------------------------------------------

geography_fields = (
    candidate_metadata[
        [
            "wikidata_id",
            "citizenship_qids",
            "birthplace_qids",
            "birth_dates",
        ]
    ]
    .drop_duplicates(
        subset=[
            "wikidata_id"
        ]
    )
)


author_geography = (
    selected_identities
    .merge(
        geography_fields,
        on="wikidata_id",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Parse QID lists
# ---------------------------------------------------------

def parse_qid_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return []


author_geography[
    "citizenship_qids_parsed"
] = (
    author_geography[
        "citizenship_qids"
    ]
    .apply(
        parse_qid_list
    )
)


author_geography[
    "birthplace_qids_parsed"
] = (
    author_geography[
        "birthplace_qids"
    ]
    .apply(
        parse_qid_list
    )
)


# ---------------------------------------------------------
# Coverage
# ---------------------------------------------------------

author_geography[
    "has_citizenship"
] = (
    author_geography[
        "citizenship_qids_parsed"
    ]
    .apply(len)
    > 0
)


author_geography[
    "has_birthplace"
] = (
    author_geography[
        "birthplace_qids_parsed"
    ]
    .apply(len)
    > 0
)


print(
    "AUTHOR GEOGRAPHIC EVIDENCE"
)
print("=" * 70)

print(
    "Selected author identities:",
    len(author_geography)
)

print(
    "Unique Open Library authors:",
    author_geography[
        "openlibrary_author_key"
    ].nunique()
)

print(
    "Identifier verified:",
    (
        author_geography[
            "identity_evidence_tier"
        ]
        == "identifier_verified"
    ).sum()
)

print(
    "Strong secondary:",
    (
        author_geography[
            "identity_evidence_tier"
        ]
        == "strong_secondary"
    ).sum()
)

print(
    "Probable secondary:",
    (
        author_geography[
            "identity_evidence_tier"
        ]
        == "probable_secondary"
    ).sum()
)

print(
    "With citizenship evidence:",
    author_geography[
        "has_citizenship"
    ].sum()
)

print(
    "With birthplace evidence:",
    author_geography[
        "has_birthplace"
    ].sum()
)


# ---------------------------------------------------------
# Coverage by identity tier
# ---------------------------------------------------------

coverage_by_tier = (
    author_geography
    .groupby(
        "identity_evidence_tier"
    )
    .agg(
        authors=(
            "openlibrary_author_key",
            "nunique",
        ),

        with_citizenship=(
            "has_citizenship",
            "sum",
        ),

        with_birthplace=(
            "has_birthplace",
            "sum",
        ),
    )
    .reset_index()
)


print(
    "\nGEOGRAPHIC COVERAGE BY IDENTITY TIER"
)
print("=" * 70)

display(
    coverage_by_tier
)


# ---------------------------------------------------------
# Preview raw QIDs
# ---------------------------------------------------------

print(
    "\nGEOGRAPHIC EVIDENCE SAMPLE"
)
print("=" * 70)

display(
    author_geography[
        [
            "openlibrary_author_key",
            "author_name",
            "wikidata_id",
            "identity_evidence_tier",
            "citizenship_qids_parsed",
            "birthplace_qids_parsed",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save preparation table
# ---------------------------------------------------------

geography_path = (
    data_root
    / "processed"
    / "wikidata_author_geographic_evidence_raw.csv"
)


author_geography.to_csv(
    geography_path,
    index=False,
)


print(
    "\nSaved:",
    geography_path
)

AUTHOR GEOGRAPHIC EVIDENCE
Selected author identities: 467
Unique Open Library authors: 467
Identifier verified: 68
Strong secondary: 351
Probable secondary: 48
With citizenship evidence: 171
With birthplace evidence: 127

GEOGRAPHIC COVERAGE BY IDENTITY TIER


,identity_evidence_tier,authors,with_citizenship,with_birthplace
0,identifier_verified,68,52,42
1,probable_secondary,48,25,16
2,strong_secondary,351,94,69



GEOGRAPHIC EVIDENCE SAMPLE


,openlibrary_author_key,author_name,wikidata_id,identity_evidence_tier,citizenship_qids_parsed,birthplace_qids_parsed
0,OL1003227A,Bernard Burnes,Q112444995,strong_secondary,[],[]
1,OL10188240A,Janine Williams,Q112574706,strong_secondary,[],[]
2,OL10520311A,Jordi Diaz,Q57413095,strong_secondary,[],[]
3,OL1076211A,R. Edward Freeman,Q3938264,strong_secondary,[Q30],[Q239870]
4,OL1078587A,Michael T. Matteson,Q139482667,strong_secondary,[],[]
5,OL1080210A,Alonzo J. Strickland,Q140330104,strong_secondary,[],[]
6,OL10856242A,Cyril O'Donnell,Q5200774,probable_secondary,[Q30],[Q28260]
7,OL10997841A,Glenn Walter,Q91890750,probable_secondary,[],[]
8,OL1101366A,Mahmoud Ezzamel,Q16728891,strong_secondary,[],[]
9,OL1121414A,Young Yun Kim,Q122582777,strong_secondary,[],[Q8684]



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_author_geographic_evidence_raw.csv


#### 18.34.19 — Resolution of Author Geographic Entities

The selected author identities contain structured Wikidata geographic identifiers for country of citizenship and place of birth.

These identifiers are resolved to human-readable Wikidata labels while preserving the original QIDs and identity-evidence tiers.

Citizenship and birthplace remain separate attributes:

- **citizenship** describes a documented national or citizenship association;
- **birthplace** describes the documented place where the author was born.

No attempt is made to convert either attribute into a generalized "author country."

Likewise, these author-level geographic attributes are not interpreted as the origin or publication country of a book.

Missing geographic information remains explicitly unavailable rather than being inferred from names, languages, publishers, ISBNs, or book subjects.

In [50]:
# ---------------------------------------------------------
# 18.34.19 Resolve geographic Wikidata QIDs
# ---------------------------------------------------------

import ast
import time
import pandas as pd


# ---------------------------------------------------------
# Load prepared geographic evidence
# ---------------------------------------------------------

geography_path = (
    data_root
    / "processed"
    / "wikidata_author_geographic_evidence_raw.csv"
)


author_geography = pd.read_csv(
    geography_path
)


# ---------------------------------------------------------
# Parse list fields
# ---------------------------------------------------------

def parse_qid_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):
            return parsed

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return []


author_geography[
    "citizenship_qids_parsed"
] = (
    author_geography[
        "citizenship_qids"
    ]
    .apply(
        parse_qid_list
    )
)


author_geography[
    "birthplace_qids_parsed"
] = (
    author_geography[
        "birthplace_qids"
    ]
    .apply(
        parse_qid_list
    )
)


# ---------------------------------------------------------
# Collect all unique geographic QIDs
# ---------------------------------------------------------

citizenship_qids = {
    qid
    for values
    in author_geography[
        "citizenship_qids_parsed"
    ]
    for qid in values
    if isinstance(qid, str)
    and qid.startswith("Q")
}


birthplace_qids = {
    qid
    for values
    in author_geography[
        "birthplace_qids_parsed"
    ]
    for qid in values
    if isinstance(qid, str)
    and qid.startswith("Q")
}


all_geographic_qids = sorted(
    citizenship_qids
    |
    birthplace_qids
)


print(
    "GEOGRAPHIC QID AUDIT"
)
print("=" * 70)

print(
    "Unique citizenship QIDs:",
    len(citizenship_qids)
)

print(
    "Unique birthplace QIDs:",
    len(birthplace_qids)
)

print(
    "Unique geographic QIDs overall:",
    len(all_geographic_qids)
)


# ---------------------------------------------------------
# Resolve QIDs through Wikidata
# ---------------------------------------------------------

geographic_rows = []

BATCH_SIZE = 25
MAX_RETRIES = 5


for batch_number, start in enumerate(
    range(
        0,
        len(all_geographic_qids),
        BATCH_SIZE,
    ),
    start=1,
):

    batch = all_geographic_qids[
        start:
        start + BATCH_SIZE
    ]


    params = {
        "action":
            "wbgetentities",

        "ids":
            "|".join(batch),

        "props":
            "labels|descriptions",

        "languages":
            "en",

        "format":
            "json",
    }


    success = False


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        response = session.get(
            wikidata_api,
            params=params,
            timeout=60,
        )


        if response.status_code == 429:

            retry_after = (
                response.headers.get(
                    "Retry-After"
                )
            )

            try:

                wait_seconds = (
                    float(retry_after)
                    if retry_after
                    else 15 * attempt
                )

            except ValueError:

                wait_seconds = (
                    15 * attempt
                )


            print(
                f"429 | batch "
                f"{batch_number} | "
                f"attempt {attempt} | "
                f"waiting "
                f"{wait_seconds:.1f}s"
            )

            time.sleep(
                wait_seconds
            )

            continue


        response.raise_for_status()

        entities = (
            response.json()
            .get(
                "entities",
                {}
            )
        )


        for qid in batch:

            entity = entities.get(
                qid,
                {}
            )

            label = (
                entity
                .get(
                    "labels",
                    {}
                )
                .get(
                    "en",
                    {}
                )
                .get(
                    "value"
                )
            )

            description = (
                entity
                .get(
                    "descriptions",
                    {}
                )
                .get(
                    "en",
                    {}
                )
                .get(
                    "value"
                )
            )


            geographic_rows.append(
                {
                    "wikidata_geo_qid":
                        qid,

                    "wikidata_geo_label":
                        label,

                    "wikidata_geo_description":
                        description,
                }
            )


        success = True
        break


    if not success:

        print(
            "FAILED BATCH:",
            batch
        )


    time.sleep(2)


geographic_lookup = pd.DataFrame(
    geographic_rows
)


# ---------------------------------------------------------
# Validate resolution
# ---------------------------------------------------------

resolved_qids = set(
    geographic_lookup[
        "wikidata_geo_qid"
    ]
)


missing_qids = (
    set(all_geographic_qids)
    -
    resolved_qids
)


print(
    "\nGEOGRAPHIC LOOKUP"
)
print("=" * 70)

print(
    "Resolved QIDs:",
    len(resolved_qids)
)

print(
    "Missing QIDs:",
    len(missing_qids)
)


if missing_qids:

    print(
        "Missing:",
        sorted(
            missing_qids
        )
    )


# ---------------------------------------------------------
# Build lookup dictionary
# ---------------------------------------------------------

geo_label_map = (
    geographic_lookup
    .dropna(
        subset=[
            "wikidata_geo_qid",
            "wikidata_geo_label",
        ]
    )
    .set_index(
        "wikidata_geo_qid"
    )[
        "wikidata_geo_label"
    ]
    .to_dict()
)


def resolve_geo_labels(qids):

    return list(
        dict.fromkeys(
            [
                geo_label_map[qid]
                for qid in qids
                if qid in geo_label_map
            ]
        )
    )


author_geography[
    "citizenship_labels"
] = (
    author_geography[
        "citizenship_qids_parsed"
    ]
    .apply(
        resolve_geo_labels
    )
)


author_geography[
    "birthplace_labels"
] = (
    author_geography[
        "birthplace_qids_parsed"
    ]
    .apply(
        resolve_geo_labels
    )
)


# ---------------------------------------------------------
# Coverage after label resolution
# ---------------------------------------------------------

print(
    "\nRESOLVED AUTHOR GEOGRAPHY"
)
print("=" * 70)

print(
    "Authors:",
    len(author_geography)
)

print(
    "With citizenship labels:",
    author_geography[
        "citizenship_labels"
    ]
    .apply(len)
    .gt(0)
    .sum()
)

print(
    "With birthplace labels:",
    author_geography[
        "birthplace_labels"
    ]
    .apply(len)
    .gt(0)
    .sum()
)


# ---------------------------------------------------------
# Preview
# ---------------------------------------------------------

display(
    author_geography[
        [
            "openlibrary_author_key",
            "author_name",
            "identity_evidence_tier",
            "citizenship_labels",
            "birthplace_labels",
        ]
    ]
    [
        (
            author_geography[
                "citizenship_labels"
            ].apply(len) > 0
        )
        |
        (
            author_geography[
                "birthplace_labels"
            ].apply(len) > 0
        )
    ]
    .head(40)
)


# ---------------------------------------------------------
# Save lookup + resolved author geography
# ---------------------------------------------------------

lookup_path = (
    data_root
    / "processed"
    / "wikidata_geographic_entity_lookup.csv"
)


resolved_path = (
    data_root
    / "processed"
    / "wikidata_author_geographic_evidence.csv"
)


geographic_lookup.to_csv(
    lookup_path,
    index=False,
)


author_geography.to_csv(
    resolved_path,
    index=False,
)


print(
    "\nSaved geographic lookup:",
    lookup_path
)

print(
    "Saved resolved author geography:",
    resolved_path
)

GEOGRAPHIC QID AUDIT
Unique citizenship QIDs: 28
Unique birthplace QIDs: 107
Unique geographic QIDs overall: 132

GEOGRAPHIC LOOKUP
Resolved QIDs: 132
Missing QIDs: 0

RESOLVED AUTHOR GEOGRAPHY
Authors: 467
With citizenship labels: 171
With birthplace labels: 127


,openlibrary_author_key,author_name,identity_evidence_tier,citizenship_labels,birthplace_labels
3,OL1076211A,R. Edward Freeman,strong_secondary,[United States],[Columbus]
6,OL10856242A,Cyril O'Donnell,probable_secondary,[United States],[Lincoln]
9,OL1121414A,Young Yun Kim,strong_secondary,[],[Seoul]
11,OL11510748A,JAMES FREDERICK BENDER,strong_secondary,[],[Dayton]
12,OL11546536A,abraham holtzman,strong_secondary,[United States],[Detroit]
13,OL1171623A,Kent M. Keith,strong_secondary,[United States],[Brooklyn]
15,OL1193951A,Aaron Antonovsky,strong_secondary,"[United States, Israel]",[Brooklyn]
19,OL1237529A,Shlomo Globerson,strong_secondary,[Israel],[Kiryat Chaim]
20,OL1239241A,Judith A. Hall,identifier_verified,[United States],[]
21,OL12578562A,Edward McDonnell,strong_secondary,[Ireland],[Dublin]



Saved geographic lookup: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_geographic_entity_lookup.csv
Saved resolved author geography: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_author_geographic_evidence.csv


### 18.35 — Book Publication Origin Enrichment

Author geography and book publication geography represent different concepts and are therefore enriched separately.

Open Library edition histories collected during post-collection enrichment contain publication dates, publication countries, publication places, publishers, and languages for individual editions.

The first stage evaluates whether geographic metadata associated with the **earliest observed publication edition(s)** can provide defensible evidence about a book's initial publication geography.

This evidence is treated conservatively.

An earliest observed publication country or place is not automatically labeled the definitive "country of origin." It represents the geographic information recorded for the earliest edition currently observed in the available Open Library edition history.

Where:

- the earliest edition has no geographic metadata;
- multiple earliest editions report different locations;
- publication evidence is incomplete; or
- the historical record remains ambiguous,

the origin remains unresolved rather than inferred.

This distinction preserves provenance between:

- observed publication geography;
- earliest observed publication geography; and
- verified original publication country.

In [51]:
# ---------------------------------------------------------
# 18.35.1 Earliest-edition publication geography audit
# ---------------------------------------------------------

import ast
import re
import pandas as pd


# ---------------------------------------------------------
# Locate edition-enrichment checkpoint
# ---------------------------------------------------------

edition_path = (
    data_root
    / "raw"
    / "openlibrary_edition_enrichment_checkpoint.csv"
)


edition_history = pd.read_csv(
    edition_path
)


print(
    "EDITION HISTORY"
)
print("=" * 70)

print(
    "Rows:",
    len(edition_history)
)

print(
    "Books:",
    edition_history[
        "book_id"
    ].nunique()
)

print(
    "\nColumns:"
)

print(
    edition_history.columns.tolist()
)


# ---------------------------------------------------------
# Detect relevant columns
# ---------------------------------------------------------

def first_existing_column(
    dataframe,
    candidates,
):

    for column in candidates:

        if column in dataframe.columns:
            return column

    return None


year_col = first_existing_column(
    edition_history,
    [
        "publication_year",
        "publish_year",
        "year",
        "publication_year_extracted",
    ],
)


date_col = first_existing_column(
    edition_history,
    [
        "publish_date",
        "publication_date",
    ],
)


country_col = first_existing_column(
    edition_history,
    [
        "publish_country",
        "publication_country",
    ],
)


place_col = first_existing_column(
    edition_history,
    [
        "publish_places",
        "publication_places",
    ],
)


publisher_col = first_existing_column(
    edition_history,
    [
        "publishers",
        "publisher",
    ],
)


print(
    "\nDETECTED FIELDS"
)
print("=" * 70)

print(
    "Year:",
    year_col
)

print(
    "Date:",
    date_col
)

print(
    "Country:",
    country_col
)

print(
    "Place:",
    place_col
)

print(
    "Publisher:",
    publisher_col
)


# ---------------------------------------------------------
# Extract year if no explicit usable year exists
# ---------------------------------------------------------

def extract_year(value):

    if pd.isna(value):
        return pd.NA

    match = re.search(
        r"\b(1[5-9]\d{2}|20\d{2})\b",
        str(value),
    )

    if match:
        return int(
            match.group(1)
        )

    return pd.NA


if year_col is not None:

    edition_history[
        "audit_publication_year"
    ] = pd.to_numeric(
        edition_history[
            year_col
        ],
        errors="coerce",
    )

else:

    edition_history[
        "audit_publication_year"
    ] = (
        edition_history[
            date_col
        ]
        .apply(
            extract_year
        )
    )


# If explicit year is partially missing,
# supplement from publication date.

if date_col is not None:

    missing_year = (
        edition_history[
            "audit_publication_year"
        ].isna()
    )

    edition_history.loc[
        missing_year,
        "audit_publication_year",
    ] = (
        edition_history.loc[
            missing_year,
            date_col,
        ]
        .apply(
            extract_year
        )
    )


# ---------------------------------------------------------
# Earliest observed year per book
# ---------------------------------------------------------

earliest_year = (
    edition_history
    .dropna(
        subset=[
            "audit_publication_year"
        ]
    )
    .groupby(
        "book_id"
    )[
        "audit_publication_year"
    ]
    .min()
    .rename(
        "earliest_observed_year"
    )
    .reset_index()
)


earliest_editions = (
    edition_history
    .merge(
        earliest_year,
        on="book_id",
        how="inner",
        validate="many_to_one",
    )
)


earliest_editions = (
    earliest_editions[
        earliest_editions[
            "audit_publication_year"
        ]
        ==
        earliest_editions[
            "earliest_observed_year"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Meaningful-value helper
# ---------------------------------------------------------

def has_meaningful_value(value):

    if pd.isna(value):
        return False

    text = str(value).strip()

    return (
        text
        not in {
            "",
            "[]",
            "{}",
            "nan",
            "None",
        }
    )


# ---------------------------------------------------------
# Coverage among earliest editions
# ---------------------------------------------------------

print(
    "\nEARLIEST-EDITION GEOGRAPHY"
)
print("=" * 70)

print(
    "Books with an observed publication year:",
    earliest_year[
        "book_id"
    ].nunique()
)

print(
    "Earliest-edition rows:",
    len(
        earliest_editions
    )
)


if country_col is not None:

    books_with_country = (
        earliest_editions[
            earliest_editions[
                country_col
            ]
            .apply(
                has_meaningful_value
            )
        ][
            "book_id"
        ]
        .nunique()
    )

    print(
        "Books with earliest-edition country evidence:",
        books_with_country
    )


if place_col is not None:

    books_with_place = (
        earliest_editions[
            earliest_editions[
                place_col
            ]
            .apply(
                has_meaningful_value
            )
        ][
            "book_id"
        ]
        .nunique()
    )

    print(
        "Books with earliest-edition place evidence:",
        books_with_place
    )


if publisher_col is not None:

    books_with_publisher = (
        earliest_editions[
            earliest_editions[
                publisher_col
            ]
            .apply(
                has_meaningful_value
            )
        ][
            "book_id"
        ]
        .nunique()
    )

    print(
        "Books with earliest-edition publisher evidence:",
        books_with_publisher
    )


# ---------------------------------------------------------
# Number of earliest-edition records per book
# ---------------------------------------------------------

earliest_record_counts = (
    earliest_editions
    .groupby(
        "book_id"
    )
    .size()
    .rename(
        "earliest_edition_record_count"
    )
    .reset_index()
)


print(
    "\nEARLIEST RECORD MULTIPLICITY"
)
print("=" * 70)

print(
    "Books with exactly one earliest-edition record:",
    (
        earliest_record_counts[
            "earliest_edition_record_count"
        ]
        == 1
    ).sum()
)

print(
    "Books with multiple records in earliest year:",
    (
        earliest_record_counts[
            "earliest_edition_record_count"
        ]
        > 1
    ).sum()
)

print(
    "Maximum earliest-year records for one book:",
    earliest_record_counts[
        "earliest_edition_record_count"
    ].max()
)


# ---------------------------------------------------------
# Preview useful evidence
# ---------------------------------------------------------

preview_columns = [
    "book_id",
    "audit_publication_year",
]


for column in [
    date_col,
    country_col,
    place_col,
    publisher_col,
]:

    if (
        column is not None
        and column not in preview_columns
    ):
        preview_columns.append(
            column
        )


print(
    "\nEARLIEST PUBLICATION EVIDENCE SAMPLE"
)
print("=" * 70)


display(
    earliest_editions[
        preview_columns
    ]
    .head(40)
)

EDITION HISTORY
Rows: 6913
Books: 948

Columns:
['book_id', 'openlibrary_work_key', 'canonical_title', 'edition_key', 'edition_title', 'full_title', 'edition_name', 'publish_date', 'publish_country', 'publish_places', 'publishers', 'languages', 'isbn_10', 'isbn_13', 'number_of_pages', 'physical_format']

DETECTED FIELDS
Year: None
Date: publish_date
Country: publish_country
Place: publish_places
Publisher: publishers

EARLIEST-EDITION GEOGRAPHY
Books with an observed publication year: 945
Earliest-edition rows: 1788
Books with earliest-edition country evidence: 479
Books with earliest-edition place evidence: 471
Books with earliest-edition publisher evidence: 941

EARLIEST RECORD MULTIPLICITY
Books with exactly one earliest-edition record: 573
Books with multiple records in earliest year: 372
Maximum earliest-year records for one book: 20

EARLIEST PUBLICATION EVIDENCE SAMPLE


,book_id,audit_publication_year,publish_date,publish_country,publish_places,publishers
18,BOOK00001,1989,April 1989,NaN,NaN,['Cedar Fort']
30,BOOK00002,1981,1981,xxk,['London'],['Prentice-Hall International']
39,BOOK00002,1981,1981,nju,"['Englewood Cliffs, N.J']",['Prentice-Hall']
47,BOOK00003,1977,1977,io,['Bogor'],['Politeia']
54,BOOK00004,1967,1967,ilu,['Chicago'],['Moody Press']
60,BOOK00005,1997,1997,cau,"['Thousand Oaks, Calif']",['Sage Publications']
71,BOOK00006,1998,1998,tnu,"['Nashville, Tenn']",['Thomas Nelson Publishers']
78,BOOK00007,2000,"July 5, 2000",NaN,NaN,"['Sage Publications, Inc']"
81,BOOK00008,1985,1985,nyu,"['New York', 'London']","['Free Press', 'Collier Macmillan']"
83,BOOK00009,2018,2018,NaN,NaN,['Pan Macmillan']


#### 18.35.2 — Earliest Observed Publication Geography Aggregation

The edition-history audit shows that multiple edition records can share the earliest observed publication year for the same book.

Therefore, LeadWise does not select a single earliest edition arbitrarily.

Instead, all publication-country codes, publication places, and publishers recorded for editions in the earliest observed year are aggregated at book level.

The resulting fields describe **earliest observed publication evidence**, not a definitive original publication country.

Each book is classified according to the consistency of its earliest-year geographic evidence:

- **single_country_evidence** — one unique publication-country code is observed;
- **multiple_country_evidence** — more than one publication-country code is observed;
- **place_only_evidence** — publication place information exists but no country code is recorded;
- **no_geographic_evidence** — neither country nor place information is available.

This approach preserves conflicting or incomplete historical metadata rather than forcing a single geographic interpretation.

In [52]:
# ---------------------------------------------------------
# 18.35.2 Aggregate earliest observed publication geography
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def parse_list_field(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if text in {
        "",
        "[]",
        "{}",
        "nan",
        "None",
    }:
        return []

    try:

        parsed = ast.literal_eval(
            text
        )

        if isinstance(parsed, list):

            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return [
        text
    ]


def parse_scalar_field(value):

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text in {
        "",
        "nan",
        "None",
    }:
        return []

    return [
        text
    ]


def unique_preserve_order(values):

    result = []

    seen = set()

    for value in values:

        if value not in seen:

            seen.add(value)

            result.append(value)

    return result


# ---------------------------------------------------------
# Parse earliest-edition metadata
# ---------------------------------------------------------

earliest_work = (
    earliest_editions
    .copy()
)


earliest_work[
    "country_values"
] = (
    earliest_work[
        country_col
    ]
    .apply(
        parse_scalar_field
    )
)


earliest_work[
    "place_values"
] = (
    earliest_work[
        place_col
    ]
    .apply(
        parse_list_field
    )
)


earliest_work[
    "publisher_values"
] = (
    earliest_work[
        publisher_col
    ]
    .apply(
        parse_list_field
    )
)


# ---------------------------------------------------------
# Aggregate one row per book
# ---------------------------------------------------------

aggregation_rows = []


for book_id, group in earliest_work.groupby(
    "book_id"
):

    country_codes = (
        unique_preserve_order(
            [
                value
                for values
                in group[
                    "country_values"
                ]
                for value in values
            ]
        )
    )


    places = (
        unique_preserve_order(
            [
                value
                for values
                in group[
                    "place_values"
                ]
                for value in values
            ]
        )
    )


    publishers = (
        unique_preserve_order(
            [
                value
                for values
                in group[
                    "publisher_values"
                ]
                for value in values
            ]
        )
    )


    earliest_year_value = (
        group[
            "earliest_observed_year"
        ]
        .iloc[0]
    )


    if len(country_codes) == 1:

        geography_status = (
            "single_country_evidence"
        )

    elif len(country_codes) > 1:

        geography_status = (
            "multiple_country_evidence"
        )

    elif len(places) > 0:

        geography_status = (
            "place_only_evidence"
        )

    else:

        geography_status = (
            "no_geographic_evidence"
        )


    aggregation_rows.append(
        {
            "book_id":
                book_id,

            "earliest_observed_year":
                earliest_year_value,

            "earliest_year_edition_records":
                len(group),

            "earliest_country_codes":
                country_codes,

            "earliest_country_code_count":
                len(country_codes),

            "earliest_publication_places":
                places,

            "earliest_publication_place_count":
                len(places),

            "earliest_publishers":
                publishers,

            "earliest_publisher_count":
                len(publishers),

            "earliest_geography_status":
                geography_status,
        }
    )


earliest_geography = pd.DataFrame(
    aggregation_rows
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "EARLIEST PUBLICATION GEOGRAPHY AGGREGATION"
)
print("=" * 70)

print(
    "Books:",
    len(
        earliest_geography
    )
)

print(
    "Unique books:",
    earliest_geography[
        "book_id"
    ].nunique()
)


# ---------------------------------------------------------
# Geography status summary
# ---------------------------------------------------------

status_summary = (
    earliest_geography[
        "earliest_geography_status"
    ]
    .value_counts()
    .rename_axis(
        "earliest_geography_status"
    )
    .reset_index(
        name="book_count"
    )
)


status_summary[
    "percentage"
] = (
    status_summary[
        "book_count"
    ]
    /
    len(
        earliest_geography
    )
    *
    100
).round(2)


print(
    "\nGEOGRAPHY STATUS"
)
print("=" * 70)

display(
    status_summary
)


# ---------------------------------------------------------
# Country-code multiplicity
# ---------------------------------------------------------

print(
    "\nCOUNTRY EVIDENCE"
)
print("=" * 70)

print(
    "Books with one country code:",
    (
        earliest_geography[
            "earliest_country_code_count"
        ]
        == 1
    ).sum()
)

print(
    "Books with multiple country codes:",
    (
        earliest_geography[
            "earliest_country_code_count"
        ]
        > 1
    ).sum()
)

print(
    "Maximum country codes for one book:",
    earliest_geography[
        "earliest_country_code_count"
    ].max()
)


# ---------------------------------------------------------
# Inspect conflicting country evidence
# ---------------------------------------------------------

country_conflicts = (
    earliest_geography[
        earliest_geography[
            "earliest_country_code_count"
        ]
        > 1
    ]
    .sort_values(
        [
            "earliest_country_code_count",
            "earliest_year_edition_records",
        ],
        ascending=False,
    )
)


print(
    "\nMULTIPLE-COUNTRY EVIDENCE SAMPLE"
)
print("=" * 70)

display(
    country_conflicts[
        [
            "book_id",
            "earliest_observed_year",
            "earliest_year_edition_records",
            "earliest_country_codes",
            "earliest_publication_places",
            "earliest_publishers",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Inspect place-only evidence
# ---------------------------------------------------------

place_only = (
    earliest_geography[
        earliest_geography[
            "earliest_geography_status"
        ]
        == "place_only_evidence"
    ]
)


print(
    "\nPLACE-ONLY EVIDENCE"
)
print("=" * 70)

print(
    "Books:",
    len(
        place_only
    )
)


display(
    place_only[
        [
            "book_id",
            "earliest_observed_year",
            "earliest_publication_places",
            "earliest_publishers",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

earliest_geo_path = (
    data_root
    / "processed"
    / "openlibrary_earliest_publication_geography.csv"
)


earliest_geography.to_csv(
    earliest_geo_path,
    index=False,
)


print(
    "\nSaved:",
    earliest_geo_path
)

EARLIEST PUBLICATION GEOGRAPHY AGGREGATION
Books: 945
Unique books: 945

GEOGRAPHY STATUS


,earliest_geography_status,book_count,percentage
0,no_geographic_evidence,462,48.89
1,single_country_evidence,422,44.66
2,multiple_country_evidence,57,6.03
3,place_only_evidence,4,0.42



COUNTRY EVIDENCE
Books with one country code: 422
Books with multiple country codes: 57
Maximum country codes for one book: 4

MULTIPLE-COUNTRY EVIDENCE SAMPLE


,book_id,earliest_observed_year,earliest_year_edition_records,earliest_country_codes,earliest_publication_places,earliest_publishers
853,BOOK00859,1998,8,"[it, enk, ch, nyu]","[Milano, London, Taibei Shi, New York]","[Bloomsbury Publishing Plc, Bantam, Rizzoli, B..."
741,BOOK00746,1965,6,"[xx, xxk, xxu, nyu]","[New York, London]",[McGraw-Hill]
286,BOOK00289,1993,9,"[nyu, onc, enk]","[New York, USA, New York, Toronto, London]","[Kiepenheuer & Witsch, Trafalgar Square, Rando..."
103,BOOK00104,1967,7,"[xxk, enk, nyu]","[London, New York, New York, NY]","[Harper & Row, HARPER & ROW, HarperCollins Pub..."
801,BOOK00806,2010,5,"[enk, pau, nyu]","[London, Philadelphia, PA, Philadelphia]","[Kogan Page, Limited, KoganPage, Kogan Page Li..."
100,BOOK00101,1953,3,"[xxu, xx, xxk]","[New York, London]","[Harper & Row, Harper, Harper and Row]"
765,BOOK00770,1995,3,"[nyu, enk, xxk]","[New York, Chichester]",[Wiley]
836,BOOK00841,1972,3,"[xxu, nyu, xx]",[New York],"[Holt, Rinehart and Winston]"
467,BOOK00471,2010,13,"[quc, nyu]","[Montréal, New York, London]","[Éditions Transcontinental, Ebury Publishing, ..."
207,BOOK00209,2008,9,"[nyu, nju]","[New York, Hoboken, N.J]","[Wiley & Sons, Incorporated, John, Brilliance ..."



PLACE-ONLY EVIDENCE
Books: 4


,book_id,earliest_observed_year,earliest_publication_places,earliest_publishers
318,BOOK00321,2019,[New Delhi],"[Sultan chand and sons, Sultan Chand & Sons]"
437,BOOK00441,2008,[Ottawa],[]
525,BOOK00529,2017,"[Newtown Square, USA]",[Project Management Institute]
530,BOOK00534,1993,[New Delhi],"[Macmillan Publishers India Limited, 2/10 Ansa..."



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_earliest_publication_geography.csv


#### 18.35.3 — Publication Geographic Code Inventory

Open Library edition metadata contains publication-country values represented by MARC-style geographic codes rather than standardized country names.

These codes can represent different geographic levels. For example, some may identify a country while others identify a state, province, or other geographic subdivision.

Consequently, different codes appearing in the same earliest publication year do not necessarily represent contradictory countries.

Before publication geography can be summarized for LeadWise, the unique observed codes are inventoried for authoritative resolution.

No geographic meaning is assigned from the code text itself.

In [53]:
# ---------------------------------------------------------
# 18.35.3 Publication geographic code inventory
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load earliest publication geography
# ---------------------------------------------------------

earliest_geo_path = (
    data_root
    / "processed"
    / "openlibrary_earliest_publication_geography.csv"
)


earliest_geography = pd.read_csv(
    earliest_geo_path
)


# ---------------------------------------------------------
# Parse country-code lists
# ---------------------------------------------------------

def parse_list_field(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:

        parsed = ast.literal_eval(
            str(value)
        )

        if isinstance(parsed, list):

            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]

    except (
        ValueError,
        SyntaxError,
    ):
        pass

    return []


earliest_geography[
    "earliest_country_codes_parsed"
] = (
    earliest_geography[
        "earliest_country_codes"
    ]
    .apply(
        parse_list_field
    )
)


# ---------------------------------------------------------
# Explode codes
# ---------------------------------------------------------

country_code_inventory = (
    earliest_geography[
        [
            "book_id",
            "earliest_observed_year",
            "earliest_country_codes_parsed",
        ]
    ]
    .explode(
        "earliest_country_codes_parsed"
    )
    .rename(
        columns={
            "earliest_country_codes_parsed":
                "publication_geo_code"
        }
    )
)


country_code_inventory = (
    country_code_inventory
    .dropna(
        subset=[
            "publication_geo_code"
        ]
    )
)


country_code_inventory[
    "publication_geo_code"
] = (
    country_code_inventory[
        "publication_geo_code"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


country_code_inventory = (
    country_code_inventory[
        country_code_inventory[
            "publication_geo_code"
        ]
        != ""
    ]
)


# ---------------------------------------------------------
# Frequency table
# ---------------------------------------------------------

code_summary = (
    country_code_inventory[
        "publication_geo_code"
    ]
    .value_counts()
    .rename_axis(
        "publication_geo_code"
    )
    .reset_index(
        name="book_occurrences"
    )
)


print(
    "PUBLICATION GEOGRAPHIC CODE INVENTORY"
)
print("=" * 70)

print(
    "Books with country-code evidence:",
    country_code_inventory[
        "book_id"
    ].nunique()
)

print(
    "Unique geographic codes:",
    code_summary[
        "publication_geo_code"
    ].nunique()
)

print(
    "Total book-code relationships:",
    len(
        country_code_inventory
    )
)


print(
    "\nMOST COMMON GEOGRAPHIC CODES"
)
print("=" * 70)

display(
    code_summary.head(50)
)


# ---------------------------------------------------------
# Codes appearing in multiple-country records
# ---------------------------------------------------------

multiple_country_books = set(
    earliest_geography.loc[
        earliest_geography[
            "earliest_country_code_count"
        ]
        > 1,
        "book_id",
    ]
)


conflict_codes = (
    country_code_inventory[
        country_code_inventory[
            "book_id"
        ].isin(
            multiple_country_books
        )
    ][
        "publication_geo_code"
    ]
    .value_counts()
    .rename_axis(
        "publication_geo_code"
    )
    .reset_index(
        name="conflict_book_occurrences"
    )
)


print(
    "\nCODES APPEARING IN MULTI-CODE BOOKS"
)
print("=" * 70)

display(
    conflict_codes.head(50)
)


# ---------------------------------------------------------
# Save inventory
# ---------------------------------------------------------

inventory_path = (
    data_root
    / "processed"
    / "openlibrary_publication_geographic_code_inventory.csv"
)


code_summary.to_csv(
    inventory_path,
    index=False,
)


print(
    "\nSaved:",
    inventory_path
)

PUBLICATION GEOGRAPHIC CODE INVENTORY
Books with country-code evidence: 479
Unique geographic codes: 54
Total book-code relationships: 546

MOST COMMON GEOGRAPHIC CODES


,publication_geo_code,book_occurrences
0,nyu,136
1,enk,75
2,mau,43
3,nju,41
4,ilu,38
5,cau,37
6,xxu,26
7,xxk,24
8,xx,17
9,pau,9



CODES APPEARING IN MULTI-CODE BOOKS


,publication_geo_code,conflict_book_occurrences
0,nyu,34
1,enk,17
2,xxk,15
3,xxu,14
4,xx,8
5,ilu,6
6,nju,5
7,cau,4
8,ch,3
9,pau,3



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_publication_geographic_code_inventory.csv


#### 18.35.4 — Authoritative Resolution of MARC Geographic Codes

The earliest-edition audit identified 54 distinct publication geographic codes.

Because MARC geographic codes may represent countries, constituent countries, states, provinces, territories, or broader geographic areas, different codes cannot be assumed to represent different countries.

The codes are therefore resolved against the authoritative Library of Congress MARC Geographic Areas reference data.

The original code is preserved alongside its authoritative label.

No geographic label is inferred from publication place, publisher, language, ISBN, or the apparent structure of the code.

This resolution stage is used to determine whether apparently conflicting earliest-edition geographic codes represent:

- different geographic subdivisions within the same country;
- different constituent countries or territories;
- genuinely different publication countries; or
- geographic codes that remain too broad or ambiguous for country-level interpretation.

In [54]:
# ---------------------------------------------------------
# 18.35.4 Resolve observed MARC geographic codes
# using Library of Congress authoritative data
# ---------------------------------------------------------

import requests
import pandas as pd


# ---------------------------------------------------------
# Load our observed code inventory
# ---------------------------------------------------------

inventory_path = (
    data_root
    / "processed"
    / "openlibrary_publication_geographic_code_inventory.csv"
)


code_inventory = pd.read_csv(
    inventory_path
)


code_inventory[
    "publication_geo_code"
] = (
    code_inventory[
        "publication_geo_code"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


observed_codes = set(
    code_inventory[
        "publication_geo_code"
    ]
)


print(
    "OBSERVED CODE INVENTORY"
)
print("=" * 70)

print(
    "Observed codes:",
    len(observed_codes)
)


# ---------------------------------------------------------
# Library of Congress MARC Geographic Areas
# ---------------------------------------------------------

loc_url = (
    "https://id.loc.gov/vocabulary/geographicAreas.json"
)


response = requests.get(
    loc_url,
    timeout=60,
    headers={
        "User-Agent":
            "LeadWise-Academic-Research/1.0"
    },
)


print(
    "LOC status:",
    response.status_code
)


response.raise_for_status()


loc_data = response.json()


print(
    "LOC records returned:",
    len(loc_data)
)


# ---------------------------------------------------------
# Inspect and extract code / label
# ---------------------------------------------------------

loc_rows = []


for item in loc_data:

    uri = item.get(
        "@id",
        ""
    )

    if not uri:
        continue


    code = (
        uri
        .rstrip("/")
        .split("/")[-1]
        .lower()
    )


    # id.loc.gov JSON commonly exposes
    # authoritative labels through
    # madsrdf:authoritativeLabel.

    raw_label = item.get(
        "http://www.loc.gov/mads/rdf/v1#authoritativeLabel"
    )


    label = None


    if isinstance(
        raw_label,
        list,
    ):

        for label_item in raw_label:

            if isinstance(
                label_item,
                dict,
            ):

                candidate = (
                    label_item.get(
                        "@value"
                    )
                )

                if candidate:

                    label = candidate
                    break


    elif isinstance(
        raw_label,
        dict,
    ):

        label = raw_label.get(
            "@value"
        )


    loc_rows.append(
        {
            "publication_geo_code":
                code,

            "marc_geographic_label":
                label,

            "loc_uri":
                uri,
        }
    )


loc_lookup = pd.DataFrame(
    loc_rows
)


# ---------------------------------------------------------
# Keep only observed LeadWise codes
# ---------------------------------------------------------

observed_lookup = (
    loc_lookup[
        loc_lookup[
            "publication_geo_code"
        ].isin(
            observed_codes
        )
    ]
    .drop_duplicates(
        subset=[
            "publication_geo_code"
        ]
    )
    .copy()
)


# ---------------------------------------------------------
# Merge with frequency inventory
# ---------------------------------------------------------

resolved_inventory = (
    code_inventory
    .merge(
        observed_lookup,
        on="publication_geo_code",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

resolved_count = (
    resolved_inventory[
        "marc_geographic_label"
    ]
    .notna()
    .sum()
)


unresolved = (
    resolved_inventory[
        resolved_inventory[
            "marc_geographic_label"
        ]
        .isna()
    ]
    .copy()
)


print(
    "\nMARC GEOGRAPHIC RESOLUTION"
)
print("=" * 70)

print(
    "Observed codes:",
    len(
        resolved_inventory
    )
)

print(
    "Resolved codes:",
    resolved_count
)

print(
    "Unresolved codes:",
    len(
        unresolved
    )
)


# ---------------------------------------------------------
# Display resolved mapping
# ---------------------------------------------------------

print(
    "\nRESOLVED MARC GEOGRAPHIC CODES"
)
print("=" * 70)


display(
    resolved_inventory[
        [
            "publication_geo_code",
            "marc_geographic_label",
            "book_occurrences",
        ]
    ]
    .sort_values(
        "book_occurrences",
        ascending=False,
    )
)


# ---------------------------------------------------------
# Display unresolved separately
# ---------------------------------------------------------

if len(
    unresolved
) > 0:

    print(
        "\nUNRESOLVED CODES"
    )
    print(
        "=" * 70
    )

    display(
        unresolved[
            [
                "publication_geo_code",
                "book_occurrences",
            ]
        ]
    )


# ---------------------------------------------------------
# Save authoritative lookup
# ---------------------------------------------------------

marc_lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_geographic_lookup.csv"
)


resolved_inventory.to_csv(
    marc_lookup_path,
    index=False,
)


print(
    "\nSaved:",
    marc_lookup_path
)

OBSERVED CODE INVENTORY
Observed codes: 54
LOC status: 200
LOC records returned: 34

MARC GEOGRAPHIC RESOLUTION
Observed codes: 54
Resolved codes: 0
Unresolved codes: 54

RESOLVED MARC GEOGRAPHIC CODES


,publication_geo_code,marc_geographic_label,book_occurrences
0,nyu,NaN,136
1,enk,NaN,75
2,mau,NaN,43
3,nju,NaN,41
4,ilu,NaN,38
5,cau,NaN,37
6,xxu,NaN,26
7,xxk,NaN,24
8,xx,NaN,17
9,pau,NaN,9



UNRESOLVED CODES


,publication_geo_code,book_occurrences
0,nyu,136
1,enk,75
2,mau,43
3,nju,41
4,ilu,38
5,cau,37
6,xxu,26
7,xxk,24
8,xx,17
9,pau,9



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/loc_marc_publication_geographic_lookup.csv


#### 18.35.4A — Correct Resolution of MARC Publication Country Codes

The previous lookup attempted to resolve Open Library `publish_country` values against the MARC Geographic Areas vocabulary.

Further validation against Library of Congress documentation shows that these edition-level values correspond instead to the **MARC Code List for Countries**.

This distinction is important because the MARC country vocabulary includes both:

- country-level codes; and
- subnational publication-place codes, such as U.S. states and Canadian provinces.

The original Open Library code is therefore preserved and resolved using the authoritative MARC country-code vocabulary.

This correction replaces the failed geographic-area lookup from Section 18.35.4. No output from that failed lookup is used downstream.

Country normalization will be performed only after the authoritative MARC labels have been resolved.

In [55]:
# ---------------------------------------------------------
# 18.35.4A Check MARC country-code support
# ---------------------------------------------------------

import importlib.util

print(
    "pymarc installed:",
    importlib.util.find_spec("pymarc") is not None
)

pymarc installed: False


#### 18.35.4B — Authoritative MARC Publication-Country Resolution

Because `pymarc` is not installed, the publication-country codes are resolved directly from the authoritative Library of Congress MARC Code List for Countries.

The Library of Congress provides a machine-readable XML representation of this controlled vocabulary for application use.

This vocabulary confirms that Open Library `publish_country` values can represent different geographic levels, including:

- countries;
- constituent countries;
- U.S. states;
- Canadian provinces;
- Australian states and territories; and
- unknown or undetermined publication locations.

The original MARC code and authoritative MARC label are preserved.

At this stage, subdivision-level labels are not yet collapsed to parent countries. Parent-country normalization is performed separately so that the original bibliographic evidence remains intact.

In [56]:
# ---------------------------------------------------------
# 18.35.4B Resolve MARC country codes from LOC XML
# ---------------------------------------------------------

import requests
import xml.etree.ElementTree as ET
import pandas as pd


# ---------------------------------------------------------
# Load observed LeadWise codes
# ---------------------------------------------------------

inventory_path = (
    data_root
    / "processed"
    / "openlibrary_publication_geographic_code_inventory.csv"
)


code_inventory = pd.read_csv(
    inventory_path
)


code_inventory[
    "publication_geo_code"
] = (
    code_inventory[
        "publication_geo_code"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


observed_codes = set(
    code_inventory[
        "publication_geo_code"
    ]
)


print(
    "OBSERVED CODE INVENTORY"
)
print("=" * 70)

print(
    "Observed codes:",
    len(observed_codes)
)


# ---------------------------------------------------------
# Authoritative LOC XML
# ---------------------------------------------------------

loc_xml_url = (
    "https://www.loc.gov/standards/codelists/countries.xml"
)


response = requests.get(
    loc_xml_url,
    timeout=60,
    headers={
        "User-Agent":
            "LeadWise-Academic-Research/1.0"
    },
)


print(
    "LOC XML status:",
    response.status_code
)


response.raise_for_status()


# ---------------------------------------------------------
# Inspect XML structure
# ---------------------------------------------------------

root = ET.fromstring(
    response.content
)


print(
    "XML root tag:",
    root.tag
)


# ---------------------------------------------------------
# Namespace-safe extraction
# ---------------------------------------------------------

loc_rows = []


for element in root.iter():

    tag = (
        element.tag
        .split("}")[-1]
        .lower()
    )

    if tag != "country":
        continue


    code = None
    name = None


    for child in element:

        child_tag = (
            child.tag
            .split("}")[-1]
            .lower()
        )

        text = (
            child.text.strip()
            if child.text
            else None
        )


        if child_tag == "code":

            code = text


        elif child_tag in {
            "name",
            "countryname",
        }:

            name = text


    if code:

        loc_rows.append(
            {
                "publication_geo_code":
                    code.lower(),

                "marc_geographic_label":
                    name,
            }
        )


loc_country_lookup = pd.DataFrame(
    loc_rows
)


print(
    "\nLOC COUNTRY VOCABULARY"
)
print("=" * 70)

print(
    "Parsed LOC codes:",
    len(
        loc_country_lookup
    )
)

print(
    "Unique LOC codes:",
    loc_country_lookup[
        "publication_geo_code"
    ].nunique()
)


# ---------------------------------------------------------
# Match LeadWise observed codes
# ---------------------------------------------------------

resolved_inventory = (
    code_inventory
    .merge(
        loc_country_lookup,
        on="publication_geo_code",
        how="left",
        validate="one_to_one",
    )
)


resolved_count = (
    resolved_inventory[
        "marc_geographic_label"
    ]
    .notna()
    .sum()
)


unresolved = (
    resolved_inventory[
        resolved_inventory[
            "marc_geographic_label"
        ]
        .isna()
    ]
    .copy()
)


print(
    "\nMARC COUNTRY-CODE RESOLUTION"
)
print("=" * 70)

print(
    "Observed codes:",
    len(
        resolved_inventory
    )
)

print(
    "Resolved codes:",
    resolved_count
)

print(
    "Unresolved codes:",
    len(
        unresolved
    )
)


# ---------------------------------------------------------
# Display mapping
# ---------------------------------------------------------

print(
    "\nRESOLVED MARC PUBLICATION CODES"
)
print("=" * 70)


display(
    resolved_inventory[
        [
            "publication_geo_code",
            "marc_geographic_label",
            "book_occurrences",
        ]
    ]
    .sort_values(
        "book_occurrences",
        ascending=False,
    )
)


# ---------------------------------------------------------
# Unresolved codes
# ---------------------------------------------------------

if len(unresolved) > 0:

    print(
        "\nUNRESOLVED CODES"
    )
    print(
        "=" * 70
    )

    display(
        unresolved[
            [
                "publication_geo_code",
                "book_occurrences",
            ]
        ]
    )


# ---------------------------------------------------------
# Save corrected authoritative lookup
# ---------------------------------------------------------

corrected_lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_country_lookup.csv"
)


resolved_inventory.to_csv(
    corrected_lookup_path,
    index=False,
)


print(
    "\nSaved:",
    corrected_lookup_path
)

OBSERVED CODE INVENTORY
Observed codes: 54
LOC XML status: 403


HTTPError: 403 Client Error: Forbidden for url: https://www.loc.gov/standards/codelists/countries.xml

#### 18.35.4C — MARC Publication-Country Resolution from the Official LOC Code List

The Library of Congress XML endpoint returned HTTP 403 in the local notebook environment.

The authoritative Library of Congress MARC Code List for Countries is, however, available through its official code-sequence page. This source contains the MARC publication-country codes and their corresponding geographic labels.

The LeadWise publication-country codes are therefore resolved against this official LOC table.

The original MARC code is retained alongside the authoritative label. Codes representing states, provinces, constituent countries, countries, or unknown locations remain at their original geographic level during this stage.

No parent-country normalization is performed yet.

In [59]:
# ---------------------------------------------------------
# 18.35.4C Resolve MARC publication-country codes
# from official Library of Congress HTML table
# ---------------------------------------------------------

import pandas as pd
import lxml

print("lxml version:", lxml.__version__)

# ---------------------------------------------------------
# Official LOC MARC country-code page
# ---------------------------------------------------------

loc_url = (
    "https://www.loc.gov/marc/countries/"
    "countries_code.html"
)


# ---------------------------------------------------------
# Read HTML tables
# ---------------------------------------------------------

tables = pd.read_html(
    loc_url
)


print(
    "LOC tables found:",
    len(tables)
)


# ---------------------------------------------------------
# Find table containing code + country
# ---------------------------------------------------------

loc_country_table = None


for table in tables:

    normalized_columns = [
        str(column)
        .strip()
        .lower()
        for column in table.columns
    ]

    if (
        "code" in normalized_columns
        and
        "country" in normalized_columns
    ):

        table = table.copy()

        table.columns = (
            normalized_columns
        )

        loc_country_table = table

        break


if loc_country_table is None:

    raise ValueError(
        "Could not locate the LOC "
        "MARC country-code table."
    )


# ---------------------------------------------------------
# Clean LOC vocabulary
# ---------------------------------------------------------

loc_country_lookup = (
    loc_country_table[
        [
            "code",
            "country",
        ]
    ]
    .rename(
        columns={
            "code":
                "publication_geo_code",

            "country":
                "marc_geographic_label",
        }
    )
    .copy()
)


loc_country_lookup[
    "publication_geo_code"
] = (
    loc_country_lookup[
        "publication_geo_code"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


loc_country_lookup[
    "marc_geographic_label"
] = (
    loc_country_lookup[
        "marc_geographic_label"
    ]
    .astype(str)
    .str.strip()
)


# ---------------------------------------------------------
# Remove discontinued-code marker
# only for identification purposes
# ---------------------------------------------------------

loc_country_lookup[
    "is_discontinued_code"
] = (
    loc_country_lookup[
        "publication_geo_code"
    ]
    .str.startswith("-")
)


loc_country_lookup[
    "publication_geo_code_clean"
] = (
    loc_country_lookup[
        "publication_geo_code"
    ]
    .str.lstrip("-")
)


print(
    "\nLOC MARC COUNTRY VOCABULARY"
)
print("=" * 70)

print(
    "Rows:",
    len(
        loc_country_lookup
    )
)

print(
    "Unique raw codes:",
    loc_country_lookup[
        "publication_geo_code"
    ].nunique()
)


# ---------------------------------------------------------
# Load LeadWise observed codes
# ---------------------------------------------------------

inventory_path = (
    data_root
    / "processed"
    / "openlibrary_publication_geographic_code_inventory.csv"
)


code_inventory = pd.read_csv(
    inventory_path
)


code_inventory[
    "publication_geo_code"
] = (
    code_inventory[
        "publication_geo_code"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ---------------------------------------------------------
# IMPORTANT:
# Match active LOC codes first.
#
# We do NOT automatically map an observed code
# to a discontinued LOC code with the same text.
# ---------------------------------------------------------

active_loc_lookup = (
    loc_country_lookup[
        ~loc_country_lookup[
            "is_discontinued_code"
        ]
    ]
    [
        [
            "publication_geo_code",
            "marc_geographic_label",
        ]
    ]
    .drop_duplicates(
        subset=[
            "publication_geo_code"
        ]
    )
)


resolved_inventory = (
    code_inventory
    .merge(
        active_loc_lookup,
        on="publication_geo_code",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# Resolution validation
# ---------------------------------------------------------

resolved_count = (
    resolved_inventory[
        "marc_geographic_label"
    ]
    .notna()
    .sum()
)


unresolved = (
    resolved_inventory[
        resolved_inventory[
            "marc_geographic_label"
        ]
        .isna()
    ]
    .copy()
)


print(
    "\nMARC COUNTRY-CODE RESOLUTION"
)
print("=" * 70)

print(
    "Observed codes:",
    len(
        resolved_inventory
    )
)

print(
    "Resolved codes:",
    resolved_count
)

print(
    "Unresolved codes:",
    len(
        unresolved
    )
)


# ---------------------------------------------------------
# Display complete LeadWise mapping
# ---------------------------------------------------------

print(
    "\nRESOLVED MARC PUBLICATION CODES"
)
print("=" * 70)


display(
    resolved_inventory[
        [
            "publication_geo_code",
            "marc_geographic_label",
            "book_occurrences",
        ]
    ]
    .sort_values(
        "book_occurrences",
        ascending=False,
    )
)


# ---------------------------------------------------------
# Unresolved codes
# ---------------------------------------------------------

if len(
    unresolved
) > 0:

    print(
        "\nUNRESOLVED CODES"
    )
    print(
        "=" * 70
    )

    display(
        unresolved[
            [
                "publication_geo_code",
                "book_occurrences",
            ]
        ]
    )


# ---------------------------------------------------------
# Critical sanity checks
# ---------------------------------------------------------

sanity_codes = [
    "nyu",
    "enk",
    "mau",
    "nju",
    "ilu",
    "cau",
    "xxu",
    "xxk",
    "xx",
]


print(
    "\nSANITY CHECK"
)
print("=" * 70)


display(
    resolved_inventory[
        resolved_inventory[
            "publication_geo_code"
        ].isin(
            sanity_codes
        )
    ]
    [
        [
            "publication_geo_code",
            "marc_geographic_label",
        ]
    ]
    .sort_values(
        "publication_geo_code"
    )
)


# ---------------------------------------------------------
# Save corrected lookup
# ---------------------------------------------------------

corrected_lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_country_lookup.csv"
)


resolved_inventory.to_csv(
    corrected_lookup_path,
    index=False,
)


print(
    "\nSaved:",
    corrected_lookup_path
)

lxml version: 6.1.3


HTTPError: HTTP Error 403: Forbidden

##### Dependency Note — HTML Parsing

The authoritative Library of Congress MARC country-code table is retrieved from an HTML page.

Pandas `read_html()` requires an HTML parsing dependency. The project environment did not currently include `lxml`, resulting in an `ImportError`.

The `lxml` package is therefore added to the project environment to support reproducible parsing of the official LOC MARC country-code table.

#### 18.35.4D — Local Authoritative MARC Publication-Code Lookup

Direct programmatic requests to the Library of Congress MARC country-code pages returned HTTP 403 in the local development environment.

To ensure reproducibility and avoid dependence on repeated external requests, the MARC codes observed in the LeadWise dataset are therefore mapped locally using the authoritative **Library of Congress MARC Code List for Countries**.

Only codes actually observed in the LeadWise dataset are included in this reference table.

The original MARC code and authoritative LOC label are preserved. No parent-country normalization is performed in this stage.

This distinction is important because the MARC vocabulary contains geographic entities at different levels, including countries, constituent countries, U.S. states, Canadian provinces, and unknown or undetermined publication locations.

In [60]:
# ---------------------------------------------------------
# 18.35.4D Local authoritative MARC lookup
# Source: Library of Congress MARC Code List for Countries
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Authoritative LOC labels for the 54 codes
# observed in the LeadWise dataset
# ---------------------------------------------------------

marc_country_mapping = {
    "aru": "Arkansas",
    "at": "Australia",
    "cau": "California",
    "cc": "China",
    "ch": "China (Republic : 1949- )",
    "cou": "Colorado",
    "ctu": "Connecticut",
    "dcu": "District of Columbia",
    "enk": "England",
    "flu": "Florida",
    "fr": "France",
    "gau": "Georgia",
    "gw": "Germany",
    "iau": "Iowa",
    "ii": "India",
    "ilu": "Illinois",
    "inu": "Indiana",
    "io": "Indonesia",
    "it": "Italy",
    "ja": "Japan",
    "ko": "Korea (South)",
    "ksu": "Kansas",
    "mau": "Massachusetts",
    "mbc": "Manitoba",
    "mdu": "Maryland",
    "meu": "Maine",
    "miu": "Michigan",
    "mnu": "Minnesota",
    "mou": "Missouri",
    "msu": "Mississippi",
    "mx": "Mexico",
    "ne": "Netherlands",
    "nhu": "New Hampshire",
    "nju": "New Jersey",
    "nyu": "New York (State)",
    "nz": "New Zealand",
    "ohu": "Ohio",
    "onc": "Ontario",
    "oru": "Oregon",
    "pau": "Pennsylvania",
    "ph": "Philippines",
    "quc": "Québec (Province)",
    "sa": "South Africa",
    "tnu": "Tennessee",
    "txu": "Texas",
    "ug": "Uganda",
    "us": None,
    "vau": "Virginia",
    "vtu": "Vermont",
    "wau": "Washington (State)",
    "xna": "New South Wales",
    "xx": "No place, unknown, or undetermined",
    "xxk": "United Kingdom",
    "xxu": "United States",
}


# ---------------------------------------------------------
# Important special case
# ---------------------------------------------------------
# LOC currently lists "-us" as the discontinued code
# for United States, while "xxu" is the active code.
#
# Because our observed value is "us" rather than "-us",
# we do NOT silently reinterpret it as United States.
# It remains unresolved pending source-level inspection.


# ---------------------------------------------------------
# Load LeadWise code inventory
# ---------------------------------------------------------

inventory_path = (
    data_root
    / "processed"
    / "openlibrary_publication_geographic_code_inventory.csv"
)


code_inventory = pd.read_csv(
    inventory_path
)


code_inventory[
    "publication_geo_code"
] = (
    code_inventory[
        "publication_geo_code"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ---------------------------------------------------------
# Apply authoritative lookup
# ---------------------------------------------------------

code_inventory[
    "marc_geographic_label"
] = (
    code_inventory[
        "publication_geo_code"
    ]
    .map(
        marc_country_mapping
    )
)


# ---------------------------------------------------------
# Validate mapping coverage
# ---------------------------------------------------------

observed_codes = set(
    code_inventory[
        "publication_geo_code"
    ]
)


mapped_codes = set(
    marc_country_mapping.keys()
)


missing_from_mapping = (
    observed_codes
    - mapped_codes
)


extra_in_mapping = (
    mapped_codes
    - observed_codes
)


resolved_count = (
    code_inventory[
        "marc_geographic_label"
    ]
    .notna()
    .sum()
)


unresolved = (
    code_inventory[
        code_inventory[
            "marc_geographic_label"
        ]
        .isna()
    ]
    .copy()
)


print(
    "MARC COUNTRY-CODE RESOLUTION"
)
print("=" * 70)

print(
    "Observed codes:",
    len(observed_codes)
)

print(
    "Codes represented in mapping:",
    len(
        observed_codes
        & mapped_codes
    )
)

print(
    "Resolved labels:",
    resolved_count
)

print(
    "Unresolved labels:",
    len(unresolved)
)

print(
    "Observed codes missing from mapping:",
    sorted(
        missing_from_mapping
    )
)

print(
    "Mapping codes not observed:",
    sorted(
        extra_in_mapping
    )
)


# ---------------------------------------------------------
# Display complete mapping
# ---------------------------------------------------------

print(
    "\nRESOLVED MARC PUBLICATION CODES"
)
print("=" * 70)


display(
    code_inventory[
        [
            "publication_geo_code",
            "marc_geographic_label",
            "book_occurrences",
        ]
    ]
    .sort_values(
        "book_occurrences",
        ascending=False,
    )
)


# ---------------------------------------------------------
# Display unresolved separately
# ---------------------------------------------------------

print(
    "\nUNRESOLVED CODES"
)
print("=" * 70)


display(
    unresolved[
        [
            "publication_geo_code",
            "book_occurrences",
        ]
    ]
)


# ---------------------------------------------------------
# Save authoritative project lookup
# ---------------------------------------------------------

lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_country_lookup.csv"
)


code_inventory.to_csv(
    lookup_path,
    index=False,
)


print(
    "\nSaved:",
    lookup_path
)

MARC COUNTRY-CODE RESOLUTION
Observed codes: 54
Codes represented in mapping: 54
Resolved labels: 53
Unresolved labels: 1
Observed codes missing from mapping: []
Mapping codes not observed: []

RESOLVED MARC PUBLICATION CODES


,publication_geo_code,marc_geographic_label,book_occurrences
0,nyu,New York (State),136
1,enk,England,75
2,mau,Massachusetts,43
3,nju,New Jersey,41
4,ilu,Illinois,38
5,cau,California,37
6,xxu,United States,26
7,xxk,United Kingdom,24
8,xx,"No place, unknown, or undetermined",17
9,pau,Pennsylvania,9



UNRESOLVED CODES


,publication_geo_code,book_occurrences
41,us,1



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/loc_marc_publication_country_lookup.csv


#### 18.35.5 — Parent-Country Normalization of MARC Publication Geography

The authoritative MARC lookup resolved 53 of the 54 publication geographic codes observed in the LeadWise dataset.

However, the MARC vocabulary operates at different geographic levels. Some codes identify countries, while others identify states, provinces, constituent countries, or other subnational areas.

For analytical consistency, resolved MARC entities are normalized to a parent-country level while preserving the original MARC code and authoritative LOC label.

Examples include:

- New York (State) → United States
- California → United States
- Massachusetts → United States
- Ontario → Canada
- Québec (Province) → Canada
- New South Wales → Australia
- England → United Kingdom

`No place, unknown, or undetermined` is retained as unresolved geographic evidence rather than treated as a country.

The single observed `us` code also remains unresolved because its exact interpretation has not yet been verified from the underlying source record.

This normalized field represents **observed publication geography**, not verified original publication country.

In [61]:
# ---------------------------------------------------------
# 18.35.5 Normalize MARC geography to parent country
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Load authoritative MARC lookup
# ---------------------------------------------------------

lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_country_lookup.csv"
)


marc_lookup = pd.read_csv(
    lookup_path
)


# ---------------------------------------------------------
# Parent-country normalization
#
# Only subdivision / constituent-country labels need
# explicit normalization.
#
# Existing country-level labels remain unchanged.
# ---------------------------------------------------------

us_subdivisions = {
    "Arkansas",
    "California",
    "Colorado",
    "Connecticut",
    "District of Columbia",
    "Florida",
    "Georgia",
    "Illinois",
    "Indiana",
    "Iowa",
    "Kansas",
    "Maine",
    "Maryland",
    "Massachusetts",
    "Michigan",
    "Minnesota",
    "Mississippi",
    "Missouri",
    "New Hampshire",
    "New Jersey",
    "New York (State)",
    "Ohio",
    "Oregon",
    "Pennsylvania",
    "Tennessee",
    "Texas",
    "Vermont",
    "Virginia",
    "Washington (State)",
}


canada_subdivisions = {
    "Manitoba",
    "Ontario",
    "Québec (Province)",
}


australia_subdivisions = {
    "New South Wales",
}


uk_constituent_countries = {
    "England",
}


# ---------------------------------------------------------
# Normalize one MARC label
# ---------------------------------------------------------

def normalize_parent_country(
    code,
    label,
):

    # Explicit unresolved cases
    if pd.isna(label):
        return None

    if code == "xx":
        return None

    if label == (
        "No place, unknown, or undetermined"
    ):
        return None

    # United States subdivisions
    if label in us_subdivisions:
        return "United States"

    # Canadian provinces
    if label in canada_subdivisions:
        return "Canada"

    # Australian subdivisions
    if label in australia_subdivisions:
        return "Australia"

    # UK constituent country
    if label in uk_constituent_countries:
        return "United Kingdom"

    # Special authoritative label normalization
    if label == "China (Republic : 1949- )":
        return "Taiwan"

    if label == "Korea (South)":
        return "South Korea"

    # Already country-level
    return label


marc_lookup[
    "normalized_parent_country"
] = marc_lookup.apply(
    lambda row:
        normalize_parent_country(
            row[
                "publication_geo_code"
            ],
            row[
                "marc_geographic_label"
            ],
        ),
    axis=1,
)


# ---------------------------------------------------------
# Geographic-level classification
# ---------------------------------------------------------

def classify_geo_level(
    code,
    label,
):

    if pd.isna(label):
        return "unresolved"

    if code == "xx":
        return "unknown"

    if label in us_subdivisions:
        return "subnational"

    if label in canada_subdivisions:
        return "subnational"

    if label in australia_subdivisions:
        return "subnational"

    if label in uk_constituent_countries:
        return "constituent_country"

    return "country"


marc_lookup[
    "marc_geographic_level"
] = marc_lookup.apply(
    lambda row:
        classify_geo_level(
            row[
                "publication_geo_code"
            ],
            row[
                "marc_geographic_label"
            ],
        ),
    axis=1,
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "PARENT-COUNTRY NORMALIZATION"
)
print("=" * 70)

print(
    "MARC codes:",
    len(marc_lookup)
)

print(
    "Parent country resolved:",
    marc_lookup[
        "normalized_parent_country"
    ].notna().sum()
)

print(
    "Parent country unresolved:",
    marc_lookup[
        "normalized_parent_country"
    ].isna().sum()
)


print(
    "\nGEOGRAPHIC LEVELS"
)
print("=" * 70)

display(
    marc_lookup[
        "marc_geographic_level"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "geographic_level"
    )
    .reset_index(
        name="code_count"
    )
)


# ---------------------------------------------------------
# Inspect normalization
# ---------------------------------------------------------

print(
    "\nNORMALIZED MARC GEOGRAPHY"
)
print("=" * 70)

display(
    marc_lookup[
        [
            "publication_geo_code",
            "marc_geographic_label",
            "marc_geographic_level",
            "normalized_parent_country",
            "book_occurrences",
        ]
    ]
    .sort_values(
        "book_occurrences",
        ascending=False,
    )
)


# ---------------------------------------------------------
# Parent-country frequency by code occurrence
# ---------------------------------------------------------

country_frequency = (
    marc_lookup[
        marc_lookup[
            "normalized_parent_country"
        ].notna()
    ]
    .groupby(
        "normalized_parent_country",
        as_index=False,
    )[
        "book_occurrences"
    ]
    .sum()
    .sort_values(
        "book_occurrences",
        ascending=False,
    )
)


print(
    "\nNORMALIZED PARENT-COUNTRY OCCURRENCES"
)
print("=" * 70)

display(
    country_frequency
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

normalized_lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_country_normalized.csv"
)


marc_lookup.to_csv(
    normalized_lookup_path,
    index=False,
)


print(
    "\nSaved:",
    normalized_lookup_path
)

PARENT-COUNTRY NORMALIZATION
MARC codes: 54
Parent country resolved: 52
Parent country unresolved: 2

GEOGRAPHIC LEVELS


,geographic_level,code_count
0,subnational,33
1,country,18
2,constituent_country,1
3,unknown,1
4,unresolved,1



NORMALIZED MARC GEOGRAPHY


,publication_geo_code,marc_geographic_label,marc_geographic_level,normalized_parent_country,book_occurrences
0,nyu,New York (State),subnational,United States,136
1,enk,England,constituent_country,United Kingdom,75
2,mau,Massachusetts,subnational,United States,43
3,nju,New Jersey,subnational,United States,41
4,ilu,Illinois,subnational,United States,38
5,cau,California,subnational,United States,37
6,xxu,United States,country,United States,26
7,xxk,United Kingdom,country,United Kingdom,24
8,xx,"No place, unknown, or undetermined",unknown,NaN,17
9,pau,Pennsylvania,subnational,United States,9



NORMALIZED PARENT-COUNTRY OCCURRENCES


,normalized_parent_country,book_occurrences
18,United States,401
17,United Kingdom,99
1,Canada,5
15,Taiwan,3
0,Australia,3
4,Germany,3
12,Philippines,2
5,India,1
6,Indonesia,1
7,Italy,1



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/loc_marc_publication_country_normalized.csv


#### 18.35.6 — Reassessment of Earliest-Publication Geographic Conflicts

The raw earliest-publication analysis identified books with multiple MARC geographic codes in the same earliest observed publication year.

Because MARC codes can represent different geographic levels, multiple raw codes do not necessarily indicate publication in multiple countries. For example, a book may contain both a U.S. state code and the general United States code.

The authoritative MARC lookup is therefore applied to each earliest-edition geographic code and normalized to parent-country level.

For every book, this stage calculates:

- the original MARC geographic codes;
- the authoritative MARC labels;
- the normalized parent countries;
- the number of distinct resolved parent countries;
- whether unresolved geographic evidence remains; and
- a revised geographic-evidence status.

The resulting classification distinguishes same-country multi-code evidence from genuine multi-country evidence.

This remains evidence about the **earliest observed publication geography in Open Library**, not proof of the work's original publication country.

In [62]:
# ---------------------------------------------------------
# 18.35.6 Reassess earliest-publication geography
# after parent-country normalization
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load earliest-publication geography
# ---------------------------------------------------------

earliest_geo_path = (
    data_root
    / "processed"
    / "openlibrary_earliest_publication_geography.csv"
)

normalized_lookup_path = (
    data_root
    / "processed"
    / "loc_marc_publication_country_normalized.csv"
)


earliest_geo = pd.read_csv(
    earliest_geo_path
)

marc_lookup = pd.read_csv(
    normalized_lookup_path
)


# ---------------------------------------------------------
# Create lookup dictionaries
# ---------------------------------------------------------

label_lookup = (
    marc_lookup
    .set_index("publication_geo_code")[
        "marc_geographic_label"
    ]
    .to_dict()
)

country_lookup = (
    marc_lookup
    .set_index("publication_geo_code")[
        "normalized_parent_country"
    ]
    .to_dict()
)


# ---------------------------------------------------------
# Safely parse list-like CSV fields
# ---------------------------------------------------------

def parse_list_field(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if (
        text == ""
        or text.lower() == "nan"
    ):
        return []

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):
        return [text]


# ---------------------------------------------------------
# Resolve one book's MARC codes
# ---------------------------------------------------------

def resolve_book_geography(value):

    codes = [
        str(code).strip().lower()
        for code in parse_list_field(value)
        if str(code).strip()
    ]

    # Preserve unique codes in observed order
    codes = list(
        dict.fromkeys(codes)
    )

    labels = []

    countries = []

    unresolved_codes = []

    for code in codes:

        label = label_lookup.get(code)

        country = country_lookup.get(code)

        if pd.notna(label):
            labels.append(label)

        if pd.notna(country):
            countries.append(country)
        else:
            unresolved_codes.append(code)

    labels = list(
        dict.fromkeys(labels)
    )

    countries = list(
        dict.fromkeys(countries)
    )

    unresolved_codes = list(
        dict.fromkeys(unresolved_codes)
    )

    return pd.Series(
        {
            "resolved_marc_labels":
                labels,

            "normalized_parent_countries":
                countries,

            "normalized_country_count":
                len(countries),

            "unresolved_geo_codes":
                unresolved_codes,

            "has_unresolved_geo_evidence":
                len(unresolved_codes) > 0,
        }
    )


resolved_geo = (
    earliest_geo[
        "earliest_country_codes"
    ]
    .apply(
        resolve_book_geography
    )
)


earliest_geo_normalized = pd.concat(
    [
        earliest_geo,
        resolved_geo,
    ],
    axis=1,
)


# ---------------------------------------------------------
# Revised evidence classification
# ---------------------------------------------------------

def classify_normalized_geography(row):

    raw_codes = parse_list_field(
        row[
            "earliest_country_codes"
        ]
    )

    raw_code_count = len(
        set(
            str(code).strip().lower()
            for code in raw_codes
            if str(code).strip()
        )
    )

    country_count = row[
        "normalized_country_count"
    ]

    has_unresolved = row[
        "has_unresolved_geo_evidence"
    ]


    if raw_code_count == 0:
        return "no_code_evidence"

    if (
        country_count == 0
        and has_unresolved
    ):
        return "unresolved_only"

    if (
        country_count == 1
        and raw_code_count == 1
        and not has_unresolved
    ):
        return "single_country_evidence"

    if (
        country_count == 1
        and raw_code_count > 1
        and not has_unresolved
    ):
        return "multi_code_same_country"

    if (
        country_count == 1
        and has_unresolved
    ):
        return "single_country_with_unresolved_code"

    if (
        country_count > 1
        and not has_unresolved
    ):
        return "multi_country_evidence"

    if (
        country_count > 1
        and has_unresolved
    ):
        return "multi_country_with_unresolved_code"

    return "other"


earliest_geo_normalized[
    "normalized_geography_status"
] = (
    earliest_geo_normalized.apply(
        classify_normalized_geography,
        axis=1,
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "NORMALIZED EARLIEST-PUBLICATION GEOGRAPHY"
)
print("=" * 70)

print(
    "Books:",
    len(
        earliest_geo_normalized
    )
)


status_summary = (
    earliest_geo_normalized[
        "normalized_geography_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "normalized_geography_status"
    )
    .reset_index(
        name="book_count"
    )
)


status_summary[
    "percentage"
] = (
    status_summary[
        "book_count"
    ]
    / len(
        earliest_geo_normalized
    )
    * 100
).round(2)


display(
    status_summary
)


# ---------------------------------------------------------
# Specifically reassess original multi-code books
# ---------------------------------------------------------

multi_code_mask = (
    earliest_geo_normalized[
        "earliest_country_codes"
    ]
    .apply(
        lambda value:
            len(
                set(
                    str(code).strip().lower()
                    for code in parse_list_field(value)
                    if str(code).strip()
                )
            ) > 1
    )
)


multi_code_books = (
    earliest_geo_normalized[
        multi_code_mask
    ]
    .copy()
)


print(
    "\nRAW MULTI-CODE REASSESSMENT"
)
print("=" * 70)

print(
    "Books originally containing multiple MARC codes:",
    len(
        multi_code_books
    )
)


display(
    multi_code_books[
        "normalized_geography_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "normalized_geography_status"
    )
    .reset_index(
        name="book_count"
    )
)


# ---------------------------------------------------------
# Genuine normalized multi-country cases
# ---------------------------------------------------------

genuine_multi_country = (
    earliest_geo_normalized[
        earliest_geo_normalized[
            "normalized_country_count"
        ] > 1
    ]
    .copy()
)


print(
    "\nBOOKS WITH MULTIPLE NORMALIZED COUNTRIES"
)
print("=" * 70)

print(
    "Books:",
    len(
        genuine_multi_country
    )
)


display(
    genuine_multi_country[
        [
            "book_id",
            "earliest_observed_year",
            "earliest_country_codes",
            "resolved_marc_labels",
            "normalized_parent_countries",
            "unresolved_geo_codes",
            "normalized_geography_status",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save enriched publication geography
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "openlibrary_earliest_publication_geography_normalized.csv"
)


earliest_geo_normalized.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

NORMALIZED EARLIEST-PUBLICATION GEOGRAPHY
Books: 945


,normalized_geography_status,book_count,percentage
0,no_code_evidence,466,49.31
1,single_country_evidence,412,43.60
2,multi_country_evidence,29,3.07
3,multi_code_same_country,20,2.12
4,unresolved_only,10,1.06
5,single_country_with_unresolved_code,6,0.63
6,multi_country_with_unresolved_code,2,0.21



RAW MULTI-CODE REASSESSMENT
Books originally containing multiple MARC codes: 57


,normalized_geography_status,book_count
0,multi_country_evidence,29
1,multi_code_same_country,20
2,single_country_with_unresolved_code,6
3,multi_country_with_unresolved_code,2



BOOKS WITH MULTIPLE NORMALIZED COUNTRIES
Books: 31


,book_id,earliest_observed_year,earliest_country_codes,resolved_marc_labels,normalized_parent_countries,unresolved_geo_codes,normalized_geography_status
1,BOOK00002,1981,"['xxk', 'nju']","[United Kingdom, New Jersey]","[United Kingdom, United States]",[],multi_country_evidence
24,BOOK00025,2002,"['ch', 'nyu']","[China (Republic : 1949- ), New York (State)]","[Taiwan, United States]",[],multi_country_evidence
42,BOOK00043,1985,"['ch', 'nyu']","[China (Republic : 1949- ), New York (State)]","[Taiwan, United States]",[],multi_country_evidence
99,BOOK00100,1991,"['vau', 'enk']","[Virginia, England]","[United States, United Kingdom]",[],multi_country_evidence
100,BOOK00101,1953,"['xxu', 'xx', 'xxk']","[United States, No place, unknown, or undeterm...","[United States, United Kingdom]",[xx],multi_country_with_unresolved_code
103,BOOK00104,1967,"['xxk', 'enk', 'nyu']","[United Kingdom, England, New York (State)]","[United Kingdom, United States]",[],multi_country_evidence
112,BOOK00113,1950,"['xxk', 'nyu']","[United Kingdom, New York (State)]","[United Kingdom, United States]",[],multi_country_evidence
147,BOOK00149,1998,"['vtu', 'enk']","[Vermont, England]","[United States, United Kingdom]",[],multi_country_evidence
286,BOOK00289,1993,"['nyu', 'onc', 'enk']","[New York (State), Ontario, England]","[United States, Canada, United Kingdom]",[],multi_country_evidence
298,BOOK00301,1911,"['nyu', 'xxk']","[New York (State), United Kingdom]","[United States, United Kingdom]",[],multi_country_evidence



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_earliest_publication_geography_normalized.csv


#### 18.35.7 — Final Earliest-Observed Publication Geography

After MARC parent-country normalization, the earliest-edition geographic evidence is converted into application-ready publication-geography fields.

A single country is assigned only when the available evidence resolves to exactly one normalized parent country.

When multiple normalized countries occur in the same earliest observed publication year, all countries are retained rather than selecting one arbitrarily.

Books with no usable country evidence remain missing.

The resulting fields describe **earliest observed publication geography in the available Open Library edition history**. They do not claim to identify the verified original publication country of the work.

This distinction is retained throughout LeadWise to prevent bibliographic evidence from being presented with greater certainty than the source supports.

In [64]:
# ---------------------------------------------------------
# 18.35.7 Final application-ready publication geography
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load normalized earliest-publication geography
# ---------------------------------------------------------

input_path = (
    data_root
    / "processed"
    / "openlibrary_earliest_publication_geography_normalized.csv"
)


publication_geo = pd.read_csv(
    input_path
)


# ---------------------------------------------------------
# Parse saved list fields
# ---------------------------------------------------------

def parse_saved_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if (
        text == ""
        or text.lower() == "nan"
        or text == "[]"
    ):
        return []

    try:

        parsed = ast.literal_eval(
            text
        )

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):

        return [text]


publication_geo[
    "normalized_parent_countries"
] = (
    publication_geo[
        "normalized_parent_countries"
    ]
    .apply(
        parse_saved_list
    )
)


publication_geo[
    "unresolved_geo_codes"
] = (
    publication_geo[
        "unresolved_geo_codes"
    ]
    .apply(
        parse_saved_list
    )
)


# ---------------------------------------------------------
# Application-ready country fields
# ---------------------------------------------------------

def single_country(countries):

    if len(countries) == 1:
        return countries[0]

    return None


def country_display(countries):

    if len(countries) == 0:
        return None

    return "; ".join(
        countries
    )


publication_geo[
    "earliest_observed_publication_country"
] = (
    publication_geo[
        "normalized_parent_countries"
    ]
    .apply(
        single_country
    )
)


publication_geo[
    "earliest_observed_publication_countries"
] = (
    publication_geo[
        "normalized_parent_countries"
    ]
    .apply(
        country_display
    )
)


# ---------------------------------------------------------
# Confidence / evidence category
# ---------------------------------------------------------

def publication_geo_evidence_type(row):

    countries = row[
        "normalized_parent_countries"
    ]

    unresolved = row[
        "unresolved_geo_codes"
    ]

    if (
        len(countries) == 0
        and len(unresolved) == 0
    ):
        return "no_country_evidence"

    if (
        len(countries) == 0
        and len(unresolved) > 0
    ):
        return "unresolved_only"

    if (
        len(countries) == 1
        and len(unresolved) == 0
    ):
        return "single_country"

    if (
        len(countries) == 1
        and len(unresolved) > 0
    ):
        return "single_country_with_unresolved_evidence"

    if (
        len(countries) > 1
        and len(unresolved) == 0
    ):
        return "multiple_countries"

    if (
        len(countries) > 1
        and len(unresolved) > 0
    ):
        return "multiple_countries_with_unresolved_evidence"

    return "other"


publication_geo[
    "publication_geo_evidence_type"
] = (
    publication_geo.apply(
        publication_geo_evidence_type,
        axis=1,
    )
)


# ---------------------------------------------------------
# Application-safe provenance label
# ---------------------------------------------------------

publication_geo[
    "publication_geo_provenance"
] = (
    "Open Library earliest observed edition year "
    + publication_geo[
        "earliest_observed_year"
    ]
    .astype("Int64")
    .astype(str)
)


# ---------------------------------------------------------
# Final compact application table
# ---------------------------------------------------------

publication_geo_final = (
    publication_geo[
        [
            "book_id",
            "earliest_observed_year",
            "earliest_observed_publication_country",
            "earliest_observed_publication_countries",
            "publication_geo_evidence_type",
            "unresolved_geo_codes",
            "publication_geo_provenance",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "FINAL PUBLICATION-GEOGRAPHY EVIDENCE"
)
print("=" * 70)

print(
    "Books:",
    len(
        publication_geo_final
    )
)

print(
    "Single-country field available:",
    publication_geo_final[
        "earliest_observed_publication_country"
    ]
    .notna()
    .sum()
)

print(
    "Any resolved country evidence:",
    publication_geo_final[
        "earliest_observed_publication_countries"
    ]
    .notna()
    .sum()
)


print(
    "\nEVIDENCE TYPES"
)
print("=" * 70)

display(
    publication_geo_final[
        "publication_geo_evidence_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "publication_geo_evidence_type"
    )
    .reset_index(
        name="book_count"
    )
)


# ---------------------------------------------------------
# Sample records
# ---------------------------------------------------------

print(
    "\nAPPLICATION-READY SAMPLE"
)
print("=" * 70)

display(
    publication_geo_final[
        publication_geo_final[
            "earliest_observed_publication_countries"
        ].notna()
    ]
    .head(20)
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "leadwise_publication_geography_evidence.csv"
)


publication_geo_final.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

FINAL PUBLICATION-GEOGRAPHY EVIDENCE
Books: 945
Single-country field available: 438
Any resolved country evidence: 469

EVIDENCE TYPES


,publication_geo_evidence_type,book_count
0,no_country_evidence,466
1,single_country,432
2,multiple_countries,29
3,unresolved_only,10
4,single_country_with_unresolved_evidence,6
5,multiple_countries_with_unresolved_evidence,2



APPLICATION-READY SAMPLE


,book_id,earliest_observed_year,earliest_observed_publication_country,earliest_observed_publication_countries,publication_geo_evidence_type,unresolved_geo_codes,publication_geo_provenance
1,BOOK00002,1981,NaN,United Kingdom; United States,multiple_countries,[],Open Library earliest observed edition year 1981
2,BOOK00003,1977,Indonesia,Indonesia,single_country,[],Open Library earliest observed edition year 1977
3,BOOK00004,1967,United States,United States,single_country,[],Open Library earliest observed edition year 1967
4,BOOK00005,1997,United States,United States,single_country,[],Open Library earliest observed edition year 1997
5,BOOK00006,1998,United States,United States,single_country,[],Open Library earliest observed edition year 1998
7,BOOK00008,1985,United States,United States,single_country,[],Open Library earliest observed edition year 1985
8,BOOK00009,2018,United States,United States,single_country,[],Open Library earliest observed edition year 2018
11,BOOK00012,1993,United States,United States,single_country,[],Open Library earliest observed edition year 1993
12,BOOK00013,1992,United States,United States,single_country,[],Open Library earliest observed edition year 1992
13,BOOK00014,1978,United States,United States,single_country,[],Open Library earliest observed edition year 1978



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_publication_geography_evidence.csv


### 18.36 — Original Language and Translation Evidence

LeadWise already contains language metadata collected from Open Library editions. However, the language of an observed edition is not necessarily the original language of the work.

For example, the presence of English, German, and French editions establishes multilingual publication evidence but does not by itself identify which language was used for the original work.

This enrichment stage therefore maintains a strict distinction between:

- **Observed publication language** — language explicitly attached to an edition;
- **Original language** — language explicitly identified as the original language of the work;
- **Translation evidence** — evidence that editions exist in languages different from a verified original language; and
- **International language reach** — the number and diversity of observed publication languages, without claiming that each represents a verified translation.

No original language is inferred from author nationality, publication country, title, publisher, ISBN, or the most common edition language.

The first step audits the existing Open Library work and edition data to determine whether explicit original-language metadata is already available before additional external enrichment is attempted.

In [65]:
# ---------------------------------------------------------
# 18.36.1 Audit existing language-related metadata
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd


# ---------------------------------------------------------
# Candidate Open Library files
# ---------------------------------------------------------

candidate_files = [
    data_root / "final" / "books_master.csv",
    data_root / "final" / "openlibrary_integrated.csv",
    data_root / "processed" / "openlibrary_work_clean.csv",
    data_root / "processed" / "openlibrary_book_edition_intelligence.csv",
]


print(
    "EXISTING LANGUAGE-METADATA AUDIT"
)
print("=" * 70)


audit_rows = []


for path in candidate_files:

    if not path.exists():

        print(
            f"\nNOT FOUND: {path}"
        )

        continue


    df = pd.read_csv(
        path,
        low_memory=False,
    )


    language_columns = [
        column
        for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in [
                "language",
                "translation",
                "original",
                "locale",
            ]
        )
    ]


    print(
        f"\nFILE: {path.name}"
    )

    print(
        "Shape:",
        df.shape
    )

    print(
        "Language/original/translation columns:"
    )

    print(
        language_columns
        if language_columns
        else "None"
    )


    for column in language_columns:

        non_null = (
            df[column]
            .notna()
            .sum()
        )

        meaningful = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
            .replace(
                {
                    "": pd.NA,
                    "[]": pd.NA,
                    "{}": pd.NA,
                    "nan": pd.NA,
                    "None": pd.NA,
                }
            )
            .notna()
            .sum()
        )


        audit_rows.append(
            {
                "file":
                    path.name,

                "column":
                    column,

                "rows":
                    len(df),

                "non_null":
                    non_null,

                "meaningful_values":
                    meaningful,

                "coverage_pct":
                    round(
                        meaningful
                        / len(df)
                        * 100,
                        2,
                    )
                    if len(df)
                    else 0,
            }
        )


# ---------------------------------------------------------
# Audit table
# ---------------------------------------------------------

language_audit = pd.DataFrame(
    audit_rows
)


print(
    "\nLANGUAGE FIELD COVERAGE"
)
print("=" * 70)


if len(language_audit):

    display(
        language_audit
        .sort_values(
            [
                "file",
                "coverage_pct",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )

else:

    print(
        "No language-related fields found."
    )


# ---------------------------------------------------------
# Search ALL column names for potentially relevant
# work-level fields that may not contain 'language'
# ---------------------------------------------------------

print(
    "\nOPEN LIBRARY WORK-LEVEL COLUMN INVENTORY"
)
print("=" * 70)


work_candidates = [
    path
    for path in candidate_files
    if (
        path.exists()
        and "work" in path.name.lower()
    )
]


for path in work_candidates:

    df = pd.read_csv(
        path,
        nrows=5,
        low_memory=False,
    )

    print(
        f"\n{path.name}"
    )

    for column in df.columns:

        print(
            " -",
            column
        )


# ---------------------------------------------------------
# Save audit
# ---------------------------------------------------------

audit_path = (
    data_root
    / "processed"
    / "leadwise_language_metadata_audit.csv"
)


language_audit.to_csv(
    audit_path,
    index=False,
)


print(
    "\nSaved:",
    audit_path
)

EXISTING LANGUAGE-METADATA AUDIT

FILE: books_master.csv
Shape: (2067, 18)
Language/original/translation columns:
None

FILE: openlibrary_integrated.csv
Shape: (950, 50)
Language/original/translation columns:
['languages']

NOT FOUND: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_work_clean.csv

FILE: openlibrary_book_edition_intelligence.csv
Shape: (948, 28)
Language/original/translation columns:
['known_publication_languages_display', 'known_publication_language_count', 'known_publication_language_codes', 'known_publication_languages']

LANGUAGE FIELD COVERAGE


,file,column,rows,non_null,meaningful_values,coverage_pct
0,openlibrary_book_edition_intelligence.csv,known_publication_language_count,948,948,948,100.00
1,openlibrary_book_edition_intelligence.csv,known_publication_languages_display,948,900,900,94.94
2,openlibrary_book_edition_intelligence.csv,known_publication_language_codes,948,948,900,94.94
3,openlibrary_book_edition_intelligence.csv,known_publication_languages,948,948,900,94.94
4,openlibrary_integrated.csv,languages,950,950,902,94.95



OPEN LIBRARY WORK-LEVEL COLUMN INVENTORY

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_language_metadata_audit.csv


#### 18.36.2 — Multilingual Publication and International Language Reach

The existing Open Library metadata does not provide an explicit original-language field.

Accordingly, observed edition languages are not interpreted as original languages or automatically classified as translations.

Instead, the edition history is used to measure **international language reach**: the number and diversity of explicitly observed publication languages associated with each book.

A book appearing in multiple publication languages provides evidence of multilingual publication, but this alone does not establish:

- which language was original;
- which editions were translations;
- the number of distinct translations; or
- the direction of translation.

This stage therefore quantifies multilingual publication evidence while preserving those limitations.

In [66]:
# ---------------------------------------------------------
# 18.36.2 Audit multilingual publication evidence
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load edition-language intelligence
# ---------------------------------------------------------

language_path = (
    data_root
    / "processed"
    / "openlibrary_book_edition_intelligence.csv"
)


language_df = pd.read_csv(
    language_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Parse saved list fields
# ---------------------------------------------------------

def parse_language_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if (
        text == ""
        or text.lower() == "nan"
        or text == "[]"
    ):
        return []

    try:

        parsed = ast.literal_eval(
            text
        )

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):

        return [text]


language_df[
    "publication_languages_list"
] = (
    language_df[
        "known_publication_languages"
    ]
    .apply(
        parse_language_list
    )
)


# ---------------------------------------------------------
# Recalculate count independently
# ---------------------------------------------------------

language_df[
    "verified_language_count"
] = (
    language_df[
        "publication_languages_list"
    ]
    .apply(
        lambda values:
            len(
                set(
                    str(value).strip()
                    for value in values
                    if str(value).strip()
                )
            )
    )
)


language_df[
    "has_language_evidence"
] = (
    language_df[
        "verified_language_count"
    ] > 0
)


language_df[
    "has_multilingual_publication_evidence"
] = (
    language_df[
        "verified_language_count"
    ] > 1
)


# ---------------------------------------------------------
# Coverage summary
# ---------------------------------------------------------

total_books = len(
    language_df
)

books_with_language = (
    language_df[
        "has_language_evidence"
    ]
    .sum()
)

multilingual_books = (
    language_df[
        "has_multilingual_publication_evidence"
    ]
    .sum()
)


print(
    "PUBLICATION-LANGUAGE EVIDENCE"
)
print("=" * 70)

print(
    "Books:",
    total_books
)

print(
    "Books with language evidence:",
    books_with_language
)

print(
    "Coverage:",
    round(
        books_with_language
        / total_books
        * 100,
        2,
    ),
    "%"
)

print(
    "Books with >1 observed publication language:",
    multilingual_books
)

print(
    "Multilingual share of all books:",
    round(
        multilingual_books
        / total_books
        * 100,
        2,
    ),
    "%"
)

print(
    "Multilingual share among books with language evidence:",
    round(
        multilingual_books
        / books_with_language
        * 100,
        2,
    ),
    "%"
)


# ---------------------------------------------------------
# Distribution of language counts
# ---------------------------------------------------------

print(
    "\nOBSERVED PUBLICATION-LANGUAGE COUNT"
)
print("=" * 70)


language_count_distribution = (
    language_df[
        "verified_language_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "observed_language_count"
    )
    .reset_index(
        name="book_count"
    )
)


language_count_distribution[
    "percentage"
] = (
    language_count_distribution[
        "book_count"
    ]
    / total_books
    * 100
).round(2)


display(
    language_count_distribution
)


# ---------------------------------------------------------
# Books with greatest multilingual reach
# ---------------------------------------------------------

print(
    "\nBOOKS WITH HIGHEST OBSERVED LANGUAGE REACH"
)
print("=" * 70)


display_columns = [
    column
    for column in [
        "book_id",
        "canonical_title",
        "known_publication_languages_display",
        "verified_language_count",
        "edition_count",
    ]
    if column in language_df.columns
]


multilingual_top = (
    language_df[
        language_df[
            "verified_language_count"
        ] > 1
    ]
    .sort_values(
        [
            "verified_language_count",
        ],
        ascending=False,
    )
)


display(
    multilingual_top[
        display_columns
    ]
    .head(30)
)


# ---------------------------------------------------------
# Language frequency
# ---------------------------------------------------------

language_exploded = (
    language_df[
        [
            "book_id",
            "publication_languages_list",
        ]
    ]
    .explode(
        "publication_languages_list"
    )
)


language_exploded[
    "publication_language"
] = (
    language_exploded[
        "publication_languages_list"
    ]
    .astype(str)
    .str.strip()
)


language_exploded = (
    language_exploded[
        language_exploded[
            "publication_language"
        ].ne("")
    ]
)


language_frequency = (
    language_exploded[
        "publication_language"
    ]
    .value_counts()
    .rename_axis(
        "publication_language"
    )
    .reset_index(
        name="book_count"
    )
)


print(
    "\nOBSERVED PUBLICATION LANGUAGES"
)
print("=" * 70)


display(
    language_frequency
    .head(30)
)


# ---------------------------------------------------------
# Save app-ready language evidence
# ---------------------------------------------------------

language_evidence = (
    language_df[
        [
            column
            for column in [
                "book_id",
                "known_publication_languages_display",
                "verified_language_count",
                "has_multilingual_publication_evidence",
            ]
            if column in language_df.columns
        ]
    ]
    .copy()
)


language_evidence = (
    language_evidence
    .rename(
        columns={
            "verified_language_count":
                "observed_publication_language_count"
        }
    )
)


output_path = (
    data_root
    / "processed"
    / "leadwise_publication_language_evidence.csv"
)


language_evidence.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

PUBLICATION-LANGUAGE EVIDENCE
Books: 948
Books with language evidence: 900
Coverage: 94.94 %
Books with >1 observed publication language: 40
Multilingual share of all books: 4.22 %
Multilingual share among books with language evidence: 4.44 %

OBSERVED PUBLICATION-LANGUAGE COUNT


,observed_language_count,book_count,percentage
0,0,48,5.06
1,1,860,90.72
2,2,23,2.43
3,3,12,1.27
4,4,1,0.11
5,5,2,0.21
6,6,1,0.11
7,9,1,0.11



BOOKS WITH HIGHEST OBSERVED LANGUAGE REACH


,book_id,canonical_title,known_publication_languages_display,verified_language_count
331,BOOK00333,The 7 Habits of Highly Effective People,"English, Portuguese, Chinese, Vietnamese, Span...",9
807,BOOK00809,Nonviolent Communication,"French, Spanish, Germanic languages, German, E...",6
856,BOOK00859,Working with Emotional Intelligence,"English, Italian, Spanish, Polish, Chinese",5
279,BOOK00281,Marketing management,"English, French, German, Chinese, Norwegian",5
349,BOOK00351,The People of the Abyss,"French, Finnish, English, Spanish",4
5,BOOK00006,The 21 Irrefutable Laws of Leadership,"Spanish, Mandarin Chinese, English",3
469,BOOK00471,Rework,"Romanian, French, English",3
287,BOOK00289,The Night Manager,"English, Chinese, German",3
148,BOOK00150,Team of Rivals,"English, Chinese, Korean",3
929,BOOK00932,Innovation and Entrepreneurship,"English, Chinese, Spanish",3



OBSERVED PUBLICATION LANGUAGES


,publication_language,book_count
0,English,893
1,Spanish,22
2,Chinese,15
3,German,10
4,French,5
5,Korean,3
6,Japanese,3
7,Indonesian,2
8,Russian,2
9,Vietnamese,2



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_publication_language_evidence.csv


#### 18.36.3 — Structured Original-Language Evidence Strategy

Observed edition languages provide strong evidence of publication-language reach but do not identify the original language of a work.

To obtain original-language evidence, LeadWise next evaluates Wikidata as a structured work-level metadata source.

The enrichment follows a conservative identity-resolution strategy. A Wikidata item is not accepted solely because its title resembles a LeadWise title.

Candidate work identities must be supported by structured identifiers or sufficiently strong bibliographic evidence before language claims are attached to a LeadWise book.

The objective is to determine whether Wikidata can provide explicit work-level language evidence for a subset of the catalog without inferring original language from:

- the most common edition language;
- author nationality;
- publication geography;
- book title;
- publisher;
- ISBN prefix; or
- the language of a particular edition.

Books without sufficiently verified work identities remain without an original-language assignment.

In [67]:
# ---------------------------------------------------------
# 18.36.3 Audit identifiers available for
# structured work-level language enrichment
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Load Open Library integrated catalog
# ---------------------------------------------------------

ol_path = (
    data_root
    / "final"
    / "openlibrary_integrated.csv"
)


ol_books = pd.read_csv(
    ol_path,
    low_memory=False,
)


print(
    "OPEN LIBRARY IDENTIFIER AUDIT"
)
print("=" * 70)

print(
    "Books:",
    len(ol_books)
)


# ---------------------------------------------------------
# Identify potentially useful identifier columns
# ---------------------------------------------------------

identifier_keywords = [
    "key",
    "work",
    "isbn",
    "oclc",
    "lccn",
    "wikidata",
    "goodreads",
    "librarything",
]


identifier_columns = [
    column
    for column in ol_books.columns
    if any(
        keyword in column.lower()
        for keyword in identifier_keywords
    )
]


print(
    "\nPotential identifier columns:"
)

for column in identifier_columns:
    print(
        " -",
        column
    )


# ---------------------------------------------------------
# Coverage
# ---------------------------------------------------------

audit_rows = []


for column in identifier_columns:

    series = (
        ol_books[column]
        .astype("string")
        .str.strip()
    )

    meaningful = (
        series
        .notna()
        & series.ne("")
        & series.ne("[]")
        & series.ne("{}")
        & series.str.lower().ne("nan")
        & series.str.lower().ne("none")
    )


    audit_rows.append(
        {
            "column":
                column,

            "books_with_value":
                int(
                    meaningful.sum()
                ),

            "coverage_pct":
                round(
                    meaningful.mean()
                    * 100,
                    2,
                ),
        }
    )


identifier_audit = (
    pd.DataFrame(
        audit_rows
    )
    .sort_values(
        "coverage_pct",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nIDENTIFIER COVERAGE"
)
print("=" * 70)

display(
    identifier_audit
)


# ---------------------------------------------------------
# Inspect all columns as a safeguard
# ---------------------------------------------------------

print(
    "\nFULL OPEN LIBRARY COLUMN INVENTORY"
)
print("=" * 70)

for column in ol_books.columns:
    print(
        " -",
        column
    )


# ---------------------------------------------------------
# Inspect sample identifier records
# ---------------------------------------------------------

sample_columns = [
    column
    for column in [
        "book_id",
        "canonical_title",
        "title",
        "authors",
        "openlibrary_work_key",
        "work_key",
        "key",
        "isbn13",
        "isbn10",
        "lccn",
        "oclc",
        "wikidata",
    ]
    if column in ol_books.columns
]


print(
    "\nIDENTIFIER SAMPLE"
)
print("=" * 70)


display(
    ol_books[
        sample_columns
    ]
    .head(20)
)


# ---------------------------------------------------------
# Save audit
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "leadwise_work_identifier_audit.csv"
)


identifier_audit.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

OPEN LIBRARY IDENTIFIER AUDIT
Books: 950

Potential identifier columns:
 - openlibrary_key
 - author_keys
 - isbn_10
 - isbn_13
 - all_isbns
 - work_status_code
 - work_subjects
 - work_covers
 - isbn13_count

IDENTIFIER COVERAGE


,column,books_with_value,coverage_pct
0,openlibrary_key,950,100.00
1,work_status_code,950,100.00
2,isbn13_count,950,100.00
3,author_keys,942,99.16
4,isbn_13,915,96.32
5,all_isbns,915,96.32
6,isbn_10,903,95.05
7,work_subjects,826,86.95
8,work_covers,743,78.21



FULL OPEN LIBRARY COLUMN INVENTORY
 - openlibrary_key
 - title
 - authors
 - author_keys
 - first_publish_year
 - publish_dates
 - publishers
 - isbn_10
 - isbn_13
 - all_isbns
 - languages
 - subjects
 - edition_count
 - ratings_average
 - ratings_count
 - ratings_count_1
 - ratings_count_2
 - ratings_count_3
 - ratings_count_4
 - ratings_count_5
 - want_to_read_count
 - currently_reading_count
 - already_read_count
 - cover_id
 - cover_url
 - ebook_access
 - has_fulltext
 - public_scan
 - source
 - source_url
 - collection_queries
 - query_match_count
 - work_status_code
 - description
 - first_sentence
 - work_subjects
 - subject_places
 - subject_people
 - subject_times
 - excerpts
 - lc_classifications
 - dewey_number
 - work_covers
 - source_openlibrary
 - source_leadershipnow
 - integration_source
 - isbn13_count
 - title_normalized
 - authors_normalized
 - book_id

IDENTIFIER SAMPLE


,book_id,title,authors
0,BOOK00001,Principle-Centered Leadership,['Stephen R. Covey']
1,BOOK00002,Leadership in Organizations,['Gary A. Yukl']
2,BOOK00003,Kepemimpinan =,['Karjadi M.']
3,BOOK00004,Spiritual leadership,['J. Oswald Sanders']
4,BOOK00005,Leadership,['Peter Guy Northouse']
5,BOOK00006,The 21 Irrefutable Laws of Leadership,['John C. Maxwell']
6,BOOK00007,Leadership,['Peter G. Northouse']
7,BOOK00008,Leadership and performance beyond expectations,['Bernard M. Bass']
8,BOOK00009,A Higher Loyalty,"['James Comey', 'James B. Comey', 'James Comey']"
9,BOOK00010,Leadership and Self Deception,"['The Arbinger Institute', 'Dick Ruhe']"



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_work_identifier_audit.csv


#### 18.36.4 — ISBN Identity Bridge for Structured Language Enrichment

The Open Library catalog contains strong bibliographic identifiers that can support more reliable external entity resolution.

Rather than matching books to Wikidata using title similarity alone, this stage constructs an ISBN identity bridge from the ISBN-10 and ISBN-13 values already associated with each Open Library work.

ISBN matching provides substantially stronger evidence than title-only matching because titles may be generic, repeated across different works, or represented by multiple editions.

The bridge retains:

- LeadWise `book_id`;
- Open Library work key;
- title and author context;
- ISBN-10 identifiers;
- ISBN-13 identifiers; and
- the source field from which each identifier was obtained.

This bridge does not itself establish a Wikidata match or an original language. It only prepares verified bibliographic identifiers for subsequent structured entity resolution.

In [68]:
# ---------------------------------------------------------
# 18.36.4 Build ISBN identity bridge
# ---------------------------------------------------------

import ast
import re
import pandas as pd


# ---------------------------------------------------------
# Reload Open Library integrated data
# ---------------------------------------------------------

ol_path = (
    data_root
    / "final"
    / "openlibrary_integrated.csv"
)

ol_books = pd.read_csv(
    ol_path,
    low_memory=False,
)


# ---------------------------------------------------------
# ISBN helpers
# ---------------------------------------------------------

def parse_identifier_values(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        values = value

    else:

        text = str(value).strip()

        if (
            text == ""
            or text.lower() == "nan"
            or text == "[]"
        ):
            return []

        try:
            parsed = ast.literal_eval(text)

            if isinstance(parsed, list):
                values = parsed
            else:
                values = [parsed]

        except (ValueError, SyntaxError):
            values = [text]


    cleaned = []

    for item in values:

        if pd.isna(item):
            continue

        identifier = re.sub(
            r"[^0-9Xx]",
            "",
            str(item),
        ).upper()

        if identifier:
            cleaned.append(identifier)


    return list(
        dict.fromkeys(cleaned)
    )


# ---------------------------------------------------------
# Collect ISBN relationships
# ---------------------------------------------------------

bridge_rows = []


for _, row in ol_books.iterrows():

    base = {
        "book_id":
            row["book_id"],

        "openlibrary_key":
            row["openlibrary_key"],

        "title":
            row["title"],

        "authors":
            row["authors"],
    }


    # ISBN-10
    for isbn in parse_identifier_values(
        row["isbn_10"]
    ):

        bridge_rows.append(
            {
                **base,
                "isbn":
                    isbn,
                "isbn_type":
                    "ISBN-10",
                "isbn_source":
                    "isbn_10",
            }
        )


    # ISBN-13
    for isbn in parse_identifier_values(
        row["isbn_13"]
    ):

        bridge_rows.append(
            {
                **base,
                "isbn":
                    isbn,
                "isbn_type":
                    "ISBN-13",
                "isbn_source":
                    "isbn_13",
            }
        )


    # all_isbns may contain either form
    for isbn in parse_identifier_values(
        row["all_isbns"]
    ):

        if len(isbn) == 10:
            isbn_type = "ISBN-10"

        elif len(isbn) == 13:
            isbn_type = "ISBN-13"

        else:
            isbn_type = "other"

        bridge_rows.append(
            {
                **base,
                "isbn":
                    isbn,
                "isbn_type":
                    isbn_type,
                "isbn_source":
                    "all_isbns",
            }
        )


isbn_bridge_raw = pd.DataFrame(
    bridge_rows
)


# ---------------------------------------------------------
# Remove duplicate book–ISBN relationships
# ---------------------------------------------------------

isbn_bridge = (
    isbn_bridge_raw
    .sort_values(
        [
            "book_id",
            "isbn",
            "isbn_source",
        ]
    )
    .drop_duplicates(
        subset=[
            "book_id",
            "isbn",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Basic structural validation
# ---------------------------------------------------------

isbn_bridge[
    "isbn_length"
] = (
    isbn_bridge[
        "isbn"
    ]
    .str.len()
)


isbn_bridge[
    "structurally_valid_length"
] = (
    (
        isbn_bridge[
            "isbn_type"
        ].eq("ISBN-10")
        & isbn_bridge[
            "isbn_length"
        ].eq(10)
    )
    |
    (
        isbn_bridge[
            "isbn_type"
        ].eq("ISBN-13")
        & isbn_bridge[
            "isbn_length"
        ].eq(13)
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "ISBN IDENTITY BRIDGE"
)
print("=" * 70)

print(
    "Book-ISBN relationships:",
    len(isbn_bridge)
)

print(
    "Books represented:",
    isbn_bridge[
        "book_id"
    ].nunique()
)

print(
    "Unique ISBNs:",
    isbn_bridge[
        "isbn"
    ].nunique()
)

print(
    "ISBN-10 relationships:",
    isbn_bridge[
        "isbn_type"
    ].eq(
        "ISBN-10"
    ).sum()
)

print(
    "ISBN-13 relationships:",
    isbn_bridge[
        "isbn_type"
    ].eq(
        "ISBN-13"
    ).sum()
)

print(
    "Other-length identifiers:",
    isbn_bridge[
        "isbn_type"
    ].eq(
        "other"
    ).sum()
)


# ---------------------------------------------------------
# ISBNs shared across different LeadWise books
# ---------------------------------------------------------

isbn_book_counts = (
    isbn_bridge
    .groupby(
        "isbn"
    )[
        "book_id"
    ]
    .nunique()
    .reset_index(
        name="leadwise_book_count"
    )
)


shared_isbns = (
    isbn_book_counts[
        isbn_book_counts[
            "leadwise_book_count"
        ] > 1
    ]
    .sort_values(
        "leadwise_book_count",
        ascending=False,
    )
)


print(
    "\nCROSS-BOOK ISBN CONFLICTS"
)
print("=" * 70)

print(
    "ISBNs attached to >1 LeadWise book:",
    len(shared_isbns)
)

display(
    shared_isbns.head(30)
)


# ---------------------------------------------------------
# ISBN type distribution
# ---------------------------------------------------------

print(
    "\nISBN TYPE DISTRIBUTION"
)
print("=" * 70)

display(
    isbn_bridge[
        "isbn_type"
    ]
    .value_counts()
    .rename_axis(
        "isbn_type"
    )
    .reset_index(
        name="relationships"
    )
)


# ---------------------------------------------------------
# Sample
# ---------------------------------------------------------

print(
    "\nISBN BRIDGE SAMPLE"
)
print("=" * 70)

display(
    isbn_bridge[
        [
            "book_id",
            "openlibrary_key",
            "title",
            "authors",
            "isbn",
            "isbn_type",
            "isbn_source",
        ]
    ]
    .head(30)
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "openlibrary_isbn_identity_bridge.csv"
)


isbn_bridge.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

ISBN IDENTITY BRIDGE
Book-ISBN relationships: 12057
Books represented: 915
Unique ISBNs: 11987
ISBN-10 relationships: 6007
ISBN-13 relationships: 6047
Other-length identifiers: 3

CROSS-BOOK ISBN CONFLICTS
ISBNs attached to >1 LeadWise book: 70


,isbn,leadwise_book_count
339,0071127305,2
6814,9780131503465,2
7819,9780314631497,2
7290,9780136589075,2
7228,9780136065616,2
7223,9780136015703,2
6882,9780132343527,2
6836,9780131869554,2
6801,9780131441392,2
8113,9780415771764,2



ISBN TYPE DISTRIBUTION


,isbn_type,relationships
0,ISBN-13,6047
1,ISBN-10,6007
2,other,3



ISBN BRIDGE SAMPLE


,book_id,openlibrary_key,title,authors,isbn,isbn_type,isbn_source
0,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],002863912X,ISBN-10,all_isbns
1,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671011138,ISBN-10,all_isbns
2,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671317032,ISBN-10,all_isbns
3,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671711164,ISBN-10,all_isbns
4,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671711350,ISBN-10,all_isbns
5,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671749102,ISBN-10,all_isbns
6,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671755455,ISBN-10,all_isbns
7,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0671792806,ISBN-10,all_isbns
8,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],068485841X,ISBN-10,all_isbns
9,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,['Stephen R. Covey'],0743468600,ISBN-10,all_isbns



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_isbn_identity_bridge.csv


#### 18.36.5 — Cross-Book ISBN Conflict Audit

The ISBN identity bridge contains a small number of ISBNs associated with more than one LeadWise/Open Library book record.

Because ISBN is intended to identify a specific publication or edition, an ISBN shared across different work records cannot automatically be treated as an unambiguous external identity key.

Before ISBN-based Wikidata matching, these conflicts are audited using:

- LeadWise book ID;
- Open Library work key;
- title;
- authors;
- ISBN;
- normalized title; and
- whether the conflicting records represent the same or different Open Library works.

The audit does not automatically merge records. Its purpose is to determine which ISBN relationships are safe for high-confidence external entity resolution.

Malformed or non-standard-length identifiers are also isolated and excluded from subsequent ISBN matching.

In [69]:
# ---------------------------------------------------------
# 18.36.5 Audit cross-book ISBN conflicts
# ---------------------------------------------------------

import re
import pandas as pd


# ---------------------------------------------------------
# Load ISBN bridge
# ---------------------------------------------------------

bridge_path = (
    data_root
    / "processed"
    / "openlibrary_isbn_identity_bridge.csv"
)

isbn_bridge = pd.read_csv(
    bridge_path,
    dtype={
        "isbn": "string"
    },
    low_memory=False,
)


# ---------------------------------------------------------
# Simple normalization for comparison only
# ---------------------------------------------------------

def normalize_compare_text(value):

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


isbn_bridge[
    "title_compare"
] = (
    isbn_bridge[
        "title"
    ]
    .apply(
        normalize_compare_text
    )
)


isbn_bridge[
    "authors_compare"
] = (
    isbn_bridge[
        "authors"
    ]
    .apply(
        normalize_compare_text
    )
)


# ---------------------------------------------------------
# Identify ISBNs assigned to >1 LeadWise book
# ---------------------------------------------------------

isbn_book_counts = (
    isbn_bridge
    .groupby(
        "isbn"
    )[
        "book_id"
    ]
    .nunique()
    .reset_index(
        name="leadwise_book_count"
    )
)


conflict_isbns = set(
    isbn_book_counts.loc[
        isbn_book_counts[
            "leadwise_book_count"
        ] > 1,
        "isbn",
    ]
)


conflict_rows = (
    isbn_bridge[
        isbn_bridge[
            "isbn"
        ].isin(
            conflict_isbns
        )
    ]
    .copy()
    .sort_values(
        [
            "isbn",
            "book_id",
        ]
    )
)


# ---------------------------------------------------------
# Summarize each conflicting ISBN
# ---------------------------------------------------------

conflict_summary = (
    conflict_rows
    .groupby(
        "isbn"
    )
    .agg(
        leadwise_book_count=(
            "book_id",
            "nunique",
        ),

        openlibrary_work_count=(
            "openlibrary_key",
            "nunique",
        ),

        normalized_title_count=(
            "title_compare",
            "nunique",
        ),

        normalized_author_count=(
            "authors_compare",
            "nunique",
        ),
    )
    .reset_index()
)


def classify_conflict(row):

    if (
        row[
            "openlibrary_work_count"
        ] == 1
    ):
        return "same_openlibrary_work"

    if (
        row[
            "normalized_title_count"
        ] == 1
        and row[
            "normalized_author_count"
        ] == 1
    ):
        return "different_work_keys_same_title_author"

    if (
        row[
            "normalized_title_count"
        ] == 1
    ):
        return "same_title_different_author_metadata"

    return "different_work_records"


conflict_summary[
    "conflict_type"
] = (
    conflict_summary.apply(
        classify_conflict,
        axis=1,
    )
)


# ---------------------------------------------------------
# Overall summary
# ---------------------------------------------------------

print(
    "CROSS-BOOK ISBN CONFLICT AUDIT"
)
print("=" * 70)

print(
    "Conflicting ISBNs:",
    len(
        conflict_summary
    )
)

print(
    "LeadWise books involved:",
    conflict_rows[
        "book_id"
    ].nunique()
)

print(
    "Open Library works involved:",
    conflict_rows[
        "openlibrary_key"
    ].nunique()
)


print(
    "\nCONFLICT CLASSIFICATION"
)
print("=" * 70)

display(
    conflict_summary[
        "conflict_type"
    ]
    .value_counts()
    .rename_axis(
        "conflict_type"
    )
    .reset_index(
        name="isbn_count"
    )
)


# ---------------------------------------------------------
# Detailed conflict examples
# ---------------------------------------------------------

print(
    "\nCONFLICT DETAILS"
)
print("=" * 70)

display(
    conflict_rows[
        [
            "isbn",
            "book_id",
            "openlibrary_key",
            "title",
            "authors",
            "isbn_type",
        ]
    ]
    .head(50)
)


# ---------------------------------------------------------
# Inspect malformed / other identifiers
# ---------------------------------------------------------

other_identifiers = (
    isbn_bridge[
        isbn_bridge[
            "isbn_type"
        ].eq(
            "other"
        )
    ]
    .copy()
)


print(
    "\nNON-STANDARD IDENTIFIERS"
)
print("=" * 70)

print(
    "Records:",
    len(
        other_identifiers
    )
)

display(
    other_identifiers[
        [
            "book_id",
            "openlibrary_key",
            "title",
            "isbn",
            "isbn_type",
            "isbn_source",
        ]
    ]
)


# ---------------------------------------------------------
# Build SAFE ISBN bridge
#
# Exclude:
# 1. ISBNs attached to >1 LeadWise book
# 2. non-standard identifier lengths
# ---------------------------------------------------------

safe_isbn_bridge = (
    isbn_bridge[
        ~isbn_bridge[
            "isbn"
        ].isin(
            conflict_isbns
        )
        &
        isbn_bridge[
            "isbn_type"
        ].isin(
            [
                "ISBN-10",
                "ISBN-13",
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "\nSAFE ISBN BRIDGE"
)
print("=" * 70)

print(
    "Relationships:",
    len(
        safe_isbn_bridge
    )
)

print(
    "Books represented:",
    safe_isbn_bridge[
        "book_id"
    ].nunique()
)

print(
    "Unique ISBNs:",
    safe_isbn_bridge[
        "isbn"
    ].nunique()
)


# ---------------------------------------------------------
# Save audit + safe bridge
# ---------------------------------------------------------

conflict_output = (
    data_root
    / "processed"
    / "openlibrary_isbn_conflict_audit.csv"
)

safe_output = (
    data_root
    / "processed"
    / "openlibrary_isbn_identity_bridge_safe.csv"
)


conflict_rows.to_csv(
    conflict_output,
    index=False,
)

safe_isbn_bridge.to_csv(
    safe_output,
    index=False,
)


print(
    "\nSaved conflict audit:",
    conflict_output
)

print(
    "Saved safe bridge:",
    safe_output
)

CROSS-BOOK ISBN CONFLICT AUDIT
Conflicting ISBNs: 70
LeadWise books involved: 48
Open Library works involved: 48

CONFLICT CLASSIFICATION


,conflict_type,isbn_count
0,same_title_different_author_metadata,36
1,different_work_keys_same_title_author,26
2,different_work_records,8



CONFLICT DETAILS


,isbn,book_id,openlibrary_key,title,authors,isbn_type
8228,0071127305,BOOK00667,/works/OL2714552W,Organizational behavior,"['John W. Newstrom', 'Keith Davis']",ISBN-10
8635,0071127305,BOOK00697,/works/OL1267341W,Organizational behavior,['Keith Davis'],ISBN-10
8745,0072487933,BOOK00708,/works/OL3266746W,Managerial Economics & Business Strategy,"['Michael R. Baye', 'Michael Baye']",ISBN-10
9136,0072487933,BOOK00749,/works/OL15843963W,Managerial economics and business strategy,['Michael R. Baye'],ISBN-10
8154,007353045X,BOOK00663,/works/OL547477W,Organizational behavior,"['Robert Kreitner', 'Angelo Kinicki']",ISBN-10
8567,007353045X,BOOK00691,/works/OL18648685W,Organizational behavior,['Robert Kreitner'],ISBN-10
5989,0130271470,BOOK00493,/works/OL7942381W,Principles of operations management,['Jay Heizer'],ISBN-10
6083,0130271470,BOOK00502,/works/OL1887969W,Principles of operations management,['Jay H. Heizer'],ISBN-10
2955,0130664928,BOOK00298,/works/OL1982791W,Human Resource Management,['Gary Dessler'],ISBN-10
7581,0130664928,BOOK00622,/works/OL15643306W,Human resource management,['Gary Dessler'],ISBN-10



NON-STANDARD IDENTIFIERS
Records: 3


,book_id,openlibrary_key,title,isbn,isbn_type,isbn_source
5648,BOOK00472,/works/OL4643074W,Production and operations management,0137718008X,other,all_isbns
8706,BOOK00703,/works/OL49758W,Organizational behavior,978452278605,other,all_isbns
10511,BOOK00818,/works/OL72511W,Looking out/looking in,780495101277,other,all_isbns



SAFE ISBN BRIDGE
Relationships: 11914
Books represented: 906
Unique ISBNs: 11914

Saved conflict audit: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_isbn_conflict_audit.csv
Saved safe bridge: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_isbn_identity_bridge_safe.csv


#### 18.36.6 — Wikidata ISBN Entity-Resolution Pilot

A conflict-free ISBN bridge has been established for Open Library books in LeadWise.

Before performing large-scale Wikidata enrichment, a limited pilot is conducted to determine:

- how frequently ISBNs resolve to Wikidata entities;
- whether the resolved entities represent books or editions;
- which structured language properties are available;
- whether ISBN-10 and ISBN-13 provide useful identity coverage; and
- whether the resulting evidence is suitable for original-language enrichment.

Only ISBNs that are uniquely associated with one LeadWise book are used.

A successful ISBN match establishes a strong bibliographic relationship between the LeadWise/Open Library record and a Wikidata entity. However, language claims are not yet interpreted as original-language evidence until the relevant Wikidata property and entity type have been examined.

This pilot therefore evaluates source suitability before any large-scale original-language assignment.

In [70]:
# ---------------------------------------------------------
# 18.36.6 Wikidata ISBN pilot
# ---------------------------------------------------------

import time
import requests
import pandas as pd


# ---------------------------------------------------------
# Load safe ISBN bridge
# ---------------------------------------------------------

safe_path = (
    data_root
    / "processed"
    / "openlibrary_isbn_identity_bridge_safe.csv"
)

safe_isbn = pd.read_csv(
    safe_path,
    dtype={
        "isbn": "string"
    },
    low_memory=False,
)


# ---------------------------------------------------------
# Select a small book-level pilot
#
# Prefer ISBN-13 where available.
# One ISBN per book only.
# ---------------------------------------------------------

pilot_candidates = (
    safe_isbn[
        safe_isbn[
            "isbn_type"
        ].eq(
            "ISBN-13"
        )
    ]
    .sort_values(
        [
            "book_id",
            "isbn",
        ]
    )
    .drop_duplicates(
        subset=[
            "book_id"
        ],
        keep="first",
    )
    .head(25)
    .copy()
)


print(
    "WIKIDATA ISBN PILOT"
)
print("=" * 70)

print(
    "Books selected:",
    pilot_candidates[
        "book_id"
    ].nunique()
)

print(
    "ISBNs selected:",
    len(
        pilot_candidates
    )
)


# ---------------------------------------------------------
# Wikidata SPARQL endpoint
# ---------------------------------------------------------

endpoint = (
    "https://query.wikidata.org/sparql"
)

headers = {
    "User-Agent":
        "LeadWiseBookResearch/1.0 "
        "(educational capstone project)"
}


# ---------------------------------------------------------
# Query one ISBN at a time for conservative pilot
#
# P212 = ISBN-13
# P957 = ISBN-10
# ---------------------------------------------------------

results = []


for counter, row in enumerate(
    pilot_candidates.itertuples(
        index=False
    ),
    start=1,
):

    isbn = str(
        row.isbn
    ).strip()

    isbn_type = row.isbn_type


    if isbn_type == "ISBN-13":
        property_id = "wdt:P212"

    elif isbn_type == "ISBN-10":
        property_id = "wdt:P957"

    else:
        continue


    query = f"""
    SELECT DISTINCT
        ?item
        ?itemLabel
        ?language
        ?languageLabel
        ?instance
        ?instanceLabel
    WHERE {{
        ?item {property_id} "{isbn}" .

        OPTIONAL {{
            ?item wdt:P407 ?language .
        }}

        OPTIONAL {{
            ?item wdt:P31 ?instance .
        }}

        SERVICE wikibase:label {{
            bd:serviceParam
                wikibase:language "en" .
        }}
    }}
    """


    try:

        response = requests.get(
            endpoint,
            params={
                "query":
                    query,

                "format":
                    "json",
            },
            headers=headers,
            timeout=30,
        )


        response.raise_for_status()

        bindings = (
            response.json()
            .get(
                "results",
                {}
            )
            .get(
                "bindings",
                []
            )
        )


        if not bindings:

            results.append(
                {
                    "book_id":
                        row.book_id,

                    "title":
                        row.title,

                    "isbn":
                        isbn,

                    "isbn_type":
                        isbn_type,

                    "wikidata_qid":
                        None,

                    "wikidata_label":
                        None,

                    "language_qid":
                        None,

                    "language_label":
                        None,

                    "instance_qid":
                        None,

                    "instance_label":
                        None,

                    "query_status":
                        "no_match",
                }
            )

        else:

            for binding in bindings:

                item_url = (
                    binding
                    .get(
                        "item",
                        {}
                    )
                    .get(
                        "value"
                    )
                )

                language_url = (
                    binding
                    .get(
                        "language",
                        {}
                    )
                    .get(
                        "value"
                    )
                )

                instance_url = (
                    binding
                    .get(
                        "instance",
                        {}
                    )
                    .get(
                        "value"
                    )
                )


                results.append(
                    {
                        "book_id":
                            row.book_id,

                        "title":
                            row.title,

                        "isbn":
                            isbn,

                        "isbn_type":
                            isbn_type,

                        "wikidata_qid":
                            (
                                item_url.split("/")[-1]
                                if item_url
                                else None
                            ),

                        "wikidata_label":
                            (
                                binding
                                .get(
                                    "itemLabel",
                                    {}
                                )
                                .get(
                                    "value"
                                )
                            ),

                        "language_qid":
                            (
                                language_url.split("/")[-1]
                                if language_url
                                else None
                            ),

                        "language_label":
                            (
                                binding
                                .get(
                                    "languageLabel",
                                    {}
                                )
                                .get(
                                    "value"
                                )
                            ),

                        "instance_qid":
                            (
                                instance_url.split("/")[-1]
                                if instance_url
                                else None
                            ),

                        "instance_label":
                            (
                                binding
                                .get(
                                    "instanceLabel",
                                    {}
                                )
                                .get(
                                    "value"
                                )
                            ),

                        "query_status":
                            "matched",
                    }
                )


    except Exception as error:

        results.append(
            {
                "book_id":
                    row.book_id,

                "title":
                    row.title,

                "isbn":
                    isbn,

                "isbn_type":
                    isbn_type,

                "wikidata_qid":
                    None,

                "wikidata_label":
                    None,

                "language_qid":
                    None,

                "language_label":
                    None,

                "instance_qid":
                    None,

                "instance_label":
                    None,

                "query_status":
                    f"error: {type(error).__name__}",
            }
        )


    print(
        f"{counter:02d}/"
        f"{len(pilot_candidates)} "
        f"{row.book_id} "
        f"{isbn}"
    )


    # Be gentle with Wikidata
    time.sleep(
        1.0
    )


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

pilot_results = pd.DataFrame(
    results
)


print(
    "\nPILOT MATCH SUMMARY"
)
print("=" * 70)


book_status = (
    pilot_results
    .groupby(
        "book_id"
    )[
        "query_status"
    ]
    .apply(
        lambda values:
            "matched"
            if (
                values == "matched"
            ).any()
            else values.iloc[0]
    )
    .value_counts()
    .rename_axis(
        "status"
    )
    .reset_index(
        name="book_count"
    )
)


display(
    book_status
)


print(
    "\nMATCHED WIKIDATA ENTITIES"
)
print("=" * 70)


display(
    pilot_results[
        pilot_results[
            "query_status"
        ].eq(
            "matched"
        )
    ][
        [
            "book_id",
            "title",
            "isbn",
            "wikidata_qid",
            "wikidata_label",
            "language_qid",
            "language_label",
            "instance_qid",
            "instance_label",
        ]
    ]
)


# ---------------------------------------------------------
# Save pilot
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "wikidata_isbn_language_pilot.csv"
)


pilot_results.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

WIKIDATA ISBN PILOT
Books selected: 25
ISBNs selected: 25
01/25 BOOK00001 9780028639123
02/25 BOOK00002 9780130323125
03/25 BOOK00004 9780551006515
04/25 BOOK00005 9780761919254
05/25 BOOK00006 9780785261360
06/25 BOOK00007 9780761925668
07/25 BOOK00008 9780029018101
08/25 BOOK00009 9781250192455
09/25 BOOK00010 9780141030067
10/25 BOOK00011 9780367374822
11/25 BOOK00012 9780072445299
12/25 BOOK00013 9780446394598
13/25 BOOK00014 9780060105884
14/25 BOOK00015 9780324041668
15/25 BOOK00016 9780470403822
16/25 BOOK00017 9780470244951
17/25 BOOK00018 9780078112652
18/25 BOOK00019 9780241300718
19/25 BOOK00020 9780881335330
20/25 BOOK00021 9780805847611
21/25 BOOK00022 9780061251306
22/25 BOOK00023 9780814459010
23/25 BOOK00024 9780786126415
24/25 BOOK00025 9780786251636
25/25 BOOK00026 9780805847628

PILOT MATCH SUMMARY


,status,book_count
0,error: ReadTimeout,12
1,no_match,9
2,error: HTTPError,4



MATCHED WIKIDATA ENTITIES


,book_id,title,isbn,wikidata_qid,wikidata_label,language_qid,language_label,instance_qid,instance_label



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/wikidata_isbn_language_pilot.csv


#### 18.36.7 — Open Library Work-Level Original-Language Audit

The Wikidata ISBN pilot did not produce usable work-level language matches and is therefore not scaled to the full catalog.

LeadWise already possesses exact Open Library work identifiers for all 950 Open Library books. The next enrichment strategy therefore returns to the primary Open Library work records and examines their raw JSON structure directly.

This stage tests whether Open Library exposes explicit work-level fields related to:

- original language;
- work language;
- translation relationships; or
- other structured language metadata.

Exact Open Library work keys are used, avoiding title-based entity resolution.

No language field is interpreted as original language unless the source explicitly represents it at the work level with appropriate semantics.

In [71]:
# ---------------------------------------------------------
# 18.36.7 Open Library Work JSON language-field audit
# ---------------------------------------------------------

import time
import requests
import pandas as pd


# ---------------------------------------------------------
# Load Open Library catalog
# ---------------------------------------------------------

ol_path = (
    data_root
    / "final"
    / "openlibrary_integrated.csv"
)

ol_books = pd.read_csv(
    ol_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Select a small sample of exact work keys
# ---------------------------------------------------------

sample_books = (
    ol_books[
        [
            "book_id",
            "openlibrary_key",
            "title",
            "authors",
        ]
    ]
    .dropna(
        subset=[
            "openlibrary_key"
        ]
    )
    .drop_duplicates(
        subset=[
            "openlibrary_key"
        ]
    )
    .head(25)
    .copy()
)


print(
    "OPEN LIBRARY WORK-LEVEL LANGUAGE PILOT"
)
print("=" * 70)

print(
    "Works selected:",
    len(sample_books)
)


# ---------------------------------------------------------
# Retrieve exact Work JSON
# ---------------------------------------------------------

headers = {
    "User-Agent":
        "LeadWiseBookResearch/1.0 "
        "(educational capstone project)"
}


audit_rows = []

all_work_fields = set()


for counter, row in enumerate(
    sample_books.itertuples(
        index=False
    ),
    start=1,
):

    work_key = str(
        row.openlibrary_key
    ).strip()

    url = (
        "https://openlibrary.org"
        + work_key
        + ".json"
    )


    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=30,
        )

        response.raise_for_status()

        payload = response.json()

        all_work_fields.update(
            payload.keys()
        )


        # ---------------------------------------------
        # Identify potentially relevant fields
        # ---------------------------------------------

        relevant_fields = {
            key: payload.get(key)
            for key in payload.keys()
            if any(
                term in key.lower()
                for term in [
                    "language",
                    "translation",
                    "original",
                    "locale",
                ]
            )
        }


        audit_rows.append(
            {
                "book_id":
                    row.book_id,

                "openlibrary_key":
                    work_key,

                "title":
                    row.title,

                "status":
                    "success",

                "relevant_field_names":
                    list(
                        relevant_fields.keys()
                    ),

                "relevant_field_values":
                    relevant_fields,

                "total_work_fields":
                    len(payload),
            }
        )


    except Exception as error:

        audit_rows.append(
            {
                "book_id":
                    row.book_id,

                "openlibrary_key":
                    work_key,

                "title":
                    row.title,

                "status":
                    f"error: {type(error).__name__}",

                "relevant_field_names":
                    [],

                "relevant_field_values":
                    {},

                "total_work_fields":
                    None,
            }
        )


    print(
        f"{counter:02d}/"
        f"{len(sample_books)} "
        f"{row.book_id} "
        f"{work_key}"
    )


    time.sleep(
        0.25
    )


work_language_audit = pd.DataFrame(
    audit_rows
)


# ---------------------------------------------------------
# Request status
# ---------------------------------------------------------

print(
    "\nREQUEST STATUS"
)
print("=" * 70)

display(
    work_language_audit[
        "status"
    ]
    .value_counts()
    .rename_axis(
        "status"
    )
    .reset_index(
        name="work_count"
    )
)


# ---------------------------------------------------------
# Relevant language/original/translation fields
# ---------------------------------------------------------

print(
    "\nWORK-LEVEL LANGUAGE-RELATED FIELDS"
)
print("=" * 70)

display(
    work_language_audit[
        [
            "book_id",
            "openlibrary_key",
            "title",
            "relevant_field_names",
            "relevant_field_values",
        ]
    ]
)


# ---------------------------------------------------------
# Full field inventory across sampled works
# ---------------------------------------------------------

print(
    "\nALL WORK-LEVEL FIELDS OBSERVED"
)
print("=" * 70)

for field in sorted(
    all_work_fields
):
    print(
        " -",
        field
    )


# ---------------------------------------------------------
# Frequency of relevant field names
# ---------------------------------------------------------

relevant_field_frequency = {}


for values in work_language_audit[
    "relevant_field_names"
]:

    for field in values:

        relevant_field_frequency[
            field
        ] = (
            relevant_field_frequency
            .get(
                field,
                0,
            )
            + 1
        )


print(
    "\nLANGUAGE-RELATED FIELD FREQUENCY"
)
print("=" * 70)


if relevant_field_frequency:

    field_frequency_df = (
        pd.DataFrame(
            [
                {
                    "field":
                        key,

                    "work_count":
                        value,
                }
                for key, value
                in relevant_field_frequency.items()
            ]
        )
        .sort_values(
            "work_count",
            ascending=False,
        )
    )

    display(
        field_frequency_df
    )

else:

    print(
        "No language/original/translation fields "
        "found in sampled Work JSON records."
    )


# ---------------------------------------------------------
# Save pilot
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "openlibrary_work_language_field_pilot.csv"
)


work_language_audit.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

OPEN LIBRARY WORK-LEVEL LANGUAGE PILOT
Works selected: 25
01/25 BOOK00001 /works/OL2630041W
02/25 BOOK00002 /works/OL2731767W
03/25 BOOK00003 /works/OL302757W
04/25 BOOK00004 /works/OL450702W
05/25 BOOK00005 /works/OL94176W
06/25 BOOK00006 /works/OL28124W
07/25 BOOK00007 /works/OL7956784W
08/25 BOOK00008 /works/OL2421691W
09/25 BOOK00009 /works/OL18146967W
10/25 BOOK00010 /works/OL8921343W
11/25 BOOK00011 /works/OL19982874W
12/25 BOOK00012 /works/OL1947324W
13/25 BOOK00013 /works/OL1807678W
14/25 BOOK00014 /works/OL30484W
15/25 BOOK00015 /works/OL1815713W
16/25 BOOK00016 /works/OL1974749W
17/25 BOOK00017 /works/OL17628850W
18/25 BOOK00018 /works/OL16296611W
19/25 BOOK00019 /works/OL17934890W
20/25 BOOK00020 /works/OL1693759W
21/25 BOOK00021 /works/OL21002118W
22/25 BOOK00022 /works/OL11659147W
23/25 BOOK00023 /works/OL2649476W
24/25 BOOK00024 /works/OL30487W
25/25 BOOK00025 /works/OL694165W

REQUEST STATUS


,status,work_count
0,success,25



WORK-LEVEL LANGUAGE-RELATED FIELDS


,book_id,openlibrary_key,title,relevant_field_names,relevant_field_values
0,BOOK00001,/works/OL2630041W,Principle-Centered Leadership,[],{}
1,BOOK00002,/works/OL2731767W,Leadership in Organizations,[],{}
2,BOOK00003,/works/OL302757W,Kepemimpinan =,[],{}
3,BOOK00004,/works/OL450702W,Spiritual leadership,[],{}
4,BOOK00005,/works/OL94176W,Leadership,[],{}
5,BOOK00006,/works/OL28124W,The 21 Irrefutable Laws of Leadership,[],{}
6,BOOK00007,/works/OL7956784W,Leadership,[],{}
7,BOOK00008,/works/OL2421691W,Leadership and performance beyond expectations,[],{}
8,BOOK00009,/works/OL18146967W,A Higher Loyalty,[],{}
9,BOOK00010,/works/OL8921343W,Leadership and Self Deception,[],{}



ALL WORK-LEVEL FIELDS OBSERVED
 - authors
 - covers
 - created
 - description
 - dewey_number
 - excerpts
 - first_publish_date
 - first_sentence
 - id
 - key
 - last_modified
 - latest_revision
 - lc_classifications
 - links
 - revision
 - subject_people
 - subject_places
 - subject_times
 - subjects
 - subtitle
 - title
 - type

LANGUAGE-RELATED FIELD FREQUENCY
No language/original/translation fields found in sampled Work JSON records.

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/openlibrary_work_language_field_pilot.csv


#### 18.36.8 — Final Application-Ready Language Evidence

The language-enrichment investigation found strong edition-level publication-language evidence but no sufficiently reliable source for assigning original language across the catalog.

Open Library edition records provide observed publication languages for most Open Library books, while a pilot inspection of exact Open Library Work records found no explicit original-language or translation fields. A separate Wikidata ISBN pilot did not produce usable structured matches.

LeadWise therefore adopts a conservative language model:

- **Observed publication languages** represent languages explicitly found across known Open Library editions.
- **Observed publication language count** measures the number of distinct languages represented in those edition records.
- **Multilingual publication evidence** identifies books observed in more than one publication language.
- **Original language** remains unassigned unless a future source provides explicit, verifiable evidence.
- **Translation count** is not inferred from multilingual edition records.
- **International language reach** is a descriptive LeadWise indicator based only on observed publication-language diversity.

This approach prevents edition-language metadata from being misrepresented as original-language or verified translation evidence.

In [72]:
# ---------------------------------------------------------
# 18.36.8 Final application-ready language evidence
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Load previously validated language evidence
# ---------------------------------------------------------

language_path = (
    data_root
    / "processed"
    / "leadwise_publication_language_evidence.csv"
)


language_final = pd.read_csv(
    language_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Rename display field for explicit app terminology
# ---------------------------------------------------------

if (
    "known_publication_languages_display"
    in language_final.columns
):

    language_final = (
        language_final.rename(
            columns={
                "known_publication_languages_display":
                    "observed_publication_languages"
            }
        )
    )


# ---------------------------------------------------------
# Add conservative interpretation fields
# ---------------------------------------------------------

language_final[
    "original_language"
] = pd.NA


language_final[
    "original_language_status"
] = (
    "not_verified"
)


language_final[
    "verified_translation_count"
] = pd.NA


language_final[
    "translation_status"
] = (
    "not_verified"
)


# ---------------------------------------------------------
# International language reach label
#
# This is descriptive only.
# It is NOT a quality score.
# ---------------------------------------------------------

def classify_language_reach(count):

    if pd.isna(count):
        return "No language evidence"

    count = int(count)

    if count == 0:
        return "No language evidence"

    if count == 1:
        return "Single observed publication language"

    return "Multilingual publication evidence"


language_final[
    "international_language_reach"
] = (
    language_final[
        "observed_publication_language_count"
    ]
    .apply(
        classify_language_reach
    )
)


# ---------------------------------------------------------
# Provenance
# ---------------------------------------------------------

language_final[
    "language_evidence_provenance"
] = (
    "Open Library observed edition metadata"
)


language_final[
    "language_evidence_note"
] = (
    "Observed publication languages do not establish "
    "original language or verified translation count."
)


# ---------------------------------------------------------
# Arrange application-ready fields
# ---------------------------------------------------------

final_columns = [
    "book_id",
    "observed_publication_languages",
    "observed_publication_language_count",
    "has_multilingual_publication_evidence",
    "international_language_reach",
    "original_language",
    "original_language_status",
    "verified_translation_count",
    "translation_status",
    "language_evidence_provenance",
    "language_evidence_note",
]


language_final = (
    language_final[
        [
            column
            for column in final_columns
            if column in language_final.columns
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "FINAL LANGUAGE EVIDENCE"
)
print("=" * 70)

print(
    "Books:",
    len(language_final)
)

print(
    "Books with observed language evidence:",
    (
        language_final[
            "observed_publication_language_count"
        ] > 0
    ).sum()
)

print(
    "Books with multilingual publication evidence:",
    language_final[
        "has_multilingual_publication_evidence"
    ].sum()
)

print(
    "Books with verified original language:",
    language_final[
        "original_language"
    ].notna().sum()
)

print(
    "Books with verified translation count:",
    language_final[
        "verified_translation_count"
    ].notna().sum()
)


# ---------------------------------------------------------
# Reach distribution
# ---------------------------------------------------------

print(
    "\nINTERNATIONAL LANGUAGE REACH"
)
print("=" * 70)


display(
    language_final[
        "international_language_reach"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "language_reach"
    )
    .reset_index(
        name="book_count"
    )
)


# ---------------------------------------------------------
# Multilingual sample
# ---------------------------------------------------------

print(
    "\nMULTILINGUAL PUBLICATION SAMPLE"
)
print("=" * 70)


display(
    language_final[
        language_final[
            "has_multilingual_publication_evidence"
        ].eq(True)
    ]
    .sort_values(
        "observed_publication_language_count",
        ascending=False,
    )
    .head(20)
)


# ---------------------------------------------------------
# Save final application-ready evidence
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "leadwise_language_evidence_final.csv"
)


language_final.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

FINAL LANGUAGE EVIDENCE
Books: 948
Books with observed language evidence: 900
Books with multilingual publication evidence: 40
Books with verified original language: 0
Books with verified translation count: 0

INTERNATIONAL LANGUAGE REACH


,language_reach,book_count
0,Single observed publication language,860
1,No language evidence,48
2,Multilingual publication evidence,40



MULTILINGUAL PUBLICATION SAMPLE


,book_id,observed_publication_languages,observed_publication_language_count,has_multilingual_publication_evidence,international_language_reach,original_language,original_language_status,verified_translation_count,translation_status,language_evidence_provenance,language_evidence_note
331,BOOK00333,"English, Portuguese, Chinese, Vietnamese, Span...",9,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
807,BOOK00809,"French, Spanish, Germanic languages, German, E...",6,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
856,BOOK00859,"English, Italian, Spanish, Polish, Chinese",5,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
279,BOOK00281,"English, French, German, Chinese, Norwegian",5,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
349,BOOK00351,"French, Finnish, English, Spanish",4,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
5,BOOK00006,"Spanish, Mandarin Chinese, English",3,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
469,BOOK00471,"Romanian, French, English",3,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
287,BOOK00289,"English, Chinese, German",3,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
148,BOOK00150,"English, Chinese, Korean",3,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...
929,BOOK00932,"English, Chinese, Spanish",3,True,Multilingual publication evidence,<NA>,not_verified,<NA>,not_verified,Open Library observed edition metadata,Observed publication languages do not establis...



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_language_evidence_final.csv


### 18.37 — Commercial and Price Metadata Enrichment

LeadWise currently contains bibliographic, reader-engagement, geographic, and publication-language metadata, but commercial price information has not yet been integrated into the application catalog.

Book prices are edition-specific, seller-specific, market-specific, currency-specific, and time-sensitive. Therefore, LeadWise does not treat a single observed price as the universal or permanent price of a book.

This enrichment stage follows a provenance-first approach and distinguishes:

- observed list price;
- observed retail price;
- currency;
- market or country when available;
- source;
- edition or ISBN associated with the price when available; and
- observation context.

No missing price is estimated from another book, edition, currency, publisher, ISBN, or market.

Before collecting new commercial metadata, the existing project files are audited to determine whether usable price and currency fields were already collected during earlier API or web-data acquisition stages.

In [73]:
# ---------------------------------------------------------
# 18.37.1 Audit existing price and commercial metadata
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd


# ---------------------------------------------------------
# Search project data files
# ---------------------------------------------------------

data_root = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System/data"
)


csv_files = sorted(
    data_root.rglob(
        "*.csv"
    )
)


commercial_terms = [
    "price",
    "currency",
    "sale",
    "retail",
    "list_price",
    "listprice",
    "amount",
    "buy",
    "purchase",
]


audit_rows = []


for file_path in csv_files:

    try:

        df = pd.read_csv(
            file_path,
            nrows=5,
            low_memory=False,
        )

    except Exception:
        continue


    matching_columns = [
        column
        for column in df.columns
        if any(
            term in column.lower()
            for term in commercial_terms
        )
    ]


    if matching_columns:

        audit_rows.append(
            {
                "file":
                    str(
                        file_path.relative_to(
                            data_root
                        )
                    ),

                "matching_columns":
                    matching_columns,
            }
        )


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print(
    "EXISTING COMMERCIAL-METADATA AUDIT"
)
print("=" * 70)

print(
    "CSV files inspected:",
    len(csv_files)
)

print(
    "Files containing possible commercial fields:",
    len(audit_rows)
)


if audit_rows:

    commercial_audit = pd.DataFrame(
        audit_rows
    )

    display(
        commercial_audit
    )

else:

    commercial_audit = pd.DataFrame(
        columns=[
            "file",
            "matching_columns",
        ]
    )

    print(
        "\nNo candidate price/currency/commercial "
        "columns found in CSV files."
    )


# ---------------------------------------------------------
# Specifically inspect likely API/raw files
# ---------------------------------------------------------

print(
    "\nLIKELY API / RAW DATA FILES"
)
print("=" * 70)


likely_files = [
    path
    for path in csv_files
    if any(
        term in path.name.lower()
        for term in [
            "google",
            "openlibrary",
            "leadership",
            "raw",
            "api",
        ]
    )
]


for path in likely_files:

    print(
        path.relative_to(
            data_root
        )
    )


# ---------------------------------------------------------
# Save audit
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "leadwise_commercial_metadata_audit.csv"
)


commercial_audit.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

EXISTING COMMERCIAL-METADATA AUDIT
CSV files inspected: 110
Files containing possible commercial fields: 0

No candidate price/currency/commercial columns found in CSV files.

LIKELY API / RAW DATA FILES
final/leadershipnow_editions.csv
final/openlibrary_integrated.csv
processed/leadershipnow_books_clean.csv
processed/openlibrary_author_book_context.csv
processed/openlibrary_author_identity_bridge.csv
processed/openlibrary_book_edition_intelligence.csv
processed/openlibrary_earliest_publication_geography.csv
processed/openlibrary_earliest_publication_geography_normalized.csv
processed/openlibrary_isbn_conflict_audit.csv
processed/openlibrary_isbn_identity_bridge.csv
processed/openlibrary_isbn_identity_bridge_safe.csv
processed/openlibrary_publication_geographic_code_inventory.csv
processed/openlibrary_work_language_field_pilot.csv
processed/wikidata_author_geographic_evidence_raw.csv
raw/leadershipnow_books_raw.csv
raw/leadershipnow_cover_anomalies.csv
raw/leadershipnow_scraping_log.cs

#### 18.37.2 — Google Books Commercial Metadata Pilot

The existing LeadWise datasets contain no commercial price or currency fields. Price therefore represents a new metadata-collection requirement rather than an integration issue.

Google Books is evaluated as a potential source because its volume metadata may contain structured `saleInfo`, including:

- saleability status;
- list price;
- retail price;
- currency;
- market/country context; and
- purchase-related metadata.

Commercial metadata is treated as an **observed market-specific record**, not as a universal or permanent book price.

The pilot uses ISBNs from the conflict-free LeadWise ISBN bridge to reduce identity ambiguity. A returned Google Books volume is retained only with its associated query ISBN and source provenance.

No missing price is estimated, converted, or inferred.

Because Google Books availability and commercial metadata may vary by market and API response, a small pilot is conducted before any larger collection.

In [74]:
# ---------------------------------------------------------
# 18.37.2 Google Books commercial metadata pilot
# ---------------------------------------------------------

import time
import requests
import pandas as pd


# ---------------------------------------------------------
# Load safe ISBN bridge
# ---------------------------------------------------------

safe_path = (
    data_root
    / "processed"
    / "openlibrary_isbn_identity_bridge_safe.csv"
)

safe_isbn = pd.read_csv(
    safe_path,
    dtype={
        "isbn": "string"
    },
    low_memory=False,
)


# ---------------------------------------------------------
# Select one ISBN-13 per book
# Small pilot: 10 books
# ---------------------------------------------------------

pilot_books = (
    safe_isbn[
        safe_isbn[
            "isbn_type"
        ].eq(
            "ISBN-13"
        )
    ]
    .sort_values(
        [
            "book_id",
            "isbn",
        ]
    )
    .drop_duplicates(
        subset=[
            "book_id"
        ],
        keep="first",
    )
    .head(10)
    .copy()
)


print(
    "GOOGLE BOOKS COMMERCIAL-METADATA PILOT"
)
print("=" * 70)

print(
    "Books selected:",
    len(pilot_books)
)


# ---------------------------------------------------------
# Google Books endpoint
# ---------------------------------------------------------

endpoint = (
    "https://www.googleapis.com/books/v1/volumes"
)

headers = {
    "User-Agent":
        "LeadWiseBookResearch/1.0"
}


results = []


# ---------------------------------------------------------
# Query by ISBN
# ---------------------------------------------------------

for counter, row in enumerate(
    pilot_books.itertuples(
        index=False
    ),
    start=1,
):

    isbn = str(
        row.isbn
    ).strip()


    try:

        response = requests.get(
            endpoint,
            params={
                "q":
                    f"isbn:{isbn}",

                "maxResults":
                    5,
            },
            headers=headers,
            timeout=30,
        )


        http_status = (
            response.status_code
        )


        response.raise_for_status()

        payload = response.json()

        items = payload.get(
            "items",
            []
        )


        if not items:

            results.append(
                {
                    "book_id":
                        row.book_id,

                    "query_isbn":
                        isbn,

                    "leadwise_title":
                        row.title,

                    "google_volume_id":
                        None,

                    "google_title":
                        None,

                    "saleability":
                        None,

                    "market_country":
                        None,

                    "list_price_amount":
                        None,

                    "list_price_currency":
                        None,

                    "retail_price_amount":
                        None,

                    "retail_price_currency":
                        None,

                    "buy_link":
                        None,

                    "query_status":
                        "no_match",

                    "http_status":
                        http_status,
                }
            )

        else:

            # For the pilot, inspect every returned item.
            for item in items:

                volume_info = (
                    item.get(
                        "volumeInfo",
                        {}
                    )
                )

                sale_info = (
                    item.get(
                        "saleInfo",
                        {}
                    )
                )

                list_price = (
                    sale_info.get(
                        "listPrice",
                        {}
                    )
                    or {}
                )

                retail_price = (
                    sale_info.get(
                        "retailPrice",
                        {}
                    )
                    or {}
                )


                results.append(
                    {
                        "book_id":
                            row.book_id,

                        "query_isbn":
                            isbn,

                        "leadwise_title":
                            row.title,

                        "google_volume_id":
                            item.get(
                                "id"
                            ),

                        "google_title":
                            volume_info.get(
                                "title"
                            ),

                        "saleability":
                            sale_info.get(
                                "saleability"
                            ),

                        "market_country":
                            sale_info.get(
                                "country"
                            ),

                        "list_price_amount":
                            list_price.get(
                                "amount"
                            ),

                        "list_price_currency":
                            list_price.get(
                                "currencyCode"
                            ),

                        "retail_price_amount":
                            retail_price.get(
                                "amount"
                            ),

                        "retail_price_currency":
                            retail_price.get(
                                "currencyCode"
                            ),

                        "buy_link":
                            sale_info.get(
                                "buyLink"
                            ),

                        "query_status":
                            "matched",

                        "http_status":
                            http_status,
                    }
                )


    except requests.exceptions.HTTPError:

        results.append(
            {
                "book_id":
                    row.book_id,

                "query_isbn":
                    isbn,

                "leadwise_title":
                    row.title,

                "google_volume_id":
                    None,

                "google_title":
                    None,

                "saleability":
                    None,

                "market_country":
                    None,

                "list_price_amount":
                    None,

                "list_price_currency":
                    None,

                "retail_price_amount":
                    None,

                "retail_price_currency":
                    None,

                "buy_link":
                    None,

                "query_status":
                    "http_error",

                "http_status":
                    response.status_code,
            }
        )


    except Exception as error:

        results.append(
            {
                "book_id":
                    row.book_id,

                "query_isbn":
                    isbn,

                "leadwise_title":
                    row.title,

                "google_volume_id":
                    None,

                "google_title":
                    None,

                "saleability":
                    None,

                "market_country":
                    None,

                "list_price_amount":
                    None,

                "list_price_currency":
                    None,

                "retail_price_amount":
                    None,

                "retail_price_currency":
                    None,

                "buy_link":
                    None,

                "query_status":
                    f"error: {type(error).__name__}",

                "http_status":
                    None,
            }
        )


    print(
        f"{counter:02d}/"
        f"{len(pilot_books)} "
        f"{row.book_id} "
        f"{isbn}"
    )


    time.sleep(
        0.5
    )


# ---------------------------------------------------------
# Results dataframe
# ---------------------------------------------------------

price_pilot = pd.DataFrame(
    results
)


# ---------------------------------------------------------
# Query status
# ---------------------------------------------------------

print(
    "\nQUERY STATUS"
)
print("=" * 70)


display(
    price_pilot[
        "query_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="records"
    )
)


# ---------------------------------------------------------
# Saleability
# ---------------------------------------------------------

print(
    "\nSALEABILITY"
)
print("=" * 70)


display(
    price_pilot[
        "saleability"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "saleability"
    )
    .reset_index(
        name="records"
    )
)


# ---------------------------------------------------------
# Price availability
# ---------------------------------------------------------

print(
    "\nPRICE AVAILABILITY"
)
print("=" * 70)

print(
    "Returned records:",
    len(price_pilot)
)

print(
    "Records with list price:",
    price_pilot[
        "list_price_amount"
    ].notna().sum()
)

print(
    "Records with retail price:",
    price_pilot[
        "retail_price_amount"
    ].notna().sum()
)

print(
    "Books with any observed price:",
    price_pilot.loc[
        (
            price_pilot[
                "list_price_amount"
            ].notna()
        )
        |
        (
            price_pilot[
                "retail_price_amount"
            ].notna()
        ),
        "book_id",
    ].nunique()
)


# ---------------------------------------------------------
# Detailed results
# ---------------------------------------------------------

print(
    "\nCOMMERCIAL METADATA RESULTS"
)
print("=" * 70)


display(
    price_pilot[
        [
            "book_id",
            "query_isbn",
            "leadwise_title",
            "google_title",
            "saleability",
            "market_country",
            "list_price_amount",
            "list_price_currency",
            "retail_price_amount",
            "retail_price_currency",
            "query_status",
            "http_status",
        ]
    ]
)


# ---------------------------------------------------------
# Save pilot
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "google_books_commercial_metadata_pilot.csv"
)


price_pilot.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

GOOGLE BOOKS COMMERCIAL-METADATA PILOT
Books selected: 10
01/10 BOOK00001 9780028639123
02/10 BOOK00002 9780130323125
03/10 BOOK00004 9780551006515
04/10 BOOK00005 9780761919254
05/10 BOOK00006 9780785261360
06/10 BOOK00007 9780761925668
07/10 BOOK00008 9780029018101
08/10 BOOK00009 9781250192455
09/10 BOOK00010 9780141030067
10/10 BOOK00011 9780367374822

QUERY STATUS


,status,records
0,http_error,10



SALEABILITY


,saleability,records
0,None,10



PRICE AVAILABILITY
Returned records: 10
Records with list price: 0
Records with retail price: 0
Books with any observed price: 0

COMMERCIAL METADATA RESULTS


,book_id,query_isbn,leadwise_title,google_title,saleability,market_country,list_price_amount,list_price_currency,retail_price_amount,retail_price_currency,query_status,http_status
0,BOOK00001,9780028639123,Principle-Centered Leadership,None,None,None,None,None,None,None,http_error,429
1,BOOK00002,9780130323125,Leadership in Organizations,None,None,None,None,None,None,None,http_error,429
2,BOOK00004,9780551006515,Spiritual leadership,None,None,None,None,None,None,None,http_error,429
3,BOOK00005,9780761919254,Leadership,None,None,None,None,None,None,None,http_error,429
4,BOOK00006,9780785261360,The 21 Irrefutable Laws of Leadership,None,None,None,None,None,None,None,http_error,429
5,BOOK00007,9780761925668,Leadership,None,None,None,None,None,None,None,http_error,429
6,BOOK00008,9780029018101,Leadership and performance beyond expectations,None,None,None,None,None,None,None,http_error,429
7,BOOK00009,9781250192455,A Higher Loyalty,None,None,None,None,None,None,None,http_error,429
8,BOOK00010,9780141030067,Leadership and Self Deception,None,None,None,None,None,None,None,http_error,429
9,BOOK00011,9780367374822,Leadership,None,None,None,None,None,None,None,http_error,429



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/google_books_commercial_metadata_pilot.csv


#### 18.37.3 — Final Commercial Metadata Status

The existing LeadWise data inventory contains no previously collected price or currency fields.

A Google Books commercial-metadata pilot was subsequently conducted using conflict-free ISBN-13 identifiers. All pilot requests returned HTTP 429 responses, indicating API rate or quota restrictions. Consequently, the pilot cannot be interpreted as evidence that the selected books lack commercial price information.

LeadWise therefore does not assign or estimate book prices from the currently available data.

For the present application version:

- price remains unavailable unless supported by an explicitly observed commercial source;
- missing price does not mean that a book is unavailable for purchase;
- prices are not inferred from ISBN, publisher, publication country, or other editions;
- currencies are not assumed or automatically converted;
- no universal or current retail-price claim is made; and
- commercial metadata remains independent of recommendation ranking.

This limitation is retained transparently in the application metadata and may be addressed through a future commercial-data source.

In [75]:
# ---------------------------------------------------------
# 18.37.3 Final application-ready commercial metadata status
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Use complete LeadWise master catalog
# ---------------------------------------------------------

master_path = (
    data_root
    / "final"
    / "books_master.csv"
)


books_master = pd.read_csv(
    master_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Create conservative commercial metadata layer
# ---------------------------------------------------------

commercial_final = (
    books_master[
        [
            "book_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


commercial_final[
    "observed_list_price"
] = pd.NA


commercial_final[
    "observed_retail_price"
] = pd.NA


commercial_final[
    "observed_price_currency"
] = pd.NA


commercial_final[
    "observed_price_market"
] = pd.NA


commercial_final[
    "observed_price_isbn"
] = pd.NA


commercial_final[
    "commercial_source"
] = pd.NA


commercial_final[
    "commercial_metadata_status"
] = (
    "not_available_from_collected_sources"
)


commercial_final[
    "commercial_metadata_note"
] = (
    "No verified observed price is available from "
    "the currently collected LeadWise sources. "
    "This does not indicate that the book is unavailable "
    "for purchase."
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "FINAL COMMERCIAL METADATA STATUS"
)
print("=" * 70)

print(
    "LeadWise books:",
    len(commercial_final)
)

print(
    "Books with observed list price:",
    commercial_final[
        "observed_list_price"
    ].notna().sum()
)

print(
    "Books with observed retail price:",
    commercial_final[
        "observed_retail_price"
    ].notna().sum()
)

print(
    "Books with observed currency:",
    commercial_final[
        "observed_price_currency"
    ].notna().sum()
)


print(
    "\nCOMMERCIAL METADATA STATUS DISTRIBUTION"
)
print("=" * 70)


display(
    commercial_final[
        "commercial_metadata_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="book_count"
    )
)


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

print(
    "\nINTEGRITY CHECKS"
)
print("=" * 70)

print(
    "Unique book IDs:",
    commercial_final[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    commercial_final[
        "book_id"
    ].duplicated().sum()
)

print(
    "Master books represented:",
    commercial_final[
        "book_id"
    ].nunique()
    ==
    books_master[
        "book_id"
    ].nunique()
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

output_path = (
    data_root
    / "processed"
    / "leadwise_commercial_metadata_final.csv"
)


commercial_final.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

FINAL COMMERCIAL METADATA STATUS
LeadWise books: 2067
Books with observed list price: 0
Books with observed retail price: 0
Books with observed currency: 0

COMMERCIAL METADATA STATUS DISTRIBUTION


,status,book_count
0,not_available_from_collected_sources,2067



INTEGRITY CHECKS
Unique book IDs: 2067
Duplicate book IDs: 0
Master books represented: True

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_commercial_metadata_final.csv


### 18.38 — Enriched LeadWise Application Catalog

The post-collection enrichment stages produced several evidence layers that extend the original LeadWise catalog without modifying the production recommendation model.

This stage integrates those layers into a single application-ready metadata catalog.

The enriched catalog combines:

- core book identity and descriptive metadata;
- authors and subjects;
- publication and edition metadata;
- ISBN identifiers;
- cover images;
- reader ratings and engagement metrics;
- topic-cluster intelligence;
- observed publication geography;
- observed publication-language reach;
- author geographic evidence;
- commercial-metadata status; and
- recommendation-system alignment.

The integration follows a **left-join architecture** anchored on the existing LeadWise application catalog. This preserves all 2,067 books and prevents enrichment availability from determining whether a book remains accessible to the recommendation system.

Missing enrichment fields remain missing rather than being inferred.

Most importantly, this integration does not retrain, reorder, or modify the production TF-IDF recommendation matrix. The existing `matrix_row` remains the authoritative mapping between application records and recommendation vectors.

In [76]:
# ---------------------------------------------------------
# 18.38.1 Inventory application-enrichment tables
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd


processed_root = (
    data_root
    / "processed"
)

final_root = (
    data_root
    / "final"
)


# ---------------------------------------------------------
# Files intended for application integration
# ---------------------------------------------------------

integration_files = {
    
    "grounding_catalog":
        processed_root
        / "llm_grounding_catalog.csv",

    "books_master":
        final_root
        / "books_master.csv",

    "openlibrary_integrated":
        final_root
        / "openlibrary_integrated.csv",

    "leadershipnow_editions":
        final_root
        / "leadershipnow_editions.csv",

    "edition_intelligence":
        processed_root
        / "openlibrary_book_edition_intelligence.csv",

    "publication_geography":
        processed_root
        / "leadwise_publication_geography_evidence.csv",

    "language_evidence":
        processed_root
        / "leadwise_language_evidence_final.csv",

    "author_geography":
        processed_root
        / "wikidata_author_geographic_evidence.csv",

    "commercial_metadata":
        processed_root
        / "leadwise_commercial_metadata_final.csv",

    "topic_clusters":
        processed_root
        / "books_with_final_topic_clusters.csv",
}


# ---------------------------------------------------------
# Inspect each file
# ---------------------------------------------------------

inventory_rows = []

loaded_tables = {}


print(
    "ENRICHMENT TABLE INVENTORY"
)
print("=" * 80)


for table_name, file_path in integration_files.items():

    if not file_path.exists():

        print(
            f"\nNOT FOUND: "
            f"{table_name} -> {file_path}"
        )

        inventory_rows.append(
            {
                "table":
                    table_name,

                "exists":
                    False,

                "rows":
                    None,

                "columns":
                    None,

                "unique_book_ids":
                    None,

                "duplicate_book_ids":
                    None,
            }
        )

        continue


    df = pd.read_csv(
        file_path,
        low_memory=False,
    )

    loaded_tables[
        table_name
    ] = df


    has_book_id = (
        "book_id"
        in df.columns
    )


    unique_books = (
        df[
            "book_id"
        ].nunique()
        if has_book_id
        else None
    )


    duplicate_books = (
        df[
            "book_id"
        ].duplicated().sum()
        if has_book_id
        else None
    )


    inventory_rows.append(
        {
            "table":
                table_name,

            "exists":
                True,

            "rows":
                len(df),

            "columns":
                len(df.columns),

            "unique_book_ids":
                unique_books,

            "duplicate_book_ids":
                duplicate_books,
        }
    )


    print(
        f"\n{table_name}"
    )

    print(
        f"Path: {file_path}"
    )

    print(
        f"Shape: {df.shape}"
    )

    print(
        f"Unique book_id: "
        f"{unique_books}"
    )

    print(
        f"Duplicate book_id rows: "
        f"{duplicate_books}"
    )

    print(
        "Columns:"
    )

    for column in df.columns:

        print(
            f" - {column}"
        )


# ---------------------------------------------------------
# Summary table
# ---------------------------------------------------------

integration_inventory = (
    pd.DataFrame(
        inventory_rows
    )
)


print(
    "\n\nINTEGRATION SUMMARY"
)
print("=" * 80)


display(
    integration_inventory
)


# ---------------------------------------------------------
# Identify tables unsafe for direct book-level join
# ---------------------------------------------------------

print(
    "\nJOIN CARDINALITY CHECK"
)
print("=" * 80)


for table_name, df in loaded_tables.items():

    if "book_id" not in df.columns:

        print(
            f"{table_name}: "
            "NO book_id"
        )

        continue


    duplicate_count = (
        df[
            "book_id"
        ]
        .duplicated()
        .sum()
    )


    if duplicate_count == 0:

        status = (
            "SAFE ONE-ROW-PER-BOOK"
        )

    else:

        status = (
            "REQUIRES AGGREGATION BEFORE JOIN"
        )


    print(
        f"{table_name}: "
        f"{status} "
        f"(duplicate rows = "
        f"{duplicate_count})"
    )


# ---------------------------------------------------------
# Save inventory
# ---------------------------------------------------------

output_path = (
    processed_root
    / "leadwise_app_integration_inventory.csv"
)


integration_inventory.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

ENRICHMENT TABLE INVENTORY

grounding_catalog
Path: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/llm_grounding_catalog.csv
Shape: (2067, 19)
Unique book_id: 2067
Duplicate book_id rows: 0
Columns:
 - matrix_row
 - book_id
 - canonical_title
 - authors
 - description
 - subjects
 - first_publish_year
 - publication_year_observed
 - average_rating
 - ratings_count
 - edition_count
 - want_to_read_count
 - currently_reading_count
 - already_read_count
 - cover_url
 - source_group
 - topic_cluster
 - cluster_label
 - enriched_zero_vector

books_master
Path: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/final/books_master.csv
Shape: (2067, 18)
Unique book_id: 2067
Duplicate book_id rows: 0
Columns:
 - book_id
 - canonical_title
 - authors
 - description
 - subjects
 - first_publish_year
 - average_rating
 - ratings_count
 - edition_count
 - want_to_read_count
 - currently_r

,table,exists,rows,columns,unique_book_ids,duplicate_book_ids
0,grounding_catalog,True,2067,19,2067.0,0.0
1,books_master,True,2067,18,2067.0,0.0
2,openlibrary_integrated,True,950,50,950.0,0.0
3,leadershipnow_editions,True,1120,23,1120.0,0.0
4,edition_intelligence,True,948,28,948.0,0.0
5,publication_geography,True,945,7,945.0,0.0
6,language_evidence,True,948,11,948.0,0.0
7,author_geography,True,467,16,NaN,NaN
8,commercial_metadata,True,2067,9,2067.0,0.0
9,topic_clusters,True,1884,73,1884.0,0.0



JOIN CARDINALITY CHECK
grounding_catalog: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
books_master: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
openlibrary_integrated: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
leadershipnow_editions: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
edition_intelligence: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
publication_geography: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
language_evidence: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
author_geography: NO book_id
commercial_metadata: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)
topic_clusters: SAFE ONE-ROW-PER-BOOK (duplicate rows = 0)

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_app_integration_inventory.csv


#### 18.38.2 — Author Geographic Evidence: Book-Level Aggregation

The validated Wikidata author-geography dataset is structured at the author level and therefore cannot be joined directly to the LeadWise application catalog.

An existing Open Library author-identity bridge provides exact relationships between `book_id` and `openlibrary_author_key`. These identifiers are used to connect validated author geographic evidence to books without fuzzy name matching.

Because a book may contain multiple authors, author-level evidence is aggregated to one record per `book_id`.

The application preserves the distinction between:

- **author citizenship evidence**, and
- **author birthplace evidence**.

These fields describe geographic attributes associated with validated author identities. They are not interpreted as:

- the book's country of origin;
- the book's publication country;
- the author's current residence; or
- a single definitive "author country."

Multiple authors and multiple geographic labels are retained where supported by the source evidence.

In [77]:
# ---------------------------------------------------------
# 18.38.2 Build book-level author geographic evidence
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Load exact author-to-book bridge
# ---------------------------------------------------------

bridge_path = (
    processed_root
    / "openlibrary_author_identity_bridge.csv"
)

author_bridge = pd.read_csv(
    bridge_path,
    low_memory=False,
)


author_geo = loaded_tables[
    "author_geography"
].copy()


print(
    "AUTHOR GEOGRAPHY BRIDGE"
)
print("=" * 80)

print(
    "Author-book relationships:",
    len(author_bridge)
)

print(
    "Unique books in bridge:",
    author_bridge[
        "book_id"
    ].nunique()
)

print(
    "Unique Open Library author keys in bridge:",
    author_bridge[
        "openlibrary_author_key"
    ].nunique()
)

print(
    "Validated geographic author records:",
    len(author_geo)
)

print(
    "Unique validated author keys:",
    author_geo[
        "openlibrary_author_key"
    ].nunique()
)


# ---------------------------------------------------------
# Exact identifier join only
# ---------------------------------------------------------

author_book_geo = (
    author_bridge.merge(
        author_geo,
        on="openlibrary_author_key",
        how="inner",
        validate="many_to_one",
        suffixes=(
            "_bridge",
            "_geo",
        ),
    )
)


print(
    "\nEXACT AUTHOR-GEOGRAPHY MATCH"
)
print("=" * 80)

print(
    "Matched author-book relationships:",
    len(author_book_geo)
)

print(
    "Books with validated author geography:",
    author_book_geo[
        "book_id"
    ].nunique()
)

print(
    "Validated authors represented:",
    author_book_geo[
        "openlibrary_author_key"
    ].nunique()
)


# ---------------------------------------------------------
# Utility:
# Convert stored labels into clean individual values
# ---------------------------------------------------------

def extract_label_values(value):

    if pd.isna(value):
        return []

    # Lists may have been stored as strings in CSV.
    if isinstance(value, str):

        cleaned = value.strip()

        if not cleaned:
            return []

        if (
            cleaned.startswith("[")
            and
            cleaned.endswith("]")
        ):

            try:
                parsed = ast.literal_eval(
                    cleaned
                )

                if isinstance(
                    parsed,
                    list,
                ):
                    return [
                        str(item).strip()
                        for item in parsed
                        if str(item).strip()
                    ]

            except Exception:
                pass

        # Fallback:
        # preserve the complete value rather than
        # making unsafe geographic assumptions.
        return [
            cleaned
        ]

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):

        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    return [
        str(value).strip()
    ]


# ---------------------------------------------------------
# Utility:
# ordered unique values
# ---------------------------------------------------------

def ordered_unique(values):

    output = []
    seen = set()

    for value in values:

        if (
            value
            and
            value not in seen
        ):

            output.append(
                value
            )

            seen.add(
                value
            )

    return output


# ---------------------------------------------------------
# Aggregate one book at a time
# ---------------------------------------------------------

book_geo_rows = []


for book_id, group in author_book_geo.groupby(
    "book_id",
    sort=False,
):

    author_names = ordered_unique(
        [
            str(value).strip()
            for value
            in group[
                "author_name"
            ].dropna()
            if str(value).strip()
        ]
    )


    citizenship_values = []

    for value in group[
        "citizenship_labels"
    ]:

        citizenship_values.extend(
            extract_label_values(
                value
            )
        )


    citizenship_values = ordered_unique(
        citizenship_values
    )


    birthplace_values = []

    for value in group[
        "birthplace_labels"
    ]:

        birthplace_values.extend(
            extract_label_values(
                value
            )
        )


    birthplace_values = ordered_unique(
        birthplace_values
    )


    evidence_tiers = ordered_unique(
        [
            str(value).strip()
            for value
            in group[
                "identity_evidence_tier"
            ].dropna()
            if str(value).strip()
        ]
    )


    book_geo_rows.append(
        {
            "book_id":
                book_id,

            "validated_geographic_author_count":
                group[
                    "openlibrary_author_key"
                ].nunique(),

            "validated_geographic_author_names":
                " | ".join(
                    author_names
                )
                if author_names
                else pd.NA,

            "author_citizenship_evidence":
                " | ".join(
                    citizenship_values
                )
                if citizenship_values
                else pd.NA,

            "author_birthplace_evidence":
                " | ".join(
                    birthplace_values
                )
                if birthplace_values
                else pd.NA,

            "author_identity_evidence_tiers":
                " | ".join(
                    evidence_tiers
                )
                if evidence_tiers
                else pd.NA,

            "author_geography_provenance":
                (
                    "Wikidata geographic claims "
                    "linked through validated "
                    "Open Library author identities"
                ),
        }
    )


book_author_geography = pd.DataFrame(
    book_geo_rows
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "\nBOOK-LEVEL AUTHOR GEOGRAPHY"
)
print("=" * 80)

print(
    "Rows:",
    len(book_author_geography)
)

print(
    "Unique book IDs:",
    book_author_geography[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    book_author_geography[
        "book_id"
    ].duplicated().sum()
)

print(
    "Books with citizenship evidence:",
    book_author_geography[
        "author_citizenship_evidence"
    ].notna().sum()
)

print(
    "Books with birthplace evidence:",
    book_author_geography[
        "author_birthplace_evidence"
    ].notna().sum()
)


# ---------------------------------------------------------
# Sample
# ---------------------------------------------------------

print(
    "\nAUTHOR GEOGRAPHY SAMPLE"
)
print("=" * 80)


display(
    book_author_geography[
        [
            "book_id",
            "validated_geographic_author_count",
            "validated_geographic_author_names",
            "author_citizenship_evidence",
            "author_birthplace_evidence",
            "author_identity_evidence_tiers",
        ]
    ]
    .head(20)
)


# ---------------------------------------------------------
# Save book-level evidence
# ---------------------------------------------------------

output_path = (
    processed_root
    / "leadwise_book_author_geography.csv"
)


book_author_geography.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

AUTHOR GEOGRAPHY BRIDGE
Author-book relationships: 1468
Unique books in bridge: 942
Unique Open Library author keys in bridge: 1237
Validated geographic author records: 467
Unique validated author keys: 467

EXACT AUTHOR-GEOGRAPHY MATCH
Matched author-book relationships: 608
Books with validated author geography: 466
Validated authors represented: 467


KeyError: 'author_name'

##### 18.38.2A — Corrected Book-Level Author Geography Aggregation

The exact identifier join between the Open Library author-book bridge and the validated Wikidata geographic evidence completed successfully.

Because both source tables contain an `author_name` field, pandas applied the specified merge suffixes and produced `author_name_bridge` and `author_name_geo`. The initial aggregation attempted to reference the pre-merge column name `author_name`, resulting in a `KeyError`.

The aggregation is corrected below using the validated geographic author name (`author_name_geo`).

No external data is recollected and no entity-resolution decisions are changed. This correction affects only the transformation of the already matched author-level evidence into one application-ready record per book.

In [78]:
# ---------------------------------------------------------
# 18.38.2A Correct book-level author geography aggregation
# ---------------------------------------------------------

import ast
import pandas as pd


# ---------------------------------------------------------
# Confirm merged author columns
# ---------------------------------------------------------

print(
    "MERGED AUTHOR COLUMNS"
)
print("=" * 80)

author_columns = [
    column
    for column in author_book_geo.columns
    if "author" in column.lower()
]

for column in author_columns:
    print(
        " -",
        column
    )


# ---------------------------------------------------------
# Select validated geographic author-name column
# ---------------------------------------------------------

if "author_name_geo" in author_book_geo.columns:

    author_name_column = (
        "author_name_geo"
    )

elif "author_name_bridge" in author_book_geo.columns:

    author_name_column = (
        "author_name_bridge"
    )

else:

    raise KeyError(
        "No merged author-name column was found."
    )


print(
    "\nAuthor name used for aggregation:",
    author_name_column
)


# ---------------------------------------------------------
# Utility:
# safely extract stored label values
# ---------------------------------------------------------

def extract_label_values(value):

    if pd.isna(value):
        return []

    if isinstance(value, str):

        cleaned = value.strip()

        if not cleaned:
            return []

        # Handle lists stored as strings
        if (
            cleaned.startswith("[")
            and
            cleaned.endswith("]")
        ):

            try:

                parsed = ast.literal_eval(
                    cleaned
                )

                if isinstance(
                    parsed,
                    list,
                ):

                    return [
                        str(item).strip()
                        for item in parsed
                        if str(item).strip()
                    ]

            except Exception:
                pass

        # Preserve non-list values exactly
        return [
            cleaned
        ]


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):

        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]


    return [
        str(value).strip()
    ]


# ---------------------------------------------------------
# Ordered unique utility
# ---------------------------------------------------------

def ordered_unique(values):

    output = []
    seen = set()

    for value in values:

        if (
            value
            and
            value not in seen
        ):

            output.append(
                value
            )

            seen.add(
                value
            )

    return output


# ---------------------------------------------------------
# Aggregate author evidence to one row per book
# ---------------------------------------------------------

book_geo_rows = []


for book_id, group in author_book_geo.groupby(
    "book_id",
    sort=False,
):

    # ---------------------------------------------
    # Validated author names
    # ---------------------------------------------

    author_names = ordered_unique(
        [
            str(value).strip()
            for value
            in group[
                author_name_column
            ].dropna()
            if str(value).strip()
        ]
    )


    # ---------------------------------------------
    # Citizenship evidence
    # ---------------------------------------------

    citizenship_values = []

    for value in group[
        "citizenship_labels"
    ]:

        citizenship_values.extend(
            extract_label_values(
                value
            )
        )


    citizenship_values = ordered_unique(
        citizenship_values
    )


    # ---------------------------------------------
    # Birthplace evidence
    # ---------------------------------------------

    birthplace_values = []

    for value in group[
        "birthplace_labels"
    ]:

        birthplace_values.extend(
            extract_label_values(
                value
            )
        )


    birthplace_values = ordered_unique(
        birthplace_values
    )


    # ---------------------------------------------
    # Identity evidence tiers
    # ---------------------------------------------

    evidence_tiers = ordered_unique(
        [
            str(value).strip()
            for value
            in group[
                "identity_evidence_tier"
            ].dropna()
            if str(value).strip()
        ]
    )


    # ---------------------------------------------
    # Book-level record
    # ---------------------------------------------

    book_geo_rows.append(
        {
            "book_id":
                book_id,

            "validated_geographic_author_count":
                group[
                    "openlibrary_author_key"
                ].nunique(),

            "validated_geographic_author_names":
                (
                    " | ".join(
                        author_names
                    )
                    if author_names
                    else pd.NA
                ),

            "author_citizenship_evidence":
                (
                    " | ".join(
                        citizenship_values
                    )
                    if citizenship_values
                    else pd.NA
                ),

            "author_birthplace_evidence":
                (
                    " | ".join(
                        birthplace_values
                    )
                    if birthplace_values
                    else pd.NA
                ),

            "author_identity_evidence_tiers":
                (
                    " | ".join(
                        evidence_tiers
                    )
                    if evidence_tiers
                    else pd.NA
                ),

            "author_geography_provenance":
                (
                    "Wikidata geographic claims linked "
                    "through validated Open Library "
                    "author identities"
                ),
        }
    )


book_author_geography = pd.DataFrame(
    book_geo_rows
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "\nBOOK-LEVEL AUTHOR GEOGRAPHY"
)
print("=" * 80)

print(
    "Rows:",
    len(book_author_geography)
)

print(
    "Unique book IDs:",
    book_author_geography[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    book_author_geography[
        "book_id"
    ].duplicated().sum()
)

print(
    "Books with citizenship evidence:",
    book_author_geography[
        "author_citizenship_evidence"
    ].notna().sum()
)

print(
    "Books with birthplace evidence:",
    book_author_geography[
        "author_birthplace_evidence"
    ].notna().sum()
)


# ---------------------------------------------------------
# Author-count distribution
# ---------------------------------------------------------

print(
    "\nVALIDATED GEOGRAPHIC AUTHORS PER BOOK"
)
print("=" * 80)

display(
    book_author_geography[
        "validated_geographic_author_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "validated_author_count"
    )
    .reset_index(
        name="book_count"
    )
)


# ---------------------------------------------------------
# Sample
# ---------------------------------------------------------

print(
    "\nAUTHOR GEOGRAPHY SAMPLE"
)
print("=" * 80)

display(
    book_author_geography[
        [
            "book_id",
            "validated_geographic_author_count",
            "validated_geographic_author_names",
            "author_citizenship_evidence",
            "author_birthplace_evidence",
            "author_identity_evidence_tiers",
        ]
    ]
    .head(20)
)


# ---------------------------------------------------------
# Save application-ready table
# ---------------------------------------------------------

output_path = (
    processed_root
    / "leadwise_book_author_geography.csv"
)


book_author_geography.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

MERGED AUTHOR COLUMNS
 - author_position
 - openlibrary_author_key
 - author_name_bridge
 - author_name_geo

Author name used for aggregation: author_name_geo

BOOK-LEVEL AUTHOR GEOGRAPHY
Rows: 466
Unique book IDs: 466
Duplicate book IDs: 0
Books with citizenship evidence: 189
Books with birthplace evidence: 149

VALIDATED GEOGRAPHIC AUTHORS PER BOOK


,validated_author_count,book_count
0,1,352
1,2,90
2,3,20
3,4,4



AUTHOR GEOGRAPHY SAMPLE


,book_id,validated_geographic_author_count,validated_geographic_author_names,author_citizenship_evidence,author_birthplace_evidence,author_identity_evidence_tiers
0,BOOK00001,1,Stephen R. Covey,United States,Salt Lake City,identifier_verified
1,BOOK00002,1,Gary A. Yukl,NaN,NaN,strong_secondary
2,BOOK00006,1,John C. Maxwell,United States,Garden City,identifier_verified
3,BOOK00008,1,Bernard M. Bass,United States,The Bronx,strong_secondary
4,BOOK00009,3,James Comey | James B. Comey,United States,Yonkers,strong_secondary | probable_secondary
5,BOOK00010,1,Dick Ruhe,NaN,NaN,strong_secondary
6,BOOK00012,2,Richard L. Hughes | Richard Hughes,United Kingdom,Weybridge,probable_secondary | identifier_verified
7,BOOK00013,1,Donald T. Phillips,United States,NaN,strong_secondary
8,BOOK00014,1,James MacGregor Burns,United States,Melrose,identifier_verified
9,BOOK00015,1,Robert N. Lussier,NaN,NaN,strong_secondary



Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_book_author_geography.csv


#### 18.38.3 — Final Enriched LeadWise Application Catalog

The validated enrichment layers are now integrated into a single application-ready LeadWise catalog.

The production grounding catalog serves as the integration anchor because it contains all 2,067 LeadWise books and preserves the authoritative `matrix_row` relationship with the TF-IDF recommendation matrix.

All enrichment sources are incorporated using left joins. This ensures that:

- every existing LeadWise book remains available;
- books are not removed because enrichment metadata is unavailable;
- missing metadata remains explicitly missing;
- enrichment does not alter recommendation vectors or similarity calculations; and
- the relationship between `book_id`, `matrix_row`, and the production recommendation matrix remains unchanged.

Only application-relevant fields are selected from the enrichment sources. Intermediate analytical, transformation, and model-development columns are excluded from the deployment catalog.

The resulting catalog supports richer Book Details, Book Explorer, comparison, topic exploration, and recommendation-result interfaces while keeping recommendation ranking independent from descriptive metadata such as geography, language, ratings, and commercial availability.

In [79]:
# ---------------------------------------------------------
# 18.38.3 Build final enriched LeadWise application catalog
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# 1. Anchor:
# production grounding catalog
# ---------------------------------------------------------

app_enriched = (
    loaded_tables[
        "grounding_catalog"
    ]
    .copy()
    .sort_values(
        "matrix_row"
    )
    .reset_index(
        drop=True
    )
)


initial_rows = len(
    app_enriched
)


print(
    "FINAL APPLICATION CATALOG INTEGRATION"
)
print("=" * 80)

print(
    "Anchor rows:",
    initial_rows
)

print(
    "Anchor unique book IDs:",
    app_enriched[
        "book_id"
    ].nunique()
)

print(
    "Anchor unique matrix rows:",
    app_enriched[
        "matrix_row"
    ].nunique()
)


# ---------------------------------------------------------
# 2. Core identity additions from books_master
# ---------------------------------------------------------

master_additions = (
    loaded_tables[
        "books_master"
    ][
        [
            "book_id",
            "openlibrary_key",
            "source_openlibrary",
            "source_leadershipnow",
        ]
    ]
    .copy()
)


app_enriched = app_enriched.merge(
    master_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 3. Open Library bibliographic metadata
# ---------------------------------------------------------

ol = loaded_tables[
    "openlibrary_integrated"
].copy()


ol_columns = [
    "book_id",
    "publishers",
    "isbn_10",
    "isbn_13",
    "all_isbns",
    "ebook_access",
    "has_fulltext",
    "public_scan",
    "source_url",
]


ol_additions = (
    ol[
        [
            column
            for column in ol_columns
            if column in ol.columns
        ]
    ]
    .copy()
)


app_enriched = app_enriched.merge(
    ol_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 4. LeadershipNow edition metadata
# ---------------------------------------------------------

ln = loaded_tables[
    "leadershipnow_editions"
].copy()


ln_columns = [
    "book_id",
    "subtitle",
    "isbn_normalized",
    "isbn_13",
    "publisher",
    "publication_date",
    "publication_year",
    "format",
    "page_count",
    "edition_note",
    "external_book_url",
]


ln_additions = (
    ln[
        [
            column
            for column in ln_columns
            if column in ln.columns
        ]
    ]
    .copy()
)


# Rename overlapping ISBN field before merge
if "isbn_13" in ln_additions.columns:

    ln_additions = (
        ln_additions.rename(
            columns={
                "isbn_13":
                    "leadershipnow_isbn_13"
            }
        )
    )


app_enriched = app_enriched.merge(
    ln_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 5. Open Library edition intelligence
# ---------------------------------------------------------

edition = loaded_tables[
    "edition_intelligence"
].copy()


edition_columns = [
    "book_id",
    "observed_edition_count",
    "earliest_observed_year",
    "latest_observed_year",
    "observed_publisher_count",
    "observed_publishers",
    "observed_isbn_10",
    "observed_isbn_10_count",
    "observed_isbn_13",
    "observed_isbn_13_count",
    "observed_physical_formats",
    "observed_format_count",
    "minimum_observed_pages",
    "maximum_observed_pages",
]


edition_additions = (
    edition[
        [
            column
            for column in edition_columns
            if column in edition.columns
        ]
    ]
    .copy()
)


app_enriched = app_enriched.merge(
    edition_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 6. Earliest observed publication geography
# ---------------------------------------------------------

publication_geo = (
    loaded_tables[
        "publication_geography"
    ]
    .copy()
)


geo_columns = [
    "book_id",
    "earliest_observed_publication_country",
    "earliest_observed_publication_countries",
    "publication_geo_evidence_type",
    "unresolved_geo_codes",
    "publication_geo_provenance",
]


geo_additions = (
    publication_geo[
        [
            column
            for column in geo_columns
            if column in publication_geo.columns
        ]
    ]
    .copy()
)


app_enriched = app_enriched.merge(
    geo_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 7. Publication-language evidence
# ---------------------------------------------------------

language = (
    loaded_tables[
        "language_evidence"
    ]
    .copy()
)


language_columns = [
    "book_id",
    "observed_publication_languages",
    "observed_publication_language_count",
    "has_multilingual_publication_evidence",
    "international_language_reach",
    "original_language",
    "original_language_status",
    "verified_translation_count",
    "translation_status",
    "language_evidence_provenance",
    "language_evidence_note",
]


language_additions = (
    language[
        [
            column
            for column in language_columns
            if column in language.columns
        ]
    ]
    .copy()
)


app_enriched = app_enriched.merge(
    language_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 8. Book-level author geographic evidence
# ---------------------------------------------------------

author_geo_path = (
    processed_root
    / "leadwise_book_author_geography.csv"
)


book_author_geo = pd.read_csv(
    author_geo_path,
    low_memory=False,
)


app_enriched = app_enriched.merge(
    book_author_geo,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 9. Commercial metadata status
# ---------------------------------------------------------

commercial = (
    loaded_tables[
        "commercial_metadata"
    ]
    .copy()
)


commercial_columns = [
    "book_id",
    "observed_list_price",
    "observed_retail_price",
    "observed_price_currency",
    "observed_price_market",
    "observed_price_isbn",
    "commercial_source",
    "commercial_metadata_status",
    "commercial_metadata_note",
]


commercial_additions = (
    commercial[
        commercial_columns
    ]
    .copy()
)


app_enriched = app_enriched.merge(
    commercial_additions,
    on="book_id",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# 10. Final ordering
# ---------------------------------------------------------

app_enriched = (
    app_enriched
    .sort_values(
        "matrix_row"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 11. Structural integrity
# ---------------------------------------------------------

print(
    "\nSTRUCTURAL INTEGRITY"
)
print("=" * 80)

print(
    "Final rows:",
    len(app_enriched)
)

print(
    "Final columns:",
    len(app_enriched.columns)
)

print(
    "Unique book IDs:",
    app_enriched[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    app_enriched[
        "book_id"
    ].duplicated().sum()
)

print(
    "Unique matrix rows:",
    app_enriched[
        "matrix_row"
    ].nunique()
)

print(
    "Duplicate matrix rows:",
    app_enriched[
        "matrix_row"
    ].duplicated().sum()
)

print(
    "Row count preserved:",
    len(app_enriched)
    ==
    initial_rows
)


# ---------------------------------------------------------
# 12. Matrix-row alignment
# ---------------------------------------------------------

expected_matrix_rows = list(
    range(
        len(app_enriched)
    )
)


actual_matrix_rows = (
    app_enriched[
        "matrix_row"
    ]
    .astype(int)
    .tolist()
)


print(
    "Matrix rows sequential:",
    actual_matrix_rows
    ==
    expected_matrix_rows
)


# ---------------------------------------------------------
# 13. Enrichment coverage
# ---------------------------------------------------------

print(
    "\nAPPLICATION METADATA COVERAGE"
)
print("=" * 80)


coverage_fields = [
    "cover_url",
    "description",
    "average_rating",
    "publishers",
    "publisher",
    "page_count",
    "observed_publication_languages",
    "earliest_observed_publication_country",
    "validated_geographic_author_names",
    "author_citizenship_evidence",
    "author_birthplace_evidence",
    "commercial_metadata_status",
]


coverage_rows = []


for column in coverage_fields:

    if column not in app_enriched.columns:
        continue

    count = (
        app_enriched[
            column
        ]
        .notna()
        .sum()
    )

    coverage_rows.append(
        {
            "field":
                column,

            "records_with_values":
                count,

            "coverage_pct":
                round(
                    (
                        count
                        /
                        len(app_enriched)
                    )
                    * 100,
                    2,
                ),
        }
    )


coverage_df = pd.DataFrame(
    coverage_rows
)


display(
    coverage_df
)


# ---------------------------------------------------------
# 14. Critical recommendation alignment checks
# ---------------------------------------------------------

print(
    "\nRECOMMENDATION ALIGNMENT"
)
print("=" * 80)

print(
    "Books with usable recommendation vectors:",
    (
        ~app_enriched[
            "enriched_zero_vector"
        ].astype(bool)
    ).sum()
)

print(
    "Books with zero recommendation vectors:",
    app_enriched[
        "enriched_zero_vector"
    ].astype(bool).sum()
)

print(
    "Books with topic cluster:",
    app_enriched[
        "topic_cluster"
    ].notna().sum()
)

print(
    "Books with cluster label:",
    app_enriched[
        "cluster_label"
    ].notna().sum()
)


# ---------------------------------------------------------
# 15. Save application catalog
# ---------------------------------------------------------

output_path = (
    processed_root
    / "leadwise_app_catalog_enriched.csv"
)


app_enriched.to_csv(
    output_path,
    index=False,
)


print(
    "\nSaved:",
    output_path
)

FINAL APPLICATION CATALOG INTEGRATION
Anchor rows: 2067
Anchor unique book IDs: 2067
Anchor unique matrix rows: 2067

STRUCTURAL INTEGRITY
Final rows: 2067
Final columns: 82
Unique book IDs: 2067
Duplicate book IDs: 0
Unique matrix rows: 2067
Duplicate matrix rows: 0
Row count preserved: True
Matrix rows sequential: True

APPLICATION METADATA COVERAGE


,field,records_with_values,coverage_pct
0,cover_url,1705,82.49
1,description,192,9.29
2,average_rating,282,13.64
3,publishers,950,45.96
4,publisher,1120,54.18
5,page_count,1119,54.14
6,observed_publication_languages,900,43.54
7,earliest_observed_publication_country,438,21.19
8,validated_geographic_author_names,466,22.54
9,author_citizenship_evidence,189,9.14



RECOMMENDATION ALIGNMENT
Books with usable recommendation vectors: 2040
Books with zero recommendation vectors: 27
Books with topic cluster: 1884
Books with cluster label: 1884

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_app_catalog_enriched.csv


### 18.39 — Streamlit Deployment Catalog Preparation

The enriched LeadWise application catalog contains 2,067 books and 82 metadata fields while preserving the production recommendation-system matrix alignment.

Before integrating the enriched catalog into the Streamlit interface, the deployment schema is validated and standardized.

This stage focuses on application behavior rather than additional data collection. It ensures that:

- identifiers remain stable;
- `matrix_row` remains integer-aligned with the recommendation matrix;
- numeric reader metrics are stored consistently;
- publication years and page counts can be displayed without decimal artifacts;
- Boolean evidence fields are interpreted consistently;
- missing metadata remains genuinely missing rather than appearing as `"nan"` or `"None"`;
- URLs are distinguishable from ordinary text;
- recommendation-critical fields remain intact; and
- optional metadata can be conditionally rendered by the interface.

No recommendation vectors, similarity calculations, topic assignments, or enrichment evidence are changed during deployment preparation.

In [80]:
# ---------------------------------------------------------
# 18.39.1 Streamlit deployment schema audit
# ---------------------------------------------------------

import pandas as pd


catalog_path = (
    processed_root
    / "leadwise_app_catalog_enriched.csv"
)


deployment_catalog = pd.read_csv(
    catalog_path,
    low_memory=False,
)


print(
    "STREAMLIT DEPLOYMENT CATALOG"
)
print("=" * 80)

print(
    "Shape:",
    deployment_catalog.shape
)

print(
    "Unique books:",
    deployment_catalog[
        "book_id"
    ].nunique()
)

print(
    "Unique matrix rows:",
    deployment_catalog[
        "matrix_row"
    ].nunique()
)


# ---------------------------------------------------------
# Data-type inventory
# ---------------------------------------------------------

print(
    "\nDATA TYPE INVENTORY"
)
print("=" * 80)


dtype_inventory = (
    deployment_catalog
    .dtypes
    .astype(str)
    .rename_axis(
        "field"
    )
    .reset_index(
        name="dtype"
    )
)


display(
    dtype_inventory
)


# ---------------------------------------------------------
# Application-critical fields
# ---------------------------------------------------------

critical_fields = [
    "matrix_row",
    "book_id",
    "canonical_title",
    "authors",
    "description",
    "subjects",
    "cover_url",
    "first_publish_year",
    "publication_year_observed",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "topic_cluster",
    "cluster_label",
    "enriched_zero_vector",
]


print(
    "\nCRITICAL FIELD CHECK"
)
print("=" * 80)


critical_rows = []


for field in critical_fields:

    exists = (
        field
        in deployment_catalog.columns
    )

    critical_rows.append(
        {
            "field":
                field,

            "exists":
                exists,

            "dtype":
                (
                    str(
                        deployment_catalog[
                            field
                        ].dtype
                    )
                    if exists
                    else None
                ),

            "non_null_records":
                (
                    deployment_catalog[
                        field
                    ].notna().sum()
                    if exists
                    else None
                ),
        }
    )


critical_audit = pd.DataFrame(
    critical_rows
)


display(
    critical_audit
)


# ---------------------------------------------------------
# Numeric display fields
# ---------------------------------------------------------

numeric_display_fields = [
    "first_publish_year",
    "publication_year_observed",
    "publication_year",
    "page_count",
    "minimum_observed_pages",
    "maximum_observed_pages",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "observed_publication_language_count",
    "validated_geographic_author_count",
]


print(
    "\nNUMERIC DISPLAY FIELD AUDIT"
)
print("=" * 80)


numeric_rows = []


for field in numeric_display_fields:

    if field not in deployment_catalog.columns:
        continue

    series = deployment_catalog[
        field
    ]


    numeric_rows.append(
        {
            "field":
                field,

            "dtype":
                str(
                    series.dtype
                ),

            "non_null":
                series.notna().sum(),

            "minimum":
                (
                    series.min()
                    if pd.api.types.is_numeric_dtype(
                        series
                    )
                    else None
                ),

            "maximum":
                (
                    series.max()
                    if pd.api.types.is_numeric_dtype(
                        series
                    )
                    else None
                ),
        }
    )


numeric_audit = pd.DataFrame(
    numeric_rows
)


display(
    numeric_audit
)


# ---------------------------------------------------------
# Boolean / status fields
# ---------------------------------------------------------

boolean_candidates = [
    "enriched_zero_vector",
    "source_openlibrary",
    "source_leadershipnow",
    "has_fulltext",
    "public_scan",
    "has_multilingual_publication_evidence",
]


print(
    "\nBOOLEAN FIELD AUDIT"
)
print("=" * 80)


for field in boolean_candidates:

    if field not in deployment_catalog.columns:
        continue

    print(
        f"\n{field}"
    )

    print(
        deployment_catalog[
            field
        ]
        .value_counts(
            dropna=False
        )
        .head(10)
    )


# ---------------------------------------------------------
# URL fields
# ---------------------------------------------------------

url_fields = [
    "cover_url",
    "source_url",
    "external_book_url",
]


print(
    "\nURL FIELD COVERAGE"
)
print("=" * 80)


for field in url_fields:

    if field not in deployment_catalog.columns:
        continue

    print(
        f"{field}: "
        f"{deployment_catalog[field].notna().sum()} "
        f"/ {len(deployment_catalog)}"
    )


# ---------------------------------------------------------
# Missing-value string audit
#
# Check whether literal text values such as
# "nan", "none", or "<na>" survived CSV processing.
# ---------------------------------------------------------

print(
    "\nLITERAL MISSING-VALUE STRING AUDIT"
)
print("=" * 80)


missing_tokens = {
    "nan",
    "none",
    "<na>",
    "null",
}


problem_rows = []


for field in deployment_catalog.select_dtypes(
    include="object"
).columns:

    normalized = (
        deployment_catalog[
            field
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )


    count = (
        normalized
        .isin(
            missing_tokens
        )
        .sum()
    )


    if count > 0:

        problem_rows.append(
            {
                "field":
                    field,

                "literal_missing_strings":
                    count,
            }
        )


if problem_rows:

    display(
        pd.DataFrame(
            problem_rows
        )
    )

else:

    print(
        "No literal missing-value strings detected."
    )

STREAMLIT DEPLOYMENT CATALOG
Shape: (2067, 82)
Unique books: 2067
Unique matrix rows: 2067

DATA TYPE INVENTORY


,field,dtype
0,matrix_row,int64
1,book_id,str
2,canonical_title,str
3,authors,str
4,description,str
...,...,...
77,observed_price_market,float64
78,observed_price_isbn,float64
79,commercial_source,float64
80,commercial_metadata_status,str



CRITICAL FIELD CHECK


,field,exists,dtype,non_null_records
0,matrix_row,True,int64,2067
1,book_id,True,str,2067
2,canonical_title,True,str,2067
3,authors,True,str,1884
4,description,True,str,192
5,subjects,True,str,1884
6,cover_url,True,str,1705
7,first_publish_year,True,float64,947
8,publication_year_observed,True,float64,934
9,average_rating,True,float64,282



NUMERIC DISPLAY FIELD AUDIT


,field,dtype,non_null,minimum,maximum
0,first_publish_year,float64,947,1900.0,2025.0
1,publication_year_observed,float64,934,2022.0,2025.0
2,publication_year,float64,1120,2022.0,2025.0
3,page_count,float64,1119,96.0,1200.0
4,minimum_observed_pages,float64,843,0.0,1216.0
5,maximum_observed_pages,float64,843,1.0,100000000.0
6,average_rating,float64,282,0.0,5.0
7,ratings_count,float64,282,0.0,278.0
8,edition_count,float64,950,1.0,95.0
9,want_to_read_count,float64,807,0.0,6754.0



BOOLEAN FIELD AUDIT

enriched_zero_vector
enriched_zero_vector
False    2040
True       27
Name: count, dtype: int64

source_openlibrary
source_openlibrary
False    1117
True      950
Name: count, dtype: int64

source_leadershipnow
source_leadershipnow
True     1120
False     947
Name: count, dtype: int64

has_fulltext
has_fulltext
NaN      1117
True      549
False     401
Name: count, dtype: int64

public_scan
public_scan
NaN      1117
False     946
True        4
Name: count, dtype: int64

has_multilingual_publication_evidence
has_multilingual_publication_evidence
NaN      1119
False     908
True       40
Name: count, dtype: int64

URL FIELD COVERAGE
cover_url: 1705 / 2067
source_url: 950 / 2067
external_book_url: 1120 / 2067

LITERAL MISSING-VALUE STRING AUDIT
No literal missing-value strings detected.


/var/folders/3_/6lgqyskd5v55bylpxy1jbmzr0000gn/T/ipykernel_46103/803504011.py:334: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for field in deployment_catalog.select_dtypes(


#### 18.39.2 — Deployment Data-Type and Display Standardization

The deployment-schema audit confirmed that the enriched catalog preserves recommendation alignment and contains no literal missing-value strings.

Several numeric metadata fields are stored as floating-point values because missing observations are present. For application display, discrete values such as publication years, page counts, reader counts, edition counts, and topic-cluster identifiers are converted to pandas nullable integer (`Int64`) fields. This preserves missing values while preventing display artifacts such as `2024.0` or `320.0`.

The audit also identified an implausible maximum observed edition page count of 100,000,000 pages. This value originates from observed Open Library edition metadata and is retained in an audit field, but implausible edition-derived page counts above 10,000 pages are suppressed from application display.

This rule applies only to deployment presentation fields and does not modify the original collected or processed source datasets.

Missing metadata remains missing and is handled conditionally by the Streamlit interface.

In [81]:
# ---------------------------------------------------------
# 18.39.2 Standardize deployment catalog
# ---------------------------------------------------------

import pandas as pd
import numpy as np


streamlit_catalog = (
    deployment_catalog
    .copy()
)


# ---------------------------------------------------------
# 1. Preserve original edition-derived page values
# ---------------------------------------------------------

for field in [
    "minimum_observed_pages",
    "maximum_observed_pages",
]:

    if field in streamlit_catalog.columns:

        streamlit_catalog[
            f"{field}_raw"
        ] = streamlit_catalog[
            field
        ]


# ---------------------------------------------------------
# 2. Flag implausible edition-derived page observations
#
# This is a deployment-display rule only.
# Original values remain in *_raw.
# ---------------------------------------------------------

PAGE_DISPLAY_UPPER_LIMIT = 10_000


streamlit_catalog[
    "edition_page_count_anomaly"
] = False


for field in [
    "minimum_observed_pages",
    "maximum_observed_pages",
]:

    if field not in streamlit_catalog.columns:
        continue

    numeric_values = pd.to_numeric(
        streamlit_catalog[field],
        errors="coerce",
    )

    anomaly_mask = (
        numeric_values
        >
        PAGE_DISPLAY_UPPER_LIMIT
    )

    streamlit_catalog.loc[
        anomaly_mask,
        "edition_page_count_anomaly"
    ] = True

    streamlit_catalog.loc[
        anomaly_mask,
        field
    ] = np.nan


# ---------------------------------------------------------
# 3. Nullable integer fields
# ---------------------------------------------------------

integer_fields = [
    "matrix_row",
    "first_publish_year",
    "publication_year_observed",
    "publication_year",
    "page_count",
    "minimum_observed_pages",
    "maximum_observed_pages",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "topic_cluster",
    "observed_edition_count",
    "earliest_observed_year",
    "latest_observed_year",
    "observed_publisher_count",
    "observed_isbn_10_count",
    "observed_isbn_13_count",
    "observed_format_count",
    "observed_publication_language_count",
    "verified_translation_count",
    "validated_geographic_author_count",
]


for field in integer_fields:

    if field not in streamlit_catalog.columns:
        continue

    streamlit_catalog[field] = (
        pd.to_numeric(
            streamlit_catalog[field],
            errors="coerce",
        )
        .round()
        .astype("Int64")
    )


# ---------------------------------------------------------
# 4. Continuous numeric fields
# ---------------------------------------------------------

float_fields = [
    "average_rating",
    "observed_list_price",
    "observed_retail_price",
]


for field in float_fields:

    if field not in streamlit_catalog.columns:
        continue

    streamlit_catalog[field] = (
        pd.to_numeric(
            streamlit_catalog[field],
            errors="coerce",
        )
    )


# ---------------------------------------------------------
# 5. Recommendation-critical Boolean
# ---------------------------------------------------------

streamlit_catalog[
    "enriched_zero_vector"
] = (
    streamlit_catalog[
        "enriched_zero_vector"
    ]
    .astype(bool)
)


# ---------------------------------------------------------
# 6. Nullable Boolean evidence fields
# ---------------------------------------------------------

nullable_boolean_fields = [
    "has_fulltext",
    "public_scan",
    "has_multilingual_publication_evidence",
]


for field in nullable_boolean_fields:

    if field in streamlit_catalog.columns:

        streamlit_catalog[field] = (
            streamlit_catalog[field]
            .astype("boolean")
        )


# ---------------------------------------------------------
# 7. Validate page-count anomaly handling
# ---------------------------------------------------------

print(
    "PAGE-COUNT DISPLAY VALIDATION"
)
print("=" * 80)

print(
    "Books flagged with edition page anomaly:",
    streamlit_catalog[
        "edition_page_count_anomaly"
    ].sum()
)

print(
    "Maximum displayed minimum_observed_pages:",
    streamlit_catalog[
        "minimum_observed_pages"
    ].max()
)

print(
    "Maximum displayed maximum_observed_pages:",
    streamlit_catalog[
        "maximum_observed_pages"
    ].max()
)


# ---------------------------------------------------------
# Show affected records
# ---------------------------------------------------------

page_anomalies = (
    streamlit_catalog[
        streamlit_catalog[
            "edition_page_count_anomaly"
        ]
    ][
        [
            "book_id",
            "canonical_title",
            "authors",
            "page_count",
            "minimum_observed_pages_raw",
            "maximum_observed_pages_raw",
            "minimum_observed_pages",
            "maximum_observed_pages",
        ]
    ]
)


print(
    "\nFLAGGED PAGE-COUNT RECORDS"
)
print("=" * 80)

display(
    page_anomalies
)


# ---------------------------------------------------------
# 8. Final dtype validation
# ---------------------------------------------------------

print(
    "\nDISPLAY DATA TYPES"
)
print("=" * 80)


display_fields = [
    "matrix_row",
    "first_publish_year",
    "publication_year_observed",
    "publication_year",
    "page_count",
    "minimum_observed_pages",
    "maximum_observed_pages",
    "average_rating",
    "ratings_count",
    "edition_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "topic_cluster",
    "observed_publication_language_count",
    "validated_geographic_author_count",
    "enriched_zero_vector",
    "has_multilingual_publication_evidence",
    "edition_page_count_anomaly",
]


dtype_check = pd.DataFrame(
    {
        "field": [
            field
            for field in display_fields
            if field in streamlit_catalog.columns
        ]
    }
)


dtype_check[
    "dtype"
] = dtype_check[
    "field"
].map(
    lambda field:
        str(
            streamlit_catalog[
                field
            ].dtype
        )
)


display(
    dtype_check
)


# ---------------------------------------------------------
# 9. Final structural checks
# ---------------------------------------------------------

print(
    "\nFINAL DEPLOYMENT INTEGRITY"
)
print("=" * 80)

print(
    "Rows:",
    len(streamlit_catalog)
)

print(
    "Unique book IDs:",
    streamlit_catalog[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    streamlit_catalog[
        "book_id"
    ].duplicated().sum()
)

print(
    "Unique matrix rows:",
    streamlit_catalog[
        "matrix_row"
    ].nunique()
)

print(
    "Duplicate matrix rows:",
    streamlit_catalog[
        "matrix_row"
    ].duplicated().sum()
)

print(
    "Usable recommendation vectors:",
    (
        ~streamlit_catalog[
            "enriched_zero_vector"
        ]
    ).sum()
)


# ---------------------------------------------------------
# 10. Save deployment-ready catalog
# ---------------------------------------------------------

deployment_output_path = (
    processed_root
    / "leadwise_streamlit_catalog.csv"
)


streamlit_catalog.to_csv(
    deployment_output_path,
    index=False,
)


print(
    "\nSaved:",
    deployment_output_path
)

PAGE-COUNT DISPLAY VALIDATION
Books flagged with edition page anomaly: 10
Maximum displayed minimum_observed_pages: 1216
Maximum displayed maximum_observed_pages: 1968

FLAGGED PAGE-COUNT RECORDS


,book_id,canonical_title,authors,page_count,minimum_observed_pages_raw,maximum_observed_pages_raw,minimum_observed_pages,maximum_observed_pages
103,BOOK00104,Effective Executive,"['Peter F. Drucker', 'Joseph A. Maciariello', ...",<NA>,148.0,35167.0,148,<NA>
281,BOOK00282,Management information systems,"['Kenneth C. Laudon', 'Jane P. Laudon', 'Jane ...",<NA>,1.0,99998.0,1,<NA>
295,BOOK00296,Strategic management and business policy,"['Thomas L. Wheelen', 'J. David Hunger', 'Tom ...",<NA>,136.0,99998.0,136,<NA>
304,BOOK00305,Operations Management,['Lee J. Krajewski'],<NA>,1.0,99998.0,1,<NA>
306,BOOK00307,Developing mangement skills,"['David A. Whetten', 'Kim S. Cameron', 'Whette...",<NA>,4.0,99999.0,4,<NA>
383,BOOK00384,Strategic management,['Fred R. David'],<NA>,290.0,99998.0,290,<NA>
428,BOOK00429,Change management in information services,['Lyndon Pugh'],<NA>,200.0,99999.0,200,<NA>
617,BOOK00618,Human Resource Management,['Gary Dessler'],<NA>,6.0,99998.0,6,<NA>
699,BOOK00700,Essentials of Organizational Behavior,"['John J. Wild', 'Kenneth L. Wild', 'Alverne B...",<NA>,326.0,99998.0,326,<NA>
914,BOOK00915,The Innovator's Dilemma,"['Clayton M. Christensen', 'L J Ganser', 'Don ...",<NA>,1.0,100000000.0,1,<NA>



DISPLAY DATA TYPES


,field,dtype
0,matrix_row,Int64
1,first_publish_year,Int64
2,publication_year_observed,Int64
3,publication_year,Int64
4,page_count,Int64
5,minimum_observed_pages,Int64
6,maximum_observed_pages,Int64
7,average_rating,float64
8,ratings_count,Int64
9,edition_count,Int64



FINAL DEPLOYMENT INTEGRITY
Rows: 2067
Unique book IDs: 2067
Duplicate book IDs: 0
Unique matrix rows: 2067
Duplicate matrix rows: 0
Usable recommendation vectors: 2040

Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_streamlit_catalog.csv


### 18.40 — Streamlit Integration of the Enriched Deployment Catalog

The final deployment catalog is now ready for integration with the LeadWise Streamlit application.

The application will replace the earlier grounding catalog with `leadwise_streamlit_catalog.csv`. The enriched catalog preserves the same 2,067-book `matrix_row` structure used by the production TF-IDF recommendation matrix while adding application-facing bibliographic, publication, language, geographic, author, and commercial metadata.

The recommendation model itself remains unchanged.

Before redesigning recommendation cards and Book Details, the enriched catalog is tested against the existing production vectorizer and TF-IDF matrix to verify that:

- catalog row count matches matrix row count;
- `matrix_row` remains sequential;
- `book_id` remains unique;
- recommendation-vector availability remains unchanged; and
- natural-language retrieval continues to operate correctly.

This separates deployment-data integration from subsequent user-interface changes and reduces the risk of introducing recommendation errors during visual development.

In [82]:
# ---------------------------------------------------------
# 18.40.1 Validate enriched catalog against production model
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# Load deployment catalog
# ---------------------------------------------------------

streamlit_catalog_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)


streamlit_catalog_test = pd.read_csv(
    streamlit_catalog_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Structural alignment
# ---------------------------------------------------------

print(
    "STREAMLIT CATALOG ↔ MODEL ALIGNMENT"
)
print("=" * 80)

print(
    "Catalog rows:",
    len(streamlit_catalog_test)
)

print(
    "TF-IDF matrix rows:",
    enriched_tfidf_matrix.shape[0]
)

print(
    "Row counts match:",
    len(streamlit_catalog_test)
    ==
    enriched_tfidf_matrix.shape[0]
)

print(
    "Unique book IDs:",
    streamlit_catalog_test[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    streamlit_catalog_test[
        "book_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# Validate matrix_row
# ---------------------------------------------------------

matrix_rows = (
    streamlit_catalog_test[
        "matrix_row"
    ]
    .astype(int)
    .to_numpy()
)


expected_rows = np.arange(
    len(
        streamlit_catalog_test
    )
)


print(
    "Matrix rows sequential:",
    np.array_equal(
        matrix_rows,
        expected_rows
    )
)


# ---------------------------------------------------------
# Vector-status validation
# ---------------------------------------------------------

zero_vector_status = (
    streamlit_catalog_test[
        "enriched_zero_vector"
    ]
)


# CSV reload should normally preserve these as booleans,
# but handle strings defensively.
if not pd.api.types.is_bool_dtype(
    zero_vector_status
):

    zero_vector_status = (
        zero_vector_status
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )


print(
    "Usable vectors:",
    (~zero_vector_status).sum()
)

print(
    "Zero vectors:",
    zero_vector_status.sum()
)


# ---------------------------------------------------------
# Replace notebook test catalog
# ---------------------------------------------------------

app_catalog_enriched = (
    streamlit_catalog_test
    .sort_values(
        "matrix_row"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Retrieval test using enriched catalog
# ---------------------------------------------------------

test_query = (
    "I want to become a better leader by improving "
    "decision making, communication, team leadership, "
    "and strategic thinking."
)


query_vector = (
    enriched_vectorizer.transform(
        [
            test_query
        ]
    )
)


query_similarities = (
    cosine_similarity(
        query_vector,
        enriched_tfidf_matrix,
    )
    .ravel()
)


# Prevent zero-vector books from being recommended
query_similarities[
    zero_vector_status.to_numpy(
        dtype=bool
    )
] = -1


ranked_indices = (
    np.argsort(
        query_similarities
    )[::-1]
)


results = []

seen_duplicate_keys = set()


for idx in ranked_indices:

    score = (
        query_similarities[
            idx
        ]
    )

    if score <= 0:
        continue

    row = (
        app_catalog_enriched
        .iloc[
            idx
        ]
    )


    title = (
        str(
            row[
                "canonical_title"
            ]
        )
        if pd.notna(
            row[
                "canonical_title"
            ]
        )
        else ""
    )


    authors = (
        str(
            row[
                "authors"
            ]
        )
        if pd.notna(
            row[
                "authors"
            ]
        )
        else ""
    )


    duplicate_key = (
        normalize_duplicate_text(
            title
        )
        + " || "
        + normalize_duplicate_text(
            authors
        )
    )


    if duplicate_key in seen_duplicate_keys:
        continue


    seen_duplicate_keys.add(
        duplicate_key
    )


    results.append(
        {
            "rank":
                len(results) + 1,

            "book_id":
                row[
                    "book_id"
                ],

            "title":
                title,

            "authors":
                authors,

            "similarity":
                round(
                    float(score),
                    6,
                ),

            "cover_available":
                pd.notna(
                    row[
                        "cover_url"
                    ]
                ),

            "description_available":
                pd.notna(
                    row[
                        "description"
                    ]
                ),

            "publication_language_evidence":
                pd.notna(
                    row.get(
                        "observed_publication_languages"
                    )
                ),

            "publication_country_evidence":
                pd.notna(
                    row.get(
                        "earliest_observed_publication_country"
                    )
                ),
        }
    )


    if len(results) >= 10:
        break


retrieval_test = pd.DataFrame(
    results
)


print(
    "\nENRICHED-CATALOG RETRIEVAL TEST"
)
print("=" * 80)


display(
    retrieval_test
)


# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

print(
    "\nDEPLOYMENT INTEGRATION CHECK"
)
print("=" * 80)

print(
    "Returned recommendations:",
    len(retrieval_test)
)

print(
    "All similarities positive:",
    (
        retrieval_test[
            "similarity"
        ]
        > 0
    ).all()
)

print(
    "Unique recommendation book IDs:",
    retrieval_test[
        "book_id"
    ].nunique()
)

print(
    "Unique title-author recommendations:",
    len(
        seen_duplicate_keys
    )
    >=
    len(
        retrieval_test
    )
)

STREAMLIT CATALOG ↔ MODEL ALIGNMENT
Catalog rows: 2067


NameError: name 'enriched_tfidf_matrix' is not defined

#### 18.40.1A — Reload Production Recommendation Artifacts

The deployment catalog loaded successfully, but the current notebook session no longer contains the in-memory production TF-IDF matrix variable used by the recommendation-system validation.

The previously saved production artifacts are therefore reloaded from the project `models` directory.

This does not retrain or modify the recommendation model. It restores the existing production TF-IDF matrix and Streamlit-compatible vectorizer so that deployment alignment can be validated against the enriched application catalog.

In [83]:
# ---------------------------------------------------------
# 18.40.1A Locate saved production recommendation artifacts
# ---------------------------------------------------------

from pathlib import Path


models_root = (
    project_root
    / "models"
)


print(
    "SAVED MODEL ARTIFACTS"
)
print("=" * 80)


model_files = sorted(
    [
        path
        for path in models_root.iterdir()
        if path.is_file()
    ],
    key=lambda path:
        path.name.lower(),
)


for path in model_files:

    print(
        path.name
    )


print(
    "\nTotal model files:",
    len(model_files)
)

SAVED MODEL ARTIFACTS
book_autoencoder_200_32.pt
core_svd_200.joblib
core_tfidf_matrix.npz
core_tfidf_vectorizer.joblib
enriched_lsa_200_tensor.pt
enriched_svd_200.joblib
enriched_tfidf_matrix.npz
enriched_tfidf_vectorizer.joblib
enriched_tfidf_vectorizer_app.joblib
enriched_tfidf_vectorizer_streamlit.joblib
final_model_architecture.json
llm_config.json
llm_system_instructions.txt
pca_model.joblib
pca_scaler.joblib

Total model files: 15


#### 18.40.1B — Reload Production TF-IDF Recommendation Artifacts

The saved model inventory confirms the availability of the production enriched TF-IDF artifacts.

For Streamlit deployment validation, LeadWise uses:

- `enriched_tfidf_matrix.npz` as the production book-content matrix; and
- `enriched_tfidf_vectorizer_streamlit.joblib` as the portable query vectorizer.

The Streamlit-compatible vectorizer preserves the same fitted TF-IDF vocabulary and transformation behavior while referencing the deployment-safe analyzer module.

The SVD, LSA tensor, PCA, and autoencoder artifacts are not used for production recommendation ranking. LeadWise continues to calculate recommendations using cosine similarity directly against the enriched TF-IDF representation.

In [84]:
# ---------------------------------------------------------
# 18.40.1B Reload production TF-IDF artifacts
# ---------------------------------------------------------

import sys
import joblib

from scipy.sparse import load_npz


# ---------------------------------------------------------
# Make deployment analyzer importable
# ---------------------------------------------------------

app_root = (
    project_root
    / "app"
)


if str(app_root) not in sys.path:

    sys.path.insert(
        0,
        str(app_root),
    )


# Import deployment analyzer module before joblib load
import nlp_utils


# ---------------------------------------------------------
# Artifact paths
# ---------------------------------------------------------

matrix_path = (
    models_root
    / "enriched_tfidf_matrix.npz"
)


vectorizer_path = (
    models_root
    / "enriched_tfidf_vectorizer_streamlit.joblib"
)


# ---------------------------------------------------------
# Load production artifacts
# ---------------------------------------------------------

enriched_tfidf_matrix = load_npz(
    matrix_path
)


enriched_vectorizer = joblib.load(
    vectorizer_path
)


# ---------------------------------------------------------
# Basic validation
# ---------------------------------------------------------

print(
    "PRODUCTION TF-IDF ARTIFACTS"
)
print("=" * 80)

print(
    "Matrix shape:",
    enriched_tfidf_matrix.shape
)

print(
    "Matrix type:",
    type(
        enriched_tfidf_matrix
    ).__name__
)

print(
    "Vectorizer type:",
    type(
        enriched_vectorizer
    ).__name__
)

print(
    "Vocabulary size:",
    len(
        enriched_vectorizer.vocabulary_
    )
)

print(
    "Feature-name count:",
    len(
        enriched_vectorizer.get_feature_names_out()
    )
)


# ---------------------------------------------------------
# Matrix ↔ vectorizer compatibility
# ---------------------------------------------------------

print(
    "\nARTIFACT COMPATIBILITY"
)
print("=" * 80)

print(
    "Matrix feature count:",
    enriched_tfidf_matrix.shape[1]
)

print(
    "Vectorizer feature count:",
    len(
        enriched_vectorizer.get_feature_names_out()
    )
)

print(
    "Feature dimensions match:",
    (
        enriched_tfidf_matrix.shape[1]
        ==
        len(
            enriched_vectorizer.get_feature_names_out()
        )
    )
)


# ---------------------------------------------------------
# Catalog ↔ matrix compatibility
# ---------------------------------------------------------

print(
    "\nCATALOG COMPATIBILITY"
)
print("=" * 80)

print(
    "Catalog rows:",
    len(
        streamlit_catalog_test
    )
)

print(
    "Matrix rows:",
    enriched_tfidf_matrix.shape[0]
)

print(
    "Catalog and matrix rows match:",
    (
        len(
            streamlit_catalog_test
        )
        ==
        enriched_tfidf_matrix.shape[0]
    )
)

PRODUCTION TF-IDF ARTIFACTS
Matrix shape: (2067, 5130)
Matrix type: csr_matrix
Vectorizer type: TfidfVectorizer
Vocabulary size: 5130
Feature-name count: 5130

ARTIFACT COMPATIBILITY
Matrix feature count: 5130
Vectorizer feature count: 5130
Feature dimensions match: True

CATALOG COMPATIBILITY
Catalog rows: 2067
Matrix rows: 2067
Catalog and matrix rows match: True


#### 18.40.1C — Final Enriched-Catalog Retrieval Validation

The production enriched TF-IDF matrix and Streamlit-compatible vectorizer have been successfully reloaded and validated.

The final deployment test now verifies that the enriched Streamlit catalog remains correctly aligned with the production recommendation system and that natural-language retrieval produces valid recommendations after the metadata integration.

This validation uses the same production architecture:

**Natural-language query → TF-IDF transformation → cosine similarity → positive-similarity filtering → duplicate suppression → ranked recommendations**

The enriched metadata is used only for presentation and contextual information. It does not alter recommendation similarity scores or ranking.

In [85]:
# ---------------------------------------------------------
# 18.40.1C Final enriched-catalog retrieval validation
# ---------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------
# Prepare deployment catalog
# ---------------------------------------------------------

app_catalog_enriched = (
    streamlit_catalog_test
    .sort_values(
        "matrix_row"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Structural alignment
# ---------------------------------------------------------

print(
    "STREAMLIT CATALOG ↔ MODEL ALIGNMENT"
)
print("=" * 80)

print(
    "Catalog rows:",
    len(app_catalog_enriched)
)

print(
    "TF-IDF matrix rows:",
    enriched_tfidf_matrix.shape[0]
)

print(
    "Row counts match:",
    len(app_catalog_enriched)
    ==
    enriched_tfidf_matrix.shape[0]
)

print(
    "Unique book IDs:",
    app_catalog_enriched[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    app_catalog_enriched[
        "book_id"
    ].duplicated().sum()
)


matrix_rows = (
    app_catalog_enriched[
        "matrix_row"
    ]
    .astype(int)
    .to_numpy()
)


expected_rows = np.arange(
    len(app_catalog_enriched)
)


print(
    "Matrix rows sequential:",
    np.array_equal(
        matrix_rows,
        expected_rows
    )
)


# ---------------------------------------------------------
# Robust zero-vector status
# ---------------------------------------------------------

zero_vector_status = (
    app_catalog_enriched[
        "enriched_zero_vector"
    ]
)


if not pd.api.types.is_bool_dtype(
    zero_vector_status
):

    zero_vector_status = (
        zero_vector_status
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )


if zero_vector_status.isna().any():

    raise ValueError(
        "Unrecognized enriched_zero_vector values detected."
    )


zero_vector_mask = (
    zero_vector_status
    .to_numpy(
        dtype=bool
    )
)


print(
    "Usable vectors:",
    (~zero_vector_mask).sum()
)

print(
    "Zero vectors:",
    zero_vector_mask.sum()
)


# ---------------------------------------------------------
# Duplicate normalization helper
# ---------------------------------------------------------

def normalize_duplicate_text(value):

    if pd.isna(value):
        return ""

    value = (
        str(value)
        .lower()
        .strip()
    )

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


# ---------------------------------------------------------
# Test natural-language query
# ---------------------------------------------------------

test_query = (
    "I want to become a better leader by improving "
    "decision making, communication, team leadership, "
    "and strategic thinking."
)


query_vector = (
    enriched_vectorizer.transform(
        [
            test_query
        ]
    )
)


print(
    "\nQUERY VECTOR VALIDATION"
)
print("=" * 80)

print(
    "Query:",
    test_query
)

print(
    "Query vector shape:",
    query_vector.shape
)

print(
    "Query non-zero features:",
    query_vector.nnz
)


# ---------------------------------------------------------
# Cosine similarity
# ---------------------------------------------------------

query_similarities = (
    cosine_similarity(
        query_vector,
        enriched_tfidf_matrix,
    )
    .ravel()
)


# Exclude books without usable vectors
query_similarities[
    zero_vector_mask
] = -1


ranked_indices = (
    np.argsort(
        query_similarities
    )[::-1]
)


# ---------------------------------------------------------
# Build duplicate-safe Top 10
# ---------------------------------------------------------

results = []

seen_duplicate_keys = set()


for idx in ranked_indices:

    score = float(
        query_similarities[
            idx
        ]
    )


    if score <= 0:
        continue


    row = (
        app_catalog_enriched
        .iloc[
            idx
        ]
    )


    title = (
        str(
            row[
                "canonical_title"
            ]
        ).strip()
        if pd.notna(
            row[
                "canonical_title"
            ]
        )
        else ""
    )


    authors = (
        str(
            row[
                "authors"
            ]
        ).strip()
        if pd.notna(
            row[
                "authors"
            ]
        )
        else ""
    )


    duplicate_key = (
        normalize_duplicate_text(
            title
        )
        + " || "
        + normalize_duplicate_text(
            authors
        )
    )


    if duplicate_key in seen_duplicate_keys:
        continue


    seen_duplicate_keys.add(
        duplicate_key
    )


    results.append(
        {
            "rank":
                len(results) + 1,

            "book_id":
                row[
                    "book_id"
                ],

            "title":
                title,

            "authors":
                authors,

            "similarity":
                round(
                    score,
                    6,
                ),

            "cover":
                pd.notna(
                    row.get(
                        "cover_url"
                    )
                ),

            "description":
                pd.notna(
                    row.get(
                        "description"
                    )
                ),

            "language_evidence":
                pd.notna(
                    row.get(
                        "observed_publication_languages"
                    )
                ),

            "publication_country":
                pd.notna(
                    row.get(
                        "earliest_observed_publication_country"
                    )
                ),

            "author_geography":
                pd.notna(
                    row.get(
                        "validated_geographic_author_names"
                    )
                ),

            "topic":
                (
                    row.get(
                        "cluster_label"
                    )
                    if pd.notna(
                        row.get(
                            "cluster_label"
                        )
                    )
                    else pd.NA
                ),
        }
    )


    if len(results) >= 10:
        break


retrieval_test = pd.DataFrame(
    results
)


print(
    "\nENRICHED-CATALOG RETRIEVAL TEST"
)
print("=" * 80)


display(
    retrieval_test
)


# ---------------------------------------------------------
# Final deployment validation
# ---------------------------------------------------------

print(
    "\nDEPLOYMENT INTEGRATION CHECK"
)
print("=" * 80)

print(
    "Returned recommendations:",
    len(retrieval_test)
)

print(
    "All similarities positive:",
    (
        retrieval_test[
            "similarity"
        ]
        > 0
    ).all()
)

print(
    "Unique recommendation book IDs:",
    retrieval_test[
        "book_id"
    ].nunique()
)

print(
    "Unique title-author recommendations:",
    (
        retrieval_test[
            [
                "title",
                "authors",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )
)

print(
    "Exactly 10 recommendations:",
    len(retrieval_test)
    == 10
)

STREAMLIT CATALOG ↔ MODEL ALIGNMENT
Catalog rows: 2067
TF-IDF matrix rows: 2067
Row counts match: True
Unique book IDs: 2067
Duplicate book IDs: 0
Matrix rows sequential: True
Usable vectors: 2040
Zero vectors: 27

QUERY VECTOR VALIDATION
Query: I want to become a better leader by improving decision making, communication, team leadership, and strategic thinking.
Query vector shape: (1, 5130)
Query non-zero features: 15

ENRICHED-CATALOG RETRIEVAL TEST


,rank,book_id,title,authors,similarity,cover,description,language_evidence,publication_country,author_geography,topic
0,1,BOOK00157,Team Leadership,['Drikus Kriek'],0.344048,False,False,True,False,False,Team Leadership
1,2,BOOK00170,Strategic Team Leadership,['Peter Saul'],0.334949,False,False,False,False,False,Team Leadership
2,3,BOOK00995,The Six Disciplines of Strategic Thinking,['Michael D. Watkins'],0.329029,True,False,False,False,False,CEO and Executive Transformation
3,4,BOOK00146,Team leadership,['John Apps'],0.297660,False,False,True,True,False,Team Leadership
4,5,BOOK01265,If You Want Something Done,['Nikki R. Haley'],0.295807,True,False,False,False,False,Future Thinking and Personal Development
5,6,BOOK00797,Creative decision making,['H. B. Gelatt'],0.290526,True,False,True,True,False,Decision Making
6,7,BOOK00785,DECISION MAKING,['Irving L. Janis'],0.246255,True,False,True,False,True,Decision Making
7,8,BOOK00187,Team Leadership Questionnaire,"['D. Kayes', 'Anna Kayes']",0.245542,False,False,True,False,False,Team Leadership
8,9,BOOK00191,Team Leadership Assessments,['LeaderTreks'],0.236848,False,False,False,False,False,Team Leadership
9,10,BOOK00782,A primer on decision making,['James G. March'],0.236787,True,False,True,True,True,Decision Making



DEPLOYMENT INTEGRATION CHECK
Returned recommendations: 10
All similarities positive: True
Unique recommendation book IDs: 10
Unique title-author recommendations: 10
Exactly 10 recommendations: True


### 18.41 — Enriched Streamlit Interface Integration

The enriched deployment catalog has been validated against the production TF-IDF recommendation system with no change to recommendation rankings or similarity scores.

LeadWise can now transition from model and metadata validation to application-interface integration.

The Streamlit application will use `leadwise_streamlit_catalog.csv` as its primary book metadata source while continuing to use the existing production TF-IDF matrix and Streamlit-compatible vectorizer for recommendation retrieval.

The next interface iteration will progressively introduce:

- book-cover rendering;
- richer recommendation cards;
- bibliographic metadata;
- reader metrics;
- topic intelligence;
- observed publication-language evidence;
- earliest observed publication geography;
- validated author geographic evidence;
- conditional metadata rendering;
- dedicated Book Details;
- book comparison; and
- clearer explanations of why a recommendation was retrieved.

Missing metadata will not produce empty interface elements. Components will be rendered only when supported by available evidence.

#### 18.41.1 — Cover Coverage Optimization

The validated Streamlit deployment catalog currently contains cover images for 1,705 of 2,067 books.

Earlier metadata auditing identified additional cover URLs in the integrated master catalog. Because book-cover imagery is an important component of the LeadWise user interface, the deployment catalog is enriched by using the master-catalog cover URL only when the existing deployment cover is unavailable.

This operation follows a conservative coalescing rule:

**Existing deployment cover → master-catalog cover → missing**

Existing deployment cover URLs are never replaced when already available.

The operation does not modify:

- `book_id`;
- `matrix_row`;
- TF-IDF features;
- recommendation vectors;
- similarity scores;
- topic assignments; or
- recommendation rankings.

The objective is solely to maximize verified cover-image availability using metadata already collected by the project.

In [86]:
# ---------------------------------------------------------
# 18.41.1 Cover Coverage Optimization
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

streamlit_catalog_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)

books_master_path = (
    data_root
    / "final"
    / "books_master.csv"
)


# ---------------------------------------------------------
# Load catalogs
# ---------------------------------------------------------

streamlit_catalog = pd.read_csv(
    streamlit_catalog_path,
    low_memory=False,
)

books_master_cover = pd.read_csv(
    books_master_path,
    usecols=[
        "book_id",
        "cover_url",
    ],
    low_memory=False,
)


print(
    "COVER SOURCE VALIDATION"
)
print("=" * 80)

print(
    "Streamlit catalog rows:",
    len(streamlit_catalog)
)

print(
    "Master catalog rows:",
    len(books_master_cover)
)

print(
    "Streamlit unique book IDs:",
    streamlit_catalog[
        "book_id"
    ].nunique()
)

print(
    "Master unique book IDs:",
    books_master_cover[
        "book_id"
    ].nunique()
)

print(
    "Master duplicate book IDs:",
    books_master_cover[
        "book_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# Normalize missing cover values
# ---------------------------------------------------------

def clean_cover_url(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if (
        not value
        or value.lower()
        in {
            "nan",
            "none",
            "null",
            "<na>",
        }
    ):
        return pd.NA

    return value


streamlit_catalog[
    "cover_url"
] = (
    streamlit_catalog[
        "cover_url"
    ]
    .apply(
        clean_cover_url
    )
)


books_master_cover[
    "cover_url"
] = (
    books_master_cover[
        "cover_url"
    ]
    .apply(
        clean_cover_url
    )
)


# ---------------------------------------------------------
# Baseline coverage
# ---------------------------------------------------------

deployment_before = int(
    streamlit_catalog[
        "cover_url"
    ]
    .notna()
    .sum()
)

master_available = int(
    books_master_cover[
        "cover_url"
    ]
    .notna()
    .sum()
)


print(
    "\nBASELINE COVER COVERAGE"
)
print("=" * 80)

print(
    "Deployment covers before:",
    deployment_before
)

print(
    "Master covers available:",
    master_available
)

print(
    "Deployment coverage before:",
    f"{deployment_before / len(streamlit_catalog):.2%}"
)

print(
    "Master coverage:",
    f"{master_available / len(books_master_cover):.2%}"
)


# ---------------------------------------------------------
# Rename master cover before merge
# ---------------------------------------------------------

books_master_cover = (
    books_master_cover
    .rename(
        columns={
            "cover_url":
                "cover_url_master"
        }
    )
)


# ---------------------------------------------------------
# Merge one-to-one by book_id
# ---------------------------------------------------------

cover_enriched_catalog = (
    streamlit_catalog
    .merge(
        books_master_cover,
        on="book_id",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# Identify newly recoverable covers
# ---------------------------------------------------------

cover_enriched_catalog[
    "cover_recovered_from_master"
] = (
    cover_enriched_catalog[
        "cover_url"
    ].isna()
    &
    cover_enriched_catalog[
        "cover_url_master"
    ].notna()
)


recovered_count = int(
    cover_enriched_catalog[
        "cover_recovered_from_master"
    ]
    .sum()
)


# ---------------------------------------------------------
# Conservative coalescing
# Existing deployment URL has priority.
# ---------------------------------------------------------

cover_enriched_catalog[
    "cover_url"
] = (
    cover_enriched_catalog[
        "cover_url"
    ]
    .combine_first(
        cover_enriched_catalog[
            "cover_url_master"
        ]
    )
)


# ---------------------------------------------------------
# Final coverage
# ---------------------------------------------------------

deployment_after = int(
    cover_enriched_catalog[
        "cover_url"
    ]
    .notna()
    .sum()
)


remaining_missing = (
    len(cover_enriched_catalog)
    - deployment_after
)


print(
    "\nCOVER ENRICHMENT RESULT"
)
print("=" * 80)

print(
    "Covers before:",
    deployment_before
)

print(
    "Recovered from master:",
    recovered_count
)

print(
    "Covers after:",
    deployment_after
)

print(
    "Remaining without cover:",
    remaining_missing
)

print(
    "Final cover coverage:",
    f"{deployment_after / len(cover_enriched_catalog):.2%}"
)


# ---------------------------------------------------------
# Show recovered examples
# ---------------------------------------------------------

recovered_examples = (
    cover_enriched_catalog.loc[
        cover_enriched_catalog[
            "cover_recovered_from_master"
        ],
        [
            "book_id",
            "canonical_title",
            "authors",
            "cover_url",
        ],
    ]
    .head(20)
)


print(
    "\nRECOVERED COVER EXAMPLES"
)
print("=" * 80)

display(
    recovered_examples
)


# ---------------------------------------------------------
# Structural integrity
# ---------------------------------------------------------

print(
    "\nPOST-ENRICHMENT INTEGRITY"
)
print("=" * 80)

print(
    "Rows:",
    len(cover_enriched_catalog)
)

print(
    "Unique book IDs:",
    cover_enriched_catalog[
        "book_id"
    ].nunique()
)

print(
    "Duplicate book IDs:",
    cover_enriched_catalog[
        "book_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Unique matrix rows:",
    cover_enriched_catalog[
        "matrix_row"
    ].nunique()
)

print(
    "Duplicate matrix rows:",
    cover_enriched_catalog[
        "matrix_row"
    ]
    .duplicated()
    .sum()
)

print(
    "Matrix rows sequential:",
    (
        cover_enriched_catalog[
            "matrix_row"
        ]
        .astype(int)
        .to_numpy()
        ==
        range(
            len(
                cover_enriched_catalog
            )
        )
    ).all()
)


# ---------------------------------------------------------
# Critical assertions
# ---------------------------------------------------------

assert (
    len(cover_enriched_catalog)
    == 2067
)

assert (
    cover_enriched_catalog[
        "book_id"
    ].is_unique
)

assert (
    cover_enriched_catalog[
        "matrix_row"
    ].is_unique
)

assert (
    deployment_after
    >= deployment_before
)


# ---------------------------------------------------------
# Remove temporary master-cover field
# Keep provenance flag for application/audit purposes.
# ---------------------------------------------------------

cover_enriched_catalog = (
    cover_enriched_catalog
    .drop(
        columns=[
            "cover_url_master",
        ]
    )
)


# ---------------------------------------------------------
# Save updated deployment catalog
# ---------------------------------------------------------

cover_enriched_catalog.to_csv(
    streamlit_catalog_path,
    index=False,
)


print(
    "\nUPDATED DEPLOYMENT CATALOG"
)
print("=" * 80)

print(
    "Saved:",
    streamlit_catalog_path
)

print(
    "Final rows:",
    len(cover_enriched_catalog)
)

print(
    "Final columns:",
    cover_enriched_catalog.shape[1]
)

COVER SOURCE VALIDATION
Streamlit catalog rows: 2067
Master catalog rows: 2067
Streamlit unique book IDs: 2067
Master unique book IDs: 2067
Master duplicate book IDs: 0

BASELINE COVER COVERAGE
Deployment covers before: 1705
Master covers available: 1888
Deployment coverage before: 82.49%
Master coverage: 91.34%

COVER ENRICHMENT RESULT
Covers before: 1705
Recovered from master: 183
Covers after: 1888
Remaining without cover: 179
Final cover coverage: 91.34%

RECOVERED COVER EXAMPLES


,book_id,canonical_title,authors,cover_url
955,BOOK00956,Jump,NaN,https://www.leadershipnow.com/leadershop/image...
961,BOOK00962,Longpath,NaN,https://www.leadershipnow.com/leadershop/image...
962,BOOK00963,Next!,NaN,https://www.leadershipnow.com/leadershop/image...
969,BOOK00970,Reunion,NaN,https://www.leadershipnow.com/leadershop/image...
975,BOOK00976,MoneyZen,NaN,https://www.leadershipnow.com/leadershop/image...
988,BOOK00989,Uptime,NaN,https://www.leadershipnow.com/leadershop/image...
1001,BOOK01002,Unfiltered,NaN,https://www.leadershipnow.com/leadershop/image...
1005,BOOK01006,Glad We Met,NaN,https://www.leadershipnow.com/leadershop/image...
1006,BOOK01007,Innovators,NaN,https://www.leadershipnow.com/leadershop/image...
1010,BOOK01011,More Than Pretty Boxes,NaN,https://www.leadershipnow.com/leadershop/image...



POST-ENRICHMENT INTEGRITY
Rows: 2067
Unique book IDs: 2067
Duplicate book IDs: 0
Unique matrix rows: 2067
Duplicate matrix rows: 0
Matrix rows sequential: True

UPDATED DEPLOYMENT CATALOG
Saved: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_streamlit_catalog.csv
Final rows: 2067
Final columns: 86


### 18.41.2 — Enriched LeadWise Application Integration

Following validation of the final deployment catalog, LeadWise transitions from metadata preparation to enriched application development.

The deployment catalog now contains 2,067 uniquely aligned books and preserves exact correspondence with the production TF-IDF recommendation matrix. Cover-image coverage has been increased from 1,705 to 1,888 books (91.34%) using previously collected master-catalog metadata.

The Streamlit application will now use `leadwise_streamlit_catalog.csv` as its primary application-facing catalog.

The enriched interface is designed around evidence-aware conditional rendering. Metadata is displayed only when supported by the collected sources. Missing descriptions, ratings, publication geography, language evidence, author geographic evidence, commercial information, or other fields are not inferred or fabricated.

Application terminology also preserves the methodological distinctions established during metadata enrichment:

- **Observed publication languages** do not imply verified original language or translation count.
- **Earliest observed publication country/countries** do not imply verified country of origin.
- **Author citizenship and birthplace evidence** are displayed as supporting geographic metadata rather than a single inferred “author country.”
- **Content similarity** represents textual similarity to the query and is not a quality score, expert rating, probability, or measure of leadership effectiveness.
- **Commercial price data** is reported as unavailable when not present in collected sources.

The upgraded application layer introduces richer recommendation cards, book-cover rendering, catalog exploration, detailed book intelligence, book comparison, and topic exploration while preserving the validated production recommendation architecture.

### 18.42 — Final Catalog Semantic Quality Audit

Visual validation of the enriched LeadWise interface identified at least one catalog record whose displayed cover does not appear to represent a conventional book cover.

This indicates that structural data validation alone is insufficient for final application deployment. A semantic quality-control stage is therefore introduced to distinguish between:

- valid books with reliable metadata;
- valid books with unreliable or mismatched cover imagery;
- ambiguous bibliographic records requiring review; and
- records that should potentially be excluded from the application catalog.

Records are not automatically removed during this stage. Each suspicious item is first traced to its source metadata, identifiers, publication evidence and model position so that corrections can be made without compromising the validated alignment between the application catalog and production TF-IDF matrix.

#### 18.42.1 — Trace a Suspicious Catalog Record

Visual inspection of the LeadWise Streamlit application identified a suspicious result titled `Kepemimpinan =` whose displayed image does not appear to be a conventional book cover.

Before excluding the record, the underlying bibliographic evidence must be examined.

The audit traces the record across the final deployment catalog and available source datasets to determine whether:

1. the bibliographic record represents a legitimate published book but contains an incorrect or unsuitable cover;
2. the record itself contains weak or inconsistent bibliographic evidence; or
3. the item should eventually be excluded from the user-facing LeadWise catalog.

No record is removed during this diagnostic stage because the deployment catalog remains positionally aligned with the production TF-IDF matrix.

In [87]:
# ---------------------------------------------------------
# 18.42.1 Trace suspicious "Kepemimpinan" record
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

streamlit_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)

master_path = (
    data_root
    / "final"
    / "books_master.csv"
)

openlibrary_path = (
    data_root
    / "final"
    / "openlibrary_integrated.csv"
)

leadershipnow_path = (
    data_root
    / "final"
    / "leadershipnow_editions.csv"
)


# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

streamlit_catalog = pd.read_csv(
    streamlit_path,
    low_memory=False,
)

books_master = pd.read_csv(
    master_path,
    low_memory=False,
)

openlibrary = pd.read_csv(
    openlibrary_path,
    low_memory=False,
)

leadershipnow = pd.read_csv(
    leadershipnow_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Search helper
# ---------------------------------------------------------

def search_title(df, term):

    title_candidates = [
        "canonical_title",
        "title",
        "book_title",
    ]

    available_title_columns = [
        col
        for col in title_candidates
        if col in df.columns
    ]

    if not available_title_columns:
        return pd.DataFrame()

    mask = pd.Series(
        False,
        index=df.index,
    )

    for col in available_title_columns:

        mask = (
            mask
            |
            df[col]
            .fillna("")
            .astype(str)
            .str.contains(
                term,
                case=False,
                regex=False,
            )
        )

    return df.loc[mask].copy()


# ---------------------------------------------------------
# Search deployment catalog
# ---------------------------------------------------------

deployment_match = search_title(
    streamlit_catalog,
    "Kepemimpinan",
)


print(
    "DEPLOYMENT CATALOG MATCH"
)
print("=" * 80)

print(
    "Matches:",
    len(deployment_match)
)


if not deployment_match.empty:

    important_columns = [
        "matrix_row",
        "book_id",
        "canonical_title",
        "authors",
        "description",
        "subjects",
        "first_publish_year",
        "publication_year_observed",
        "publisher",
        "publishers",
        "isbn_10",
        "isbn_13",
        "page_count",
        "cover_url",
        "source_group",
        "source_url",
        "source_openlibrary",
        "source_leadershipnow",
        "topic_cluster",
        "cluster_label",
        "enriched_zero_vector",
        "observed_publication_languages",
        "earliest_observed_publication_country",
        "commercial_metadata_status",
    ]

    available_columns = [
        col
        for col in important_columns
        if col in deployment_match.columns
    ]

    display(
        deployment_match[
            available_columns
        ].T
    )


# ---------------------------------------------------------
# Extract matched book IDs
# ---------------------------------------------------------

matched_book_ids = (
    deployment_match[
        "book_id"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


print(
    "\nMATCHED BOOK IDs"
)
print("=" * 80)

print(
    matched_book_ids
)


# ---------------------------------------------------------
# Trace in books_master
# ---------------------------------------------------------

master_match = (
    books_master[
        books_master[
            "book_id"
        ]
        .astype(str)
        .isin(
            matched_book_ids
        )
    ]
    .copy()
)


print(
    "\nBOOKS_MASTER TRACE"
)
print("=" * 80)

print(
    "Matches:",
    len(master_match)
)

if not master_match.empty:

    display(
        master_match.T
    )


# ---------------------------------------------------------
# Trace Open Library source
# ---------------------------------------------------------

if "book_id" in openlibrary.columns:

    openlibrary_match = (
        openlibrary[
            openlibrary[
                "book_id"
            ]
            .astype(str)
            .isin(
                matched_book_ids
            )
        ]
        .copy()
    )

else:

    openlibrary_match = search_title(
        openlibrary,
        "Kepemimpinan",
    )


print(
    "\nOPEN LIBRARY TRACE"
)
print("=" * 80)

print(
    "Matches:",
    len(openlibrary_match)
)

if not openlibrary_match.empty:

    display(
        openlibrary_match.T
    )


# ---------------------------------------------------------
# Trace LeadershipNow source
# ---------------------------------------------------------

if "book_id" in leadershipnow.columns:

    leadershipnow_match = (
        leadershipnow[
            leadershipnow[
                "book_id"
            ]
            .astype(str)
            .isin(
                matched_book_ids
            )
        ]
        .copy()
    )

else:

    leadershipnow_match = search_title(
        leadershipnow,
        "Kepemimpinan",
    )


print(
    "\nLEADERSHIPNOW TRACE"
)
print("=" * 80)

print(
    "Matches:",
    len(leadershipnow_match)
)

if not leadershipnow_match.empty:

    display(
        leadershipnow_match.T
    )


# ---------------------------------------------------------
# Compact diagnostic summary
# ---------------------------------------------------------

print(
    "\nDIAGNOSTIC SUMMARY"
)
print("=" * 80)


for _, row in deployment_match.iterrows():

    print(
        "Book ID:",
        row.get(
            "book_id"
        )
    )

    print(
        "Matrix row:",
        row.get(
            "matrix_row"
        )
    )

    print(
        "Title:",
        row.get(
            "canonical_title"
        )
    )

    print(
        "Authors:",
        row.get(
            "authors"
        )
    )

    print(
        "Source group:",
        row.get(
            "source_group"
        )
    )

    print(
        "Open Library source:",
        row.get(
            "source_openlibrary"
        )
    )

    print(
        "LeadershipNow source:",
        row.get(
            "source_leadershipnow"
        )
    )

    print(
        "ISBN-10:",
        row.get(
            "isbn_10"
        )
    )

    print(
        "ISBN-13:",
        row.get(
            "isbn_13"
        )
    )

    print(
        "Publisher:",
        row.get(
            "publisher"
        )
    )

    print(
        "First publication year:",
        row.get(
            "first_publish_year"
        )
    )

    print(
        "Observed publication year:",
        row.get(
            "publication_year_observed"
        )
    )

    print(
        "Page count:",
        row.get(
            "page_count"
        )
    )

    print(
        "Subjects:",
        row.get(
            "subjects"
        )
    )

    print(
        "Topic:",
        row.get(
            "cluster_label"
        )
    )

    print(
        "Cover URL:",
        row.get(
            "cover_url"
        )
    )

    print(
        "Source URL:",
        row.get(
            "source_url"
        )
    )

    print("-" * 80)

DEPLOYMENT CATALOG MATCH
Matches: 1


,2
matrix_row,2
book_id,BOOK00003
canonical_title,Kepemimpinan =
authors,['Karjadi M.']
description,NaN
subjects,['Leadership']
first_publish_year,1977.0
publication_year_observed,NaN
publisher,NaN
publishers,['Politeia']



MATCHED BOOK IDs
['BOOK00003']

BOOKS_MASTER TRACE
Matches: 1


,2
book_id,BOOK00003
canonical_title,Kepemimpinan =
authors,['Karjadi M.']
description,NaN
subjects,['Leadership']
first_publish_year,1977.0
average_rating,1.0
ratings_count,1.0
edition_count,1.0
want_to_read_count,41.0



OPEN LIBRARY TRACE
Matches: 1


,2
openlibrary_key,/works/OL302757W
title,Kepemimpinan =
authors,['Karjadi M.']
author_keys,['OL1268A']
first_publish_year,1977.0
publish_dates,['1977']
publishers,['Politeia']
isbn_10,[]
isbn_13,[]
all_isbns,[]



LEADERSHIPNOW TRACE
Matches: 0

DIAGNOSTIC SUMMARY
Book ID: BOOK00003
Matrix row: 2
Title: Kepemimpinan =
Authors: ['Karjadi M.']
Source group: Open Library only
Open Library source: True
LeadershipNow source: False
ISBN-10: []
ISBN-13: []
Publisher: nan
First publication year: 1977.0
Observed publication year: nan
Page count: nan
Subjects: ['Leadership']
Topic: Transformational Leadership
Cover URL: https://covers.openlibrary.org/b/id/14420782-L.jpg
Source URL: https://openlibrary.org/works/OL302757W
--------------------------------------------------------------------------------


#### 18.42.2 — Correction of BOOK00003

The diagnostic trace indicates that `BOOK00003` represents a bibliographic Open Library work rather than a non-book application record.

Supporting evidence includes an identified author, publisher, publication year, language, leadership subject classification, Open Library work identifier and Library of Congress classification.

The visual anomaly is therefore treated as a metadata-quality problem associated with the displayed cover rather than sufficient evidence for excluding the underlying book.

Two conservative deployment corrections are applied:

1. the trailing cataloging punctuation in `Kepemimpinan =` is removed from the application-facing title; and
2. the visually unreliable cover URL is suppressed.

The original values are retained in audit fields. The book remains recommendation-eligible and its `matrix_row` is unchanged, preserving exact alignment with the production TF-IDF matrix.

In [88]:
# ---------------------------------------------------------
# 18.42.2 Correct BOOK00003 deployment metadata
# ---------------------------------------------------------

import pandas as pd


catalog_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)


catalog = pd.read_csv(
    catalog_path,
    low_memory=False,
)


target_book_id = "BOOK00003"


target_mask = (
    catalog["book_id"]
    .astype(str)
    .eq(target_book_id)
)


assert target_mask.sum() == 1


# ---------------------------------------------------------
# Preserve original values
# ---------------------------------------------------------

if "canonical_title_original" not in catalog.columns:

    catalog[
        "canonical_title_original"
    ] = pd.NA


if "cover_url_original" not in catalog.columns:

    catalog[
        "cover_url_original"
    ] = pd.NA


if "metadata_quality_note" not in catalog.columns:

    catalog[
        "metadata_quality_note"
    ] = pd.NA


catalog.loc[
    target_mask,
    "canonical_title_original"
] = (
    catalog.loc[
        target_mask,
        "canonical_title"
    ]
)


catalog.loc[
    target_mask,
    "cover_url_original"
] = (
    catalog.loc[
        target_mask,
        "cover_url"
    ]
)


# ---------------------------------------------------------
# Application-facing corrections
# ---------------------------------------------------------

catalog.loc[
    target_mask,
    "canonical_title"
] = "Kepemimpinan"


catalog.loc[
    target_mask,
    "cover_url"
] = pd.NA


catalog.loc[
    target_mask,
    "metadata_quality_note"
] = (
    "Application-facing title cleaned; "
    "Open Library cover suppressed after "
    "visual quality review."
)


# ---------------------------------------------------------
# Validate corrected record
# ---------------------------------------------------------

audit_columns = [
    "matrix_row",
    "book_id",
    "canonical_title",
    "canonical_title_original",
    "authors",
    "publishers",
    "first_publish_year",
    "observed_publication_languages",
    "earliest_observed_publication_country",
    "cover_url",
    "cover_url_original",
    "metadata_quality_note",
    "enriched_zero_vector",
]


print(
    "CORRECTED BOOK RECORD"
)
print("=" * 80)

display(
    catalog.loc[
        target_mask,
        [
            col
            for col in audit_columns
            if col in catalog.columns
        ],
    ].T
)


# ---------------------------------------------------------
# Revalidate catalog integrity
# ---------------------------------------------------------

print(
    "\nCATALOG INTEGRITY"
)
print("=" * 80)

print(
    "Rows:",
    len(catalog)
)

print(
    "Unique book IDs:",
    catalog["book_id"].nunique()
)

print(
    "Duplicate book IDs:",
    catalog["book_id"].duplicated().sum()
)

print(
    "Unique matrix rows:",
    catalog["matrix_row"].nunique()
)

print(
    "Duplicate matrix rows:",
    catalog["matrix_row"].duplicated().sum()
)

print(
    "Target matrix row:",
    int(
        catalog.loc[
            target_mask,
            "matrix_row"
        ].iloc[0]
    )
)


assert len(catalog) == 2067
assert catalog["book_id"].is_unique
assert catalog["matrix_row"].is_unique

assert (
    int(
        catalog.loc[
            target_mask,
            "matrix_row"
        ].iloc[0]
    )
    == 2
)


# ---------------------------------------------------------
# Save corrected deployment catalog
# ---------------------------------------------------------

catalog.to_csv(
    catalog_path,
    index=False,
)


print(
    "\nSAVED"
)
print("=" * 80)

print(catalog_path)

CORRECTED BOOK RECORD


,2
matrix_row,2
book_id,BOOK00003
canonical_title,Kepemimpinan
canonical_title_original,Kepemimpinan =
authors,['Karjadi M.']
publishers,['Politeia']
first_publish_year,1977.0
observed_publication_languages,Indonesian
earliest_observed_publication_country,Indonesia
cover_url,NaN



CATALOG INTEGRITY
Rows: 2067
Unique book IDs: 2067
Duplicate book IDs: 0
Unique matrix rows: 2067
Duplicate matrix rows: 0
Target matrix row: 2

SAVED
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_streamlit_catalog.csv


#### 18.42.3 — Catalog-Wide Semantic Quality Screening

Following the manual identification and correction of `BOOK00003`, a catalog-wide semantic quality screening is performed to identify records that may require additional review before final deployment.

The purpose of this screening is not to automatically determine whether a record is a valid book. Instead, transparent diagnostic indicators are used to prioritize records for manual inspection.

The screening considers several types of evidence:

- weak bibliographic metadata;
- missing author information;
- missing publication evidence;
- absence of ISBN identifiers;
- missing publisher information;
- missing page-count information;
- unusually short or structurally suspicious titles;
- title patterns that may indicate non-standard publication objects;
- absence of descriptive metadata;
- weak subject evidence; and
- anomalous edition page-count metadata identified during earlier enrichment.

No book is removed from the catalog during this stage.

The original 2,067-row catalog and its `matrix_row` alignment with the production TF-IDF matrix are preserved. Records receiving multiple diagnostic flags are prioritized for manual review.

In [89]:
# ---------------------------------------------------------
# 18.42.3 Catalog-Wide Semantic Quality Screening
# ---------------------------------------------------------

import re
import numpy as np
import pandas as pd


catalog_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)


quality_catalog = pd.read_csv(
    catalog_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def has_meaningful_value(value):

    if pd.isna(value):
        return False

    text = str(value).strip()

    return (
        text != ""
        and text.lower()
        not in {
            "nan",
            "none",
            "null",
            "<na>",
            "[]",
            "{}",
        }
    )


def clean_title_for_audit(value):

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
    )


# ---------------------------------------------------------
# Basic evidence flags
# ---------------------------------------------------------

quality_catalog[
    "qa_missing_author"
] = ~quality_catalog[
    "authors"
].apply(
    has_meaningful_value
)


quality_catalog[
    "qa_missing_description"
] = ~quality_catalog[
    "description"
].apply(
    has_meaningful_value
)


quality_catalog[
    "qa_missing_subjects"
] = ~quality_catalog[
    "subjects"
].apply(
    has_meaningful_value
)


# ---------------------------------------------------------
# ISBN evidence
# ---------------------------------------------------------

isbn10_available = (
    quality_catalog[
        "isbn_10"
    ].apply(
        has_meaningful_value
    )
    if "isbn_10" in quality_catalog.columns
    else pd.Series(
        False,
        index=quality_catalog.index,
    )
)


isbn13_available = (
    quality_catalog[
        "isbn_13"
    ].apply(
        has_meaningful_value
    )
    if "isbn_13" in quality_catalog.columns
    else pd.Series(
        False,
        index=quality_catalog.index,
    )
)


quality_catalog[
    "qa_missing_isbn"
] = ~(
    isbn10_available
    |
    isbn13_available
)


# ---------------------------------------------------------
# Publisher evidence
# ---------------------------------------------------------

publisher_available = (
    quality_catalog[
        "publisher"
    ].apply(
        has_meaningful_value
    )
    if "publisher" in quality_catalog.columns
    else pd.Series(
        False,
        index=quality_catalog.index,
    )
)


publishers_available = (
    quality_catalog[
        "publishers"
    ].apply(
        has_meaningful_value
    )
    if "publishers" in quality_catalog.columns
    else pd.Series(
        False,
        index=quality_catalog.index,
    )
)


quality_catalog[
    "qa_missing_publisher"
] = ~(
    publisher_available
    |
    publishers_available
)


# ---------------------------------------------------------
# Publication-year evidence
# ---------------------------------------------------------

first_year_available = (
    quality_catalog[
        "first_publish_year"
    ].notna()
)


observed_year_available = (
    quality_catalog[
        "publication_year_observed"
    ].notna()
)


quality_catalog[
    "qa_missing_publication_year"
] = ~(
    first_year_available
    |
    observed_year_available
)


# ---------------------------------------------------------
# Page evidence
# ---------------------------------------------------------

page_fields = [
    field
    for field in [
        "page_count",
        "minimum_observed_pages",
        "maximum_observed_pages",
    ]
    if field in quality_catalog.columns
]


if page_fields:

    quality_catalog[
        "qa_missing_page_evidence"
    ] = (
        quality_catalog[
            page_fields
        ]
        .isna()
        .all(
            axis=1
        )
    )

else:

    quality_catalog[
        "qa_missing_page_evidence"
    ] = True


# ---------------------------------------------------------
# Title diagnostics
# ---------------------------------------------------------

quality_catalog[
    "_qa_title"
] = quality_catalog[
    "canonical_title"
].apply(
    clean_title_for_audit
)


quality_catalog[
    "qa_very_short_title"
] = (
    quality_catalog[
        "_qa_title"
    ]
    .str.len()
    <= 2
)


quality_catalog[
    "qa_title_trailing_symbol"
] = (
    quality_catalog[
        "_qa_title"
    ]
    .str.contains(
        r"[=|_/\\<>]+$",
        regex=True,
        na=False,
    )
)


quality_catalog[
    "qa_title_url_like"
] = (
    quality_catalog[
        "_qa_title"
    ]
    .str.contains(
        r"https?://|www\.",
        case=False,
        regex=True,
        na=False,
    )
)


quality_catalog[
    "qa_title_mostly_numeric"
] = (
    quality_catalog[
        "_qa_title"
    ]
    .str.replace(
        r"\D",
        "",
        regex=True,
    )
    .str.len()
    >=
    (
        quality_catalog[
            "_qa_title"
        ]
        .str.replace(
            r"\s",
            "",
            regex=True,
        )
        .str.len()
        * 0.70
    )
)


# ---------------------------------------------------------
# Potential non-standard-object title terms
#
# IMPORTANT:
# These terms DO NOT establish that an item is not a book.
# They only prioritize records for inspection.
# ---------------------------------------------------------

review_terms = [
    "questionnaire",
    "assessment",
    "workbook",
    "manual",
    "handbook",
    "guidebook",
    "catalog",
    "catalogue",
    "directory",
    "proceedings",
    "report",
    "journal",
    "magazine",
    "newsletter",
    "poster",
    "cards",
    "card deck",
    "game",
    "software",
    "dvd",
    "video",
    "audio",
]


review_pattern = (
    r"\b(?:"
    + "|".join(
        re.escape(term)
        for term in review_terms
    )
    + r")\b"
)


quality_catalog[
    "qa_nonstandard_title_term"
] = (
    quality_catalog[
        "_qa_title"
    ]
    .str.contains(
        review_pattern,
        case=False,
        regex=True,
        na=False,
    )
)


# ---------------------------------------------------------
# Existing page-count anomaly
# ---------------------------------------------------------

if (
    "edition_page_count_anomaly"
    in quality_catalog.columns
):

    quality_catalog[
        "qa_page_anomaly"
    ] = (
        quality_catalog[
            "edition_page_count_anomaly"
        ]
        .fillna(False)
        .astype(bool)
    )

else:

    quality_catalog[
        "qa_page_anomaly"
    ] = False


# ---------------------------------------------------------
# Evidence-strength diagnostic
#
# Major indicators receive more weight than ordinary
# missing descriptive metadata.
# ---------------------------------------------------------

major_flags = [
    "qa_missing_author",
    "qa_missing_subjects",
    "qa_missing_publication_year",
    "qa_very_short_title",
    "qa_title_url_like",
    "qa_title_mostly_numeric",
    "qa_page_anomaly",
]


minor_flags = [
    "qa_missing_description",
    "qa_missing_isbn",
    "qa_missing_publisher",
    "qa_missing_page_evidence",
    "qa_title_trailing_symbol",
    "qa_nonstandard_title_term",
]


quality_catalog[
    "qa_major_flag_count"
] = (
    quality_catalog[
        major_flags
    ]
    .sum(
        axis=1
    )
)


quality_catalog[
    "qa_minor_flag_count"
] = (
    quality_catalog[
        minor_flags
    ]
    .sum(
        axis=1
    )
)


quality_catalog[
    "qa_review_score"
] = (
    quality_catalog[
        "qa_major_flag_count"
    ] * 2
    +
    quality_catalog[
        "qa_minor_flag_count"
    ]
)


# ---------------------------------------------------------
# Review priority
#
# This is a REVIEW PRIORITY, not a book-validity rating.
# ---------------------------------------------------------

quality_catalog[
    "qa_review_priority"
] = np.select(
    [
        quality_catalog[
            "qa_major_flag_count"
        ] >= 2,

        (
            quality_catalog[
                "qa_major_flag_count"
            ] >= 1
        )
        &
        (
            quality_catalog[
                "qa_review_score"
            ] >= 3
        ),

        quality_catalog[
            "qa_review_score"
        ] >= 4,

        quality_catalog[
            "qa_review_score"
        ] >= 2,
    ],
    [
        "Priority review",
        "Priority review",
        "Review",
        "Watch",
    ],
    default="Routine",
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "CATALOG QUALITY SCREENING"
)
print("=" * 80)

print(
    "Catalog rows:",
    len(quality_catalog)
)

print(
    "Unique book IDs:",
    quality_catalog[
        "book_id"
    ].nunique()
)


print(
    "\nREVIEW PRIORITY DISTRIBUTION"
)
print("=" * 80)

display(
    quality_catalog[
        "qa_review_priority"
    ]
    .value_counts()
    .rename_axis(
        "review_priority"
    )
    .reset_index(
        name="books"
    )
)


# ---------------------------------------------------------
# Individual flag frequencies
# ---------------------------------------------------------

all_flags = (
    major_flags
    +
    minor_flags
)


flag_summary = pd.DataFrame(
    {
        "flag": all_flags,
        "books_flagged": [
            int(
                quality_catalog[
                    flag
                ].sum()
            )
            for flag in all_flags
        ],
    }
)


flag_summary[
    "catalog_pct"
] = (
    flag_summary[
        "books_flagged"
    ]
    /
    len(
        quality_catalog
    )
    * 100
).round(2)


print(
    "\nFLAG FREQUENCIES"
)
print("=" * 80)

display(
    flag_summary
    .sort_values(
        "books_flagged",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Highest-priority records
# ---------------------------------------------------------

review_columns = [
    "matrix_row",
    "book_id",
    "canonical_title",
    "authors",
    "source_group",
    "first_publish_year",
    "publication_year_observed",
    "publishers",
    "isbn_13",
    "subjects",
    "cluster_label",
    "cover_url",
    "qa_major_flag_count",
    "qa_minor_flag_count",
    "qa_review_score",
    "qa_review_priority",
]


review_columns = [
    col
    for col in review_columns
    if col in quality_catalog.columns
]


priority_review = (
    quality_catalog[
        quality_catalog[
            "qa_review_priority"
        ]
        .isin(
            [
                "Priority review",
                "Review",
            ]
        )
    ]
    .sort_values(
        [
            "qa_review_score",
            "qa_major_flag_count",
        ],
        ascending=False,
    )
    [
        review_columns
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\nPRIORITY / REVIEW CANDIDATES"
)
print("=" * 80)

print(
    "Books:",
    len(priority_review)
)


display(
    priority_review.head(
        50
    )
)


# ---------------------------------------------------------
# Save audit outputs separately.
# Do NOT overwrite deployment catalog yet.
# ---------------------------------------------------------

audit_path = (
    data_root
    / "processed"
    / "leadwise_catalog_quality_audit.csv"
)


review_path = (
    data_root
    / "processed"
    / "leadwise_catalog_manual_review_queue.csv"
)


quality_catalog.drop(
    columns=[
        "_qa_title",
    ]
).to_csv(
    audit_path,
    index=False,
)


priority_review.to_csv(
    review_path,
    index=False,
)


print(
    "\nSAVED"
)
print("=" * 80)

print(
    "Full quality audit:",
    audit_path
)

print(
    "Manual review queue:",
    review_path
)


# ---------------------------------------------------------
# Integrity check
# ---------------------------------------------------------

assert len(
    quality_catalog
) == 2067

assert quality_catalog[
    "book_id"
].is_unique

assert quality_catalog[
    "matrix_row"
].is_unique


print(
    "\nProduction row alignment preserved: True"
)

CATALOG QUALITY SCREENING
Catalog rows: 2067
Unique book IDs: 2067

REVIEW PRIORITY DISTRIBUTION


,review_priority,books
0,Priority review,1251
1,Routine,733
2,Watch,83



FLAG FREQUENCIES


,flag,books_flagged,catalog_pct
0,qa_missing_description,1875,90.71
1,qa_missing_subjects,1241,60.04
2,qa_missing_isbn,1152,55.73
3,qa_missing_author,191,9.24
4,qa_missing_publication_year,186,9.00
5,qa_missing_page_evidence,107,5.18
6,qa_nonstandard_title_term,35,1.69
7,qa_page_anomaly,10,0.48
8,qa_missing_publisher,6,0.29
9,qa_title_mostly_numeric,1,0.05



PRIORITY / REVIEW CANDIDATES
Books: 1251


,matrix_row,book_id,canonical_title,authors,source_group,first_publish_year,publication_year_observed,publishers,isbn_13,subjects,cluster_label,cover_url,qa_major_flag_count,qa_minor_flag_count,qa_review_score,qa_review_priority
0,1106,BOOK01107,1929,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,4,2,10,Priority review
1,713,BOOK00714,Your Next Five Moves,[],Open Library only,NaN,NaN,[],['9781982168964'],[],CEO and Executive Transformation,https://covers.openlibrary.org/b/id/13227022-L...,3,3,9,Priority review
2,955,BOOK00956,Jump,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
3,961,BOOK00962,Longpath,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
4,962,BOOK00963,Next!,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
5,969,BOOK00970,Reunion,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
6,975,BOOK00976,MoneyZen,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
7,988,BOOK00989,Uptime,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
8,1001,BOOK01002,Unfiltered,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review
9,1005,BOOK01006,Glad We Met,NaN,LeadershipNow only,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,3,2,8,Priority review



SAVED
Full quality audit: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_catalog_quality_audit.csv
Manual review queue: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_catalog_manual_review_queue.csv

Production row alignment preserved: True


#### 18.42.4 — Refined Targeted Semantic Review

The initial catalog-wide quality screening demonstrated that missing metadata should not be used as a primary indicator of record validity.

Large proportions of the LeadWise catalog legitimately lack descriptions, subjects, ISBNs, or other metadata because coverage differs substantially between source systems. Consequently, treating these missing fields as validity indicators produced an excessively broad review queue.

The quality-control strategy is therefore refined to distinguish **metadata completeness** from **semantic or structural anomaly detection**.

Missing descriptions, ISBNs, subjects, authors, publishers, publication years, and page counts remain useful metadata-quality statistics but do not independently make a book suspicious.

Manual semantic review is instead restricted to records exhibiting specific structural indicators, including:

- unusual title structure;
- predominantly numeric titles;
- trailing cataloging symbols;
- URL-like titles;
- potentially non-standard publication-object terminology; and
- previously identified page-count anomalies.

These indicators remain screening signals rather than automatic exclusion criteria. Final corrections require record-level evidence.

In [90]:
# ---------------------------------------------------------
# 18.42.4 Refined Targeted Semantic Review
# ---------------------------------------------------------

import pandas as pd


audit_path = (
    data_root
    / "processed"
    / "leadwise_catalog_quality_audit.csv"
)


audit = pd.read_csv(
    audit_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Structural indicators only
# ---------------------------------------------------------

structural_flags = [
    "qa_very_short_title",
    "qa_title_trailing_symbol",
    "qa_title_url_like",
    "qa_title_mostly_numeric",
    "qa_nonstandard_title_term",
    "qa_page_anomaly",
]


for flag in structural_flags:

    if flag not in audit.columns:
        audit[flag] = False

    audit[flag] = (
        audit[flag]
        .fillna(False)
        .astype(bool)
    )


audit[
    "structural_flag_count"
] = (
    audit[
        structural_flags
    ]
    .sum(axis=1)
)


targeted_review = (
    audit[
        audit[
            "structural_flag_count"
        ] > 0
    ]
    .copy()
)


# ---------------------------------------------------------
# Human-readable reasons
# ---------------------------------------------------------

reason_labels = {
    "qa_very_short_title":
        "Very short title",

    "qa_title_trailing_symbol":
        "Trailing cataloging symbol",

    "qa_title_url_like":
        "URL-like title",

    "qa_title_mostly_numeric":
        "Predominantly numeric title",

    "qa_nonstandard_title_term":
        "Potential non-standard publication term",

    "qa_page_anomaly":
        "Edition page-count anomaly",
}


def build_review_reason(row):

    reasons = [
        label
        for flag, label
        in reason_labels.items()
        if bool(row.get(flag, False))
    ]

    return "; ".join(reasons)


targeted_review[
    "review_reason"
] = targeted_review.apply(
    build_review_reason,
    axis=1,
)


# ---------------------------------------------------------
# Review categories
# ---------------------------------------------------------

def classify_review(row):

    if (
        row["qa_title_mostly_numeric"]
        or row["qa_title_url_like"]
        or row["qa_very_short_title"]
        or row["qa_title_trailing_symbol"]
    ):
        return "Title review"

    if row["qa_nonstandard_title_term"]:
        return "Publication-type review"

    if row["qa_page_anomaly"]:
        return "Metadata correction"

    return "Other review"


targeted_review[
    "review_category"
] = targeted_review.apply(
    classify_review,
    axis=1,
)


# ---------------------------------------------------------
# Output columns
# ---------------------------------------------------------

display_columns = [
    "matrix_row",
    "book_id",
    "canonical_title",
    "authors",
    "source_group",
    "first_publish_year",
    "publication_year_observed",
    "subjects",
    "cluster_label",
    "cover_url",
    "structural_flag_count",
    "review_category",
    "review_reason",
]


display_columns = [
    col
    for col in display_columns
    if col in targeted_review.columns
]


targeted_review = (
    targeted_review
    .sort_values(
        [
            "structural_flag_count",
            "book_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "REFINED TARGETED QUALITY REVIEW"
)
print("=" * 80)

print(
    "Total catalog books:",
    len(audit)
)

print(
    "Books requiring targeted review:",
    len(targeted_review)
)

print(
    "Share of catalog:",
    f"{len(targeted_review) / len(audit) * 100:.2f}%"
)


print(
    "\nREVIEW CATEGORY DISTRIBUTION"
)
print("=" * 80)

display(
    targeted_review[
        "review_category"
    ]
    .value_counts()
    .rename_axis(
        "review_category"
    )
    .reset_index(
        name="books"
    )
)


print(
    "\nTARGETED REVIEW RECORDS"
)
print("=" * 80)

display(
    targeted_review[
        display_columns
    ]
)


# ---------------------------------------------------------
# Save refined queue
# ---------------------------------------------------------

refined_path = (
    data_root
    / "processed"
    / "leadwise_catalog_targeted_review.csv"
)


targeted_review[
    display_columns
].to_csv(
    refined_path,
    index=False,
)


print(
    "\nSAVED"
)
print("=" * 80)

print(refined_path)


# ---------------------------------------------------------
# Production integrity remains unchanged
# ---------------------------------------------------------

assert len(audit) == 2067
assert audit["book_id"].is_unique
assert audit["matrix_row"].is_unique

print(
    "\nProduction catalog modified: False"
)
print(
    "Production row alignment preserved: True"
)

REFINED TARGETED QUALITY REVIEW
Total catalog books: 2067
Books requiring targeted review: 47
Share of catalog: 2.27%

REVIEW CATEGORY DISTRIBUTION


,review_category,books
0,Publication-type review,35
1,Metadata correction,10
2,Title review,2



TARGETED REVIEW RECORDS


,matrix_row,book_id,canonical_title,authors,source_group,first_publish_year,publication_year_observed,subjects,cluster_label,cover_url,structural_flag_count,review_category,review_reason
58,58,BOOK00059,The Center for Creative Leadership handbook of...,"['Cynthia D. McCauley', 'Ellen Van Velsor']",Open Library only,2003.0,NaN,"['Creative ability in business', 'Leadership']",Leadership Development,https://covers.openlibrary.org/b/id/10216345-L...,1,Publication-type review,Potential non-standard publication term
67,67,BOOK00068,The Center for Creative Leadership handbook of...,['Ellen Van Velsor'],Open Library only,2010.0,NaN,"['Creative ability in business', 'Leadership',...",Leadership Development,https://covers.openlibrary.org/b/id/9271912-L.jpg,1,Publication-type review,Potential non-standard publication term
79,79,BOOK00080,The handbook for student leadership development,['Susan R. Komives'],Open Library only,2011.0,NaN,"['Counseling in higher education', 'College st...",Leadership Development,https://covers.openlibrary.org/b/id/9296574-L.jpg,1,Publication-type review,Potential non-standard publication term
103,103,BOOK00104,Effective Executive,"['Peter F. Drucker', 'Joseph A. Maciariello', ...",Open Library only,1967.0,NaN,"['Decision making', 'Executives', 'Decision-ma...",General Business and Management,https://covers.openlibrary.org/b/id/4941284-L.jpg,1,Metadata correction,Edition page-count anomaly
107,107,BOOK00108,Executive Leadership Council Journal,"['The Executive Leadership Council', 'The Exec...",Open Library only,2020.0,NaN,['Business'],Executive Leadership,NaN,1,Publication-type review,Potential non-standard publication term
154,154,BOOK00155,Team leadership in the game industry,"['Spaulding, Seth II.']",Open Library only,2009.0,NaN,"['Teams in the workplace', 'Leadership', 'Elec...",Leadership Development,https://covers.openlibrary.org/b/id/9953222-L.jpg,1,Publication-type review,Potential non-standard publication term
181,181,BOOK00182,Tim lidŏsip =,['Sŭng-gak Yi'],Open Library only,1996.0,NaN,"['Leadership', 'Social group work']",Workplace and Generational Dynamics,NaN,1,Title review,Trailing cataloging symbol
186,186,BOOK00187,Team Leadership Questionnaire,"['D. Kayes', 'Anna Kayes']",Open Library only,2011.0,NaN,['Leadership'],Team Leadership,NaN,1,Publication-type review,Potential non-standard publication term
246,246,BOOK00247,Handbook for secular Franciscan servant leader...,[],Open Library only,1998.0,NaN,"['Secular Franciscan Order', 'Servant leadersh...",Servant Leadership,NaN,1,Publication-type review,Potential non-standard publication term
281,281,BOOK00282,Management information systems,"['Kenneth C. Laudon', 'Jane P. Laudon', 'Jane ...",Open Library only,1945.0,NaN,"['Information technology', 'Management', 'Mana...",Information Technology Management,https://covers.openlibrary.org/b/id/1104118-L.jpg,1,Metadata correction,Edition page-count anomaly



SAVED
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_catalog_targeted_review.csv

Production catalog modified: False
Production row alignment preserved: True


#### 18.42.5 — Targeted Verification of Ambiguous Publication Records

The refined semantic review identified 47 records with structural screening indicators. Manual inspection demonstrates that most flags are caused by legitimate terminology such as *handbook* or *game* and do not indicate invalid publication records.

The final verification stage therefore focuses on six genuinely ambiguous records whose titles or publication types warrant closer inspection:

- `BOOK00108` — Executive Leadership Council Journal
- `BOOK00182` — Tim lidŏsip =
- `BOOK00187` — Team Leadership Questionnaire
- `BOOK01107` — 1929
- `BOOK01517` — Change Your Questions, Change Your Life Workbook:
- `BOOK01952` — Get Clear Career Assessment

These records are traced against their original source metadata before any application-level inclusion, exclusion, title correction, or publication-type decision is made.

No production record is removed during this stage.

In [91]:
# ---------------------------------------------------------
# 18.42.5 Trace genuinely ambiguous publication records
# ---------------------------------------------------------

import pandas as pd


TARGET_BOOK_IDS = [
    "BOOK00108",
    "BOOK00182",
    "BOOK00187",
    "BOOK01107",
    "BOOK01517",
    "BOOK01952",
]


# ---------------------------------------------------------
# Load source datasets
# ---------------------------------------------------------

catalog = pd.read_csv(
    data_root / "processed" / "leadwise_streamlit_catalog.csv",
    low_memory=False,
)

master = pd.read_csv(
    data_root / "final" / "books_master.csv",
    low_memory=False,
)

openlibrary = pd.read_csv(
    data_root / "final" / "openlibrary_integrated.csv",
    low_memory=False,
)

leadershipnow = pd.read_csv(
    data_root / "final" / "leadershipnow_editions.csv",
    low_memory=False,
)


# ---------------------------------------------------------
# Helper
# ---------------------------------------------------------

def trace_records(df, book_ids):

    if "book_id" not in df.columns:
        return pd.DataFrame()

    return (
        df[
            df["book_id"]
            .astype(str)
            .isin(book_ids)
        ]
        .copy()
        .sort_values("book_id")
    )


# ---------------------------------------------------------
# Deployment catalog
# ---------------------------------------------------------

deployment_trace = trace_records(
    catalog,
    TARGET_BOOK_IDS,
)

print("DEPLOYMENT CATALOG")
print("=" * 80)

deployment_columns = [
    "matrix_row",
    "book_id",
    "canonical_title",
    "authors",
    "description",
    "subjects",
    "first_publish_year",
    "publication_year_observed",
    "publisher",
    "publishers",
    "isbn_10",
    "isbn_13",
    "page_count",
    "cover_url",
    "source_group",
    "source_url",
    "topic_cluster",
    "cluster_label",
]

deployment_columns = [
    c for c in deployment_columns
    if c in deployment_trace.columns
]

display(
    deployment_trace[
        deployment_columns
    ]
)


# ---------------------------------------------------------
# Books master
# ---------------------------------------------------------

master_trace = trace_records(
    master,
    TARGET_BOOK_IDS,
)

print("\nBOOKS MASTER")
print("=" * 80)

display(master_trace)


# ---------------------------------------------------------
# Open Library records
# ---------------------------------------------------------

ol_trace = trace_records(
    openlibrary,
    TARGET_BOOK_IDS,
)

print("\nOPEN LIBRARY SOURCE")
print("=" * 80)

print("Records:", len(ol_trace))

if not ol_trace.empty:

    ol_columns = [
        "book_id",
        "openlibrary_key",
        "title",
        "authors",
        "author_keys",
        "first_publish_year",
        "publish_dates",
        "publishers",
        "isbn_10",
        "isbn_13",
        "all_isbns",
        "languages",
        "subjects",
        "edition_count",
        "ratings_average",
        "ratings_count",
        "cover_id",
        "cover_url",
        "ebook_access",
        "has_fulltext",
        "public_scan",
        "description",
        "work_subjects",
        "lc_classifications",
        "dewey_number",
        "source_url",
    ]

    ol_columns = [
        c for c in ol_columns
        if c in ol_trace.columns
    ]

    display(
        ol_trace[
            ol_columns
        ]
    )


# ---------------------------------------------------------
# LeadershipNow records
# ---------------------------------------------------------

ln_trace = trace_records(
    leadershipnow,
    TARGET_BOOK_IDS,
)

print("\nLEADERSHIPNOW SOURCE")
print("=" * 80)

print("Records:", len(ln_trace))

if not ln_trace.empty:

    display(ln_trace)


# ---------------------------------------------------------
# Source coverage
# ---------------------------------------------------------

summary = (
    deployment_trace[
        [
            "matrix_row",
            "book_id",
            "canonical_title",
            "source_group",
        ]
    ]
    .copy()
)


summary[
    "found_in_openlibrary"
] = summary[
    "book_id"
].isin(
    ol_trace["book_id"]
    if not ol_trace.empty
    else []
)


summary[
    "found_in_leadershipnow"
] = summary[
    "book_id"
].isin(
    ln_trace["book_id"]
    if not ln_trace.empty
    else []
)


print("\nSOURCE TRACE SUMMARY")
print("=" * 80)

display(summary)


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

print("\nINTEGRITY")
print("=" * 80)

print(
    "Requested records:",
    len(TARGET_BOOK_IDS)
)

print(
    "Deployment records found:",
    len(deployment_trace)
)

print(
    "Unique deployment book IDs:",
    deployment_trace["book_id"].nunique()
)

assert len(deployment_trace) == len(TARGET_BOOK_IDS)
assert deployment_trace["book_id"].is_unique

print(
    "Production catalog modified: False"
)

DEPLOYMENT CATALOG


,matrix_row,book_id,canonical_title,authors,description,subjects,first_publish_year,publication_year_observed,publisher,publishers,isbn_10,isbn_13,page_count,cover_url,source_group,source_url,topic_cluster,cluster_label
107,107,BOOK00108,Executive Leadership Council Journal,"['The Executive Leadership Council', 'The Exec...",NaN,['Business'],2020.0,NaN,NaN,"['Executive Leadership Council, The']","['0578640600', '0578768100', '0578984180']","['9780578984186', '9780578768106', '9798218115...",NaN,NaN,Open Library only,https://openlibrary.org/works/OL20764205W,19.0,Executive Leadership
181,181,BOOK00182,Tim lidŏsip =,['Sŭng-gak Yi'],NaN,"['Leadership', 'Social group work']",1996.0,NaN,NaN,['Hanʾguk Nŭngnyul Hyŏphoe'],['8972771007'],['9788972771005'],NaN,NaN,Open Library only,https://openlibrary.org/works/OL12107242W,15.0,Workplace and Generational Dynamics
186,186,BOOK00187,Team Leadership Questionnaire,"['D. Kayes', 'Anna Kayes']",NaN,['Leadership'],2011.0,NaN,NaN,['CreateSpace Independent Publishing Platform'],['1466466294'],['9781466466296'],NaN,NaN,Open Library only,https://openlibrary.org/works/OL40036856W,2.0,Team Leadership
1106,1106,BOOK01107,1929,NaN,NaN,NaN,NaN,NaN,Viking,NaN,NaN,NaN,352.0,https://www.leadershipnow.com/leadershop/image...,LeadershipNow only,NaN,NaN,NaN
1516,1516,BOOK01517,"Change Your Questions, Change Your Life Workbook:",['Marilee Adams with Andrea F. Lipton'],NaN,[],NaN,2022.0,Berrett-Koehler Publishers,NaN,NaN,NaN,128.0,https://www.leadershipnow.com/leadershop/image...,LeadershipNow only,NaN,9.0,Change Management
1951,1951,BOOK01952,Get Clear Career Assessment,['Ken Coleman'],NaN,[],NaN,2024.0,Ramsey Press,NaN,NaN,NaN,160.0,https://www.leadershipnow.com/leadershop/image...,LeadershipNow only,NaN,11.0,Organizational Culture and Trust



BOOKS MASTER


,book_id,canonical_title,authors,description,subjects,first_publish_year,average_rating,ratings_count,edition_count,want_to_read_count,currently_reading_count,already_read_count,cover_url,openlibrary_key,source_openlibrary,source_leadershipnow,publication_year_observed,title_normalized
107,BOOK00108,Executive Leadership Council Journal,"['The Executive Leadership Council', 'The Exec...",NaN,['Business'],2020.0,NaN,NaN,4.0,0.0,0.0,0.0,NaN,/works/OL20764205W,True,False,NaN,executive leadership council journal
181,BOOK00182,Tim lidŏsip =,['Sŭng-gak Yi'],NaN,"['Leadership', 'Social group work']",1996.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,/works/OL12107242W,True,False,NaN,tim lidŏsip =
186,BOOK00187,Team Leadership Questionnaire,"['D. Kayes', 'Anna Kayes']",NaN,['Leadership'],2011.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,/works/OL40036856W,True,False,NaN,team leadership questionnaire
1106,BOOK01107,1929,['Andrew Ross Sorkin'],NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,NaN,False,True,2025.0,1929
1516,BOOK01517,"Change Your Questions, Change Your Life Workbook:",['Marilee Adams with Andrea F. Lipton'],NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,NaN,False,True,2022.0,"change your questions, change your life workbook:"
1951,BOOK01952,Get Clear Career Assessment,['Ken Coleman'],NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.leadershipnow.com/leadershop/image...,NaN,False,True,2024.0,get clear career assessment



OPEN LIBRARY SOURCE
Records: 3


...


LEADERSHIPNOW SOURCE
Records: 3


,book_id,scrape_id,title,subtitle,author,isbn_source,isbn_normalized,isbn_13,isbn_status,publisher,...,page_count,edition_note,cover_url,external_book_url,release_pages,source_metadata_conflict,source,openlibrary_key,match_method,match_confidence
156,BOOK01107,42,1929,Inside the Greatest Crash in Wall Street Histo...,Andrew Ross Sorkin,9780593296967,9780593296967,9780593296967,Valid ISBN-13,Viking,...,352.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/465mxDZ,['2025 Releases'],False,LeadershipNow,NaN,NaN,NaN
569,BOOK01517,950,"Change Your Questions, Change Your Life Workbook:",Master Your Mindset Using Question Thinking,Marilee Adams with Andrea F. Lipton,9781523091201,9781523091201,9781523091201,Valid ISBN-13,Berrett-Koehler Publishers,...,128.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3pmtUAL,['2022 Releases'],False,LeadershipNow,NaN,NaN,NaN
1004,BOOK01952,415,Get Clear Career Assessment,Find the Work You're Wired to Do,Ken Coleman,9788887820231,9788887820231,9788887820231,Invalid check digit,Ramsey Press,...,160.0,NaN,https://www.leadershipnow.com/leadershop/image...,https://amzn.to/3xQIIyZ,['2024 Releases'],False,LeadershipNow,NaN,NaN,NaN



SOURCE TRACE SUMMARY


,matrix_row,book_id,canonical_title,source_group,found_in_openlibrary,found_in_leadershipnow
107,107,BOOK00108,Executive Leadership Council Journal,Open Library only,True,False
181,181,BOOK00182,Tim lidŏsip =,Open Library only,True,False
186,186,BOOK00187,Team Leadership Questionnaire,Open Library only,True,False
1106,1106,BOOK01107,1929,LeadershipNow only,False,True
1516,1516,BOOK01517,"Change Your Questions, Change Your Life Workbook:",LeadershipNow only,False,True
1951,1951,BOOK01952,Get Clear Career Assessment,LeadershipNow only,False,True



INTEGRITY
Requested records: 6
Deployment records found: 6
Unique deployment book IDs: 6
Production catalog modified: False


#### 18.42.6 — Final Application Metadata Corrections

Targeted source verification found no evidence requiring removal of the six ambiguous publication records from the LeadWise catalog.

The investigation instead identified several application-facing metadata corrections:

- `BOOK00182` contains trailing cataloging punctuation in its title and is displayed as `Tim lidŏsip`.
- `BOOK01107` is a valid LeadershipNow book titled `1929`; author and publication metadata available in the collected source are restored to the deployment catalog.
- `BOOK01952` remains a valid collected publication record, but its source identifier failed ISBN-13 check-digit validation and must not be represented as a validated ISBN.
- `BOOK00108`, `BOOK00187`, and `BOOK01517` remain eligible because collected bibliographic evidence supports them as legitimate leadership/management-related publications.

These corrections affect application-facing metadata only. No book or matrix row is removed, and the production TF-IDF recommendation architecture remains unchanged.

In [92]:
# ---------------------------------------------------------
# 18.42.6 Final evidence-supported metadata corrections
# ---------------------------------------------------------

import pandas as pd


catalog_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)


catalog = pd.read_csv(
    catalog_path,
    low_memory=False,
)


# ---------------------------------------------------------
# Helper
# ---------------------------------------------------------

def ensure_audit_column(column):

    if column not in catalog.columns:
        catalog[column] = pd.NA


for column in [
    "canonical_title_original",
    "metadata_quality_note",
]:
    ensure_audit_column(column)


# ---------------------------------------------------------
# BOOK00182
# Remove application-facing trailing catalog punctuation
# ---------------------------------------------------------

mask = catalog["book_id"].eq("BOOK00182")

assert mask.sum() == 1


if pd.isna(
    catalog.loc[
        mask,
        "canonical_title_original"
    ].iloc[0]
):

    catalog.loc[
        mask,
        "canonical_title_original"
    ] = catalog.loc[
        mask,
        "canonical_title"
    ]


catalog.loc[
    mask,
    "canonical_title"
] = "Tim lidŏsip"


catalog.loc[
    mask,
    "metadata_quality_note"
] = (
    "Application-facing trailing cataloging "
    "punctuation removed; original title preserved."
)


# ---------------------------------------------------------
# BOOK01107
# Restore collected LeadershipNow metadata
# ---------------------------------------------------------

mask = catalog["book_id"].eq("BOOK01107")

assert mask.sum() == 1


catalog.loc[
    mask,
    "authors"
] = "['Andrew Ross Sorkin']"


catalog.loc[
    mask,
    "publication_year_observed"
] = 2025


# Preserve existing publisher if already present.
if "publisher" in catalog.columns:

    catalog.loc[
        mask,
        "publisher"
    ] = "Viking"


if "page_count" in catalog.columns:

    catalog.loc[
        mask,
        "page_count"
    ] = 352


catalog.loc[
    mask,
    "metadata_quality_note"
] = (
    "LeadershipNow metadata restored to deployment "
    "catalog: author, publication year, publisher "
    "and page count."
)


# ---------------------------------------------------------
# BOOK01952
# Preserve publication but record invalid ISBN evidence
# ---------------------------------------------------------

mask = catalog["book_id"].eq("BOOK01952")

assert mask.sum() == 1


catalog.loc[
    mask,
    "metadata_quality_note"
] = (
    "Collected LeadershipNow publication retained. "
    "Source identifier 9788887820231 failed ISBN-13 "
    "check-digit validation and must not be displayed "
    "as a validated ISBN."
)


# ---------------------------------------------------------
# Final integrity
# ---------------------------------------------------------

print(
    "FINAL CORRECTED RECORDS"
)
print("=" * 80)


check_ids = [
    "BOOK00003",
    "BOOK00182",
    "BOOK01107",
    "BOOK01952",
]


check_columns = [
    "matrix_row",
    "book_id",
    "canonical_title",
    "canonical_title_original",
    "authors",
    "publication_year_observed",
    "publisher",
    "page_count",
    "cover_url",
    "metadata_quality_note",
]


check_columns = [
    col
    for col in check_columns
    if col in catalog.columns
]


display(
    catalog[
        catalog["book_id"]
        .isin(check_ids)
    ][
        check_columns
    ]
    .sort_values(
        "matrix_row"
    )
)


print(
    "\nCATALOG INTEGRITY"
)
print("=" * 80)

print("Rows:", len(catalog))
print(
    "Unique book IDs:",
    catalog["book_id"].nunique()
)
print(
    "Duplicate book IDs:",
    catalog["book_id"].duplicated().sum()
)
print(
    "Unique matrix rows:",
    catalog["matrix_row"].nunique()
)
print(
    "Duplicate matrix rows:",
    catalog["matrix_row"].duplicated().sum()
)


assert len(catalog) == 2067
assert catalog["book_id"].is_unique
assert catalog["matrix_row"].is_unique


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

catalog.to_csv(
    catalog_path,
    index=False,
)


print(
    "\nFINAL DEPLOYMENT CATALOG SAVED"
)
print("=" * 80)

print(catalog_path)

print(
    "\nProduction matrix alignment preserved: True"
)

FINAL CORRECTED RECORDS


,matrix_row,book_id,canonical_title,canonical_title_original,authors,publication_year_observed,publisher,page_count,cover_url,metadata_quality_note
2,2,BOOK00003,Kepemimpinan,Kepemimpinan =,['Karjadi M.'],NaN,NaN,NaN,NaN,Application-facing title cleaned; Open Library...
181,181,BOOK00182,Tim lidŏsip,Tim lidŏsip =,['Sŭng-gak Yi'],NaN,NaN,NaN,NaN,Application-facing trailing cataloging punctua...
1106,1106,BOOK01107,1929,NaN,['Andrew Ross Sorkin'],2025.0,Viking,352.0,https://www.leadershipnow.com/leadershop/image...,LeadershipNow metadata restored to deployment ...
1951,1951,BOOK01952,Get Clear Career Assessment,NaN,['Ken Coleman'],2024.0,Ramsey Press,160.0,https://www.leadershipnow.com/leadershop/image...,Collected LeadershipNow publication retained. ...



CATALOG INTEGRITY
Rows: 2067
Unique book IDs: 2067
Duplicate book IDs: 0
Unique matrix rows: 2067
Duplicate matrix rows: 0

FINAL DEPLOYMENT CATALOG SAVED
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_streamlit_catalog.csv

Production matrix alignment preserved: True


### 18.43 — Final Deployment Catalog Integration

The semantic quality-control stage is complete.

The final LeadWise deployment catalog contains 2,067 unique publication records and remains exactly aligned with the 2,067 rows of the production enriched TF-IDF matrix.

Final application-facing quality corrections include:

- suppression of a visually unreliable Open Library cover for `BOOK00003`;
- removal of trailing cataloging punctuation from selected display titles while preserving their original values;
- restoration of available LeadershipNow metadata for `BOOK01107`;
- documentation of an invalid ISBN-like identifier associated with `BOOK01952`;
- preservation of legitimate alternative publication types, including handbooks, workbooks, questionnaires, journals, and assessment-oriented publications when supported by collected bibliographic evidence; and
- suppression of previously identified anomalous edition page-count values from application display fields.

The final deployment catalog is:

`data/processed/leadwise_streamlit_catalog.csv`

No publication was removed from the production recommendation matrix during semantic quality control.

The application will therefore preserve the validated recommendation architecture while conditionally presenting only metadata supported by collected evidence.

At this stage, the deployment dataset is considered **frozen for Streamlit application development**. Subsequent work focuses on the LeadWise user experience rather than further modification of the recommendation dataset.

### 18.42.2 — Source Verification of the Suspicious *Kepemimpinan* Record

The Streamlit application identified a suspicious visual asset associated with the record *Kepemimpinan* by Karjadi M.

The objective of this validation is to distinguish between:

- a legitimate bibliographic record with an incorrect or mismatched cover;
- an unreliable bibliographic record; and
- a record that should be excluded from user-facing discovery.

No record will be deleted from the deployment catalog during this audit because the catalog must remain positionally aligned with the production TF-IDF matrix.

In [94]:
# =========================================================
# 18.42.2 — TRACE KEPEMIMPINAN SOURCE RECORD
# =========================================================

import pandas as pd
from pathlib import Path

data_root = project_root / "data"

streamlit_catalog = pd.read_csv(
    data_root / "processed" / "leadwise_streamlit_catalog.csv"
)

books_master = pd.read_csv(
    data_root / "final" / "books_master.csv"
)

openlibrary_integrated = pd.read_csv(
    data_root / "final" / "openlibrary_integrated.csv"
)


# ---------------------------------------------------------
# 1. Locate the deployment record
# ---------------------------------------------------------

deployment_match = streamlit_catalog[
    streamlit_catalog["canonical_title"]
    .fillna("")
    .str.contains(
        "Kepemimpinan",
        case=False,
        regex=False,
    )
].copy()

print("DEPLOYMENT MATCHES:", len(deployment_match))

display(
    deployment_match.T
)


# ---------------------------------------------------------
# 2. Extract the book ID
# ---------------------------------------------------------

if deployment_match.empty:

    print("No Kepemimpinan record found.")

else:

    suspicious_book_id = (
        deployment_match.iloc[0]["book_id"]
    )

    print(
        "\nSuspicious book_id:",
        suspicious_book_id,
    )


    # -----------------------------------------------------
    # 3. Trace in master catalog
    # -----------------------------------------------------

    master_match = books_master[
        books_master["book_id"]
        .astype(str)
        .eq(str(suspicious_book_id))
    ]

    print(
        "\nBOOKS MASTER MATCHES:",
        len(master_match),
    )

    display(
        master_match.T
    )


    # -----------------------------------------------------
    # 4. Trace in Open Library integrated data
    # -----------------------------------------------------

    if "book_id" in openlibrary_integrated.columns:

        ol_match = openlibrary_integrated[
            openlibrary_integrated["book_id"]
            .astype(str)
            .eq(str(suspicious_book_id))
        ]

    else:

        ol_match = openlibrary_integrated[
            openlibrary_integrated["title"]
            .fillna("")
            .str.contains(
                "Kepemimpinan",
                case=False,
                regex=False,
            )
        ]


    print(
        "\nOPEN LIBRARY MATCHES:",
        len(ol_match),
    )

    display(
        ol_match.T
    )


    # -----------------------------------------------------
    # 5. Compact diagnostic
    # -----------------------------------------------------

    row = deployment_match.iloc[0]

    print("\n" + "=" * 70)
    print("LEADWISE QUALITY DIAGNOSTIC")
    print("=" * 70)

    fields = [
        "book_id",
        "matrix_row",
        "canonical_title",
        "authors",
        "first_publish_year",
        "publication_year_observed",
        "publisher",
        "publishers",
        "isbn_10",
        "isbn_13",
        "cover_url",
        "source_group",
        "source_url",
        "observed_publication_languages",
        "earliest_observed_publication_country",
        "cluster_label",
        "average_rating",
        "ratings_count",
    ]

    for field in fields:

        if field in row.index:

            print(
                f"{field}:",
                row[field],
            )

DEPLOYMENT MATCHES: 1


,2
matrix_row,2
book_id,BOOK00003
canonical_title,Kepemimpinan
authors,['Karjadi M.']
description,NaN
...,...
edition_page_count_anomaly,False
cover_recovered_from_master,False
canonical_title_original,Kepemimpinan =
cover_url_original,https://covers.openlibrary.org/b/id/14420782-L...



Suspicious book_id: BOOK00003

BOOKS MASTER MATCHES: 1


,2
book_id,BOOK00003
canonical_title,Kepemimpinan =
authors,['Karjadi M.']
description,NaN
subjects,['Leadership']
first_publish_year,1977.0
average_rating,1.0
ratings_count,1.0
edition_count,1.0
want_to_read_count,41.0



OPEN LIBRARY MATCHES: 1


,2
openlibrary_key,/works/OL302757W
title,Kepemimpinan =
authors,['Karjadi M.']
author_keys,['OL1268A']
first_publish_year,1977.0
publish_dates,['1977']
publishers,['Politeia']
isbn_10,[]
isbn_13,[]
all_isbns,[]



LEADWISE QUALITY DIAGNOSTIC
book_id: BOOK00003
matrix_row: 2
canonical_title: Kepemimpinan
authors: ['Karjadi M.']
first_publish_year: 1977.0
publication_year_observed: nan
publisher: nan
publishers: ['Politeia']
isbn_10: []
isbn_13: []
cover_url: nan
source_group: Open Library only
source_url: https://openlibrary.org/works/OL302757W
observed_publication_languages: Indonesian
earliest_observed_publication_country: Indonesia
cluster_label: Transformational Leadership
average_rating: 1.0
ratings_count: 1.0


#### 18.43.2 — Streamlit Deployment Synchronization Validation

The final deployment catalog confirms that the unreliable cover associated with `BOOK00003 — Kepemimpinan` has already been suppressed.

This validation verifies that the Streamlit application is reading the finalized deployment catalog rather than an earlier cached catalog version.

The expected application-facing state for `BOOK00003` is:

- title: `Kepemimpinan`
- author: `Karjadi M.`
- matrix row: `2`
- cover URL: unavailable
- original unreliable cover retained only in the audit field
- recommendation eligibility retained

No recommendation-model modification is required.

In [95]:
# =========================================================
# 18.43.2 — STREAMLIT DEPLOYMENT SYNCHRONIZATION CHECK
# =========================================================

import pandas as pd


deployment_path = (
    data_root
    / "processed"
    / "leadwise_streamlit_catalog.csv"
)


deployment = pd.read_csv(
    deployment_path,
    low_memory=False,
)


target = deployment[
    deployment["book_id"]
    .astype(str)
    .eq("BOOK00003")
].copy()


assert len(target) == 1


row = target.iloc[0]


print("STREAMLIT DEPLOYMENT SYNCHRONIZATION")
print("=" * 75)

print("Deployment file:")
print(deployment_path)

print("\nCatalog rows:", len(deployment))
print(
    "Unique matrix rows:",
    deployment["matrix_row"].nunique()
)

print("\nBOOK00003")
print("-" * 75)

print(
    "Matrix row:",
    row["matrix_row"]
)

print(
    "Title:",
    row["canonical_title"]
)

print(
    "Original title:",
    row.get(
        "canonical_title_original"
    )
)

print(
    "Authors:",
    row["authors"]
)

print(
    "Current cover URL:",
    row.get(
        "cover_url"
    )
)

print(
    "Original cover URL:",
    row.get(
        "cover_url_original"
    )
)

print(
    "Metadata note:",
    row.get(
        "metadata_quality_note"
    )
)


# ---------------------------------------------------------
# Expected deployment state
# ---------------------------------------------------------

assert int(
    row["matrix_row"]
) == 2

assert (
    row["canonical_title"]
    == "Kepemimpinan"
)

assert pd.isna(
    row["cover_url"]
)

assert (
    pd.notna(
        row.get(
            "cover_url_original"
        )
    )
)


print("\n" + "=" * 75)
print("FINAL STATUS")
print("=" * 75)

print(
    "Corrected title: PASS"
)

print(
    "Bad cover suppressed: PASS"
)

print(
    "Original cover retained for audit: PASS"
)

print(
    "Matrix alignment preserved: PASS"
)

print(
    "\nDeployment catalog is ready for Streamlit."
)

STREAMLIT DEPLOYMENT SYNCHRONIZATION
Deployment file:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/leadwise_streamlit_catalog.csv

Catalog rows: 2067
Unique matrix rows: 2067

BOOK00003
---------------------------------------------------------------------------
Matrix row: 2
Title: Kepemimpinan
Original title: Kepemimpinan =
Authors: ['Karjadi M.']
Current cover URL: nan
Original cover URL: https://covers.openlibrary.org/b/id/14420782-L.jpg
Metadata note: Application-facing title cleaned; Open Library cover suppressed after visual quality review.

FINAL STATUS
Corrected title: PASS
Bad cover suppressed: PASS
Original cover retained for audit: PASS
Matrix alignment preserved: PASS

Deployment catalog is ready for Streamlit.


#### 18.43.2 — Deployment Synchronization Validation — Complete

The finalized LeadWise deployment catalog was successfully validated against the application-facing metadata corrections.

The catalog contains:

- **2,067 publication records**
- **2,067 unique matrix rows**
- preserved one-to-one alignment with the production TF-IDF matrix

A targeted validation of `BOOK00003 — Kepemimpinan` confirmed that:

- the cleaned application-facing title is retained;
- the original bibliographic title is preserved for audit purposes;
- the visually unreliable Open Library cover has been suppressed from application display;
- the original cover URL remains preserved as provenance;
- the publication remains eligible for content-based recommendation; and
- matrix row `2` remains unchanged.

This demonstrates that LeadWise can apply application-facing metadata quality controls without altering the underlying recommendation architecture.

The finalized deployment catalog is therefore synchronized and ready for continued Streamlit application development.